# Fase 4 — estimação horária de CO₂ em Iatan 2, 2025

**Cópia revisada.** O notebook original guarda as saídas históricas e seus resultados exploratórios. Esta cópia contém código revisado e saídas limpas. A auditoria executou verificações pontuais e não reajustou os modelos do grupo.

A variável-alvo é CO₂ em toneladas métricas por hora. A tarefa utiliza a geração da própria hora e o histórico operacional disponível até esse instante. O estudo de 2025 compara B0, P1 e P2. Os contrastes controlados M0, M2 e M4 separam a contribuição da memória de geração e a contribuição adicional dos atributos de processo.

**Executar em ordem.** Informe os nomes completos dos diretórios das execuções das Fases 2 e 3 em IATAN_PHASE2_RUN_ID e IATAN_PHASE3_RUN_ID. Ajuste IATAN_PROJECT_ROOT se os dados estiverem em outra raiz. Os três Parquets devem estar disponíveis antes da integração. A configuração imprime a pasta exclusiva de saída. Se houver interrupção ou for preciso recalcular um experimento concluído, reinicie e gere uma nova pasta de saída.

**Interpretação.** A filtragem HMM forward restringe a inferência por hora, enquanto o ajuste do HMM e as escolhas de processo também precisam respeitar o corte de cada janela. O teste de 2025 já foi consultado no projeto. Os Experimentos 30 a 32 são exploratórios e a Fase 5 deve construir validação temporal independente da seleção.

[Gu et al. (2023), emissões e geração com dados públicos](https://doi.org/10.1039/D3VA00191A) · [Cerqueira et al. (2020), avaliação temporal](https://arxiv.org/abs/1905.11744) · [Cawley e Talbot (2010), viés de seleção](https://jmlr.org/papers/v11/cawley10a.html)

In [ ]:
# Registros do experimento usam caminhos relativos à raiz configurada.
# Versões de bibliotecas e resultados são preservados sem identidade da sessão.
import builtins as _builtins
import json as _json
import os as _os
import re as _re
import warnings as _warnings
from pathlib import Path as _Path

def _project_root():
    for name in ("PROJECT_ROOT", "ROOT33", "ROOT", "PROJECT", "SOURCE", "root"):
        value = globals().get(name)
        if isinstance(value, _Path) and value.is_dir():
            return value.resolve()
    return _Path.cwd()

def portable_path(path, root=None):
    path = _Path(path).expanduser()
    root = _Path(root) if root is not None else _project_root()
    try:
        return path.resolve().relative_to(root.resolve()).as_posix()
    except (ValueError, OSError):
        return "external/" + path.name

def public_record(value):
    if isinstance(value, _Path):
        return portable_path(value)
    if isinstance(value, dict):
        return {k: public_record(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [public_record(v) for v in value]
    if isinstance(value, str):
        root = _project_root()
        for prefix in {str(root), root.as_posix()}:
            if value.startswith(prefix + "/") or value.startswith(prefix + "\\"):
                return value[len(prefix)+1:].replace("\\", "/")
            value = value.replace(prefix + "/", "").replace(prefix + "\\", "")
            value = value.replace(prefix, ".")
        if _re.match(r"^(?:[A-Za-z]:[\\/]|/(?:content|tmp|home|Users|mnt)/)", value):
            return "external/" + value.replace("\\", "/").rstrip("/").rsplit("/", 1)[-1]
        return value
    # Cópias de tabelas exportadas normalizam somente colunas textuais.
    if type(value).__module__.startswith("pandas.") and hasattr(value, "columns"):
        result = value.copy()
        for column in result.select_dtypes(include=["object", "string"]).columns:
            result[column] = result[column].map(public_record)
        return result
    return value

def _project_print(*values, **kwargs):
    root = _project_root()
    if root.is_dir():
        _os.chdir(root)
    return _builtins.print(*(public_record(v) for v in values), **kwargs)

def _project_dump(value, *args, **kwargs):
    return _json.dump(public_record(value), *args, **kwargs)

def _project_dumps(value, *args, **kwargs):
    return _json.dumps(public_record(value), *args, **kwargs)

def _project_csv(frame, *args, **kwargs):
    return public_record(frame).to_csv(*args, **kwargs)

def _project_write_text(path, text, *args, **kwargs):
    return path.write_text(public_record(text), *args, **kwargs)

def _project_display(*values, **kwargs):
    from IPython.display import display as _display
    return _display(*(public_record(v) for v in values), **kwargs)

# Avisos mantêm o conteúdo sem diretórios temporários ou IDs do kernel.
_warnings.formatwarning = lambda message, category, *args, **kwargs: (
    f"{category.__name__}: {public_record(str(message))}\n"
)
_os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
_context = _json.loads(_os.environ.get("MLFLOW_RUN_CONTEXT", "{}"))
_context.update({
    "mlflow.user": "equipe", "mlflow.source.name": "iatan",
    "mlflow.source.type": "LOCAL", "mlflow.source.git.commit": "",
    "mlflow.source.git.branch": "", "mlflow.source.git.repoURL": "",
    "mlflow.source.git.diff": "", "mlflow.source.git.dirty": "",
})
_os.environ["MLFLOW_RUN_CONTEXT"] = _json.dumps(_context)

# Instale as versões de requirements-iatan2025.txt separadamente.
# O ambiente da execução histórica está registrado no notebook original.
# A configuração a seguir grava as versões realmente usadas nesta execução.

In [ ]:
# FASE 4 V2 — CÉLULA 01: AMBIENTE, CONFIGURAÇÃO E REPRODUTIBILIDADE
# Cada execução recebe uma pasta própria sob o diretório original da fase.
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
import hashlib
import os
import sys, json, platform, importlib.metadata as md

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    _project_print('Fora do Colab: configure IATAN_PROJECT_ROOT com a pasta local do projeto.')

DEFAULT_COLAB_ROOT = (
    '/content/drive/MyDrive/Ciência de Dados - Iatan - Process Mining e Experimentos/'
    'Iatan 2025 - entrega revisada/execucao'
)
PROJECT_ROOT = Path(os.environ.get('IATAN_PROJECT_ROOT', DEFAULT_COLAB_ROOT)).expanduser().resolve()
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f'Pasta do projeto não encontrada: {PROJECT_ROOT}. Ajuste IATAN_PROJECT_ROOT.')

def require_run_id(env_var):
    value = os.environ.get(env_var, '').strip()
    if not value.startswith('iatan_6065_unit2_2025_') or any(c in value for c in '/\\'):
        raise ValueError(f'{env_var}: informe o nome completo do diretório da execução da fase.')
    return value

PHASE2_REFERENCE_RUN = require_run_id('IATAN_PHASE2_RUN_ID')
PHASE3_REFERENCE_RUN = require_run_id('IATAN_PHASE3_RUN_ID')
PHASE4_RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid4().hex[:8]
PHASE_ROOT = PROJECT_ROOT / 'outputs' / 'phase_04_predictive_modeling' / PHASE4_RUN_ID
PHASE_ROOT.mkdir(parents=True, exist_ok=False)
EXPERIMENT_NAME = 'iatan_6065_unit2_2025_process_mining_comparison'
SOURCE_NOTEBOOK = next((p for p in (
    PROJECT_ROOT / 'notebooks' / '05_modelagem_co2_mlflow.ipynb',
    PROJECT_ROOT.parent / 'notebooks' / '05_modelagem_co2_mlflow.ipynb',
) if p.is_file()), None)
SOURCE_NOTEBOOK_SHA256 = (
    hashlib.sha256(SOURCE_NOTEBOOK.read_bytes()).hexdigest()
    if SOURCE_NOTEBOOK is not None else None
)
DIRS = {
    'root': PHASE_ROOT,
    'tables': PHASE_ROOT / 'tables',
    'reports': PHASE_ROOT / 'reports',
    'figures': PHASE_ROOT / 'figures',
    'manifests': PHASE_ROOT / 'manifests',
    'models': PHASE_ROOT / 'models',
    'logs': PHASE_ROOT / 'logs',
    'processed': PHASE_ROOT / 'data' / 'processed',
}
for p in DIRS.values(): p.mkdir(parents=True, exist_ok=True)

# Os notebooks anteriores podem depender de versões específicas de bibliotecas.
# NÃO instala nem altera pacotes silenciosamente.
required_packages = ['numpy','pandas','scikit-learn','xgboost','mlflow','joblib','pyarrow','matplotlib','scipy']
versions = {}
for name in required_packages:
    try: versions[name] = md.version(name)
    except md.PackageNotFoundError: versions[name] = 'NOT_INSTALLED'
_project_print('Versões:', versions)
missing = [k for k,v in versions.items() if v == 'NOT_INSTALLED']
if missing:
    raise RuntimeError('Instale as dependências antes de executar a Fase 4: '+', '.join(missing))
with open(DIRS['manifests'] / 'phase04_v2_environment.json','w',encoding='utf-8') as f:
    _project_dump({'python':sys.version.split()[0],'packages':versions,
               'project_root':str(PROJECT_ROOT),'phase_root':str(PHASE_ROOT),
               'phase4_run_id': PHASE4_RUN_ID, 'phase2_run_id': PHASE2_REFERENCE_RUN,
               'phase3_run_id': PHASE3_REFERENCE_RUN, 'notebook_file_sha256': SOURCE_NOTEBOOK_SHA256,
               'test_status':'2025 Nov-Dec previously examined; exploratory'},f,indent=2)
_project_print('Configuração concluída:', PHASE_ROOT)

In [ ]:
# Registro de parâmetros, métricas e artefatos. Aula 08, pp. 31–32 e 53–56.
# Zaharia et al. (2018): https://people.eecs.berkeley.edu/~matei/papers/2018/ieee_mlflow.pdf
# MLflow Tracking: https://mlflow.org/docs/latest/ml/tracking/tracking-api/
# SQLite Backup API: https://docs.python.org/3/library/sqlite3.html#sqlite3.Connection.backup
import atexit
import sqlite3
from contextlib import closing
from tempfile import mkdtemp

import mlflow
from mlflow.tracking import MlflowClient

MLFLOW_LOCAL_DIR = Path(mkdtemp(prefix='iatan_phase04_'))
MLFLOW_DB_LOCAL = MLFLOW_LOCAL_DIR / 'mlflow.db'
MLFLOW_DB_BACKUP = DIRS['logs'] / 'mlflow.db'
MLFLOW_ARTIFACTS_DIR = DIRS['root'] / 'mlflow_artifacts'
MLFLOW_ARTIFACTS_DIR.mkdir(exist_ok=True)
MLFLOW_TRACKING_URI = f'sqlite:///{MLFLOW_DB_LOCAL.as_posix()}'
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

def activate_phase4_experiment(name, artifact_folder):
    experiment = client.get_experiment_by_name(name)
    if experiment is None:
        artifact_path = MLFLOW_ARTIFACTS_DIR / artifact_folder
        artifact_path.mkdir(exist_ok=True)
        experiment_id = client.create_experiment(name, artifact_location=artifact_path.resolve().as_uri())
    else:
        experiment_id = experiment.experiment_id
    return mlflow.set_experiment(experiment_id=experiment_id)

experiment = activate_phase4_experiment(EXPERIMENT_NAME, 'comparacao_b0_p1_p2')

def backup_mlflow_database():
    if not MLFLOW_DB_LOCAL.is_file():
        return
    temporary = MLFLOW_DB_BACKUP.with_suffix('.db.tmp')
    with closing(sqlite3.connect(MLFLOW_DB_LOCAL)) as source:
        with closing(sqlite3.connect(temporary)) as destination:
            source.backup(destination)
            if destination.execute('PRAGMA integrity_check').fetchone() != ('ok',):
                raise RuntimeError('A cópia do SQLite não passou na checagem de integridade.')
    temporary.replace(MLFLOW_DB_BACKUP)

backup_mlflow_database()
atexit.register(backup_mlflow_database)
assert MLFLOW_DB_BACKUP.is_file()
_project_print('Experimento:', experiment.name, '| ID:', experiment.experiment_id)
_project_print('Artefatos:', MLFLOW_ARTIFACTS_DIR)
_project_print('Backup SQLite:', MLFLOW_DB_BACKUP)

### Proveniência de dados e registro de entradas

Aula 08, pp. 53–56, e [MLflow Tracking](https://mlflow.org/docs/latest/ml/tracking/tracking-api/) fundamentam a trilha. SHA-256 identifica o arquivo; a inspeção de rótulos e chaves testa a compatibilidade.


In [ ]:
# ============================================================
# FASE 4 — CÉLULA 02
# IDENTIFICAÇÃO E RASTREABILIDADE DOS DATASETS
# ============================================================
#
# OBJETIVOS:
#
# 1. Localizar arquivos das Fases 2 e 3.
# 2. Identificar datasets de geração, CO2 e processos.
# 3. Verificar arquivos de partições temporais.
# 4. Registrar metadados dos arquivos encontrados.
# 5. Gerar manifesto reproduzível.
#
# IMPORTANTE:
# - Não modifica arquivos anteriores.
# - Não executa modelos.
# - Não escolhe automaticamente uma versão dos dados.
# ============================================================

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

# ------------------------------------------------------------
# 1. VERIFICAÇÃO DO AMBIENTE
# ------------------------------------------------------------

required_variables = [
    "PROJECT_ROOT",
    "PHASE_ROOT",
    "DIRS",
    "EXPERIMENT_NAME",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Execute a Célula 01 antes desta célula. "
        f"Variáveis ausentes: {missing_variables}"
    )

# ------------------------------------------------------------
# 2. PADRÕES DE ARQUIVOS ESPERADOS
# ------------------------------------------------------------

EXPECTED_FILES = {
    "phase2_process_features": [
        "cell30_process_features_all.parquet",
    ],
    "phase3_temporal_partitions": [
        "cell03_experimental_partitions.parquet",
    ],
    "phase3_feature_engineering": [
        "step05_feature_engineering.parquet",
    ],
}

# 4. METADADOS E HASH SHA256
# ------------------------------------------------------------

def calculate_sha256(file_path, chunk_size=1024 * 1024):
    sha256 = hashlib.sha256()

    with open(file_path, "rb") as file:
        while True:
            chunk = file.read(chunk_size)

            if not chunk:
                break

            sha256.update(chunk)

    return sha256.hexdigest()


def file_metadata(file_path, category):
    stat = file_path.stat()

    return {
        "category": category,
        "filename": file_path.name,
        "path": str(file_path),
        "size_mb": round(
            stat.st_size / (1024 ** 2), 3
        ),
        "modified_utc": datetime.fromtimestamp(
            stat.st_mtime,
            tz=timezone.utc
        ).isoformat(),
        "sha256": calculate_sha256(file_path),
    }

# ------------------------------------------------------------
# 5. INVENTÁRIO DOS ARQUIVOS
# ------------------------------------------------------------

inventory = []

for category, filenames in EXPECTED_FILES.items():

    if category == 'phase2_process_features':
        pattern = 'outputs/fase2_hmm_process_mining/*/data/processed/'
    elif category == 'phase3_temporal_partitions':
        pattern = 'outputs/lag_selection_predictive/*/data/processed/'
    else:
        pattern = 'data/interim/'
    matches = sorted(p for filename in filenames
                     for p in PROJECT_ROOT.glob(pattern + filename) if p.is_file())

    _project_print("\n" + "=" * 65)
    _project_print("Categoria:", category)
    _project_print("=" * 65)

    if not matches:
        _project_print("NENHUM ARQUIVO ENCONTRADO")
        continue

    for file_path in matches:

        metadata = file_metadata(
            file_path,
            category
        )

        inventory.append(metadata)

        _project_print("Arquivo:", file_path.name)
        _project_print("Caminho:", file_path)
        _project_print("Tamanho (MB):", metadata["size_mb"])
        _project_print("SHA256:", metadata["sha256"][:16], "...")
        _project_print("-" * 65)

# ------------------------------------------------------------
# 6. DATAFRAME DO INVENTÁRIO
# ------------------------------------------------------------

inventory_df = pd.DataFrame(inventory)

if not inventory_df.empty:

    _project_display(
        inventory_df[
            [
                "category",
                "filename",
                "size_mb",
                "modified_utc",
                "path",
            ]
        ]
    )

else:
    _project_print(
        "\nNenhum arquivo esperado foi localizado."
    )

# ------------------------------------------------------------
# 7. VERIFICAÇÃO DAS CATEGORIAS
# ------------------------------------------------------------

found_categories = set(
    inventory_df["category"]
) if not inventory_df.empty else set()

category_status = {}

for category in EXPECTED_FILES:

    category_status[category] = (
        category in found_categories
    )

_project_print("\nSTATUS DOS ARQUIVOS:")

for category, found in category_status.items():

    status = (
        "ENCONTRADO"
        if found
        else "AUSENTE"
    )

    _project_print(f"{category}: {status}")

# ------------------------------------------------------------
# 8. SALVAMENTO DO MANIFESTO
# ------------------------------------------------------------

manifest = {
    "phase": "phase_04_predictive_modeling",
    "cell": "02",
    "generated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "expected_files": EXPECTED_FILES,
    "category_status": category_status,
    "inventory": inventory,
}

MANIFEST_FILE = (
    DIRS["manifests"]
    / "cell02_input_inventory.json"
)

with open(
    MANIFEST_FILE,
    "w",
    encoding="utf-8"
) as file:

    _project_dump(
        manifest,
        file,
        indent=4,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 9. RESUMO FINAL
# ------------------------------------------------------------

_project_print("\n" + "=" * 65)
_project_print("FASE 4 — CÉLULA 02")
_project_print("=" * 65)

_project_print(
    "Arquivos encontrados:",
    len(inventory)
)

_project_print(
    "Categorias identificadas:",
    len(found_categories),
    "/",
    len(EXPECTED_FILES)
)

_project_print("\nManifesto salvo em:")
_project_print(MANIFEST_FILE)

if all(category_status.values()):

    _project_print(
        "\nSTATUS: INVENTÁRIO COMPLETO"
    )

else:

    _project_print(
        "\nSTATUS: INVENTÁRIO PARCIAL"
    )
    _project_print(
        "Verificar arquivos ausentes antes da integração."
    )

_project_print("=" * 65)

In [ ]:

import json
from pathlib import Path

manifest_path = DIRS["manifests"] / "cell02_input_inventory.json"

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

matches = [
    item for item in manifest["inventory"]
    if item["category"] == "phase2_process_features"
]

_project_print(f"Arquivos encontrados: {len(matches)}\n")

for i, item in enumerate(matches, 1):
    path = Path(item["path"])

    _project_print(f"ARQUIVO {i}")
    _project_print("Nome:", path.name)
    _project_print("Caminho:", path)
    _project_print("Existe:", path.exists())

    if path.exists():
        _project_print("Tamanho:", round(path.stat().st_size / 1024**2, 2), "MB")
        _project_print("Modificado:", pd.Timestamp(
            path.stat().st_mtime, unit="s", tz="UTC"
        ))

    _project_print("-" * 70)


In [ ]:

import pandas as pd
import hashlib
from pathlib import Path

arquivos = [
    Path(item["path"])
    for item in manifest["inventory"]
    if item["category"] == "phase2_process_features"
]

datasets_fase2 = {}
resumo = []

for i, caminho in enumerate(arquivos, 1):
    df = pd.read_parquet(caminho)
    datasets_fase2[i] = df

    resumo.append({
        "Execução": i,
        "Pasta": caminho.parent.parent.parent.name,
        "Linhas": len(df),
        "Colunas": len(df.columns),
        "Nulos": int(df.isna().sum().sum()),
        "SHA256": hashlib.sha256(
            caminho.read_bytes()
        ).hexdigest()[:16]
    })

_project_display(pd.DataFrame(resumo))

_project_print("\nCOMPARAÇÃO DE CONTEÚDO")

for i in range(1, len(datasets_fase2)):
    a = datasets_fase2[1]
    b = datasets_fase2[i + 1]

    mesmas_colunas = list(a.columns) == list(b.columns)
    mesmo_conteudo = a.equals(b)

    _project_print(f"\nExecução 1 × Execução {i + 1}")
    _project_print("Mesmas colunas:", mesmas_colunas)
    _project_print("Mesmo conteúdo:", mesmo_conteudo)

    if not mesmas_colunas:
        _project_print("Colunas exclusivas da primeira:",
              sorted(set(a.columns) - set(b.columns)))
        _project_print("Colunas exclusivas da outra:",
              sorted(set(b.columns) - set(a.columns)))


In [ ]:
if len(datasets_fase2) >= 2 and datasets_fase2[1].shape == datasets_fase2[2].shape and list(datasets_fase2[1]) == list(datasets_fase2[2]) and datasets_fase2[1]['timestamp'].equals(datasets_fase2[2]['timestamp']):

    import numpy as np
    import pandas as pd

    a = datasets_fase2[1].reset_index(drop=True)
    b = datasets_fase2[2].reset_index(drop=True)

    assert a.shape == b.shape
    assert list(a.columns) == list(b.columns)

    # Verifica diferenças, considerando NaN == NaN
    diferencas = a.ne(b) & ~(a.isna() & b.isna())

    resumo = []

    for coluna in a.columns:
        mascara = diferencas[coluna]
        quantidade = int(mascara.sum())

        if quantidade == 0:
            continue

        registro = {
            "coluna": coluna,
            "linhas_diferentes": quantidade,
            "percentual": round(
                100 * quantidade / len(a), 2
            ),
        }

        if (
            pd.api.types.is_numeric_dtype(a[coluna])
            and pd.api.types.is_numeric_dtype(b[coluna])
        ):
            delta = (
                pd.to_numeric(a[coluna], errors="coerce")
                - pd.to_numeric(b[coluna], errors="coerce")
            ).abs()

            registro["diferenca_media"] = delta[mascara].mean()
            registro["diferenca_maxima"] = delta[mascara].max()

        resumo.append(registro)

    resultado = pd.DataFrame(resumo)

    if resultado.empty:
        _project_print("Nenhuma diferença identificada.")
    else:
        resultado = resultado.sort_values(
            "linhas_diferentes", ascending=False
        )
        _project_display(resultado)

    _project_print(
        "Linhas com pelo menos uma diferença:",
        int(diferencas.any(axis=1).sum())
    )

else:
    _project_print('Comparação auxiliar omitida: requer dois arquivos com os mesmos campos e timestamps alinhados.')

In [ ]:
if len(datasets_fase2) >= 2 and all({'timestamp', 'hmm_state'}.issubset(datasets_fase2[k]) for k in (1, 2)) and datasets_fase2[1]['timestamp'].equals(datasets_fase2[2]['timestamp']):

    import numpy as np
    import pandas as pd
    from scipy.optimize import linear_sum_assignment
    from sklearn.metrics import adjusted_rand_score

    a = datasets_fase2[1].reset_index(drop=True)
    b = datasets_fase2[2].reset_index(drop=True)

    # Confirma alinhamento temporal antes da comparação.
    time_candidates = [
        c for c in a.columns
        if any(k in c.lower() for k in ["timestamp", "datetime", "date_utc"])
    ]

    if time_candidates:
        time_col = time_candidates[0]
        assert a[time_col].equals(b[time_col]), (
            f"As linhas não estão alinhadas por {time_col}."
        )
    else:
        _project_print("ATENÇÃO: nenhuma coluna temporal identificada.")
        _project_print("Confirme o alinhamento das linhas antes de interpretar.")

    # Tabela de correspondência dos estados
    contingencia = pd.crosstab(
        a["hmm_state"],
        b["hmm_state"]
    )

    _project_print("MATRIZ DE CORRESPONDÊNCIA")
    _project_display(contingencia)

    # Melhor correspondência um-para-um
    matriz = contingencia.to_numpy()
    linhas, colunas = linear_sum_assignment(-matriz)

    mapeamento = {
        contingencia.columns[j]: contingencia.index[i]
        for i, j in zip(linhas, colunas)
    }

    b_alinhado = b["hmm_state"].map(mapeamento)

    concordancia = (a["hmm_state"] == b_alinhado).mean()
    ari = adjusted_rand_score(a["hmm_state"], b["hmm_state"])

    _project_print("\nMAPEAMENTO DOS ESTADOS:")
    _project_print(mapeamento)

    _project_print(f"\nConcordância após alinhamento: {concordancia:.4%}")
    _project_print(f"Adjusted Rand Index (ARI): {ari:.6f}")

else:
    _project_print('Alinhamento auxiliar de estados omitido: requer dois arquivos com timestamps alinhados.')

A seleção efetiva usa os IDs explícitos das Fases 2 e 3 e a tabela de engenharia da Fase 1. Os diagnósticos anteriores apenas comparam arquivos disponíveis e não selecionam insumos.

In [ ]:
# ============================================================
# FASE 4 — CÉLULA 03
# CARREGAMENTO, DIAGNÓSTICO E AUDITORIA DOS DATASETS
# ============================================================
#
# OBJETIVOS:
# 1. Recuperar o inventário da Célula 02;
# 2. Identificar os arquivos disponíveis;
# 3. Selecionar as execuções de referência;
# 4. Verificar integridade e cobertura temporal;
# 5. Registrar os arquivos efetivamente utilizados.
#
# Não modifica os datasets originais.
# ============================================================

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from IPython.display import display


# ============================================================
# 1. RECUPERAÇÃO DO MANIFESTO
# ============================================================

MANIFEST_FILE = (
    DIRS["manifests"]
    / "cell02_input_inventory.json"
)

if not MANIFEST_FILE.exists():
    raise FileNotFoundError(
        f"Manifesto não encontrado: {MANIFEST_FILE}"
    )

with open(MANIFEST_FILE, "r", encoding="utf-8") as file:
    manifest = json.load(file)

inventory = manifest["inventory"]

required_categories = [
    "phase2_process_features",
    "phase3_temporal_partitions",
    "phase3_feature_engineering",
]


# ============================================================
# 2. REFERÊNCIAS E DIAGNÓSTICO
# ============================================================

# Os IDs de F2 e F3 são definidos na configuração inicial.
# A engenharia de atributos usa o único caminho canônico data/interim.
PHASE3_FEATURE_REFERENCE_RUN = None

inventory_records = []

for category in required_categories:

    matches = [
        Path(item["path"])
        for item in inventory
        if item["category"] == category
    ]

    _project_print("\n" + "=" * 75)
    _project_print("CATEGORIA:", category)
    _project_print("ARQUIVOS ENCONTRADOS:", len(matches))
    _project_print("=" * 75)

    for path in matches:

        run_id = next(
            (
                part for part in path.parts
                if part.startswith(
                    "iatan_6065_unit2_2025_"
                )
            ),
            "não identificado"
        )

        inventory_records.append({
            "category": category,
            "run_id": run_id,
            "filename": path.name,
            "exists": path.exists(),
            "path": str(path),
        })

        _project_print("Execução:", run_id)
        _project_print("Arquivo:", path.name)
        _project_print("Existe:", path.exists())
        _project_print("Caminho:", path)
        _project_print("-" * 50)

diagnostic_df = pd.DataFrame(
    inventory_records,
    columns=[
        "category", "run_id", "filename",
        "exists", "path"
    ]
)

_project_print("\nRESUMO DO INVENTÁRIO")
_project_display(diagnostic_df)


# ============================================================
# 3. SELEÇÃO CONTROLADA
# ============================================================

reference_runs = {
    "phase2_process_features": PHASE2_REFERENCE_RUN,
    "phase3_temporal_partitions": PHASE3_REFERENCE_RUN,
    "phase3_feature_engineering": (
        PHASE3_FEATURE_REFERENCE_RUN
    ),
}

dataset_paths = {}
selection_issues = []

for category in required_categories:

    candidates = diagnostic_df.loc[
        diagnostic_df["category"] == category
    ]

    reference = reference_runs[category]

    if reference is None:

        if len(candidates) == 1:
            selected = candidates.iloc[0]
        else:
            selection_issues.append(
                f"{category}: referência não definida; "
                f"{len(candidates)} candidatos."
            )
            continue

    else:
        selected_rows = candidates.loc[
            candidates["run_id"] == reference
        ]

        if len(selected_rows) != 1:
            selection_issues.append(
                f"{category}: referência {reference}; "
                f"{len(selected_rows)} correspondências."
            )
            continue

        selected = selected_rows.iloc[0]

    path = Path(selected["path"])

    if not path.is_file():
        selection_issues.append(
            f"{category}: arquivo inexistente: {path}"
        )
        continue

    dataset_paths[category] = path

if selection_issues:

    _project_print("\n" + "=" * 75)
    _project_print("SELEÇÃO PENDENTE")
    _project_print("=" * 75)

    for issue in selection_issues:
        _project_print("-", issue)

    _project_print("\nCANDIDATOS DISPONÍVEIS:")
    _project_display(diagnostic_df)

    raise RuntimeError(
        "Não foi possível selecionar os três datasets "
        "com segurança. Confira as execuções acima."
    )

_project_print("\nDATASETS SELECIONADOS")

for category, path in dataset_paths.items():
    _project_print(f"{category}: {path}")


# ============================================================
# 4. CARREGAMENTO DOS DATASETS
# ============================================================

datasets = {}
file_metadata = {}

for category, path in dataset_paths.items():

    if path.suffix.lower() != ".parquet":
        raise ValueError(
            f"Formato inesperado: {path}"
        )

    df = pd.read_parquet(path)
    datasets[category] = df

    sha256 = hashlib.sha256()

    with open(path, "rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            sha256.update(chunk)

    file_metadata[category] = {
        "path": str(path),
        "filename": path.name,
        "sha256": sha256.hexdigest(),
        "size_bytes": path.stat().st_size,
        "rows": int(len(df)),
        "columns": int(len(df.columns)),
    }

    _project_print(
        f"\nCarregado: {category} | "
        f"{len(df):,} linhas | "
        f"{len(df.columns)} colunas"
    )


# ============================================================
# 5. FUNÇÕES DE AUDITORIA
# ============================================================

def identify_datetime_columns(df):

    patterns = [
        "timestamp", "datetime", "date",
        "time", "data", "hora", "utc"
    ]

    candidates = []

    for col in df.columns:

        name = str(col).lower()

        is_datetime = (
            pd.api.types.is_datetime64_any_dtype(
                df[col]
            )
        )

        name_matches = any(
            pattern in name
            for pattern in patterns
        )

        if is_datetime or name_matches:
            candidates.append(str(col))

    return candidates


def identify_variable_groups(df):

    patterns = {
        "co2": [
            "co2", "carbon", "emission"
        ],
        "generation": [
            "generation", "gross_load",
            "load_mw", "power_mw", "mw"
        ],
        "hmm_states": [
            "hmm", "state", "regime"
        ],
        "process": [
            "process", "activity", "event",
            "transition", "cycle", "duration"
        ],
        "lags": [
            "lag", "rolling", "moving"
        ],
    }

    groups = {}

    for group, keywords in patterns.items():

        groups[group] = [
            str(col)
            for col in df.columns
            if any(
                keyword in str(col).lower()
                for keyword in keywords
            )
        ]

    return groups


def audit_dataset(name, df):

    _project_print("\n" + "=" * 75)
    _project_print("DATASET:", name)
    _project_print("=" * 75)

    rows, cols = df.shape

    _project_print(f"Dimensões: {rows:,} x {cols}")

    schema = pd.DataFrame({
        "column": [str(c) for c in df.columns],
        "dtype": [str(t) for t in df.dtypes],
        "null_count": df.isna().sum().values,
        "null_pct": (
            df.isna().mean().values * 100
        ).round(2),
    })

    _project_print("\nCOLUNAS E TIPOS:")
    _project_display(schema)

    duplicated_rows = int(
        df.duplicated().sum()
    )

    duplicated_columns = [
        str(c)
        for c in df.columns[
            df.columns.duplicated()
        ]
    ]

    _project_print("\nINTEGRIDADE:")
    _project_print(
        "Linhas inteiramente duplicadas:",
        duplicated_rows
    )
    _project_print(
        "Nomes de colunas duplicados:",
        duplicated_columns
    )

    datetime_candidates = (
        identify_datetime_columns(df)
    )

    _project_print("\nCOLUNAS TEMPORAIS CANDIDATAS:")
    _project_print(datetime_candidates)

    temporal_summary = {}

    for col in datetime_candidates:

        try:
            parsed = pd.to_datetime(
                df[col],
                errors="coerce",
                utc=True
            )

            valid = parsed.dropna()

            temporal_summary[col] = {
                "valid_count": int(valid.size),
                "min_utc": (
                    valid.min().isoformat()
                    if not valid.empty else None
                ),
                "max_utc": (
                    valid.max().isoformat()
                    if not valid.empty else None
                ),
                "duplicate_timestamps": int(
                    valid.duplicated().sum()
                ),
            }

        except Exception as error:
            temporal_summary[col] = {
                "error": str(error)
            }

    _project_print("\nCOBERTURA TEMPORAL:")

    if temporal_summary:
        _project_display(
            pd.DataFrame.from_dict(
                temporal_summary,
                orient="index"
            )
        )
    else:
        _project_print("Nenhuma coluna temporal identificada.")

    variable_groups = (
        identify_variable_groups(df)
    )

    _project_print("\nGRUPOS DE VARIÁVEIS:")

    for group, columns in variable_groups.items():

        _project_print(
            f"\n{group.upper()} "
            f"({len(columns)} colunas):"
        )
        _project_print(columns)

    numeric = df.select_dtypes(
        include=[np.number]
    )

    _project_print("\nRESUMO NUMÉRICO:")

    if numeric.shape[1] > 0:

        numeric_summary = (
            numeric.describe().T[
                ["count", "mean", "std", "min", "max"]
            ]
        )

        _project_display(numeric_summary)

    else:
        _project_print("Nenhuma variável numérica.")

    return {
        "dataset": name,
        "rows": int(rows),
        "columns": int(cols),
        "duplicated_rows": duplicated_rows,
        "duplicated_column_names": (
            duplicated_columns
        ),
        "datetime_candidates": (
            datetime_candidates
        ),
        "temporal_summary": (
            temporal_summary
        ),
        "variable_groups": (
            variable_groups
        ),
        "schema": schema.to_dict(
            orient="records"
        ),
    }


# ============================================================
# 6. EXECUÇÃO DA AUDITORIA
# ============================================================

audit_results = {}

for name, df in datasets.items():

    audit_results[name] = audit_dataset(
        name,
        df
    )


# ============================================================
# 7. QUADRO COMPARATIVO
# ============================================================

comparison_rows = []

for name, result in audit_results.items():

    comparison_rows.append({
        "dataset": name,
        "rows": result["rows"],
        "columns": result["columns"],
        "duplicated_rows": (
            result["duplicated_rows"]
        ),
        "datetime_candidates": ", ".join(
            result["datetime_candidates"]
        ),
        "co2_candidates": len(
            result["variable_groups"]["co2"]
        ),
        "generation_candidates": len(
            result["variable_groups"]["generation"]
        ),
        "hmm_candidates": len(
            result["variable_groups"]["hmm_states"]
        ),
        "process_candidates": len(
            result["variable_groups"]["process"]
        ),
    })

comparison_df = pd.DataFrame(
    comparison_rows
)

_project_print("\nQUADRO COMPARATIVO DOS DATASETS")
_project_display(comparison_df)


# ============================================================
# 8. SALVAMENTO DOS RELATÓRIOS
# ============================================================

AUDIT_FILE = (
    DIRS["reports"]
    / "cell03_dataset_audit.json"
)

COMPARISON_FILE = (
    DIRS["tables"]
    / "cell03_dataset_comparison.csv"
)

SELECTION_FILE = (
    DIRS["manifests"]
    / "cell03_selected_inputs.json"
)

for path in [
    AUDIT_FILE,
    COMPARISON_FILE,
    SELECTION_FILE
]:
    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

audit_report = {
    "phase": "phase_04_predictive_modeling",
    "cell": "03",
    "generated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "source_manifest": str(MANIFEST_FILE),
    "datasets": audit_results,
}

with open(
    AUDIT_FILE,
    "w",
    encoding="utf-8"
) as file:

    _project_dump(
        audit_report,
        file,
        indent=4,
        ensure_ascii=False,
        default=str
    )

_project_csv(comparison_df, 
    COMPARISON_FILE,
    index=False
)

selection_report = {
    "phase": "phase_04_predictive_modeling",
    "cell": "03",
    "phase2_reference_run": (
        PHASE2_REFERENCE_RUN
    ),
    "phase3_temporal_reference_run": (
        PHASE3_REFERENCE_RUN
    ),
    "phase3_feature_reference_run": (
        str(dataset_paths[
            "phase3_feature_engineering"
        ].parent)
    ),
    "generated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "selected_files": file_metadata,
}

with open(
    SELECTION_FILE,
    "w",
    encoding="utf-8"
) as file:

    _project_dump(
        selection_report,
        file,
        indent=4,
        ensure_ascii=False
    )


# ============================================================
# 9. VERIFICAÇÕES FINAIS
# ============================================================

assert len(datasets) == 3
assert AUDIT_FILE.exists()
assert COMPARISON_FILE.exists()
assert SELECTION_FILE.exists()

assert (
    datasets["phase2_process_features"].shape[0]
    == 6249
), "Número inesperado de horas operacionais na Fase 2."

_project_print("\n" + "=" * 75)
_project_print("FASE 4 — CÉLULA 03")
_project_print("=" * 75)

_project_print("Datasets carregados:", len(datasets))

_project_print("\nRelatório:")
_project_print(AUDIT_FILE)

_project_print("\nTabela comparativa:")
_project_print(COMPARISON_FILE)

_project_print("\nManifesto dos arquivos selecionados:")
_project_print(SELECTION_FILE)

_project_print("\nSTATUS: AUDITORIA ESTRUTURAL CONCLUÍDA")
_project_print("=" * 75)

### Base comum para os três cenários

[EPA CAM API](https://www.epa.gov/power-sector/cam-api-portal) fornece a fonte e as unidades. A conversão para tonelada métrica e as contagens desta comparação são escolhas documentadas para Iatan 2. A igualdade de chaves e alvo sustenta a comparação pareada.


In [ ]:
# ============================================================
# FASE 4 — CÉLULA 04
# INTEGRAÇÃO E VALIDAÇÃO DO DATASET MESTRE
# ============================================================
#
# OBJETIVOS:
# 1. Padronizar timestamps.
# 2. Preservar o calendário completo de 2025.
# 3. Integrar dados de CO2, geração e processos.
# 4. Verificar consistência entre as fontes.
# 5. Preservar as partições da Fase 3.
# 6. Exportar dataset e relatório de integração.
#
# NÃO REALIZA:
# - imputação;
# - seleção de atributos;
# - treinamento;
# - alteração dos dados originais.
# ============================================================

import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd

# ============================================================
# 1. RECUPERAÇÃO DOS DATASETS
# ============================================================

required = [
    "phase2_process_features",
    "phase3_temporal_partitions",
    "phase3_feature_engineering",
]

if "datasets" not in globals():
    raise RuntimeError("Execute primeiro a Célula 03.")

for name in required:
    if name not in datasets:
        raise RuntimeError(f"Dataset ausente: {name}")

df_process = datasets["phase2_process_features"].copy()
df_partition = datasets["phase3_temporal_partitions"].copy()
df_engineering = datasets["phase3_feature_engineering"].copy()


# ============================================================
# 2. PADRONIZAÇÃO TEMPORAL
# ============================================================
#
# Convenção do protocolo histórico: timestamps sem timezone recebem
# a etiqueta UTC para a junção, preservando o rótulo de hora do CEMS.
# A concordância das chaves e da geração verifica o alinhamento das tabelas,
# mas não estabelece o fuso físico da medição.
# ============================================================

for df in [df_process, df_partition, df_engineering]:

    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        utc=True,
        errors="raise"
    )

    if df["timestamp"].isna().any():
        raise ValueError("Timestamp inválido encontrado.")

    if df["timestamp"].duplicated().any():
        raise ValueError(
            "Timestamps duplicados em uma fonte."
        )


# ============================================================
# 3. CONSTRUÇÃO DA BASE PRINCIPAL
# ============================================================
#
# A base de partições é a referência do calendário,
# do CO2 observado e da geração original.
# ============================================================

master = df_partition.copy()

assert len(master) == 8760

master = master.sort_values(
    "timestamp"
).reset_index(drop=True)

master["source_process_available"] = False


# ============================================================
# 4. PREPARAÇÃO DOS ATRIBUTOS DA FASE 2
# ============================================================

process_columns = [
    col for col in df_process.columns
    if col not in [
        "timestamp",
        "partition",
        "load_mw"
    ]
]

process_join = df_process[
    ["timestamp", "load_mw"] + process_columns
].copy()

process_join = process_join.rename(
    columns={
        "load_mw": "phase2_load_mw"
    }
)

process_join["source_process_available"] = True

master = master.drop(
    columns=["source_process_available"]
)

master = master.merge(
    process_join,
    on="timestamp",
    how="left",
    validate="one_to_one",
    indicator="_merge_process"
)

master["source_process_available"] = (
    master["_merge_process"] == "both"
)

master = master.drop(
    columns="_merge_process"
)


# ============================================================
# 5. INTEGRAÇÃO DA ENGENHARIA DE ATRIBUTOS
# ============================================================
#
# Os campos da engenharia anterior recebem prefixo
# para evitar colisões e preservar a proveniência.
# ============================================================

engineering_columns = [
    col for col in df_engineering.columns
    if col not in [
        "timestamp",
        "Date",
        "Hour",
        "Facility ID",
        "Unit ID",
        "facilityId_norm",
        "unitId_norm",
    ]
]

engineering_join = df_engineering[
    ["timestamp"] + engineering_columns
].copy()

engineering_join = engineering_join.rename(
    columns={
        col: f"eng_{col}"
        for col in engineering_columns
    }
)

master = master.merge(
    engineering_join,
    on="timestamp",
    how="left",
    validate="one_to_one",
    indicator="_merge_engineering"
)

master["source_engineering_available"] = (
    master["_merge_engineering"] == "both"
)

master = master.drop(
    columns="_merge_engineering"
)


# ============================================================
# 6. VERIFICAÇÃO DA INTEGRIDADE TEMPORAL
# ============================================================

expected_index = pd.date_range(
    "2025-01-01 00:00:00",
    "2025-12-31 23:00:00",
    freq="h",
    tz="UTC"
)

actual_index = pd.DatetimeIndex(
    master["timestamp"]
)

assert len(master) == len(expected_index)
assert actual_index.is_unique
assert actual_index.equals(expected_index)

_project_print("Calendário completo de 2025: OK")

# As partições F2 (HMM) e F3 (regressores) têm finalidades distintas.
# Um corte de treino F2 posterior ao início da validação F4 é evidência
# de sobreposição entre as partições registradas. Cortes coerentes são
# necessários, mas não certificam a origem do ajuste de um Parquet.
def partition_bounds(frame, eligible=None):
    labels = frame['partition']
    if eligible is not None:
        frame = frame.loc[eligible]
        labels = frame['partition']
    return {
        label: {'first': str(group['timestamp'].min()), 'last': str(group['timestamp'].max())}
        for label in ('train', 'validation', 'test')
        if not (group := frame.loc[labels.eq(label)]).empty
    }

bounds_f2 = partition_bounds(df_process) if 'partition' in df_process else {}
bounds_f4 = partition_bounds(df_partition, df_partition['eligible_for_modeling'])
overlap_fit_f2 = None
overlap_selection_f2 = None
if 'train' in bounds_f2 and 'validation' in bounds_f4:
    overlap_fit_f2 = pd.Timestamp(bounds_f2['train']['last']) >= pd.Timestamp(bounds_f4['validation']['first'])
if 'validation' in bounds_f2 and 'validation' in bounds_f4:
    start_f2 = pd.Timestamp(bounds_f2['validation']['first'])
    end_f2 = pd.Timestamp(bounds_f2['validation']['last'])
    start_f4 = pd.Timestamp(bounds_f4['validation']['first'])
    end_f4 = pd.Timestamp(bounds_f4['validation']['last'])
    overlap_selection_f2 = bool(start_f2 <= end_f4 and start_f4 <= end_f2)
phase2_export_path = dataset_paths['phase2_process_features']
phase2_manifest_path = phase2_export_path.parents[2] / 'reports' / 'cell30_process_features_manifest.json'
phase2_manifest_summary = {'status': 'not_available'}
if phase2_manifest_path.is_file():
    phase2_manifest = json.loads(phase2_manifest_path.read_text(encoding='utf-8'))
    declared_hash = phase2_manifest.get('output_sha256', {}).get('all')
    actual_hash = file_metadata['phase2_process_features']['sha256']
    if declared_hash is not None and declared_hash != actual_hash:
        raise RuntimeError('O SHA-256 do Parquet F2 diverge do manifesto C30.')
    declared_run = phase2_manifest.get('run_id')
    if declared_run is not None and declared_run != PHASE2_REFERENCE_RUN:
        raise RuntimeError('O ID do manifesto C30 diverge da execução F2 selecionada.')
    phase2_manifest_summary = {
        'status': 'checked_parquet_sha256' if declared_hash is not None else 'no_parquet_sha256',
        'path': str(phase2_manifest_path),
        'sha256': calculate_sha256(phase2_manifest_path),
        'hmm_temporal_provenance': phase2_manifest.get('hmm_temporal_provenance'),
    }

phase3_export_path = dataset_paths['phase3_temporal_partitions']
phase3_manifest_path = phase3_export_path.parents[2] / 'reports' / 'cell03_experimental_protocol.json'
phase3_manifest_summary = {'status': 'not_available'}
if phase3_manifest_path.is_file():
    phase3_manifest = json.loads(phase3_manifest_path.read_text(encoding='utf-8'))
    verified_fields = []
    for field, category in (
        ('partition_parquet_sha256', 'phase3_temporal_partitions'),
        ('source_sha256', 'phase3_feature_engineering'),
    ):
        declared_hash = phase3_manifest.get(field)
        if declared_hash is not None:
            if declared_hash != file_metadata[category]['sha256']:
                raise RuntimeError(
                    f'{field} diverge do manifesto F3. Use o mesmo step05 canônico em F3 e F4.'
                )
            verified_fields.append(field)
    phase3_manifest_summary = {
        'status': 'hashes_checked' if verified_fields else 'no_hashes_declared',
        'sha256': calculate_sha256(phase3_manifest_path), 'fields_checked': verified_fields,
    }

TEMPORAL_SOURCE_AUDIT = {
    'phase2_partitions': bounds_f2, 'phase4_partitions': bounds_f4,
    'phase2_feature_manifest': phase2_manifest_summary,
    'phase3_partition_manifest': phase3_manifest_summary,
    'phase2_train_cutoff_reaches_phase4_validation': overlap_fit_f2,
    'phase2_validation_intersects_phase4_validation': overlap_selection_f2,
    'foldwise_refit_verified': False,
    'interpretation': 'exploratory',
}
_project_write_text(DIRS['manifests'] / 'phase04_temporal_source_audit.json', 
    _project_dumps(TEMPORAL_SOURCE_AUDIT, ensure_ascii=False, indent=2), encoding='utf-8')
_project_print('Proveniência temporal:', TEMPORAL_SOURCE_AUDIT)


# ============================================================
# 7. VERIFICAÇÃO DA CORRESPONDÊNCIA DE GERAÇÃO
# ============================================================

overlap = master[
    master["source_process_available"]
].copy()

generation_difference = (
    overlap["generation_mw"]
    - overlap["phase2_load_mw"]
).abs()

valid_comparison = generation_difference.notna()

generation_mismatches = int(
    (
        generation_difference[valid_comparison]
        > 1e-6
    ).sum()
)

generation_max_difference = (
    float(generation_difference.max())
    if valid_comparison.any()
    else None
)

_project_print("\nCOMPARAÇÃO DE GERAÇÃO")

_project_print("Horas com atributos de processos:", len(overlap))
_project_print("Comparações válidas:", int(valid_comparison.sum()))
_project_print("Divergências:", generation_mismatches)
_project_print("Diferença máxima (MW):", generation_max_difference)

if generation_mismatches > 0:
    raise ValueError(
        "Divergência entre geração da Fase 2 "
        "e geração da Fase 3. Revisar timestamps."
    )

if not valid_comparison.all():
    raise ValueError(
        "Há registros de processos sem geração "
        "comparável na Fase 3."
    )


# ============================================================
# 8. VERIFICAÇÃO DO CO2 ENTRE AS FONTES
# ============================================================

SHORT_TON_TO_METRIC_TON = 0.90718474

co2_from_short_tons = (
    master["co2_short_tons"]
    * SHORT_TON_TO_METRIC_TON
)

co2_difference = (
    master["co2_metric_tons"]
    - co2_from_short_tons
).abs()

co2_valid = co2_difference.notna()

co2_mismatches = int(
    (co2_difference[co2_valid] > 1e-5).sum()
)

_project_print("\nVALIDAÇÃO DE UNIDADES DE CO2")
_project_print("Comparações válidas:", int(co2_valid.sum()))
_project_print("Divergências:", co2_mismatches)

if co2_mismatches > 0:
    raise ValueError(
        "Inconsistência na conversão de CO2 "
        "de short tons para toneladas métricas."
    )


# ============================================================
# 9. VERIFICAÇÃO DAS PARTIÇÕES
# ============================================================

_project_print("\nPARTIÇÕES ORIGINAIS")

partition_summary = (
    master.groupby(
        "partition",
        dropna=False
    )
    .agg(
        hours=("timestamp", "size"),
        eligible_hours=("eligible_for_modeling", "sum"),
        process_hours=("source_process_available", "sum"),
    )
    .reset_index()
)

_project_display(partition_summary)

# A partição original é preservada sem alterações.
assert master["partition"].notna().all()


# ============================================================
# 10. VERIFICAÇÃO DA COBERTURA
# ============================================================

coverage_summary = {
    "calendar_hours": int(len(master)),
    "process_hours": int(
        master["source_process_available"].sum()
    ),
    "engineering_hours": int(
        master["source_engineering_available"].sum()
    ),
    "generation_observed_hours": int(
        master["generation_mw"].notna().sum()
    ),
    "co2_observed_hours": int(
        master["co2_metric_tons"].notna().sum()
    ),
    "eligible_modeling_hours": int(
        master["eligible_for_modeling"].sum()
    ),
    "generation_mismatches": generation_mismatches,
    "co2_conversion_mismatches": co2_mismatches,
}

_project_print("\nCOBERTURA DO DATASET MESTRE")

_project_display(
    pd.DataFrame(
        coverage_summary.items(),
        columns=["indicator", "value"]
    )
)


# ============================================================
# 11. VALIDAÇÃO DOS ATRIBUTOS DE PROCESSOS
# ============================================================

process_without_eligibility = int(
    (
        master["source_process_available"]
        & ~master["eligible_for_modeling"]
    ).sum()
)

eligible_without_process = int(
    (
        master["eligible_for_modeling"]
        & ~master["source_process_available"]
    ).sum()
)

_project_print("\nCOMPATIBILIDADE COM A AMOSTRA MODELÁVEL")

_project_print(
    "Processos fora da amostra elegível:",
    process_without_eligibility
)

_project_print(
    "Horas elegíveis sem processos:",
    eligible_without_process
)

# Não eliminar linhas nesta etapa.
# Divergências de cobertura serão examinadas antes
# da construção dos conjuntos B0, P1 e P2.


# ============================================================
# 12. EXPORTAÇÃO DO DATASET MESTRE
# ============================================================

MASTER_FILE = (
    DIRS["processed"]
    / "cell04_master_dataset.parquet"
)

master.to_parquet(
    MASTER_FILE,
    index=False
)

PARTITION_SUMMARY_FILE = (
    DIRS["tables"]
    / "cell04_partition_summary.csv"
)

_project_csv(partition_summary, 
    PARTITION_SUMMARY_FILE,
    index=False
)


# ============================================================
# 13. MANIFESTO DE INTEGRAÇÃO
# ============================================================

integration_report = {
    "phase": "phase_04_predictive_modeling",
    "cell": "04",
    "generated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "timestamp_assumption": (
        "Naive CEMS hour labels tagged UTC for historical alignment; physical timezone not inferred"
    ),
    "master_file": str(MASTER_FILE),
    "rows": int(len(master)),
    "columns": int(master.shape[1]),
    "coverage": coverage_summary,
    "process_without_eligibility": (
        process_without_eligibility
    ),
    "eligible_without_process": (
        eligible_without_process
    ),
    "partition_summary": (
        partition_summary.to_dict(
            orient="records"
        )
    ),
}

INTEGRATION_REPORT_FILE = (
    DIRS["reports"]
    / "cell04_integration_report.json"
)

with open(
    INTEGRATION_REPORT_FILE,
    "w",
    encoding="utf-8"
) as file:
    _project_dump(
        integration_report,
        file,
        indent=4,
        ensure_ascii=False,
        default=str
    )


# ============================================================
# 14. VERIFICAÇÕES FINAIS
# ============================================================

assert MASTER_FILE.exists()
assert INTEGRATION_REPORT_FILE.exists()
assert len(master) == 8760

_project_print("\n" + "=" * 70)
_project_print("FASE 4 — CÉLULA 04")
_project_print("=" * 70)

_project_print("Dataset mestre:", master.shape)
_project_print("Arquivo:", MASTER_FILE)

_project_print("\nRelatório:")
_project_print(INTEGRATION_REPORT_FILE)

_project_print("\nSTATUS: INTEGRAÇÃO CONCLUÍDA")
_project_print("=" * 70)

### Disponibilidade temporal dos atributos

[Rabiner (1989)](https://doi.org/10.1109/5.18626) distingue filtragem forward e inferência retrospectiva. [Diba et al. (2020)](https://doi.org/10.1002/widm.1346) descrevem extração, correlação e abstração de eventos. As horas e os ciclos de Iatan definem as regras locais.


In [ ]:

# ============================================================
# FASE 4 — CÉLULA 05
# AUDITORIA DE VAZAMENTO E ADMISSIBILIDADE DOS ATRIBUTOS
# ============================================================
#
# OBJETIVO:
# Classificar os atributos do dataset mestre conforme
# sua disponibilidade, proveniência e risco de leakage.
#
# NÃO REALIZA:
# - seleção definitiva de variáveis;
# - treinamento;
# - exclusão de linhas;
# - modificação dos arquivos originais.
#
# REGRA:
# A classificação automática é preliminar.
# A admissibilidade de atributos HMM depende também
# da metodologia de estimação utilizada na Fase 2.
# ============================================================

import json
import re
from datetime import datetime, timezone

import numpy as np
import pandas as pd


# ============================================================
# 1. CARREGAMENTO DO DATASET MESTRE
# ============================================================

MASTER_FILE = (
    DIRS["processed"]
    / "cell04_master_dataset.parquet"
)

if not MASTER_FILE.exists():
    raise FileNotFoundError(
        "Execute a Célula 04 antes desta etapa."
    )

df = pd.read_parquet(MASTER_FILE)

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    utc=True
)

df = df.sort_values("timestamp").reset_index(drop=True)

assert len(df) == 8760
assert df["timestamp"].is_unique

eligible = df.loc[
    df["eligible_for_modeling"]
].copy()

assert len(eligible) == 6249

_project_print("Dataset mestre:", df.shape)
_project_print("Amostra elegível:", eligible.shape)


# ============================================================
# 2. DEFINIÇÃO DO PROBLEMA PREDITIVO
# ============================================================
#
# Estimativa contemporânea:
#
# CO2(t) = f(informações operacionais disponíveis até t)
#
# A geração G(t) é admitida como variável explicativa.
#
# Portanto, trata-se de estimar emissões da hora t
# a partir de medições operacionais dessa mesma hora,
# e NÃO de prever emissões antes de a hora começar.
#
# Dados de t+1, t+2 etc. não são admissíveis.
# ============================================================

TARGET = "co2_metric_tons"

BASELINE_FEATURE = "generation_mw"

PREDICTION_PROTOCOL = (
    "Contemporaneous hourly CO2 estimation using "
    "operational information available through hour t"
)

assert TARGET in df.columns
assert BASELINE_FEATURE in df.columns


# ============================================================
# 3. GRUPOS DE VARIÁVEIS
# ============================================================

# Variáveis derivadas diretamente do CO2.
# Nunca serão utilizadas como preditores.

TARGET_DERIVED = {
    "co2_metric_tons",
    "co2_short_tons",
    "co2_missing",
    "co2_zero",
    "co2_positive",
    "negative_co2",
    "generation_positive_co2_zero",
    "generation_zero_co2_positive",
}

# Variáveis administrativas ou de particionamento.

METADATA_EXCLUDED = {
    "timestamp",
    "partition",
    "eligible_for_modeling",
    "case_id",
    "cycle_index",
    "provisional_on_cycle",
    "source_process_available",
    "source_engineering_available",
}

# Variáveis convencionais inicialmente admissíveis.

CONVENTIONAL_PRELIMINARY = {
    "generation_mw",
}

# Estados HMM e suas representações.
# Dependem de auditoria da construção do HMM.

HMM_DEPENDENT = {
    "hmm_state",
    "hmm_confidence",
    "hmm_entropy",
    "previous_hmm_state",
    "has_previous_state",
    "state_changed",
    "cumulative_state_changes",
    "hours_in_current_state",
    "hmm_state_lag_1h",
    "hmm_state_lag_2h",
    "hmm_state_lag_3h",
    "hmm_state_lag_6h",
    "hmm_state_lag_12h",
    "hmm_state_lag_24h",
    "state_changes_last_3h",
    "state_changes_last_6h",
    "state_changes_last_12h",
    "state_changes_last_24h",
    "is_regime_s0",
    "is_regime_s1",
    "is_regime_s2",
    "is_regime_s3",
    "is_regime_s4",
    "regime_activity",
    "combined_activity",
}

# Atributos de processo baseados em geração e
# histórico operacional. Sua construção ainda
# precisa ser verificada quanto à causalidade.

PROCESS_DEPENDENT = {
    "hours_since_cycle_start",
    "ramp_defined",
    "delta_load_mw",
    "hours_in_current_ramp",
    "load_mean_last_3h",
    "load_std_last_3h",
    "load_mean_last_6h",
    "load_std_last_6h",
    "load_mean_last_12h",
    "load_std_last_12h",
    "load_mean_last_24h",
    "load_std_last_24h",
    "load_lag_1h",
    "load_lag_2h",
    "load_lag_3h",
    "load_lag_6h",
    "load_lag_12h",
    "load_lag_24h",
    "is_ramp_up",
    "is_ramp_down",
    "is_ramp_stable",
    "is_ramp_undefined",
    "ramp_activity",
}

# Variáveis temporais calculáveis sem observar CO2.

CALENDAR_FEATURES = {
    "hour_of_day",
    "day_of_week",
    "month",
    "hour_sin",
    "hour_cos",
}


# ============================================================
# 4. CLASSIFICAÇÃO DOS ATRIBUTOS
# ============================================================

def classify_feature(column):

    if column in TARGET_DERIVED:
        return "excluded_target_derived"

    if column in METADATA_EXCLUDED:
        return "excluded_metadata"

    if column.startswith("eng_"):
        return "review_engineering_provenance"

    if column in CONVENTIONAL_PRELIMINARY:
        return "preliminary_conventional"

    if column in HMM_DEPENDENT:
        return "review_hmm_causality"

    if column in PROCESS_DEPENDENT:
        return "review_process_causality"

    if column in CALENDAR_FEATURES:
        return "preliminary_calendar"

    return "review_unclassified"


feature_audit = pd.DataFrame({
    "feature": list(df.columns)
})

feature_audit["classification"] = (
    feature_audit["feature"].apply(
        classify_feature
    )
)

feature_audit["missing_eligible_pct"] = (
    feature_audit["feature"]
    .map(
        eligible.isna().mean().mul(100).to_dict()
    )
    .round(3)
)

feature_audit["dtype"] = (
    feature_audit["feature"]
    .map(df.dtypes.astype(str).to_dict())
)


# ============================================================
# 5. VERIFICAÇÃO DE DISPONIBILIDADE DOS ATRIBUTOS
# ============================================================

_project_print("\nCLASSIFICAÇÃO DOS ATRIBUTOS")

classification_summary = (
    feature_audit.groupby(
        "classification"
    )
    .size()
    .reset_index(name="n_features")
    .sort_values("classification")
)

_project_display(classification_summary)

_project_print("\nATRIBUTOS QUE EXIGEM REVISÃO")

review_features = feature_audit[
    feature_audit["classification"].str.startswith(
        "review_"
    )
]

_project_display(review_features)


# ============================================================
# 6. TESTES DE CONSISTÊNCIA OPERACIONAL
# ============================================================
#
# Esses testes identificam inconsistências observáveis.
# Não demonstram, isoladamente, ausência de leakage.
# ============================================================

checks = {}

# ------------------------------------------------------------
# 6.1. Correspondência da geração
# ------------------------------------------------------------

checks["generation_phase2_consistent"] = bool(
    np.allclose(
        eligible["generation_mw"],
        eligible["phase2_load_mw"],
        rtol=0,
        atol=1e-6,
        equal_nan=False
    )
)

# ------------------------------------------------------------
# 6.2. Valores finitos do alvo
# ------------------------------------------------------------

checks["target_finite"] = bool(
    np.isfinite(
        eligible[TARGET].to_numpy(dtype=float)
    ).all()
)

# ------------------------------------------------------------
# 6.3. Valores finitos da geração
# ------------------------------------------------------------

checks["generation_finite"] = bool(
    np.isfinite(
        eligible[BASELINE_FEATURE].to_numpy(
            dtype=float
        )
    ).all()
)

# ------------------------------------------------------------
# 6.4. Cobertura dos atributos de processos
# ------------------------------------------------------------

checks["process_coverage_complete"] = bool(
    eligible["source_process_available"].all()
)

# ------------------------------------------------------------
# 6.5. Estados HMM válidos
# ------------------------------------------------------------

observed_states = sorted(
    eligible["hmm_state"].dropna().unique().tolist()
)

checks["hmm_states_valid"] = (
    observed_states == [0, 1, 2, 3, 4]
)

# ------------------------------------------------------------
# 6.6. Coerência dos indicadores de estado
# ------------------------------------------------------------

regime_columns = [
    f"is_regime_s{i}"
    for i in range(5)
]

regime_matrix = eligible[
    regime_columns
].to_numpy()

checks["regime_one_hot_consistent"] = bool(
    (regime_matrix.sum(axis=1) == 1).all()
)

# ------------------------------------------------------------
# 6.7. Coerência dos indicadores de rampa
# ------------------------------------------------------------

ramp_columns = [
    "is_ramp_up",
    "is_ramp_down",
    "is_ramp_stable",
    "is_ramp_undefined",
]

checks["ramp_one_hot_consistent"] = bool(
    (
        eligible[ramp_columns]
        .sum(axis=1) == 1
    ).all()
)

# ------------------------------------------------------------
# 6.8. Identificação de infinitos
# ------------------------------------------------------------

numeric_columns = eligible.select_dtypes(
    include=[np.number]
).columns

infinite_counts = {}

for col in numeric_columns:
    values = eligible[col].to_numpy(
        dtype=float,
        na_value=np.nan
    )
    count = int(np.isinf(values).sum())

    if count > 0:
        infinite_counts[col] = count

checks["no_infinite_numeric_values"] = (
    len(infinite_counts) == 0
)

_project_print("\nVERIFICAÇÕES OPERACIONAIS")

for check, passed in checks.items():
    _project_print(
        f"{'OK' if passed else 'REVISAR'}: {check}"
    )

if infinite_counts:
    _project_print("\nValores infinitos encontrados:")
    _project_print(infinite_counts)


# ============================================================
# 7. AUDITORIA PRELIMINAR DOS LAGS
# ============================================================
#
# Verificamos se os lags de geração da Fase 2
# correspondem ao histórico dentro do mesmo ciclo.
#
# A comparação utiliza o deslocamento de linhas
# dentro de cada case_id, conforme a estrutura
# observada da Fase 2.
#
# Não substitui uma auditoria do código de origem.
# ============================================================

process_only = eligible.sort_values(
    ["case_id", "timestamp"]
).copy()

lag_checks = []

for lag in [1, 2, 3, 6, 12, 24]:

    column = f"load_lag_{lag}h"

    expected = (
        process_only.groupby(
            "case_id",
            sort=False
        )["phase2_load_mw"]
        .shift(lag)
    )

    observed = process_only[column]

    both_valid = (
        expected.notna()
        & observed.notna()
    )

    difference = (
        expected[both_valid]
        - observed[both_valid]
    ).abs()

    mismatches = int(
        (difference > 1e-6).sum()
    )

    missing_pattern_mismatches = int(
        (
            expected.isna()
            != observed.isna()
        ).sum()
    )

    lag_checks.append({
        "lag_hours": lag,
        "comparisons": int(both_valid.sum()),
        "value_mismatches": mismatches,
        "missing_pattern_mismatches": (
            missing_pattern_mismatches
        ),
    })

lag_audit_df = pd.DataFrame(lag_checks)

_project_print("\nAUDITORIA DOS LAGS DA FASE 2")
_project_display(lag_audit_df)


# ============================================================
# 8. PENDÊNCIAS METODOLÓGICAS
# ============================================================

methodological_checks = {
    "hmm_fit_train_only": "not_verified",
    "hmm_states_filtered_not_smoothed": "not_verified",
    "hmm_scaler_train_only": "not_verified",
    "hmm_state_mapping_fixed_before_validation": "not_verified",
    "process_features_no_future_information": "not_verified",
    "process_features_no_target_dependency": "not_verified",
    "temporal_partitions_preserved": "verified",
    "same_eligible_rows_for_comparison": "verified",
}

_project_print("\nPENDÊNCIAS METODOLÓGICAS")

for item, status in methodological_checks.items():
    _project_print(f"{item}: {status}")


# ============================================================
# 9. EXPORTAÇÃO DOS RESULTADOS
# ============================================================

FEATURE_AUDIT_FILE = (
    DIRS["tables"]
    / "cell05_feature_admissibility.csv"
)

LAG_AUDIT_FILE = (
    DIRS["tables"]
    / "cell05_lag_consistency.csv"
)

AUDIT_REPORT_FILE = (
    DIRS["reports"]
    / "cell05_leakage_audit.json"
)

_project_csv(feature_audit, 
    FEATURE_AUDIT_FILE,
    index=False
)

_project_csv(lag_audit_df, 
    LAG_AUDIT_FILE,
    index=False
)

audit_report = {
    "phase": "phase_04_predictive_modeling",
    "cell": "05",
    "generated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "prediction_protocol": PREDICTION_PROTOCOL,
    "target": TARGET,
    "baseline_feature": BASELINE_FEATURE,
    "operational_checks": checks,
    "observed_hmm_states": observed_states,
    "lag_checks": lag_checks,
    "methodological_checks": methodological_checks,
    "feature_classification": (
        feature_audit.to_dict(
            orient="records"
        )
    ),
}

with open(
    AUDIT_REPORT_FILE,
    "w",
    encoding="utf-8"
) as file:
    _project_dump(
        audit_report,
        file,
        indent=4,
        ensure_ascii=False,
        default=str
    )


# ============================================================
# 10. RESULTADO FINAL
# ============================================================

_project_print("\n" + "=" * 70)
_project_print("FASE 4 — CÉLULA 05")
_project_print("=" * 70)

_project_print("Atributos auditados:", len(feature_audit))
_project_print("Testes operacionais:", len(checks))
_project_print("Testes aprovados:", sum(checks.values()))

_project_print("\nRelatório:")
_project_print(AUDIT_REPORT_FILE)

_project_print("\nSTATUS: AUDITORIA PRELIMINAR CONCLUÍDA")
_project_print(
    "ATENÇÃO: admissibilidade temporal do HMM "
    "ainda requer verificação metodológica."
)

_project_print("=" * 70)


In [ ]:

# ============================================================
# FASE 4 — CÉLULA 06
# CONSTRUÇÃO DOS CONJUNTOS EXPERIMENTAIS B0, P1 E P2
# ============================================================
#
# OBJETIVO:
# Definir explicitamente os atributos dos três cenários
# de modelagem preditiva de CO2.
#
# B0: Geração elétrica contemporânea.
# P1: B0 + regimes operacionais HMM filtrados.
# P2: P1 + atributos de mineração de processos.
#
# PRINCÍPIOS:
# - Mesmo alvo e mesmas observações nos três cenários.
# - Nenhuma variável derivada de CO2 como preditor.
# - Nenhuma informação de partição como preditor.
# - Nenhuma seleção automática de todas as colunas.
# - Sem treinamento ou ajuste de pré-processamento.
# ============================================================

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. CARREGAMENTO DO DATASET MESTRE
# ============================================================

MASTER_FILE = (
    DIRS["processed"]
    / "cell04_master_dataset.parquet"
)

if not MASTER_FILE.exists():
    raise FileNotFoundError(
        "Dataset mestre não encontrado. Execute a Célula 04."
    )

master = pd.read_parquet(MASTER_FILE)

master["timestamp"] = pd.to_datetime(
    master["timestamp"],
    utc=True
)

master = master.sort_values(
    "timestamp"
).reset_index(drop=True)


# ============================================================
# 2. DEFINIÇÃO DO ALVO
# ============================================================

TARGET = "co2_metric_tons"

BASELINE_FEATURES = [
    "generation_mw",
]


# ============================================================
# 3. P1 — ESTADOS OPERACIONAIS HMM
# ============================================================
#
# Utilizamos uma representação categórica do estado.
#
# As variáveis is_regime_s0 ... is_regime_s4
# representam os cinco estados HMM filtrados.
#
# Não incluímos simultaneamente hmm_state como
# variável numérica, evitando impor uma ordenação
# artificial aos estados.
#
# A confiança e a entropia ficam reservadas para
# análises adicionais de ablação.
# ============================================================

HMM_FEATURES = [
    "is_regime_s0",
    "is_regime_s1",
    "is_regime_s2",
    "is_regime_s3",
    "is_regime_s4",
]


# ============================================================
# 4. P2 — ATRIBUTOS DE MINERAÇÃO DE PROCESSOS
# ============================================================
#
# Incluímos atributos derivados da sequência operacional,
# do histórico de geração e da evolução dos regimes.
#
# Excluímos:
# - identificadores de ciclos;
# - atributos derivados de CO2;
# - variáveis de calendário;
# - representações textuais redundantes das atividades;
# - lags de HMM com muitos valores ausentes.
#
# Os atributos abaixo representam uma especificação
# inicial explícita e reproduzível de P2.
# ============================================================

PROCESS_FEATURES = [

    # --------------------------------------------------------
    # Contexto do ciclo operacional
    # --------------------------------------------------------

    "hours_since_cycle_start",

    # --------------------------------------------------------
    # Transições entre regimes
    # --------------------------------------------------------

    "state_changed",
    "cumulative_state_changes",
    "hours_in_current_state",

    # --------------------------------------------------------
    # Histórico de mudanças de regime
    # --------------------------------------------------------

    "state_changes_last_3h",
    "state_changes_last_6h",
    "state_changes_last_12h",
    "state_changes_last_24h",

    # --------------------------------------------------------
    # Comportamento das rampas
    # --------------------------------------------------------

    "ramp_defined",
    "delta_load_mw",
    "hours_in_current_ramp",

    "is_ramp_up",
    "is_ramp_down",
    "is_ramp_stable",
    "is_ramp_undefined",

    # --------------------------------------------------------
    # Histórico recente de geração
    # --------------------------------------------------------

    "load_mean_last_3h",
    "load_std_last_3h",

    "load_mean_last_6h",
    "load_std_last_6h",

    "load_mean_last_12h",
    "load_std_last_12h",

    "load_mean_last_24h",
    "load_std_last_24h",

    # --------------------------------------------------------
    # Defasagens de geração
    # --------------------------------------------------------

    "load_lag_1h",
    "load_lag_2h",
    "load_lag_3h",
    "load_lag_6h",
    "load_lag_12h",
    "load_lag_24h",
]


# ============================================================
# 5. CONSTRUÇÃO DOS CENÁRIOS ANINHADOS
# ============================================================

FEATURE_SETS = {

    "B0": BASELINE_FEATURES,

    "P1": (
        BASELINE_FEATURES
        + HMM_FEATURES
    ),

    "P2": (
        BASELINE_FEATURES
        + HMM_FEATURES
        + PROCESS_FEATURES
    ),
}

assert set(FEATURE_SETS["B0"]).issubset(
    FEATURE_SETS["P1"]
)

assert set(FEATURE_SETS["P1"]).issubset(
    FEATURE_SETS["P2"]
)

for scenario, features in FEATURE_SETS.items():

    if len(features) != len(set(features)):
        raise ValueError(
            f"Atributos duplicados em {scenario}."
        )

    missing_columns = [
        col for col in features
        if col not in master.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Colunas ausentes em {scenario}: "
            f"{missing_columns}"
        )


# ============================================================
# 6. SELEÇÃO DA AMOSTRA ELEGÍVEL
# ============================================================

modeling_data = master.loc[
    master["eligible_for_modeling"]
].copy()

modeling_data = modeling_data.sort_values(
    "timestamp"
).reset_index(drop=True)

assert len(modeling_data) == 6249
assert modeling_data["timestamp"].is_unique

assert modeling_data[TARGET].notna().all()
assert modeling_data["generation_mw"].notna().all()

assert modeling_data[
    "source_process_available"
].all()


# ============================================================
# 7. BLOQUEIO DE VARIÁVEIS PROIBIDAS
# ============================================================

FORBIDDEN_FEATURES = {
    "timestamp",
    "partition",
    "eligible_for_modeling",
    "co2_metric_tons",
    "co2_short_tons",
    "co2_missing",
    "co2_zero",
    "co2_positive",
    "negative_co2",
    "generation_positive_co2_zero",
    "generation_zero_co2_positive",
    "case_id",
    "cycle_index",
    "provisional_on_cycle",
}

for scenario, features in FEATURE_SETS.items():

    forbidden_found = (
        set(features) & FORBIDDEN_FEATURES
    )

    if forbidden_found:
        raise ValueError(
            f"Variáveis proibidas em {scenario}: "
            f"{sorted(forbidden_found)}"
        )

    if any(
        feature.startswith("eng_")
        for feature in features
    ):
        raise ValueError(
            f"Variáveis de engenharia não auditadas "
            f"encontradas em {scenario}."
        )


# ============================================================
# 8. CRIAÇÃO DAS MATRIZES EXPERIMENTAIS
# ============================================================
#
# Ainda não aplicamos imputação ou normalização.
# Isso será realizado exclusivamente a partir
# dos dados de treinamento na Célula 08.
# ============================================================

X_sets = {}

for scenario, features in FEATURE_SETS.items():

    X_sets[scenario] = modeling_data[
        features
    ].copy()

y = modeling_data[TARGET].copy()

modeling_timestamps = modeling_data[
    "timestamp"
].copy()

modeling_partitions = modeling_data[
    "partition"
].copy()


# ============================================================
# 9. VERIFICAÇÃO DE TIPOS
# ============================================================

for scenario, X in X_sets.items():

    non_numeric = X.select_dtypes(
        exclude=[np.number, "bool"]
    ).columns.tolist()

    if non_numeric:
        raise TypeError(
            f"Colunas não numéricas em {scenario}: "
            f"{non_numeric}"
        )

    infinite_columns = []

    for col in X.columns:

        values = pd.to_numeric(
            X[col],
            errors="raise"
        ).to_numpy(dtype=float)

        if np.isinf(values).any():
            infinite_columns.append(col)

    if infinite_columns:
        raise ValueError(
            f"Infinitos em {scenario}: "
            f"{infinite_columns}"
        )


# ============================================================
# 10. AUDITORIA DE VALORES AUSENTES
# ============================================================

missing_summary = []

for scenario, X in X_sets.items():

    for feature in X.columns:

        missing_summary.append({
            "scenario": scenario,
            "feature": feature,
            "missing_count": int(
                X[feature].isna().sum()
            ),
            "missing_pct": round(
                100 * X[feature].isna().mean(),
                3
            ),
        })

missing_df = pd.DataFrame(missing_summary)

_project_print("\nATRIBUTOS COM VALORES AUSENTES")

_project_display(
    missing_df.loc[
        missing_df["missing_count"] > 0
    ]
)

# Não excluímos linhas com lags ausentes.
# O tratamento ocorrerá dentro do pipeline,
# ajustado apenas no treinamento.


# ============================================================
# 11. RESUMO DOS CENÁRIOS
# ============================================================

scenario_summary = []

for scenario, X in X_sets.items():

    scenario_summary.append({
        "scenario": scenario,
        "n_observations": len(X),
        "n_features": X.shape[1],
        "missing_cells": int(
            X.isna().sum().sum()
        ),
        "target": TARGET,
    })

scenario_summary_df = pd.DataFrame(
    scenario_summary
)

_project_print("\nRESUMO DOS CENÁRIOS")

_project_display(scenario_summary_df)


# ============================================================
# 12. VERIFICAÇÃO DAS PARTIÇÕES
# ============================================================

partition_summary = (
    modeling_data.groupby("partition")
    .size()
    .rename("n_observations")
    .reset_index()
)

_project_print("\nPARTIÇÕES PRESERVADAS")

_project_display(partition_summary)

EXPECTED_PARTITIONS = {
    "train": 4499,
    "validation": 687,
    "test": 1063,
}

actual_partitions = dict(
    zip(
        partition_summary["partition"],
        partition_summary["n_observations"]
    )
)

if actual_partitions != EXPECTED_PARTITIONS:
    raise ValueError(
        "As partições diferem da Fase 3. "
        f"Obtido: {actual_partitions}"
    )


# ============================================================
# 13. EXPORTAÇÃO DOS CENÁRIOS
# ============================================================

export_paths = {}

for scenario, features in FEATURE_SETS.items():

    export_df = modeling_data[
        ["timestamp", "partition", TARGET]
        + features
    ].copy()

    output_path = (
        DIRS["processed"]
        / f"cell06_{scenario.lower()}_dataset.parquet"
    )

    export_df.to_parquet(
        output_path,
        index=False
    )

    export_paths[scenario] = str(output_path)


# ============================================================
# 14. SALVAMENTO DO MANIFESTO EXPERIMENTAL
# ============================================================

FEATURE_MANIFEST_FILE = (
    DIRS["manifests"]
    / "cell06_feature_sets.json"
)

feature_manifest = {
    "phase": "phase_04_predictive_modeling",
    "cell": "06",
    "generated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "target": TARGET,
    "target_unit": "metric tons CO2/hour",
    "prediction_protocol": (
        "Contemporaneous estimation using "
        "operational information through hour t"
    ),
    "feature_sets": FEATURE_SETS,
    "n_observations": int(
        len(modeling_data)
    ),
    "partition_counts": EXPECTED_PARTITIONS,
    "export_paths": export_paths,
    "notes": [
        "HMM states represented by one-hot indicators.",
        "No CO2-derived features used as predictors.",
        "Same observations for all scenarios.",
        "Missing values retained for train-only imputation.",
        "Process history features will be assessed by ablation.",
        "HMM was trained on train partition in Phase 2.",
    ],
}

with open(
    FEATURE_MANIFEST_FILE,
    "w",
    encoding="utf-8"
) as file:

    _project_dump(
        feature_manifest,
        file,
        indent=4,
        ensure_ascii=False
    )


# ============================================================
# 15. EXPORTAÇÃO DAS TABELAS DE AUDITORIA
# ============================================================

_project_csv(scenario_summary_df, 
    DIRS["tables"]
    / "cell06_scenario_summary.csv",
    index=False
)

_project_csv(missing_df, 
    DIRS["tables"]
    / "cell06_missing_features.csv",
    index=False
)


# ============================================================
# 16. VERIFICAÇÕES FINAIS
# ============================================================

assert len(X_sets) == 3

assert all(
    len(X) == 6249
    for X in X_sets.values()
)

assert len(y) == 6249

assert all(
    Path(path).exists()
    for path in export_paths.values()
)

_project_print("\n" + "=" * 70)
_project_print("FASE 4 — CÉLULA 06")
_project_print("=" * 70)

for scenario, X in X_sets.items():
    _project_print(
        f"{scenario}: "
        f"{X.shape[0]} observações × "
        f"{X.shape[1]} atributos"
    )

_project_print("\nManifesto:")
_project_print(FEATURE_MANIFEST_FILE)

_project_print("\nSTATUS: CENÁRIOS EXPERIMENTAIS CONSTRUÍDOS")
_project_print("=" * 70)


In [ ]:

# ============================================================
# FASE 4 — CÉLULA 07
# RECUPERAÇÃO E VALIDAÇÃO DAS PARTIÇÕES TEMPORAIS
# ============================================================
#
# OBJETIVOS:
# 1. Recuperar train, validation e test da Fase 3.
# 2. Confirmar a identidade das amostras B0, P1 e P2.
# 3. Verificar ordenação temporal e ausência de sobreposição.
# 4. Criar índices reutilizáveis pelos modelos.
# 5. Exportar relatório e índices das partições.
#
# NÃO REALIZA:
# - normalização;
# - imputação;
# - seleção de modelos;
# - treinamento;
# - avaliação de desempenho.
# ============================================================

import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd


# ============================================================
# 1. CARREGAMENTO DOS TRÊS CENÁRIOS
# ============================================================

SCENARIOS = ["B0", "P1", "P2"]

scenario_data = {}

for scenario in SCENARIOS:

    path = (
        DIRS["processed"]
        / f"cell06_{scenario.lower()}_dataset.parquet"
    )

    if not path.exists():
        raise FileNotFoundError(path)

    data = pd.read_parquet(path)

    data["timestamp"] = pd.to_datetime(
        data["timestamp"],
        utc=True
    )

    data = data.sort_values(
        "timestamp"
    ).reset_index(drop=True)

    scenario_data[scenario] = data

    _project_print(
        f"{scenario}: {len(data)} observações, "
        f"{len(data.columns) - 3} atributos"
    )


# ============================================================
# 2. VERIFICAÇÃO DE IDENTIDADE DAS OBSERVAÇÕES
# ============================================================

reference = scenario_data["B0"]

TARGET = "co2_metric_tons"

for scenario, data in scenario_data.items():

    assert len(data) == len(reference)

    pd.testing.assert_series_equal(
        data["timestamp"],
        reference["timestamp"],
        check_names=False
    )

    pd.testing.assert_series_equal(
        data["partition"],
        reference["partition"],
        check_names=False
    )

    pd.testing.assert_series_equal(
        data[TARGET],
        reference[TARGET],
        check_names=False
    )

    assert data["timestamp"].is_unique

    assert data["timestamp"].is_monotonic_increasing

_project_print("\nIdentidade de timestamps, partições e alvo: OK")


# ============================================================
# 3. RECUPERAÇÃO DOS ÍNDICES
# ============================================================

PARTITION_ORDER = [
    "train",
    "validation",
    "test",
]

EXPECTED_COUNTS = {
    "train": 4499,
    "validation": 687,
    "test": 1063,
}

partition_indices = {}

for partition in PARTITION_ORDER:

    indices = np.flatnonzero(
        reference["partition"].to_numpy() == partition
    )

    if len(indices) != EXPECTED_COUNTS[partition]:
        raise ValueError(
            f"Contagem incorreta em {partition}: "
            f"{len(indices)}"
        )

    partition_indices[partition] = indices

all_indices = np.concatenate(
    [partition_indices[p] for p in PARTITION_ORDER]
)

assert len(all_indices) == len(reference)
assert len(np.unique(all_indices)) == len(reference)

_project_print("\nPartições recuperadas corretamente.")


# ============================================================
# 4. VERIFICAÇÃO CRONOLÓGICA
# ============================================================

temporal_summary = []

for partition in PARTITION_ORDER:

    indices = partition_indices[partition]

    subset = reference.iloc[indices]

    temporal_summary.append({
        "partition": partition,
        "n_observations": len(subset),
        "first_timestamp": subset["timestamp"].min(),
        "last_timestamp": subset["timestamp"].max(),
        "first_index": int(indices.min()),
        "last_index": int(indices.max()),
    })

temporal_summary_df = pd.DataFrame(
    temporal_summary
)

_project_display(temporal_summary_df)

train_end = reference.iloc[
    partition_indices["train"]
]["timestamp"].max()

validation_start = reference.iloc[
    partition_indices["validation"]
]["timestamp"].min()

validation_end = reference.iloc[
    partition_indices["validation"]
]["timestamp"].max()

test_start = reference.iloc[
    partition_indices["test"]
]["timestamp"].min()

assert train_end < validation_start
assert validation_end < test_start

# Como as horas não elegíveis foram retiradas,
# não exigimos continuidade horária absoluta.

for partition in PARTITION_ORDER:

    indices = partition_indices[partition]

    if not np.array_equal(
        indices,
        np.arange(indices.min(), indices.max() + 1)
    ):
        raise ValueError(
            f"A partição {partition} não forma "
            "um bloco cronológico na amostra elegível."
        )

_project_print("\nOrdem cronológica: OK")
_project_print("Train < Validation < Test: OK")


# ============================================================
# 5. VERIFICAÇÃO COM A FONTE ORIGINAL DA FASE 3
# ============================================================

source = datasets[
    "phase3_temporal_partitions"
].copy()

source["timestamp"] = pd.to_datetime(
    source["timestamp"],
    utc=True
)

source_eligible = source.loc[
    source["eligible_for_modeling"],
    ["timestamp", "partition", TARGET]
].sort_values("timestamp").reset_index(drop=True)

pd.testing.assert_frame_equal(
    reference[
        ["timestamp", "partition", TARGET]
    ].reset_index(drop=True),
    source_eligible.reset_index(drop=True),
    check_dtype=False
)

_project_print("Correspondência com a Fase 3: OK")


# ============================================================
# 6. MATRIZES POR CENÁRIO E PARTIÇÃO
# ============================================================
#
# Os dados permanecem brutos.
# A imputação e a normalização serão aplicadas
# na Célula 08, com ajuste somente em train.
# ============================================================

split_data = {}

for scenario, data in scenario_data.items():

    feature_columns = [
        col for col in data.columns
        if col not in [
            "timestamp",
            "partition",
            TARGET,
        ]
    ]

    split_data[scenario] = {}

    for partition in PARTITION_ORDER:

        indices = partition_indices[partition]

        subset = data.iloc[indices]

        X = subset[
            feature_columns
        ].copy().reset_index(drop=True)

        y = subset[
            TARGET
        ].copy().reset_index(drop=True)

        timestamps = subset[
            "timestamp"
        ].copy().reset_index(drop=True)

        split_data[scenario][partition] = {
            "X": X,
            "y": y,
            "timestamps": timestamps,
        }


# ============================================================
# 7. VERIFICAÇÃO FINAL DA IDENTIDADE DOS ALVOS
# ============================================================

for partition in PARTITION_ORDER:

    reference_y = split_data["B0"][
        partition
    ]["y"]

    reference_time = split_data["B0"][
        partition
    ]["timestamps"]

    for scenario in SCENARIOS:

        pd.testing.assert_series_equal(
            split_data[scenario][partition]["y"],
            reference_y
        )

        pd.testing.assert_series_equal(
            split_data[scenario][partition]["timestamps"],
            reference_time
        )

_project_print("Mesmos alvos e timestamps em B0, P1 e P2: OK")


# ============================================================
# 8. EXPORTAÇÃO DOS ÍNDICES
# ============================================================

indices_df = reference[
    ["timestamp", "partition"]
].copy()

indices_df["modeling_row_id"] = np.arange(
    len(indices_df)
)

INDICES_FILE = (
    DIRS["processed"]
    / "cell07_partition_indices.parquet"
)

indices_df.to_parquet(
    INDICES_FILE,
    index=False
)

SUMMARY_FILE = (
    DIRS["tables"]
    / "cell07_temporal_partition_summary.csv"
)

_project_csv(temporal_summary_df, 
    SUMMARY_FILE,
    index=False
)


# ============================================================
# 9. RELATÓRIO DE AUDITORIA
# ============================================================

REPORT_FILE = (
    DIRS["reports"]
    / "cell07_temporal_partitions.json"
)

report = {
    "phase": "phase_04_predictive_modeling",
    "cell": "07",
    "generated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "n_observations": int(len(reference)),
    "partition_counts": EXPECTED_COUNTS,
    "scenarios": SCENARIOS,
    "same_timestamps": True,
    "same_targets": True,
    "same_partitions": True,
    "chronological_order_verified": True,
    "source_phase3_verified": True,
    "indices_file": str(INDICES_FILE),
    "summary": temporal_summary_df.astype(
        str
    ).to_dict(orient="records"),
}

with open(
    REPORT_FILE,
    "w",
    encoding="utf-8"
) as file:

    _project_dump(
        report,
        file,
        indent=4,
        ensure_ascii=False
    )


# ============================================================
# 10. RESULTADO FINAL
# ============================================================

_project_print("\n" + "=" * 70)
_project_print("FASE 4 — CÉLULA 07")
_project_print("=" * 70)

for partition in PARTITION_ORDER:

    _project_print(
        f"{partition}: "
        f"{len(partition_indices[partition])} observações"
    )

_project_print("\nCenários:", ", ".join(SCENARIOS))
_project_print("Arquivo de índices:", INDICES_FILE)
_project_print("Relatório:", REPORT_FILE)

_project_print("\nSTATUS: PARTIÇÕES TEMPORAIS VALIDADAS")
_project_print("=" * 70)


### Transformações e treino

A imputação e a escala usam somente parâmetros do treino. A [documentação do scikit-learn sobre armadilhas comuns](https://scikit-learn.org/stable/common_pitfalls.html) explica a separação de fit e transform. [Cerqueira et al. (2020)](https://arxiv.org/abs/1905.11744) fundamentam a avaliação em ordem temporal.


In [ ]:

# ============================================================
# FASE 4 — CÉLULA 08
# PIPELINES DE PRÉ-PROCESSAMENTO
# ============================================================
#
# OBJETIVO:
# Preparar pipelines reproduzíveis para os três cenários
# e quatro algoritmos da Fase 4.
#
# REGRAS:
# - Imputação ajustada somente em train.
# - Escalonamento ajustado somente em train.
# - Nenhuma linha removida.
# - Nenhuma informação de validação/teste usada no fit.
# - Nenhum modelo preditivo treinado nesta célula.
# ============================================================

import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# ============================================================
# 1. VERIFICAÇÃO DAS ESTRUTURAS DA CÉLULA 07
# ============================================================

if "split_data" not in globals():
    raise RuntimeError(
        "Execute a Célula 07 antes desta célula."
    )

SCENARIOS = ["B0", "P1", "P2"]

ALGORITHMS = [
    "ElasticNet",
    "RandomForest",
    "XGBoost",
    "HistGradientBoosting",
]

for scenario in SCENARIOS:
    assert scenario in split_data

    for partition in ["train", "validation", "test"]:
        assert partition in split_data[scenario]


# ============================================================
# 2. IDENTIFICAÇÃO DAS VARIÁVEIS BINÁRIAS
# ============================================================
#
# As variáveis de estado HMM e indicadores de rampa
# permanecem em escala 0/1 no ElasticNet.
#
# As demais variáveis numéricas são padronizadas.
# ============================================================

BINARY_FEATURES = {
    "is_regime_s0",
    "is_regime_s1",
    "is_regime_s2",
    "is_regime_s3",
    "is_regime_s4",
    "state_changed",
    "ramp_defined",
    "is_ramp_up",
    "is_ramp_down",
    "is_ramp_stable",
    "is_ramp_undefined",
}


# ============================================================
# 3. CONSTRUÇÃO DOS PRÉ-PROCESSADORES
# ============================================================

def build_preprocessor(
    feature_columns,
    algorithm
):

    binary_columns = [
        col for col in feature_columns
        if col in BINARY_FEATURES
    ]

    continuous_columns = [
        col for col in feature_columns
        if col not in BINARY_FEATURES
    ]

    if algorithm == "ElasticNet":

        continuous_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "scaler",
                StandardScaler()
            ),
        ])

        binary_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
        ])

    else:

        continuous_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
        ])

        binary_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
        ])

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "continuous",
                continuous_pipeline,
                continuous_columns
            ),
            (
                "binary",
                binary_pipeline,
                binary_columns
            ),
        ],
        remainder="drop",
        verbose_feature_names_out=False
    )

    return preprocessor


# ============================================================
# 4. AJUSTE EXCLUSIVAMENTE NO TREINAMENTO
# ============================================================

preprocessors = {}
processed_data = {}
preprocessing_summary = []

for scenario in SCENARIOS:

    preprocessors[scenario] = {}
    processed_data[scenario] = {}

    X_train = split_data[scenario]["train"]["X"]

    feature_columns = X_train.columns.tolist()

    for algorithm in ALGORITHMS:

        preprocessor = build_preprocessor(
            feature_columns,
            algorithm
        )

        # FIT SOMENTE NO TREINAMENTO
        preprocessor.fit(X_train)

        transformed = {}

        for partition in [
            "train",
            "validation",
            "test"
        ]:

            X_original = split_data[
                scenario
            ][partition]["X"]

            X_processed = preprocessor.transform(
                X_original
            )

            X_processed = np.asarray(
                X_processed,
                dtype=float
            )

            if not np.isfinite(X_processed).all():
                raise ValueError(
                    f"Valores não finitos em "
                    f"{scenario}/{algorithm}/{partition}"
                )

            assert len(X_processed) == len(X_original)

            transformed[partition] = X_processed

        preprocessors[scenario][algorithm] = (
            preprocessor
        )

        processed_data[scenario][algorithm] = (
            transformed
        )

        preprocessing_summary.append({
            "scenario": scenario,
            "algorithm": algorithm,
            "n_features": int(
                transformed["train"].shape[1]
            ),
            "train_rows": int(
                transformed["train"].shape[0]
            ),
            "validation_rows": int(
                transformed["validation"].shape[0]
            ),
            "test_rows": int(
                transformed["test"].shape[0]
            ),
            "train_missing_after": int(
                np.isnan(
                    transformed["train"]
                ).sum()
            ),
            "validation_missing_after": int(
                np.isnan(
                    transformed["validation"]
                ).sum()
            ),
            "test_missing_after": int(
                np.isnan(
                    transformed["test"]
                ).sum()
            ),
        })


# ============================================================
# 5. RESUMO DOS PIPELINES
# ============================================================

preprocessing_summary_df = pd.DataFrame(
    preprocessing_summary
)

_project_print("\nRESUMO DO PRÉ-PROCESSAMENTO")

_project_display(preprocessing_summary_df)


# ============================================================
# 6. VERIFICAÇÃO DOS VALORES IMPUTADOS
# ============================================================
#
# As medianas são extraídas do pré-processador
# ajustado exclusivamente em train.
# ============================================================

imputation_records = []

for scenario in SCENARIOS:

    preprocessor = preprocessors[
        scenario
    ]["ElasticNet"]

    continuous_columns = [
        col
        for col in split_data[
            scenario
        ]["train"]["X"].columns
        if col not in BINARY_FEATURES
    ]

    imputer = (
        preprocessor
        .named_transformers_["continuous"]
        .named_steps["imputer"]
    )

    for feature, median in zip(
        continuous_columns,
        imputer.statistics_
    ):

        imputation_records.append({
            "scenario": scenario,
            "feature": feature,
            "train_median": float(median),
        })

imputation_df = pd.DataFrame(
    imputation_records
)

_project_print("\nMEDIANAS DE IMPUTAÇÃO (TREINAMENTO)")

_project_display(
    imputation_df[
        imputation_df["feature"].str.contains(
            "lag|delta",
            case=False,
            regex=True
        )
    ]
)


# ============================================================
# 7. VERIFICAÇÃO DE ISOLAMENTO TEMPORAL
# ============================================================

for scenario in SCENARIOS:

    train_end = split_data[
        scenario
    ]["train"]["timestamps"].max()

    validation_start = split_data[
        scenario
    ]["validation"]["timestamps"].min()

    validation_end = split_data[
        scenario
    ]["validation"]["timestamps"].max()

    test_start = split_data[
        scenario
    ]["test"]["timestamps"].min()

    assert train_end < validation_start
    assert validation_end < test_start

_project_print("\nIsolamento cronológico: OK")


# ============================================================
# 8. EXPORTAÇÃO DOS PRÉ-PROCESSADORES
# ============================================================

preprocessor_paths = {}

for scenario in SCENARIOS:

    for algorithm in ALGORITHMS:

        filename = (
            f"cell08_preprocessor_"
            f"{scenario.lower()}_"
            f"{algorithm.lower()}.joblib"
        )

        path = DIRS["models"] / filename

        joblib.dump(
            preprocessors[scenario][algorithm],
            path
        )

        preprocessor_paths[
            f"{scenario}_{algorithm}"
        ] = str(path)


# ============================================================
# 9. EXPORTAÇÃO DO RELATÓRIO
# ============================================================

SUMMARY_FILE = (
    DIRS["tables"]
    / "cell08_preprocessing_summary.csv"
)

_project_csv(preprocessing_summary_df, 
    SUMMARY_FILE,
    index=False
)

IMPUTATION_FILE = (
    DIRS["tables"]
    / "cell08_imputation_medians.csv"
)

_project_csv(imputation_df, 
    IMPUTATION_FILE,
    index=False
)

REPORT_FILE = (
    DIRS["reports"]
    / "cell08_preprocessing_report.json"
)

report = {
    "phase": "phase_04_predictive_modeling",
    "cell": "08",
    "generated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "scenarios": SCENARIOS,
    "algorithms": ALGORITHMS,
    "fit_partition": "train",
    "imputation": {
        "continuous": "median",
        "binary": "most_frequent",
    },
    "scaling": {
        "ElasticNet": "continuous_only",
        "RandomForest": "none",
        "XGBoost": "none",
        "HistGradientBoosting": "none",
    },
    "preprocessor_paths": preprocessor_paths,
    "summary": preprocessing_summary,
}

with open(
    REPORT_FILE,
    "w",
    encoding="utf-8"
) as file:

    _project_dump(
        report,
        file,
        indent=4,
        ensure_ascii=False
    )


# ============================================================
# 10. RESULTADO FINAL
# ============================================================

assert len(preprocessing_summary_df) == 12

assert (
    preprocessing_summary_df[
        [
            "train_missing_after",
            "validation_missing_after",
            "test_missing_after",
        ]
    ].to_numpy() == 0
).all()

_project_print("\n" + "=" * 70)
_project_print("FASE 4 — CÉLULA 08")
_project_print("=" * 70)

_project_print("Cenários:", len(SCENARIOS))
_project_print("Algoritmos:", len(ALGORITHMS))
_project_print("Pipelines ajustados:", len(preprocessing_summary_df))

_project_print("\nSTATUS: PRÉ-PROCESSAMENTO CONCLUÍDO")
_project_print("=" * 70)


In [ ]:

# ============================================================
# FASE 4 — CÉLULA 09
# ANÁLISE DESCRITIVA DOS CENÁRIOS EXPERIMENTAIS
# ============================================================
#
# OBJETIVOS:
# 1. Caracterizar geração e emissões de CO2.
# 2. Examinar a relação geração × CO2.
# 3. Descrever os cinco estados HMM.
# 4. Caracterizar rampas e transições operacionais.
# 5. Registrar valores ausentes originais de P2.
# 6. Exportar tabelas e figuras.
#
# IMPORTANTE:
# - Exploração do alvo somente em train/validation.
# - Teste permanece reservado.
# - Nenhum modelo é treinado.
# - Nenhum atributo é selecionado por desempenho.
# ============================================================

import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 1. CARREGAMENTO E VERIFICAÇÃO
# ============================================================

MASTER_FILE = (
    DIRS["processed"]
    / "cell04_master_dataset.parquet"
)

FEATURE_FILE = (
    DIRS["manifests"]
    / "cell06_feature_sets.json"
)

df = pd.read_parquet(MASTER_FILE)

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    utc=True
)

with open(FEATURE_FILE, "r", encoding="utf-8") as f:
    feature_manifest = json.load(f)

feature_sets = feature_manifest["feature_sets"]

eligible = df.loc[
    df["eligible_for_modeling"]
].copy()

development = eligible.loc[
    eligible["partition"].isin(["train", "validation"])
].copy()

assert len(eligible) == 6249
assert len(development) == 4499 + 687

assert set(development["partition"].unique()) == {
    "train", "validation"
}

TARGET = "co2_metric_tons"
GENERATION = "generation_mw"

_project_print("Amostra elegível:", len(eligible))
_project_print("Amostra exploratória (train + validation):",
      len(development))
_project_print("Teste reservado:", int(
    (eligible["partition"] == "test").sum()
))


# ============================================================
# 2. ESTATÍSTICAS DESCRITIVAS POR PARTIÇÃO
# ============================================================

descriptive_rows = []

for partition in ["train", "validation"]:

    subset = development.loc[
        development["partition"] == partition
    ]

    for variable in [GENERATION, TARGET]:

        values = subset[variable].dropna()

        descriptive_rows.append({
            "partition": partition,
            "variable": variable,
            "n": int(len(values)),
            "mean": float(values.mean()),
            "std": float(values.std()),
            "min": float(values.min()),
            "p25": float(values.quantile(0.25)),
            "median": float(values.median()),
            "p75": float(values.quantile(0.75)),
            "max": float(values.max()),
        })

descriptive_df = pd.DataFrame(descriptive_rows)

_project_print("\nESTATÍSTICAS DESCRITIVAS")
_project_display(descriptive_df.round(3))


# ============================================================
# 3. RELAÇÃO ENTRE GERAÇÃO E CO2
# ============================================================

correlation_rows = []

for partition in ["train", "validation"]:

    subset = development.loc[
        development["partition"] == partition
    ]

    correlation_rows.append({
        "partition": partition,
        "n": int(len(subset)),
        "pearson": float(
            subset[[GENERATION, TARGET]]
            .corr(method="pearson")
            .iloc[0, 1]
        ),
        "spearman": float(
            subset[[GENERATION, TARGET]]
            .corr(method="spearman")
            .iloc[0, 1]
        ),
    })

correlation_df = pd.DataFrame(correlation_rows)

_project_print("\nCORRELAÇÃO GERAÇÃO × CO2")
_project_display(correlation_df.round(4))

fig, ax = plt.subplots(figsize=(9, 6))

for partition in ["train", "validation"]:

    subset = development.loc[
        development["partition"] == partition
    ]

    ax.scatter(
        subset[GENERATION],
        subset[TARGET],
        s=10,
        alpha=0.35,
        label=partition
    )

ax.set_xlabel("Geração elétrica (MW)")
ax.set_ylabel("Emissões de CO₂ (t/h)")
ax.set_title("Iatan 2 — Relação entre geração e emissões")
ax.legend()
ax.grid(alpha=0.2)

fig.tight_layout()

fig.savefig(
    DIRS["figures"] / "cell09_generation_vs_co2.png",
    dpi=180,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 4. DISTRIBUIÇÃO DOS ESTADOS HMM
# ============================================================

hmm_summary = (
    development.groupby(
        ["partition", "hmm_state"],
        observed=True
    )
    .agg(
        hours=("timestamp", "size"),
        mean_generation=(GENERATION, "mean"),
        median_generation=(GENERATION, "median"),
        mean_co2=(TARGET, "mean"),
        median_co2=(TARGET, "median"),
    )
    .reset_index()
)

hmm_summary["share_pct"] = (
    100
    * hmm_summary["hours"]
    / hmm_summary.groupby("partition")["hours"]
      .transform("sum")
)

_project_print("\nDISTRIBUIÇÃO DOS ESTADOS HMM")
_project_display(hmm_summary.round(3))

hmm_counts = pd.crosstab(
    development["hmm_state"],
    development["partition"],
    normalize="columns"
).mul(100)

hmm_counts = hmm_counts.reindex(
    columns=["train", "validation"]
)

ax = hmm_counts.plot(
    kind="bar",
    figsize=(9, 5),
    rot=0
)

ax.set_xlabel("Estado HMM")
ax.set_ylabel("Participação (%)")
ax.set_title("Distribuição dos regimes operacionais")
ax.grid(axis="y", alpha=0.2)

plt.tight_layout()

plt.savefig(
    DIRS["figures"] / "cell09_hmm_distribution.png",
    dpi=180,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 5. EMISSÕES POR ESTADO OPERACIONAL
# ============================================================

fig, ax = plt.subplots(figsize=(9, 5))

states = sorted(
    development["hmm_state"].dropna().unique()
)

state_co2 = [
    development.loc[
        development["hmm_state"] == state,
        TARGET
    ].dropna().to_numpy()
    for state in states
]

ax.boxplot(
    state_co2,
    tick_labels=[str(s) for s in states],
    showfliers=False
)

ax.set_xlabel("Estado HMM")
ax.set_ylabel("Emissões de CO₂ (t/h)")
ax.set_title("Emissões por regime operacional")
ax.grid(axis="y", alpha=0.2)

fig.tight_layout()

fig.savefig(
    DIRS["figures"] / "cell09_co2_by_hmm_state.png",
    dpi=180,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 6. CARACTERIZAÇÃO DAS RAMPAS
# ============================================================

ramp_indicators = {
    "up": "is_ramp_up",
    "down": "is_ramp_down",
    "stable": "is_ramp_stable",
    "undefined": "is_ramp_undefined",
}

ramp_rows = []

for partition in ["train", "validation"]:

    subset = development.loc[
        development["partition"] == partition
    ]

    for ramp_name, indicator in ramp_indicators.items():

        selected = subset.loc[
            subset[indicator].astype(bool)
        ]

        ramp_rows.append({
            "partition": partition,
            "ramp": ramp_name,
            "hours": int(len(selected)),
            "share_pct": (
                100 * len(selected) / len(subset)
            ),
            "mean_generation": (
                float(selected[GENERATION].mean())
                if len(selected) else np.nan
            ),
            "mean_co2": (
                float(selected[TARGET].mean())
                if len(selected) else np.nan
            ),
        })

ramp_summary_df = pd.DataFrame(ramp_rows)

_project_print("\nCARACTERIZAÇÃO DAS RAMPAS")
_project_display(ramp_summary_df.round(3))


# ============================================================
# 7. TRANSIÇÕES OPERACIONAIS
# ============================================================

transition_rows = []

for partition in ["train", "validation"]:

    subset = development.loc[
        development["partition"] == partition
    ]

    changed = subset["state_changed"].fillna(False).astype(bool)

    transition_rows.append({
        "partition": partition,
        "hours": int(len(subset)),
        "state_change_hours": int(changed.sum()),
        "state_change_pct": float(
            100 * changed.mean()
        ),
        "mean_co2_transition": float(
            subset.loc[changed, TARGET].mean()
        ) if changed.any() else np.nan,
        "mean_co2_no_transition": float(
            subset.loc[~changed, TARGET].mean()
        ) if (~changed).any() else np.nan,
    })

transition_df = pd.DataFrame(transition_rows)

_project_print("\nTRANSIÇÕES ENTRE ESTADOS")
_project_display(transition_df.round(3))


# ============================================================
# 8. VALORES AUSENTES ORIGINAIS EM P2
# ============================================================
#
# Examina os dados antes da imputação.
# O conjunto de teste não é usado nesta análise.
# ============================================================

p2_features = feature_sets["P2"]

missing_rows = []

for partition in ["train", "validation"]:

    subset = development.loc[
        development["partition"] == partition
    ]

    for feature in p2_features:

        count = int(subset[feature].isna().sum())

        if count > 0:
            missing_rows.append({
                "partition": partition,
                "feature": feature,
                "missing_count": count,
                "missing_pct": float(
                    100 * count / len(subset)
                ),
            })

missing_df = pd.DataFrame(missing_rows)

_project_print("\nVALORES AUSENTES ORIGINAIS — P2")

if len(missing_df):
    _project_display(missing_df.round(3))
else:
    _project_print("Nenhum valor ausente.")


# ============================================================
# 9. RESUMO DOS CENÁRIOS EXPERIMENTAIS
# ============================================================

scenario_rows = []

for scenario, features in feature_sets.items():

    scenario_rows.append({
        "scenario": scenario,
        "n_features": len(features),
        "train_rows": 4499,
        "validation_rows": 687,
        "test_rows_reserved": 1063,
    })

scenario_df = pd.DataFrame(scenario_rows)

_project_print("\nCENÁRIOS EXPERIMENTAIS")
_project_display(scenario_df)


# ============================================================
# 10. EXPORTAÇÃO DAS TABELAS
# ============================================================

tables = {
    "descriptive_statistics": descriptive_df,
    "generation_co2_correlation": correlation_df,
    "hmm_summary": hmm_summary,
    "ramp_summary": ramp_summary_df,
    "transition_summary": transition_df,
    "missing_p2": missing_df,
    "scenario_summary": scenario_df,
}

table_paths = {}

for name, table in tables.items():

    path = (
        DIRS["tables"]
        / f"cell09_{name}.csv"
    )

    _project_csv(table, path, index=False)

    table_paths[name] = str(path)


# ============================================================
# 11. RELATÓRIO DE ANÁLISE DESCRITIVA
# ============================================================

REPORT_FILE = (
    DIRS["reports"]
    / "cell09_descriptive_analysis.json"
)

report = {
    "phase": "phase_04_predictive_modeling",
    "cell": "09",
    "generated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "plant": "Iatan",
    "facility_id": 6065,
    "unit_id": "2",
    "year": 2025,
    "exploratory_partitions": [
        "train",
        "validation",
    ],
    "test_target_inspected": False,
    "development_rows": int(len(development)),
    "test_rows_reserved": 1063,
    "tables": table_paths,
    "figures": [
        "cell09_generation_vs_co2.png",
        "cell09_hmm_distribution.png",
        "cell09_co2_by_hmm_state.png",
    ],
}

with open(
    REPORT_FILE,
    "w",
    encoding="utf-8"
) as f:

    _project_dump(
        report,
        f,
        indent=4,
        ensure_ascii=False
    )


# ============================================================
# 12. RESULTADO FINAL
# ============================================================

assert len(development) == 5186
assert len(scenario_df) == 3
assert REPORT_FILE.exists()

_project_print("\n" + "=" * 70)
_project_print("FASE 4 — CÉLULA 09")
_project_print("=" * 70)

_project_print("Horas analisadas:", len(development))
_project_print("Horas de teste preservadas:", 1063)
_project_print("Tabelas exportadas:", len(table_paths))
_project_print("Figuras exportadas:", 3)
_project_print("Relatório:", REPORT_FILE)

_project_print("\nSTATUS: ANÁLISE DESCRITIVA CONCLUÍDA")
_project_print("=" * 70)


### Controles, métricas e interpretação

[Hyndman e Koehler (2006)](https://doi.org/10.1016/j.ijforecast.2006.03.001) discutem escalas e propriedades de métricas. RMSE e MAE aqui têm unidade t CO₂/h. O viés usa predito menos observado. A média do treino e a regressão linear são controles.


In [ ]:

# ============================================================
# FASE 4 — CÉLULA 10
# BASELINES CONVENCIONAIS E REFERÊNCIAS DE DESEMPENHO
# ============================================================
#
# OBJETIVOS:
# 1. Construir baseline ingênuo pela média do treinamento.
# 2. Ajustar regressão linear simples: CO2(t) ~ G(t).
# 3. Avaliar ambos exclusivamente na validação.
# 4. Registrar RMSE, MAE, R2 e viés.
# 5. Salvar modelos, previsões, métricas e figura.
#
# NÃO REALIZA:
# - avaliação no teste;
# - busca de hiperparâmetros;
# - seleção entre P1 e P2.
#
# NOTA:
# O baseline linear desta célula é uma referência
# explícita da Fase 4. Não presumimos que sua
# especificação seja idêntica à da Fase 3.
# ============================================================

import json
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)


# ============================================================
# 1. RECUPERAÇÃO DOS DADOS B0
# ============================================================

if "split_data" not in globals():
    raise RuntimeError(
        "Execute a Célula 07 antes da Célula 10."
    )

X_train = split_data["B0"]["train"]["X"].copy()
y_train = split_data["B0"]["train"]["y"].to_numpy()

X_val = split_data["B0"]["validation"]["X"].copy()
y_val = split_data["B0"]["validation"]["y"].to_numpy()

timestamps_val = split_data["B0"]["validation"][
    "timestamps"
].reset_index(drop=True)

assert X_train.columns.tolist() == ["generation_mw"]
assert X_val.columns.tolist() == ["generation_mw"]

assert len(X_train) == 4499
assert len(X_val) == 687

assert np.isfinite(X_train.to_numpy(dtype=float)).all()
assert np.isfinite(X_val.to_numpy(dtype=float)).all()
assert np.isfinite(y_train).all()
assert np.isfinite(y_val).all()


# ============================================================
# 2. DEFINIÇÃO DOS MODELOS
# ============================================================

baseline_models = {
    "DummyMean": DummyRegressor(
        strategy="mean"
    ),
    "LinearRegression_B0": LinearRegression(),
}


# ============================================================
# 3. FUNÇÃO DE AVALIAÇÃO
# ============================================================

def evaluate_baseline(y_true, y_pred):

    errors = np.asarray(y_pred) - np.asarray(y_true)

    return {
        "rmse": float(
            np.sqrt(mean_squared_error(y_true, y_pred))
        ),
        "mae": float(
            mean_absolute_error(y_true, y_pred)
        ),
        "r2": float(
            r2_score(y_true, y_pred)
        ),
        "bias": float(errors.mean()),
    }


# ============================================================
# 4. TREINAMENTO E VALIDAÇÃO
# ============================================================

baseline_results = []
baseline_predictions = pd.DataFrame({
    "timestamp": timestamps_val,
    "partition": "validation",
    "y_true": y_val,
    "generation_mw": X_val["generation_mw"].to_numpy(),
})

for model_name, model in baseline_models.items():

    # FIT exclusivamente em treinamento
    model.fit(X_train, y_train)

    # PREDICT exclusivamente em validação
    predictions = model.predict(X_val)

    metrics = evaluate_baseline(
        y_val,
        predictions
    )

    baseline_results.append({
        "model": model_name,
        "scenario": "B0",
        "partition": "validation",
        "n": len(y_val),
        **metrics,
    })

    baseline_predictions[
        f"prediction_{model_name}"
    ] = predictions

    model_path = (
        DIRS["models"]
        / f"cell10_{model_name}.joblib"
    )

    joblib.dump(model, model_path)


baseline_metrics_df = pd.DataFrame(
    baseline_results
).sort_values("rmse").reset_index(drop=True)

_project_print("\nDESEMPENHO DOS BASELINES — VALIDAÇÃO")
_project_display(baseline_metrics_df.round(4))


# ============================================================
# 5. COEFICIENTES DA REGRESSÃO LINEAR
# ============================================================

linear_model = baseline_models["LinearRegression_B0"]

linear_intercept = float(linear_model.intercept_)
linear_slope = float(linear_model.coef_[0])

_project_print("\nEQUAÇÃO ESTIMADA NO TREINAMENTO")
_project_print(
    f"CO2_estimado(t) = {linear_intercept:.4f} "
    f"+ {linear_slope:.6f} × G(t)"
)

_project_print("Unidade de CO2: toneladas métricas por hora")
_project_print("Unidade de G(t): MW")


# ============================================================
# 6. COMPARAÇÃO VISUAL NA VALIDAÇÃO
# ============================================================

fig, ax = plt.subplots(figsize=(11, 5))

ax.plot(
    baseline_predictions["timestamp"],
    baseline_predictions["y_true"],
    label="CO2 observado",
    linewidth=1.4,
    alpha=0.9,
)

ax.plot(
    baseline_predictions["timestamp"],
    baseline_predictions[
        "prediction_LinearRegression_B0"
    ],
    label="Baseline linear B0",
    linewidth=1.2,
    alpha=0.8,
)

ax.set_title(
    "Iatan 2 — Baseline convencional na validação"
)
ax.set_xlabel("Data")
ax.set_ylabel("Emissões de CO₂ (t/h)")
ax.legend()
ax.grid(alpha=0.2)

fig.autofmt_xdate()
fig.tight_layout()

FIGURE_FILE = (
    DIRS["figures"]
    / "cell10_baseline_validation.png"
)

fig.savefig(
    FIGURE_FILE,
    dpi=180,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 7. RESÍDUOS DA REGRESSÃO LINEAR
# ============================================================

baseline_predictions["residual_linear"] = (
    baseline_predictions["y_true"]
    - baseline_predictions[
        "prediction_LinearRegression_B0"
    ]
)

fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(
    baseline_predictions["generation_mw"],
    baseline_predictions["residual_linear"],
    s=12,
    alpha=0.5,
)

ax.axhline(
    0,
    linestyle="--",
    linewidth=1,
)

ax.set_title(
    "Resíduos do baseline linear por geração"
)
ax.set_xlabel("Geração elétrica (MW)")
ax.set_ylabel("Resíduo: observado − previsto (t/h)")
ax.grid(alpha=0.2)

fig.tight_layout()

RESIDUAL_FIGURE_FILE = (
    DIRS["figures"]
    / "cell10_baseline_residuals.png"
)

fig.savefig(
    RESIDUAL_FIGURE_FILE,
    dpi=180,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 8. EXPORTAÇÃO DE MÉTRICAS E PREVISÕES
# ============================================================

METRICS_FILE = (
    DIRS["tables"]
    / "cell10_baseline_validation_metrics.csv"
)

PREDICTIONS_FILE = (
    DIRS["processed"]
    / "cell10_baseline_validation_predictions.parquet"
)

_project_csv(baseline_metrics_df, 
    METRICS_FILE,
    index=False
)

baseline_predictions.to_parquet(
    PREDICTIONS_FILE,
    index=False
)


# ============================================================
# 9. RELATÓRIO METODOLÓGICO
# ============================================================

REPORT_FILE = (
    DIRS["reports"]
    / "cell10_baseline_report.json"
)

report = {
    "phase": "phase_04_predictive_modeling",
    "cell": "10",
    "generated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "scenario": "B0",
    "feature": "generation_mw",
    "target": "co2_metric_tons",
    "fit_partition": "train",
    "evaluation_partition": "validation",
    "test_evaluated": False,
    "models": list(baseline_models.keys()),
    "linear_intercept": linear_intercept,
    "linear_slope": linear_slope,
    "validation_metrics": baseline_results,
    "historical_phase3_test_rmse_reference": 31.096,
    "historical_reference_note": (
        "Historical Phase 3 test RMSE is not directly "
        "comparable with Phase 4 validation RMSE."
    ),
    "metrics_file": str(METRICS_FILE),
    "predictions_file": str(PREDICTIONS_FILE),
    "figures": [
        str(FIGURE_FILE),
        str(RESIDUAL_FIGURE_FILE),
    ],
}

with open(
    REPORT_FILE,
    "w",
    encoding="utf-8"
) as f:
    _project_dump(
        report,
        f,
        indent=4,
        ensure_ascii=False
    )


# ============================================================
# 10. RESULTADO FINAL
# ============================================================

assert len(baseline_metrics_df) == 2
assert len(baseline_predictions) == 687

assert np.isfinite(
    baseline_metrics_df[
        ["rmse", "mae", "r2", "bias"]
    ].to_numpy()
).all()

_project_print("\n" + "=" * 70)
_project_print("FASE 4 — CÉLULA 10")
_project_print("=" * 70)

_project_print("Modelos treinados:", len(baseline_models))
_project_print("Observações de treinamento:", len(y_train))
_project_print("Observações de validação:", len(y_val))
_project_print("Observações de teste avaliadas: 0")

_project_print("\nMétricas:", METRICS_FILE)
_project_print("Previsões:", PREDICTIONS_FILE)
_project_print("Relatório:", REPORT_FILE)

_project_print("\nSTATUS: BASELINES VALIDADOS")
_project_print("=" * 70)


### Plano de experimentação e famílias de modelos

Aula 08, pp. 23–32. [Zou e Hastie (2005)](https://doi.org/10.1111/j.1467-9868.2005.00503.x) para Elastic Net, [Breiman (2001)](https://doi.org/10.1023/A:1010933404324) para Random Forest, [Chen e Guestrin (2016)](https://arxiv.org/abs/1603.02754) para XGBoost e [Friedman (2001)](https://doi.org/10.1214/aos/1013203451) para gradient boosting. O estimador de histograma segue a [documentação do scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html). Os grids pertencem ao projeto.


In [ ]:
# ============================================================
# FASE 4 — CÉLULA 11
# MATRIZ EXPERIMENTAL E CONFIGURAÇÃO DO MLFLOW
# ============================================================
#
# OBJETIVOS:
# 1. Definir 12 combinações cenário × algoritmo.
# 2. Congelar espaços de hiperparâmetros.
# 3. Definir métricas e critério de seleção.
# 4. Confirmar a configuração do MLflow.
# 5. Exportar matriz e protocolo experimental.
#
# NÃO REALIZA:
# - treinamento;
# - otimização;
# - previsão;
# - avaliação do teste.
#
# SELEÇÃO:
# RMSE mínimo na validação.
#
# REPRODUTIBILIDADE:
# Seed fixa e protocolo salvo em JSON.
# ============================================================

import json
import itertools
from datetime import datetime, timezone

import pandas as pd
import mlflow


# ============================================================
# 1. CONFIGURAÇÕES GERAIS
# ============================================================

RANDOM_STATE = 42

SCENARIOS = ["B0", "P1", "P2"]

ALGORITHMS = [
    "ElasticNet",
    "RandomForest",
    "XGBoost",
    "HistGradientBoosting",
]

PRIMARY_METRIC = "rmse"

SECONDARY_METRICS = [
    "mae",
    "r2",
    "bias",
]

SELECTION_PARTITION = "validation"
FINAL_EVALUATION_PARTITION = "test"


# ============================================================
# 2. ESPAÇOS DE HIPERPARÂMETROS
# ============================================================
#
# Grades compactas para manter o experimento
# computacionalmente viável.
#
# Os mesmos hiperparâmetros candidatos serão
# utilizados em B0, P1 e P2.
#
# O número de árvores e as demais configurações
# foram definidos antes de executar os experimentos.
# ============================================================

PARAM_GRIDS = {

    "ElasticNet": {
        "alpha": [0.01, 0.1, 1.0, 10.0],
        "l1_ratio": [0.0, 0.5, 1.0],
    },

    "RandomForest": {
        "n_estimators": [200],
        "max_depth": [8, None],
        "min_samples_leaf": [1, 5],
    },

    "XGBoost": {
        "n_estimators": [200, 400],
        "max_depth": [3, 5],
        "learning_rate": [0.05],
        "subsample": [0.9],
        "colsample_bytree": [0.9],
    },

    "HistGradientBoosting": {
        "max_iter": [200, 400],
        "max_leaf_nodes": [15, 31],
        "learning_rate": [0.05],
        "l2_regularization": [0.0],
    },
}


# ============================================================
# 3. CONTAGEM DE CONFIGURAÇÕES
# ============================================================

def count_combinations(grid):

    counts = [
        len(values)
        for values in grid.values()
    ]

    result = 1

    for count in counts:
        result *= count

    return result


algorithm_summary = []

for algorithm in ALGORITHMS:

    n_configs = count_combinations(
        PARAM_GRIDS[algorithm]
    )

    algorithm_summary.append({
        "algorithm": algorithm,
        "configurations_per_scenario": n_configs,
        "n_scenarios": len(SCENARIOS),
        "total_candidate_fits": (
            n_configs * len(SCENARIOS)
        ),
    })

algorithm_summary_df = pd.DataFrame(
    algorithm_summary
)

_project_print("\nCONFIGURAÇÕES POR ALGORITMO")
_project_display(algorithm_summary_df)


# ============================================================
# 4. MATRIZ DAS 12 COMBINAÇÕES PRINCIPAIS
# ============================================================

experimental_rows = []

for scenario, algorithm in itertools.product(
    SCENARIOS,
    ALGORITHMS
):

    n_features = split_data[
        scenario
    ]["train"]["X"].shape[1]

    experimental_rows.append({
        "experiment_id": (
            f"{scenario}_{algorithm}"
        ),
        "scenario": scenario,
        "algorithm": algorithm,
        "n_features": int(n_features),
        "train_rows": 4499,
        "validation_rows": 687,
        "test_rows_reserved": 1063,
        "candidate_configurations": (
            count_combinations(
                PARAM_GRIDS[algorithm]
            )
        ),
        "selection_metric": PRIMARY_METRIC,
        "selection_partition": (
            SELECTION_PARTITION
        ),
        "status": "planned",
    })

experimental_matrix_df = pd.DataFrame(
    experimental_rows
)

assert len(experimental_matrix_df) == 12

_project_print("\nMATRIZ EXPERIMENTAL")
_project_display(experimental_matrix_df)


# ============================================================
# 5. VERIFICAÇÃO DAS PARTIÇÕES
# ============================================================

for scenario in SCENARIOS:

    for partition, expected in {
        "train": 4499,
        "validation": 687,
        "test": 1063,
    }.items():

        actual = len(
            split_data[scenario][partition]["X"]
        )

        assert actual == expected, (
            scenario,
            partition,
            actual,
        )

_project_print("\nPartições experimentais: OK")


# ============================================================
# 6. VERIFICAÇÃO DO MLFLOW
# ============================================================
#
# Reutilizamos a configuração persistente da
# Célula 01. Não criamos outro experimento.
# ============================================================

experiment = mlflow.get_experiment_by_name(
    EXPERIMENT_NAME
)

if experiment is None:
    raise RuntimeError(
        "Experimento MLflow não encontrado. "
        "Execute a configuração da Célula 01."
    )

mlflow.set_experiment(EXPERIMENT_NAME)

_project_print("\nMLFLOW")
_project_print("Experimento:", EXPERIMENT_NAME)
_project_print("Experiment ID:", experiment.experiment_id)
_project_print("Tracking URI:", mlflow.get_tracking_uri())


# ============================================================
# 7. METADADOS DO EXPERIMENTO
# ============================================================

mlflow.set_experiment_tags({
    "project": "iatan-process-data-science",
    "phase": "04",
    "facility_id": "6065",
    "unit_id": "2",
    "year": "2025",
    "primary_metric": PRIMARY_METRIC,
    "selection_partition": SELECTION_PARTITION,
    "evaluation_protocol": (
        "contemporaneous_hourly_estimation"
    ),
})


# ============================================================
# 8. EXPORTAÇÃO DA MATRIZ
# ============================================================

MATRIX_FILE = (
    DIRS["tables"]
    / "cell11_experimental_matrix.csv"
)

ALGORITHM_FILE = (
    DIRS["tables"]
    / "cell11_algorithm_search_summary.csv"
)

_project_csv(experimental_matrix_df, 
    MATRIX_FILE,
    index=False
)

_project_csv(algorithm_summary_df, 
    ALGORITHM_FILE,
    index=False
)


# ============================================================
# 9. EXPORTAÇÃO DO PROTOCOLO
# ============================================================

PROTOCOL_FILE = (
    DIRS["manifests"]
    / "cell11_experimental_protocol.json"
)

protocol = {
    "phase": "phase_04_predictive_modeling",
    "cell": "11",
    "generated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "facility_id": 6065,
    "unit_id": "2",
    "year": 2025,
    "random_state": RANDOM_STATE,
    "scenarios": SCENARIOS,
    "algorithms": ALGORITHMS,
    "parameter_grids": PARAM_GRIDS,
    "primary_metric": PRIMARY_METRIC,
    "secondary_metrics": SECONDARY_METRICS,
    "selection_partition": SELECTION_PARTITION,
    "final_evaluation_partition": (
        FINAL_EVALUATION_PARTITION
    ),
    "n_primary_combinations": 12,
    "total_candidate_fits": int(
        algorithm_summary_df[
            "total_candidate_fits"
        ].sum()
    ),
    "selection_rule": (
        "Minimum validation RMSE within each "
        "scenario and algorithm."
    ),
    "test_policy": (
        "No test predictions or metrics "
        "during model selection."
    ),
    "mlflow_experiment_name": EXPERIMENT_NAME,
    "mlflow_experiment_id": (
        experiment.experiment_id
    ),
}

with open(
    PROTOCOL_FILE,
    "w",
    encoding="utf-8"
) as f:

    _project_dump(
        protocol,
        f,
        indent=4,
        ensure_ascii=False
    )


# ============================================================
# 10. RESULTADO FINAL
# ============================================================

TOTAL_FITS = int(
    algorithm_summary_df[
        "total_candidate_fits"
    ].sum()
)

assert TOTAL_FITS == 72

_project_print("\n" + "=" * 70)
_project_print("FASE 4 — CÉLULA 11")
_project_print("=" * 70)

_project_print("Cenários:", len(SCENARIOS))
_project_print("Algoritmos:", len(ALGORITHMS))
_project_print("Combinações principais:", len(experimental_matrix_df))
_project_print("Ajustes candidatos previstos:", TOTAL_FITS)

_project_print("\nMétrica principal:", PRIMARY_METRIC)
_project_print("Seleção:", SELECTION_PARTITION)
_project_print("Teste utilizado: NÃO")

_project_print("\nMatriz:", MATRIX_FILE)
_project_print("Protocolo:", PROTOCOL_FILE)

_project_print("\nSTATUS: MATRIZ EXPERIMENTAL CONFIGURADA")
_project_print("=" * 70)

In [ ]:

# ============================================================
# FASE 4 — CÉLULA 12
# TREINAMENTO E SELEÇÃO DOS MODELOS B0
# ============================================================
#
# CENÁRIO:
# B0 — Geração elétrica contemporânea G(t)
#
# ALGORITMOS:
# - ElasticNet
# - RandomForest
# - XGBoost
# - HistGradientBoosting
#
# PROTOCOLO:
# - Treinamento: 4.499 observações
# - Validação: 687 observações
# - Teste: não utilizado
# - Seleção: menor RMSE de validação
#
# TOTAL ESPERADO: 24 configurações candidatas.
#
# Os pré-processadores foram ajustados somente em
# treinamento na Célula 08.
# ============================================================

import json
import time
import warnings
import itertools
from datetime import datetime, timezone

import joblib
import mlflow
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import ElasticNet
from sklearn.ensemble import (
    RandomForestRegressor,
    HistGradientBoostingRegressor,
)
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)
from sklearn.exceptions import ConvergenceWarning

from xgboost import XGBRegressor


# ============================================================
# 1. VERIFICAÇÃO DO PROTOCOLO EXPERIMENTAL
# ============================================================

SCENARIO = "B0"

PROTOCOL_FILE = (
    DIRS["manifests"]
    / "cell11_experimental_protocol.json"
)

if not PROTOCOL_FILE.exists():
    raise FileNotFoundError(
        "Protocolo da Célula 11 não encontrado."
    )

with open(PROTOCOL_FILE, "r", encoding="utf-8") as f:
    protocol = json.load(f)

PARAM_GRIDS_B0 = protocol["parameter_grids"]
RANDOM_STATE_B0 = int(protocol["random_state"])

ALGORITHMS_B0 = protocol["algorithms"]

assert protocol["primary_metric"] == "rmse"
assert protocol["selection_partition"] == "validation"

assert SCENARIO in protocol["scenarios"]

assert len(ALGORITHMS_B0) == 4

if mlflow.active_run() is not None:
    raise RuntimeError(
        "Existe uma execução MLflow ativa. "
        "Finalize-a antes de iniciar a Célula 12."
    )

mlflow.set_experiment(
    protocol["mlflow_experiment_name"]
)


# ============================================================
# 2. RECUPERAÇÃO DOS DADOS PRÉ-PROCESSADOS
# ============================================================

if "processed_data" not in globals():
    raise RuntimeError(
        "Execute a Célula 08 antes da Célula 12."
    )

y_train = split_data[SCENARIO]["train"][
    "y"
].to_numpy(dtype=float)

y_val = split_data[SCENARIO]["validation"][
    "y"
].to_numpy(dtype=float)

timestamps_val = split_data[SCENARIO][
    "validation"
]["timestamps"].reset_index(drop=True)

assert len(y_train) == 4499
assert len(y_val) == 687

assert np.isfinite(y_train).all()
assert np.isfinite(y_val).all()

assert split_data[SCENARIO]["train"][
    "X"
].columns.tolist() == ["generation_mw"]


# ============================================================
# 3. EXPANSÃO DAS GRADES DE HIPERPARÂMETROS
# ============================================================

def expand_parameter_grid(grid):

    keys = list(grid.keys())
    values = [grid[key] for key in keys]

    for combination in itertools.product(*values):
        yield dict(zip(keys, combination))


# ============================================================
# 4. CONSTRUÇÃO DOS ALGORITMOS
# ============================================================

def build_b0_model(algorithm, params, seed):

    if algorithm == "ElasticNet":

        # l1_ratio=0 corresponde à penalização L2.
        # Usamos ElasticNet para manter a grade
        # previamente registrada na Célula 11.
        return ElasticNet(
            **params,
            fit_intercept=True,
            max_iter=30000,
            tol=1e-5,
            selection="cyclic",
            random_state=seed,
        )

    if algorithm == "RandomForest":

        return RandomForestRegressor(
            **params,
            random_state=seed,
            n_jobs=-1,
        )

    if algorithm == "XGBoost":

        return XGBRegressor(
            **params,
            objective="reg:squarederror",
            random_state=seed,
            n_jobs=2,
            tree_method="hist",
            verbosity=0,
        )

    if algorithm == "HistGradientBoosting":

        return HistGradientBoostingRegressor(
            **params,
            random_state=seed,
            early_stopping=False,
        )

    raise ValueError(
        f"Algoritmo desconhecido: {algorithm}"
    )


# ============================================================
# 5. FUNÇÃO DE MÉTRICAS
# ============================================================

def calculate_b0_metrics(y_true, y_pred):

    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    if not np.isfinite(y_pred).all():
        raise ValueError(
            "Previsões não finitas."
        )

    return {
        "rmse": float(
            np.sqrt(
                mean_squared_error(y_true, y_pred)
            )
        ),
        "mae": float(
            mean_absolute_error(y_true, y_pred)
        ),
        "r2": float(
            r2_score(y_true, y_pred)
        ),
        "bias": float(
            np.mean(y_pred - y_true)
        ),
    }


# ============================================================
# 6. TREINAMENTO DAS 24 CONFIGURAÇÕES
# ============================================================
#
# Cada candidato recebe uma execução própria no
# MLflow, permitindo rastrear todos os resultados.
#
# Não acessamos X_test nem y_test.
# ============================================================

candidate_records = []
best_models_b0 = {}
best_predictions_b0 = {}

for algorithm in ALGORITHMS_B0:

    X_train = processed_data[
        SCENARIO
    ][algorithm]["train"]

    X_val = processed_data[
        SCENARIO
    ][algorithm]["validation"]

    assert X_train.shape == (4499, 1)
    assert X_val.shape == (687, 1)

    candidates = list(
        expand_parameter_grid(
            PARAM_GRIDS_B0[algorithm]
        )
    )

    _project_print("\n" + "-" * 65)
    _project_print(f"B0 — {algorithm}")
    _project_print(f"Configurações: {len(candidates)}")
    _project_print("-" * 65)

    algorithm_results = []
    fitted_candidates = {}

    for candidate_index, params in enumerate(
        candidates,
        start=1
    ):

        run_name = (
            f"B0_{algorithm}_"
            f"candidate_{candidate_index:02d}"
        )

        model = build_b0_model(
            algorithm,
            params,
            RANDOM_STATE_B0
        )

        with mlflow.start_run(
            run_name=run_name
        ) as run:

            mlflow.set_tags({
                "phase": "04",
                "cell": "12",
                "scenario": SCENARIO,
                "algorithm": algorithm,
                "partition": "validation",
                "run_type": "candidate",
            })

            mlflow.log_params({
                "scenario": SCENARIO,
                "algorithm": algorithm,
                "candidate_index": candidate_index,
                "random_state": RANDOM_STATE_B0,
                "n_features": 1,
                "train_rows": 4499,
                "validation_rows": 687,
                **params,
            })

            start_time = time.perf_counter()

            # Ajuste exclusivamente no treinamento.
            with warnings.catch_warnings(
                record=True
            ) as captured_warnings:

                warnings.simplefilter(
                    "always",
                    ConvergenceWarning
                )

                model.fit(
                    X_train,
                    y_train
                )

            fit_seconds = (
                time.perf_counter() - start_time
            )

            convergence_warnings = [
                str(w.message)
                for w in captured_warnings
                if issubclass(
                    w.category,
                    ConvergenceWarning
                )
            ]

            # Avaliação exclusivamente na validação.
            predictions = model.predict(X_val)

            metrics = calculate_b0_metrics(
                y_val,
                predictions
            )

            mlflow.log_metrics({
                f"validation_{key}": value
                for key, value in metrics.items()
            })

            mlflow.log_metric(
                "fit_seconds",
                fit_seconds
            )

            mlflow.log_metric(
                "convergence_warning_count",
                len(convergence_warnings)
            )

            record = {
                "scenario": SCENARIO,
                "algorithm": algorithm,
                "candidate_index": candidate_index,
                "run_id": run.info.run_id,
                "params_json": _project_dumps(
                    params,
                    sort_keys=True
                ),
                "fit_seconds": fit_seconds,
                "convergence_warning_count": (
                    len(convergence_warnings)
                ),
                **metrics,
            }

            candidate_records.append(record)
            algorithm_results.append(record)

            fitted_candidates[
                candidate_index
            ] = {
                "model": model,
                "predictions": predictions,
                "params": params,
            }

        _project_print(
            f"[{candidate_index:02d}/"
            f"{len(candidates):02d}] "
            f"RMSE={metrics['rmse']:.4f} | "
            f"MAE={metrics['mae']:.4f} | "
            f"R²={metrics['r2']:.4f}"
        )

    # --------------------------------------------------------
    # Seleção da melhor configuração do algoritmo
    # --------------------------------------------------------

    algorithm_df = pd.DataFrame(
        algorithm_results
    )

    algorithm_df = algorithm_df.sort_values(
        ["rmse", "mae", "candidate_index"]
    )

    best_record = algorithm_df.iloc[0]

    best_index = int(
        best_record["candidate_index"]
    )

    best_entry = fitted_candidates[
        best_index
    ]

    best_models_b0[algorithm] = (
        best_entry["model"]
    )

    best_predictions_b0[algorithm] = (
        best_entry["predictions"]
    )

    # Salvar apenas o melhor modelo do algoritmo.
    MODEL_FILE = (
        DIRS["models"]
        / f"cell12_b0_best_{algorithm.lower()}.joblib"
    )

    joblib.dump(
        best_entry["model"],
        MODEL_FILE
    )

    _project_print(
        f"\nMelhor {algorithm}: "
        f"candidato {best_index} | "
        f"RMSE={best_record['rmse']:.4f}"
    )


# ============================================================
# 7. TABELA COMPLETA DE CANDIDATOS
# ============================================================

b0_candidates_df = pd.DataFrame(
    candidate_records
)

EXPECTED_FITS = sum(
    len(
        list(
            expand_parameter_grid(
                PARAM_GRIDS_B0[algorithm]
            )
        )
    )
    for algorithm in ALGORITHMS_B0
)

assert EXPECTED_FITS == 24
assert len(b0_candidates_df) == 24

_project_print("\nTODOS OS CANDIDATOS B0")

_project_display(
    b0_candidates_df[
        [
            "algorithm",
            "candidate_index",
            "rmse",
            "mae",
            "r2",
            "bias",
            "fit_seconds",
            "convergence_warning_count",
        ]
    ].round(4)
)


# ============================================================
# 8. SELEÇÃO DO MELHOR MODELO DE CADA ALGORITMO
# ============================================================

b0_best_df = (
    b0_candidates_df
    .sort_values(
        ["rmse", "mae", "candidate_index"]
    )
    .groupby("algorithm", as_index=False)
    .first()
    .sort_values("rmse")
    .reset_index(drop=True)
)

_project_print("\nMELHOR CONFIGURAÇÃO POR ALGORITMO")

_project_display(
    b0_best_df[
        [
            "algorithm",
            "candidate_index",
            "rmse",
            "mae",
            "r2",
            "bias",
            "params_json",
        ]
    ].round(4)
)


# ============================================================
# 9. COMPARAÇÃO COM BASELINE LINEAR DA CÉLULA 10
# ============================================================

BASELINE_FILE = (
    DIRS["tables"]
    / "cell10_baseline_validation_metrics.csv"
)

baseline_df = pd.read_csv(
    BASELINE_FILE
)

linear_baseline_rmse = float(
    baseline_df.loc[
        baseline_df["model"]
        == "LinearRegression_B0",
        "rmse"
    ].iloc[0]
)

b0_best_df["rmse_improvement_vs_linear_pct"] = (
    100
    * (
        linear_baseline_rmse
        - b0_best_df["rmse"]
    )
    / linear_baseline_rmse
)

_project_print("\nCOMPARAÇÃO COM REGRESSÃO LINEAR B0")
_project_print(
    f"RMSE da regressão linear: "
    f"{linear_baseline_rmse:.4f} t CO2/h"
)

_project_display(
    b0_best_df[
        [
            "algorithm",
            "rmse",
            "rmse_improvement_vs_linear_pct",
        ]
    ].round(3)
)


# ============================================================
# 10. FIGURA — RMSE POR CANDIDATO
# ============================================================

fig, ax = plt.subplots(
    figsize=(11, 5)
)

for algorithm in ALGORITHMS_B0:

    subset = b0_candidates_df.loc[
        b0_candidates_df["algorithm"] == algorithm
    ]

    ax.plot(
        subset["candidate_index"],
        subset["rmse"],
        marker="o",
        linewidth=1.4,
        label=algorithm
    )

ax.axhline(
    linear_baseline_rmse,
    linestyle="--",
    linewidth=1.2,
    label="Regressão linear B0"
)

ax.set_xlabel(
    "Índice da configuração dentro do algoritmo"
)

ax.set_ylabel(
    "RMSE de validação (t CO₂/h)"
)

ax.set_title(
    "Iatan 2 — Busca de hiperparâmetros B0"
)

ax.grid(alpha=0.2)
ax.legend()

fig.tight_layout()

FIG_CANDIDATES = (
    DIRS["figures"]
    / "cell12_b0_candidate_rmse.png"
)

fig.savefig(
    FIG_CANDIDATES,
    dpi=180,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 11. FIGURA — MELHORES ALGORITMOS B0
# ============================================================

fig, ax = plt.subplots(
    figsize=(9, 5)
)

ax.bar(
    b0_best_df["algorithm"],
    b0_best_df["rmse"]
)

ax.axhline(
    linear_baseline_rmse,
    linestyle="--",
    linewidth=1.2,
    label="Regressão linear B0"
)

ax.set_ylabel(
    "RMSE de validação (t CO₂/h)"
)

ax.set_title(
    "Iatan 2 — Melhor configuração por algoritmo B0"
)

ax.tick_params(
    axis="x",
    rotation=15
)

ax.grid(
    axis="y",
    alpha=0.2
)

ax.legend()

fig.tight_layout()

FIG_BEST = (
    DIRS["figures"]
    / "cell12_b0_best_algorithms.png"
)

fig.savefig(
    FIG_BEST,
    dpi=180,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 12. PREVISÕES DE VALIDAÇÃO DOS MELHORES MODELOS
# ============================================================

b0_predictions_df = pd.DataFrame({
    "timestamp": timestamps_val,
    "partition": "validation",
    "y_true": y_val,
})

for algorithm in ALGORITHMS_B0:

    b0_predictions_df[
        f"pred_{algorithm}"
    ] = best_predictions_b0[algorithm]


# ============================================================
# 13. EXPORTAÇÃO DOS RESULTADOS
# ============================================================

CANDIDATES_FILE = (
    DIRS["tables"]
    / "cell12_b0_all_candidates.csv"
)

BEST_FILE = (
    DIRS["tables"]
    / "cell12_b0_best_by_algorithm.csv"
)

PREDICTIONS_FILE = (
    DIRS["processed"]
    / "cell12_b0_validation_predictions.parquet"
)

_project_csv(b0_candidates_df, 
    CANDIDATES_FILE,
    index=False
)

_project_csv(b0_best_df, 
    BEST_FILE,
    index=False
)

b0_predictions_df.to_parquet(
    PREDICTIONS_FILE,
    index=False
)


# ============================================================
# 14. RELATÓRIO DE TREINAMENTO
# ============================================================

REPORT_FILE = (
    DIRS["reports"]
    / "cell12_b0_training_report.json"
)

report = {
    "phase": "phase_04_predictive_modeling",
    "cell": "12",
    "generated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "scenario": SCENARIO,
    "feature_set": ["generation_mw"],
    "train_rows": 4499,
    "validation_rows": 687,
    "test_evaluated": False,
    "primary_metric": "rmse",
    "candidate_fits": int(
        len(b0_candidates_df)
    ),
    "linear_baseline_validation_rmse": (
        linear_baseline_rmse
    ),
    "best_by_algorithm": (
        b0_best_df.to_dict(
            orient="records"
        )
    ),
    "artifacts": {
        "all_candidates": str(CANDIDATES_FILE),
        "best_models": str(BEST_FILE),
        "validation_predictions": str(
            PREDICTIONS_FILE
        ),
        "candidate_figure": str(
            FIG_CANDIDATES
        ),
        "best_figure": str(
            FIG_BEST
        ),
    },
}

with open(
    REPORT_FILE,
    "w",
    encoding="utf-8"
) as f:

    _project_dump(
        report,
        f,
        indent=4,
        ensure_ascii=False
    )


# ============================================================
# 15. BACKUP DO MLFLOW
# ============================================================

if "backup_mlflow_database" in globals():
    backup_mlflow_database()


# ============================================================
# 16. RESULTADO FINAL
# ============================================================

assert len(b0_best_df) == 4
assert len(b0_predictions_df) == 687

assert b0_best_df["rmse"].notna().all()

_project_print("\n" + "=" * 70)
_project_print("FASE 4 — CÉLULA 12")
_project_print("=" * 70)

_project_print("Cenário:", SCENARIO)
_project_print("Algoritmos:", len(ALGORITHMS_B0))
_project_print("Candidatos avaliados:", len(b0_candidates_df))
_project_print("Melhores modelos salvos:", len(best_models_b0))

_project_print("Treinamento:", len(y_train))
_project_print("Validação:", len(y_val))
_project_print("Teste avaliado: NÃO")

_project_print("\nMelhor algoritmo B0:")
_project_print(
    b0_best_df.iloc[0]["algorithm"]
)

_project_print(
    "RMSE de validação:",
    round(
        float(b0_best_df.iloc[0]["rmse"]),
        4
    )
)

_project_print("\nRelatório:", REPORT_FILE)
_project_print("Resultados:", BEST_FILE)

_project_print("\nSTATUS: TREINAMENTO B0 CONCLUÍDO")
_project_print("=" * 70)


In [ ]:

# ============================================================
# FASE 4 — CÉLULA 13
# TREINAMENTO E SELEÇÃO DOS MODELOS P1
# ============================================================
#
# P1 = Geração elétrica + 5 indicadores de estados HMM
#
# 24 configurações candidatas:
#   ElasticNet             12
#   RandomForest            4
#   XGBoost                 4
#   HistGradientBoosting    4
#
# FIT: somente treinamento
# SELEÇÃO: RMSE na validação
# TESTE: não acessado
#
# COMPARAÇÃO:
# P1 versus B0, algoritmo a algoritmo.
# ============================================================

import json
import time
import warnings
import itertools
from datetime import datetime, timezone

import joblib
import mlflow
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import ElasticNet
from sklearn.ensemble import (
    RandomForestRegressor,
    HistGradientBoostingRegressor,
)
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)
from xgboost import XGBRegressor


# ============================================================
# 1. PROTOCOLO EXPERIMENTAL
# ============================================================

SCENARIO_P1 = "P1"

PROTOCOL_FILE = (
    DIRS["manifests"]
    / "cell11_experimental_protocol.json"
)

with open(PROTOCOL_FILE, "r", encoding="utf-8") as f:
    protocol_p1 = json.load(f)

ALGORITHMS_P1 = protocol_p1["algorithms"]
PARAM_GRIDS_P1 = protocol_p1["parameter_grids"]
SEED_P1 = int(protocol_p1["random_state"])

assert SCENARIO_P1 in protocol_p1["scenarios"]
assert protocol_p1["primary_metric"] == "rmse"
assert protocol_p1["selection_partition"] == "validation"

mlflow.set_experiment(
    protocol_p1["mlflow_experiment_name"]
)

if mlflow.active_run() is not None:
    raise RuntimeError(
        "Finalize a execução MLflow ativa antes de continuar."
    )


# ============================================================
# 2. DADOS E PRÉ-PROCESSAMENTO
# ============================================================

assert "split_data" in globals()
assert "processed_data" in globals()

X_train_raw_p1 = split_data["P1"]["train"]["X"]
X_val_raw_p1 = split_data["P1"]["validation"]["X"]

y_train_p1 = np.asarray(
    split_data["P1"]["train"]["y"],
    dtype=float
)

y_val_p1 = np.asarray(
    split_data["P1"]["validation"]["y"],
    dtype=float
)

timestamps_val_p1 = pd.to_datetime(
    split_data["P1"]["validation"]["timestamps"],
    utc=True
).reset_index(drop=True)

assert X_train_raw_p1.shape == (4499, 6)
assert X_val_raw_p1.shape == (687, 6)
assert len(y_train_p1) == 4499
assert len(y_val_p1) == 687

assert np.isfinite(y_train_p1).all()
assert np.isfinite(y_val_p1).all()

assert "generation_mw" in X_train_raw_p1.columns

# Garantir comparação pareada com B0.
assert np.array_equal(
    y_train_p1,
    np.asarray(
        split_data["B0"]["train"]["y"],
        dtype=float
    )
)

assert np.array_equal(
    y_val_p1,
    np.asarray(
        split_data["B0"]["validation"]["y"],
        dtype=float
    )
)

_project_print("Atributos P1:")
_project_print(X_train_raw_p1.columns.tolist())


# ============================================================
# 3. FUNÇÕES EXPERIMENTAIS
# ============================================================

def p1_parameter_combinations(grid):

    keys = list(grid.keys())

    for values in itertools.product(
        *(grid[key] for key in keys)
    ):
        yield dict(zip(keys, values))


def p1_build_model(algorithm, params, seed):

    if algorithm == "ElasticNet":
        return ElasticNet(
            **params,
            fit_intercept=True,
            max_iter=30000,
            tol=1e-5,
            selection="cyclic",
            random_state=seed,
        )

    if algorithm == "RandomForest":
        return RandomForestRegressor(
            **params,
            random_state=seed,
            n_jobs=-1,
        )

    if algorithm == "XGBoost":
        return XGBRegressor(
            **params,
            objective="reg:squarederror",
            random_state=seed,
            n_jobs=2,
            tree_method="hist",
            verbosity=0,
        )

    if algorithm == "HistGradientBoosting":
        return HistGradientBoostingRegressor(
            **params,
            random_state=seed,
            early_stopping=False,
        )

    raise ValueError(algorithm)


def p1_metrics(y_true, y_pred):

    y_pred = np.asarray(y_pred, dtype=float)

    assert np.isfinite(y_pred).all()

    return {
        "rmse": float(
            np.sqrt(mean_squared_error(y_true, y_pred))
        ),
        "mae": float(
            mean_absolute_error(y_true, y_pred)
        ),
        "r2": float(
            r2_score(y_true, y_pred)
        ),
        "bias": float(
            np.mean(y_pred - y_true)
        ),
    }


# ============================================================
# 4. TREINAMENTO DOS CANDIDATOS P1
# ============================================================

p1_records = []
best_models_p1 = {}
best_predictions_p1 = {}

for algorithm in ALGORITHMS_P1:

    X_train = processed_data["P1"][algorithm]["train"]
    X_val = processed_data["P1"][algorithm]["validation"]

    assert X_train.shape[0] == 4499
    assert X_val.shape[0] == 687
    assert X_train.shape[1] == 6
    assert X_val.shape[1] == 6

    candidates = list(
        p1_parameter_combinations(
            PARAM_GRIDS_P1[algorithm]
        )
    )

    _project_print("\n" + "=" * 65)
    _project_print(f"P1 — {algorithm}: {len(candidates)} candidatos")
    _project_print("=" * 65)

    algorithm_records = []
    fitted_candidates = {}

    for candidate_index, params in enumerate(
        candidates, start=1
    ):

        model = p1_build_model(
            algorithm,
            params,
            SEED_P1
        )

        run_name = (
            f"P1_{algorithm}_"
            f"candidate_{candidate_index:02d}"
        )

        with mlflow.start_run(run_name=run_name) as run:

            mlflow.set_tags({
                "phase": "04",
                "cell": "13",
                "scenario": "P1",
                "algorithm": algorithm,
                "partition": "validation",
                "run_type": "candidate",
            })

            mlflow.log_params({
                "scenario": "P1",
                "algorithm": algorithm,
                "candidate_index": candidate_index,
                "random_state": SEED_P1,
                "n_features": 6,
                "train_rows": 4499,
                "validation_rows": 687,
                **params,
            })

            start = time.perf_counter()

            with warnings.catch_warnings(
                record=True
            ) as captured:

                warnings.simplefilter(
                    "always",
                    ConvergenceWarning
                )

                model.fit(X_train, y_train_p1)

            fit_seconds = time.perf_counter() - start

            convergence_warnings = [
                str(w.message)
                for w in captured
                if issubclass(
                    w.category,
                    ConvergenceWarning
                )
            ]

            predictions = model.predict(X_val)

            metrics = p1_metrics(
                y_val_p1,
                predictions
            )

            mlflow.log_metrics({
                f"validation_{key}": value
                for key, value in metrics.items()
            })

            mlflow.log_metric(
                "fit_seconds",
                fit_seconds
            )

            mlflow.log_metric(
                "convergence_warning_count",
                len(convergence_warnings)
            )

            record = {
                "scenario": "P1",
                "algorithm": algorithm,
                "candidate_index": candidate_index,
                "run_id": run.info.run_id,
                "params_json": _project_dumps(
                    params,
                    sort_keys=True
                ),
                "fit_seconds": fit_seconds,
                "convergence_warning_count": len(
                    convergence_warnings
                ),
                **metrics,
            }

            p1_records.append(record)
            algorithm_records.append(record)

            fitted_candidates[candidate_index] = {
                "model": model,
                "predictions": predictions,
            }

        _project_print(
            f"[{candidate_index:02d}/{len(candidates):02d}] "
            f"RMSE={metrics['rmse']:.4f} | "
            f"MAE={metrics['mae']:.4f} | "
            f"Viés={metrics['bias']:+.4f}"
        )

    # Melhor candidato: RMSE, depois MAE.
    algorithm_df = pd.DataFrame(
        algorithm_records
    ).sort_values(
        ["rmse", "mae", "candidate_index"]
    )

    best_record = algorithm_df.iloc[0]
    best_index = int(best_record["candidate_index"])

    best_models_p1[algorithm] = (
        fitted_candidates[best_index]["model"]
    )

    best_predictions_p1[algorithm] = (
        fitted_candidates[best_index]["predictions"]
    )

    model_path = (
        DIRS["models"]
        / f"cell13_p1_best_{algorithm.lower()}.joblib"
    )

    joblib.dump(
        best_models_p1[algorithm],
        model_path
    )

    _project_print(
        f"\nMelhor {algorithm}: "
        f"candidato {best_index} | "
        f"RMSE={best_record['rmse']:.4f}"
    )


# ============================================================
# 5. TABELAS DOS RESULTADOS
# ============================================================

p1_candidates_df = pd.DataFrame(p1_records)

assert len(p1_candidates_df) == 24

p1_best_df = (
    p1_candidates_df
    .sort_values(["rmse", "mae", "candidate_index"])
    .groupby("algorithm", as_index=False)
    .first()
    .sort_values("rmse")
    .reset_index(drop=True)
)

_project_print("\nMELHORES CONFIGURAÇÕES P1")
_project_display(
    p1_best_df[
        [
            "algorithm",
            "candidate_index",
            "rmse",
            "mae",
            "r2",
            "bias",
            "params_json",
            "convergence_warning_count",
        ]
    ].round(4)
)


# ============================================================
# 6. COMPARAÇÃO PAREADA P1 VERSUS B0
# ============================================================

B0_FILE = (
    DIRS["tables"]
    / "cell12_b0_best_by_algorithm.csv"
)

b0_reference_df = pd.read_csv(B0_FILE)

comparison_p1_b0 = (
    b0_reference_df[
        ["algorithm", "rmse", "mae", "bias"]
    ]
    .rename(columns={
        "rmse": "rmse_b0",
        "mae": "mae_b0",
        "bias": "bias_b0",
    })
    .merge(
        p1_best_df[
            ["algorithm", "rmse", "mae", "bias"]
        ].rename(columns={
            "rmse": "rmse_p1",
            "mae": "mae_p1",
            "bias": "bias_p1",
        }),
        on="algorithm",
        validate="one_to_one"
    )
)

comparison_p1_b0["rmse_reduction_pct"] = (
    100
    * (
        comparison_p1_b0["rmse_b0"]
        - comparison_p1_b0["rmse_p1"]
    )
    / comparison_p1_b0["rmse_b0"]
)

comparison_p1_b0["mae_reduction_pct"] = (
    100
    * (
        comparison_p1_b0["mae_b0"]
        - comparison_p1_b0["mae_p1"]
    )
    / comparison_p1_b0["mae_b0"]
)

comparison_p1_b0 = comparison_p1_b0.sort_values(
    "rmse_reduction_pct",
    ascending=False
).reset_index(drop=True)

_project_print("\nCOMPARAÇÃO PAREADA — B0 VERSUS P1")
_project_display(comparison_p1_b0.round(4))


# ============================================================
# 7. FIGURA — RMSE B0 VERSUS P1
# ============================================================

plot_df = comparison_p1_b0.sort_values(
    "algorithm"
).reset_index(drop=True)

x = np.arange(len(plot_df))
width = 0.36

fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(
    x - width / 2,
    plot_df["rmse_b0"],
    width,
    label="B0 — geração"
)

ax.bar(
    x + width / 2,
    plot_df["rmse_p1"],
    width,
    label="P1 — geração + HMM"
)

ax.set_xticks(x)
ax.set_xticklabels(
    plot_df["algorithm"],
    rotation=12
)

ax.set_ylabel("RMSE de validação (t CO₂/h)")
ax.set_title("Iatan 2 — Comparação B0 versus P1")
ax.legend()
ax.grid(axis="y", alpha=0.2)

fig.tight_layout()

FIG_COMPARISON = (
    DIRS["figures"]
    / "cell13_b0_vs_p1_rmse.png"
)

fig.savefig(
    FIG_COMPARISON,
    dpi=180,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 8. FIGURA — REDUÇÃO PERCENTUAL DO RMSE
# ============================================================

fig, ax = plt.subplots(figsize=(9, 5))

ax.barh(
    plot_df["algorithm"],
    plot_df["rmse_reduction_pct"]
)

ax.axvline(
    0,
    linestyle="--",
    linewidth=1
)

ax.set_xlabel("Redução do RMSE de P1 versus B0 (%)")
ax.set_title("Ganho incremental dos estados HMM")
ax.grid(axis="x", alpha=0.2)

fig.tight_layout()

FIG_IMPROVEMENT = (
    DIRS["figures"]
    / "cell13_p1_rmse_improvement.png"
)

fig.savefig(
    FIG_IMPROVEMENT,
    dpi=180,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 9. PREVISÕES DE VALIDAÇÃO
# ============================================================

p1_predictions_df = pd.DataFrame({
    "timestamp": timestamps_val_p1,
    "partition": "validation",
    "y_true": y_val_p1,
})

for algorithm in ALGORITHMS_P1:
    p1_predictions_df[
        f"pred_{algorithm}"
    ] = best_predictions_p1[algorithm]


# ============================================================
# 10. EXPORTAÇÃO DOS ARTEFATOS
# ============================================================

CANDIDATES_FILE = (
    DIRS["tables"]
    / "cell13_p1_all_candidates.csv"
)

BEST_FILE = (
    DIRS["tables"]
    / "cell13_p1_best_by_algorithm.csv"
)

COMPARISON_FILE = (
    DIRS["tables"]
    / "cell13_b0_vs_p1_comparison.csv"
)

PREDICTIONS_FILE = (
    DIRS["processed"]
    / "cell13_p1_validation_predictions.parquet"
)

REPORT_FILE = (
    DIRS["reports"]
    / "cell13_p1_training_report.json"
)

_project_csv(p1_candidates_df, 
    CANDIDATES_FILE,
    index=False
)

_project_csv(p1_best_df, 
    BEST_FILE,
    index=False
)

_project_csv(comparison_p1_b0, 
    COMPARISON_FILE,
    index=False
)

p1_predictions_df.to_parquet(
    PREDICTIONS_FILE,
    index=False
)

report_p1 = {
    "phase": "phase_04_predictive_modeling",
    "cell": "13",
    "generated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "scenario": "P1",
    "train_rows": 4499,
    "validation_rows": 687,
    "test_evaluated": False,
    "feature_names": X_train_raw_p1.columns.tolist(),
    "primary_metric": "rmse",
    "candidate_fits": len(p1_candidates_df),
    "best_by_algorithm": p1_best_df.to_dict(
        orient="records"
    ),
    "comparison_with_b0": comparison_p1_b0.to_dict(
        orient="records"
    ),
    "files": {
        "candidates": str(CANDIDATES_FILE),
        "best": str(BEST_FILE),
        "comparison": str(COMPARISON_FILE),
        "predictions": str(PREDICTIONS_FILE),
        "figure_comparison": str(FIG_COMPARISON),
        "figure_improvement": str(FIG_IMPROVEMENT),
    },
}

with open(REPORT_FILE, "w", encoding="utf-8") as f:
    _project_dump(
        report_p1,
        f,
        indent=4,
        ensure_ascii=False
    )

if "backup_mlflow_database" in globals():
    backup_mlflow_database()


# ============================================================
# 11. VERIFICAÇÕES FINAIS
# ============================================================

assert len(p1_best_df) == 4
assert len(comparison_p1_b0) == 4
assert len(p1_predictions_df) == 687

assert np.isfinite(
    p1_best_df[["rmse", "mae", "r2", "bias"]]
    .to_numpy(dtype=float)
).all()

_project_print("\n" + "=" * 70)
_project_print("FASE 4 — CÉLULA 13")
_project_print("=" * 70)

_project_print("Cenário: P1")
_project_print("Atributos:", X_train_raw_p1.shape[1])
_project_print("Candidatos avaliados:", len(p1_candidates_df))
_project_print("Melhores modelos salvos:", len(p1_best_df))
_project_print("Teste avaliado: NÃO")

_project_print("\nMelhor algoritmo P1:")
_project_print(p1_best_df.iloc[0]["algorithm"])

_project_print(
    "RMSE de validação:",
    round(float(p1_best_df.iloc[0]["rmse"]), 4)
)

_project_print("\nComparação:", COMPARISON_FILE)
_project_print("Relatório:", REPORT_FILE)

_project_print("\nSTATUS: TREINAMENTO P1 CONCLUÍDO")
_project_print("=" * 70)


In [ ]:

# ============================================================
# FASE 4 — CÉLULA 14
# TREINAMENTO E SELEÇÃO DOS MODELOS P2
# ============================================================
#
# OBJETIVO:
# Verificar o ganho preditivo dos atributos de processos
# em relação aos cenários B0 e P1.
#
# P2: 35 atributos
# B0: geração elétrica
# P1: geração + indicadores HMM
#
# 24 candidatos:
# ElasticNet:             12
# RandomForest:            4
# XGBoost:                 4
# HistGradientBoosting:    4
#
# Treinamento: 4.499 horas
# Validação:     687 horas
# Teste: NÃO UTILIZADO
#
# O pré-processamento é recuperado da Célula 08.
# ============================================================

import json
import time
import warnings
import itertools
from datetime import datetime, timezone

import joblib
import mlflow
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import ElasticNet
from sklearn.ensemble import (
    RandomForestRegressor,
    HistGradientBoostingRegressor
)
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)
from xgboost import XGBRegressor


# ============================================================
# 1. RECUPERAÇÃO DO PROTOCOLO
# ============================================================

SCENARIO_P2 = "P2"

PROTOCOL_FILE = (
    DIRS["manifests"]
    / "cell11_experimental_protocol.json"
)

with open(PROTOCOL_FILE, "r", encoding="utf-8") as f:
    protocol_p2 = json.load(f)

ALGORITHMS_P2 = protocol_p2["algorithms"]
PARAM_GRIDS_P2 = protocol_p2["parameter_grids"]
SEED_P2 = int(protocol_p2["random_state"])

assert SCENARIO_P2 in protocol_p2["scenarios"]
assert protocol_p2["primary_metric"] == "rmse"
assert protocol_p2["selection_partition"] == "validation"

if mlflow.active_run() is not None:
    raise RuntimeError(
        "Finalize a execução MLflow ativa antes de continuar."
    )

mlflow.set_experiment(
    protocol_p2["mlflow_experiment_name"]
)


# ============================================================
# 2. RECUPERAÇÃO E VERIFICAÇÃO DOS DADOS
# ============================================================

if "split_data" not in globals():
    raise RuntimeError("Execute a Célula 07.")

if "processed_data" not in globals():
    raise RuntimeError("Execute a Célula 08.")

X_train_raw_p2 = split_data["P2"]["train"]["X"]
X_val_raw_p2 = split_data["P2"]["validation"]["X"]

y_train_p2 = np.asarray(
    split_data["P2"]["train"]["y"],
    dtype=float
)

y_val_p2 = np.asarray(
    split_data["P2"]["validation"]["y"],
    dtype=float
)

timestamps_val_p2 = pd.to_datetime(
    split_data["P2"]["validation"]["timestamps"],
    utc=True
).reset_index(drop=True)

assert X_train_raw_p2.shape == (4499, 35)
assert X_val_raw_p2.shape == (687, 35)

assert np.isfinite(y_train_p2).all()
assert np.isfinite(y_val_p2).all()

# Identidade do alvo com os cenários anteriores.
for scenario in ["B0", "P1"]:

    np.testing.assert_array_equal(
        y_train_p2,
        split_data[scenario]["train"]["y"].to_numpy()
    )

    np.testing.assert_array_equal(
        y_val_p2,
        split_data[scenario]["validation"]["y"].to_numpy()
    )

# Conferência de que as matrizes P2 possuem
# os atributos esperados e não incluem o alvo.
assert "generation_mw" in X_train_raw_p2.columns
assert "co2_metric_tons" not in X_train_raw_p2.columns
assert X_train_raw_p2.columns.is_unique

_project_print("ATRIBUTOS P2:")
for i, col in enumerate(X_train_raw_p2.columns, 1):
    _project_print(f"{i:02d}. {col}")


# ============================================================
# 3. FUNÇÕES EXPERIMENTAIS
# ============================================================

def p2_parameter_combinations(grid):

    keys = list(grid)

    for values in itertools.product(
        *(grid[key] for key in keys)
    ):
        yield dict(zip(keys, values))


def p2_build_model(algorithm, params, seed):

    if algorithm == "ElasticNet":

        return ElasticNet(
            **params,
            fit_intercept=True,
            max_iter=30000,
            tol=1e-5,
            selection="cyclic",
            random_state=seed
        )

    if algorithm == "RandomForest":

        return RandomForestRegressor(
            **params,
            random_state=seed,
            n_jobs=-1
        )

    if algorithm == "XGBoost":

        return XGBRegressor(
            **params,
            objective="reg:squarederror",
            random_state=seed,
            n_jobs=2,
            tree_method="hist",
            verbosity=0
        )

    if algorithm == "HistGradientBoosting":

        return HistGradientBoostingRegressor(
            **params,
            random_state=seed,
            early_stopping=False
        )

    raise ValueError(algorithm)


def p2_calculate_metrics(y_true, y_pred):

    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    if not np.isfinite(y_pred).all():
        raise ValueError("Previsões não finitas.")

    return {
        "rmse": float(
            np.sqrt(mean_squared_error(y_true, y_pred))
        ),
        "mae": float(
            mean_absolute_error(y_true, y_pred)
        ),
        "r2": float(
            r2_score(y_true, y_pred)
        ),
        "bias": float(
            np.mean(y_pred - y_true)
        )
    }


# ============================================================
# 4. TREINAMENTO DOS 24 CANDIDATOS P2
# ============================================================

p2_records = []
best_models_p2 = {}
best_predictions_p2 = {}

for algorithm in ALGORITHMS_P2:

    X_train = processed_data["P2"][algorithm]["train"]
    X_val = processed_data["P2"][algorithm]["validation"]

    assert X_train.shape == (4499, 35)
    assert X_val.shape == (687, 35)

    candidates = list(
        p2_parameter_combinations(
            PARAM_GRIDS_P2[algorithm]
        )
    )

    algorithm_records = []
    fitted_candidates = {}

    _project_print("\n" + "=" * 65)
    _project_print(f"P2 — {algorithm}: {len(candidates)} candidatos")
    _project_print("=" * 65)

    for candidate_index, params in enumerate(
        candidates, start=1
    ):

        model = p2_build_model(
            algorithm,
            params,
            SEED_P2
        )

        run_name = (
            f"P2_{algorithm}_"
            f"candidate_{candidate_index:02d}"
        )

        with mlflow.start_run(run_name=run_name) as run:

            mlflow.set_tags({
                "phase": "04",
                "cell": "14",
                "scenario": "P2",
                "algorithm": algorithm,
                "partition": "validation",
                "run_type": "candidate"
            })

            mlflow.log_params({
                "scenario": "P2",
                "algorithm": algorithm,
                "candidate_index": candidate_index,
                "random_state": SEED_P2,
                "n_features": 35,
                "train_rows": 4499,
                "validation_rows": 687,
                **params
            })

            start = time.perf_counter()

            with warnings.catch_warnings(
                record=True
            ) as captured:

                warnings.simplefilter(
                    "always",
                    ConvergenceWarning
                )

                model.fit(X_train, y_train_p2)

            fit_seconds = time.perf_counter() - start

            convergence_warnings = [
                str(w.message)
                for w in captured
                if issubclass(
                    w.category,
                    ConvergenceWarning
                )
            ]

            predictions = model.predict(X_val)

            metrics = p2_calculate_metrics(
                y_val_p2,
                predictions
            )

            mlflow.log_metrics({
                f"validation_{key}": value
                for key, value in metrics.items()
            })

            mlflow.log_metric(
                "fit_seconds",
                fit_seconds
            )

            mlflow.log_metric(
                "convergence_warning_count",
                len(convergence_warnings)
            )

            record = {
                "scenario": "P2",
                "algorithm": algorithm,
                "candidate_index": candidate_index,
                "run_id": run.info.run_id,
                "params_json": _project_dumps(
                    params,
                    sort_keys=True
                ),
                "fit_seconds": fit_seconds,
                "convergence_warning_count": (
                    len(convergence_warnings)
                ),
                **metrics
            }

            p2_records.append(record)
            algorithm_records.append(record)

            fitted_candidates[candidate_index] = {
                "model": model,
                "predictions": predictions
            }

        _project_print(
            f"[{candidate_index:02d}/{len(candidates):02d}] "
            f"RMSE={metrics['rmse']:.4f} | "
            f"MAE={metrics['mae']:.4f} | "
            f"Viés={metrics['bias']:+.4f}"
        )

    # Seleção por RMSE de validação.
    algorithm_df = pd.DataFrame(
        algorithm_records
    ).sort_values(
        ["rmse", "mae", "candidate_index"]
    )

    best_record = algorithm_df.iloc[0]
    best_index = int(best_record["candidate_index"])

    best_models_p2[algorithm] = (
        fitted_candidates[best_index]["model"]
    )

    best_predictions_p2[algorithm] = (
        fitted_candidates[best_index]["predictions"]
    )

    MODEL_FILE = (
        DIRS["models"]
        / f"cell14_p2_best_{algorithm.lower()}.joblib"
    )

    joblib.dump(
        best_models_p2[algorithm],
        MODEL_FILE
    )

    _project_print(
        f"\nMelhor {algorithm}: "
        f"candidato {best_index} | "
        f"RMSE={best_record['rmse']:.4f}"
    )


# ============================================================
# 5. RESULTADOS DOS CANDIDATOS
# ============================================================

p2_candidates_df = pd.DataFrame(p2_records)

assert len(p2_candidates_df) == 24

p2_best_df = (
    p2_candidates_df
    .sort_values(["rmse", "mae", "candidate_index"])
    .groupby("algorithm", as_index=False)
    .first()
    .sort_values("rmse")
    .reset_index(drop=True)
)

_project_print("\nMELHORES CONFIGURAÇÕES P2")

_project_display(
    p2_best_df[
        [
            "algorithm",
            "candidate_index",
            "rmse",
            "mae",
            "r2",
            "bias",
            "params_json",
            "convergence_warning_count"
        ]
    ].round(4)
)


# ============================================================
# 6. COMPARAÇÃO B0 × P1 × P2
# ============================================================

b0_reference = pd.read_csv(
    DIRS["tables"]
    / "cell12_b0_best_by_algorithm.csv"
)

p1_reference = pd.read_csv(
    DIRS["tables"]
    / "cell13_p1_best_by_algorithm.csv"
)

comparison_p2 = (
    b0_reference[
        ["algorithm", "rmse", "mae", "bias"]
    ]
    .rename(columns={
        "rmse": "rmse_b0",
        "mae": "mae_b0",
        "bias": "bias_b0"
    })
    .merge(
        p1_reference[
            ["algorithm", "rmse", "mae", "bias"]
        ].rename(columns={
            "rmse": "rmse_p1",
            "mae": "mae_p1",
            "bias": "bias_p1"
        }),
        on="algorithm",
        validate="one_to_one"
    )
    .merge(
        p2_best_df[
            ["algorithm", "rmse", "mae", "bias"]
        ].rename(columns={
            "rmse": "rmse_p2",
            "mae": "mae_p2",
            "bias": "bias_p2"
        }),
        on="algorithm",
        validate="one_to_one"
    )
)

comparison_p2["gain_p2_vs_b0_pct"] = (
    100 * (
        comparison_p2["rmse_b0"]
        - comparison_p2["rmse_p2"]
    ) / comparison_p2["rmse_b0"]
)

comparison_p2["gain_p2_vs_p1_pct"] = (
    100 * (
        comparison_p2["rmse_p1"]
        - comparison_p2["rmse_p2"]
    ) / comparison_p2["rmse_p1"]
)

comparison_p2 = comparison_p2.sort_values(
    "gain_p2_vs_b0_pct",
    ascending=False
).reset_index(drop=True)

_project_print("\nCOMPARAÇÃO B0 × P1 × P2")

_project_display(
    comparison_p2[
        [
            "algorithm",
            "rmse_b0",
            "rmse_p1",
            "rmse_p2",
            "gain_p2_vs_b0_pct",
            "gain_p2_vs_p1_pct",
            "bias_b0",
            "bias_p1",
            "bias_p2"
        ]
    ].round(4)
)


# ============================================================
# 7. FIGURA — RMSE POR CENÁRIO E ALGORITMO
# ============================================================

plot_df = comparison_p2.sort_values(
    "algorithm"
).reset_index(drop=True)

x = np.arange(len(plot_df))
width = 0.25

fig, ax = plt.subplots(figsize=(11, 5))

ax.bar(
    x - width,
    plot_df["rmse_b0"],
    width,
    label="B0"
)

ax.bar(
    x,
    plot_df["rmse_p1"],
    width,
    label="P1"
)

ax.bar(
    x + width,
    plot_df["rmse_p2"],
    width,
    label="P2"
)

ax.set_xticks(x)
ax.set_xticklabels(
    plot_df["algorithm"],
    rotation=12
)

ax.set_ylabel("RMSE de validação (t CO₂/h)")
ax.set_title("Iatan 2 — Comparação B0 × P1 × P2")
ax.legend()
ax.grid(axis="y", alpha=0.2)

fig.tight_layout()

FIG_COMPARISON_P2 = (
    DIRS["figures"]
    / "cell14_b0_p1_p2_rmse.png"
)

fig.savefig(
    FIG_COMPARISON_P2,
    dpi=180,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 8. FIGURA — GANHO PERCENTUAL DE P2
# ============================================================

fig, ax = plt.subplots(figsize=(9, 5))

plot_gain = comparison_p2.sort_values(
    "gain_p2_vs_b0_pct"
)

ax.barh(
    plot_gain["algorithm"],
    plot_gain["gain_p2_vs_b0_pct"]
)

ax.axvline(
    0,
    linestyle="--",
    linewidth=1
)

ax.set_xlabel("Redução do RMSE de P2 versus B0 (%)")
ax.set_title("Ganho incremental dos atributos de processos")
ax.grid(axis="x", alpha=0.2)

fig.tight_layout()

FIG_GAIN_P2 = (
    DIRS["figures"]
    / "cell14_p2_incremental_gain.png"
)

fig.savefig(
    FIG_GAIN_P2,
    dpi=180,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 9. PREVISÕES DOS MELHORES MODELOS P2
# ============================================================

p2_predictions_df = pd.DataFrame({
    "timestamp": timestamps_val_p2,
    "partition": "validation",
    "y_true": y_val_p2
})

for algorithm in ALGORITHMS_P2:

    p2_predictions_df[
        f"pred_{algorithm}"
    ] = best_predictions_p2[algorithm]


# ============================================================
# 10. EXPORTAÇÃO
# ============================================================

CANDIDATES_FILE = (
    DIRS["tables"]
    / "cell14_p2_all_candidates.csv"
)

BEST_FILE = (
    DIRS["tables"]
    / "cell14_p2_best_by_algorithm.csv"
)

COMPARISON_FILE = (
    DIRS["tables"]
    / "cell14_b0_p1_p2_comparison.csv"
)

PREDICTIONS_FILE = (
    DIRS["processed"]
    / "cell14_p2_validation_predictions.parquet"
)

REPORT_FILE = (
    DIRS["reports"]
    / "cell14_p2_training_report.json"
)

_project_csv(p2_candidates_df, 
    CANDIDATES_FILE,
    index=False
)

_project_csv(p2_best_df, 
    BEST_FILE,
    index=False
)

_project_csv(comparison_p2, 
    COMPARISON_FILE,
    index=False
)

p2_predictions_df.to_parquet(
    PREDICTIONS_FILE,
    index=False
)

report_p2 = {
    "phase": "phase_04_predictive_modeling",
    "cell": "14",
    "generated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "scenario": "P2",
    "n_features": 35,
    "feature_names": X_train_raw_p2.columns.tolist(),
    "train_rows": 4499,
    "validation_rows": 687,
    "test_evaluated": False,
    "primary_metric": "rmse",
    "candidate_fits": int(len(p2_candidates_df)),
    "best_by_algorithm": p2_best_df.to_dict(
        orient="records"
    ),
    "comparison": comparison_p2.to_dict(
        orient="records"
    ),
    "files": {
        "candidates": str(CANDIDATES_FILE),
        "best": str(BEST_FILE),
        "comparison": str(COMPARISON_FILE),
        "predictions": str(PREDICTIONS_FILE),
        "figure_comparison": str(FIG_COMPARISON_P2),
        "figure_gain": str(FIG_GAIN_P2)
    }
}

with open(REPORT_FILE, "w", encoding="utf-8") as f:

    _project_dump(
        report_p2,
        f,
        indent=4,
        ensure_ascii=False
    )


# ============================================================
# 11. BACKUP DO MLFLOW
# ============================================================

if "backup_mlflow_database" in globals():
    backup_mlflow_database()


# ============================================================
# 12. VERIFICAÇÕES FINAIS
# ============================================================

assert len(p2_best_df) == 4
assert len(comparison_p2) == 4
assert len(p2_predictions_df) == 687

assert np.isfinite(
    p2_best_df[
        ["rmse", "mae", "r2", "bias"]
    ].to_numpy(dtype=float)
).all()

_project_print("\n" + "=" * 70)
_project_print("FASE 4 — CÉLULA 14")
_project_print("=" * 70)

_project_print("Cenário: P2")
_project_print("Atributos:", X_train_raw_p2.shape[1])
_project_print("Candidatos avaliados:", len(p2_candidates_df))
_project_print("Melhores modelos salvos:", len(p2_best_df))
_project_print("Teste avaliado: NÃO")

_project_print("\nMelhor algoritmo P2:")
_project_print(p2_best_df.iloc[0]["algorithm"])

_project_print(
    "RMSE de validação:",
    round(float(p2_best_df.iloc[0]["rmse"]), 4)
)

_project_print("\nComparação:", COMPARISON_FILE)
_project_print("Relatório:", REPORT_FILE)

_project_print("\nSTATUS: TREINAMENTO P2 CONCLUÍDO")
_project_print("=" * 70)


### Congelamento da seleção

[Cawley e Talbot (2010)](https://jmlr.org/papers/v11/cawley10a.html) justificam separar a seleção por validação da avaliação. O congelamento preserva a escolha e a origem dos modelos, com o teste de 2025 tratado como retrospectivo.


In [ ]:

# ============================================================
# FASE 4 — CÉLULA 15
# CONSOLIDAÇÃO E CONGELAMENTO DA SELEÇÃO
# ============================================================
#
# 1. Reunir os 72 candidatos B0/P1/P2.
# 2. Validar unicidade, parâmetros e MLflow.
# 3. Comparar as 12 melhores combinações.
# 4. Congelar o melhor modelo de cada cenário.
# 5. Produzir gráficos e manifesto reproduzível.
#
# Não treina modelos.
# Não acessa dados de teste.
# ============================================================

import json
import hashlib
from datetime import datetime, timezone
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 1. CARREGAR PROTOCOLO E RESULTADOS
# ============================================================

with open(
    DIRS["manifests"] / "cell11_experimental_protocol.json",
    encoding="utf-8"
) as f:
    protocol15 = json.load(f)

scenarios15 = ["B0", "P1", "P2"]
algorithms15 = protocol15["algorithms"]

candidate_files15 = {
    "B0": DIRS["tables"] / "cell12_b0_all_candidates.csv",
    "P1": DIRS["tables"] / "cell13_p1_all_candidates.csv",
    "P2": DIRS["tables"] / "cell14_p2_all_candidates.csv",
}

frames15 = []

for scenario, path in candidate_files15.items():
    if not path.exists():
        raise FileNotFoundError(path)

    frame = pd.read_csv(path)

    assert len(frame) == 24, (scenario, len(frame))
    assert (frame["scenario"] == scenario).all()

    frames15.append(frame)

all_trials15 = pd.concat(
    frames15, ignore_index=True
)

assert len(all_trials15) == 72

assert not all_trials15.duplicated(
    ["scenario", "algorithm", "candidate_index"]
).any()

assert all_trials15["run_id"].is_unique

assert np.isfinite(
    all_trials15[
        ["rmse", "mae", "r2", "bias", "fit_seconds"]
    ].to_numpy(dtype=float)
).all()

_project_print("Candidatos consolidados:", len(all_trials15))


# ============================================================
# 2. VALIDAR CONFIGURAÇÕES E REGISTROS MLFLOW
# ============================================================

experiment15 = mlflow.get_experiment_by_name(
    protocol15["mlflow_experiment_name"]
)

if experiment15 is None:
    raise RuntimeError("Experimento MLflow indisponível.")

expected15 = {
    "ElasticNet": 12,
    "RandomForest": 4,
    "XGBoost": 4,
    "HistGradientBoosting": 4,
}

checks15 = []

for scenario in scenarios15:
    for algorithm in algorithms15:

        subset = all_trials15.loc[
            (all_trials15["scenario"] == scenario)
            & (all_trials15["algorithm"] == algorithm)
        ]

        expected_n = expected15[algorithm]

        assert len(subset) == expected_n

        assert sorted(
            subset["candidate_index"].astype(int)
        ) == list(range(1, expected_n + 1))

        checks15.append({
            "scenario": scenario,
            "algorithm": algorithm,
            "candidates": len(subset),
            "status": "OK"
        })

# Conferir os 72 run IDs contra o MLflow.
mlflow_missing15 = []
mlflow_mismatches15 = []

for row in all_trials15.itertuples(index=False):
    try:
        run = mlflow.get_run(str(row.run_id))
    except Exception:
        mlflow_missing15.append(str(row.run_id))
        continue

    if (
        str(run.info.experiment_id)
        != str(experiment15.experiment_id)
        or run.data.tags.get("scenario") != row.scenario
        or run.data.tags.get("algorithm") != row.algorithm
        or not np.isclose(
            run.data.metrics.get("validation_rmse", np.nan),
            float(row.rmse),
            rtol=1e-7,
            atol=1e-7
        )
    ):
        mlflow_mismatches15.append(str(row.run_id))

_project_print("Execuções ausentes no MLflow:", len(mlflow_missing15))
_project_print("Execuções divergentes:", len(mlflow_mismatches15))

if mlflow_missing15 or mlflow_mismatches15:
    raise RuntimeError(
        "Inconsistência entre CSVs e MLflow. "
        "Verifique o banco restaurado antes de continuar."
    )


# ============================================================
# 3. SELECIONAR OS 12 MELHORES MODELOS
# ============================================================

best12_15 = (
    all_trials15
    .sort_values(
        ["rmse", "mae", "candidate_index"],
        kind="stable"
    )
    .groupby(
        ["scenario", "algorithm"],
        as_index=False
    )
    .first()
    .sort_values("rmse")
    .reset_index(drop=True)
)

assert len(best12_15) == 12

_project_print("\nMELHORES CONFIGURAÇÕES — 12 COMBINAÇÕES")
_project_display(
    best12_15[
        [
            "scenario", "algorithm", "candidate_index",
            "rmse", "mae", "r2", "bias",
            "convergence_warning_count"
        ]
    ].round(4)
)


# ============================================================
# 4. SELECIONAR UM MODELO POR CENÁRIO
# ============================================================

champions15 = (
    best12_15
    .sort_values(["rmse", "mae"], kind="stable")
    .groupby("scenario", as_index=False)
    .first()
    .set_index("scenario")
    .loc[scenarios15]
    .reset_index()
)

_project_print("\nCAMPEÕES POR CENÁRIO")
_project_display(
    champions15[
        [
            "scenario", "algorithm", "candidate_index",
            "rmse", "mae", "r2", "bias"
        ]
    ].round(4)
)

global_champion15 = champions15.sort_values(
    "rmse"
).iloc[0]

_project_print(
    "\nMelhor modelo global na validação:",
    global_champion15["scenario"],
    global_champion15["algorithm"],
    f"RMSE={global_champion15['rmse']:.4f}"
)


# ============================================================
# 5. LOCALIZAR E VERIFICAR OS MODELOS SALVOS
# ============================================================

model_cells15 = {"B0": 12, "P1": 13, "P2": 14}

def sha256_file15(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

model_inventory15 = []

for row in best12_15.itertuples(index=False):

    model_path = (
        DIRS["models"]
        / (
            f"cell{model_cells15[row.scenario]}_"
            f"{row.scenario.lower()}_best_"
            f"{row.algorithm.lower()}.joblib"
        )
    )

    if not model_path.exists():
        raise FileNotFoundError(model_path)

    model_inventory15.append({
        "scenario": row.scenario,
        "algorithm": row.algorithm,
        "candidate_index": int(row.candidate_index),
        "run_id": str(row.run_id),
        "model_path": str(model_path),
        "model_sha256": sha256_file15(model_path),
    })

model_inventory15 = pd.DataFrame(model_inventory15)


# ============================================================
# 6. GRÁFICO — RMSE DOS 72 CANDIDATOS
# ============================================================

fig, ax = plt.subplots(figsize=(11, 6))

positions15 = np.arange(len(algorithms15))
offsets15 = {"B0": -0.24, "P1": 0.0, "P2": 0.24}

for scenario in scenarios15:
    for i, algorithm in enumerate(algorithms15):

        subset = all_trials15.loc[
            (all_trials15["scenario"] == scenario)
            & (all_trials15["algorithm"] == algorithm)
        ]

        center = positions15[i] + offsets15[scenario]

        # Cada ponto representa um candidato.
        ax.scatter(
            np.full(len(subset), center),
            subset["rmse"],
            s=23,
            alpha=0.45,
            label=scenario if i == 0 else None
        )

        best_rmse = subset["rmse"].min()

        ax.scatter(
            center, best_rmse,
            marker="D", s=65,
            edgecolors="black",
            linewidths=0.6,
            zorder=4
        )

ax.set_xticks(positions15)
ax.set_xticklabels(algorithms15, rotation=12)
ax.set_ylabel("RMSE de validação (t CO₂/h)")
ax.set_title(
    "Iatan 2 — Distribuição dos 72 candidatos"
)
ax.legend(title="Cenário")
ax.grid(axis="y", alpha=0.2)

fig.tight_layout()

fig_trials15 = (
    DIRS["figures"] / "cell15_all_trials_rmse.png"
)

fig.savefig(fig_trials15, dpi=180, bbox_inches="tight")
plt.show()


# ============================================================
# 7. GRÁFICO — 12 MELHORES COMBINAÇÕES
# ============================================================

pivot15 = best12_15.pivot(
    index="algorithm",
    columns="scenario",
    values="rmse"
).loc[algorithms15, scenarios15]

ax = pivot15.plot.bar(figsize=(11, 5), width=0.75)

ax.set_title(
    "Iatan 2 — Melhor RMSE por algoritmo e cenário"
)
ax.set_xlabel("Algoritmo")
ax.set_ylabel("RMSE de validação (t CO₂/h)")
ax.tick_params(axis="x", rotation=12)
ax.grid(axis="y", alpha=0.2)
ax.legend(title="Cenário")

plt.tight_layout()

fig_best15 = (
    DIRS["figures"] / "cell15_best12_rmse.png"
)

plt.savefig(fig_best15, dpi=180, bbox_inches="tight")
plt.show()


# ============================================================
# 8. EXPORTAR RESULTADOS
# ============================================================

all_file15 = (
    DIRS["tables"] / "cell15_all_72_trials.csv"
)

best_file15 = (
    DIRS["tables"] / "cell15_best_12_models.csv"
)

champions_file15 = (
    DIRS["tables"] / "cell15_scenario_champions.csv"
)

inventory_file15 = (
    DIRS["tables"] / "cell15_model_inventory.csv"
)

_project_csv(all_trials15, all_file15, index=False)
_project_csv(best12_15, best_file15, index=False)
_project_csv(champions15, champions_file15, index=False)
_project_csv(model_inventory15, inventory_file15, index=False)


# ============================================================
# 9. MANIFESTO DE SELEÇÃO
# ============================================================
#
# Este arquivo congela:
# - algoritmo;
# - hiperparâmetros;
# - run ID;
# - arquivo e hash do modelo;
# - RMSE de validação.
#
# A auditoria temporal HMM permanece pendente.
# ============================================================

selected_models15 = []

for row in champions15.itertuples(index=False):

    match = model_inventory15.loc[
        (model_inventory15["scenario"] == row.scenario)
        & (model_inventory15["algorithm"] == row.algorithm)
    ].iloc[0]

    selected_models15.append({
        "scenario": row.scenario,
        "algorithm": row.algorithm,
        "candidate_index": int(row.candidate_index),
        "parameters": json.loads(row.params_json),
        "validation_rmse": float(row.rmse),
        "validation_mae": float(row.mae),
        "validation_r2": float(row.r2),
        "validation_bias": float(row.bias),
        "run_id": str(row.run_id),
        "model_path": match["model_path"],
        "model_sha256": match["model_sha256"],
    })

manifest15 = {
    "phase": "phase_04_predictive_modeling",
    "cell": "15",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "selection_status": "frozen_on_validation",
    "selection_metric": "rmse",
    "selection_partition": "validation",
    "test_evaluated": False,
    "n_candidates": 72,
    "n_primary_combinations": 12,
    "n_scenario_champions": 3,
    "selected_models": selected_models15,
    "global_validation_champion": {
        "scenario": str(global_champion15["scenario"]),
        "algorithm": str(global_champion15["algorithm"]),
        "validation_rmse": float(global_champion15["rmse"])
    },
    "temporal_provenance": {
        "hmm_train_only_fit": "not_verified",
        "hmm_forward_only_inference": "not_verified",
        "p2_features_no_future_information": "not_verified",
    },
    "test_release_status": "blocked_pending_temporal_audit",
    "methodological_note": (
        "Model selection is frozen, but temporal provenance "
        "of HMM-derived features must be verified before "
        "interpreting final test results as prospective "
        "predictive performance."
    )
}

manifest_file15 = (
    DIRS["manifests"]
    / "cell15_frozen_model_selection.json"
)

with open(manifest_file15, "w", encoding="utf-8") as f:
    _project_dump(
        manifest15,
        f,
        indent=4,
        ensure_ascii=False
    )


# ============================================================
# 10. BACKUP E RESUMO FINAL
# ============================================================

if "backup_mlflow_database" in globals():
    backup_mlflow_database()

_project_print("\n" + "=" * 70)
_project_print("FASE 4 — CÉLULA 15")
_project_print("=" * 70)

_project_print("Candidatos consolidados:", len(all_trials15))
_project_print("Combinações principais:", len(best12_15))
_project_print("Campeões por cenário:", len(champions15))
_project_print("Registros MLflow conferidos: 72")
_project_print("Teste avaliado: NÃO")

_project_print("\nManifesto:", manifest_file15)
_project_print("Matriz:", best_file15)
_project_print("Inventário dos modelos:", inventory_file15)

_project_print("\nSeleção estatística: CONGELADA")
_project_print("Auditoria temporal HMM: PENDENTE")
_project_print("Liberação para teste: BLOQUEADA")

_project_print("\nSTATUS: CONSOLIDAÇÃO CONCLUÍDA")
_project_print("=" * 70)


In [ ]:
# ============================================================
# FASE 4 — CÉLULA 15A
# AUDITORIA TEMPORAL E LIBERAÇÃO CONTROLADA DO TESTE
# ============================================================
#
# Objetivos:
# 1. Recuperar a seleção congelada na Célula 15.
# 2. Registrar a auditoria do notebook da Fase 2.
# 3. Verificar integridade dos modelos selecionados.
# 4. Preservar o manifesto original.
# 5. Criar autorização documentada para a Célula 16.
#
# NÃO treina modelos.
# NÃO modifica hiperparâmetros.
# NÃO acessa observações do teste.
#
# Protocolo: estimação contemporânea de CO2(t)
# usando geração e informações operacionais até t.
# ============================================================

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd


# ============================================================
# 1. RECUPERAR MANIFESTO CONGELADO
# ============================================================

manifest_path = (
    DIRS["manifests"]
    / "cell15_frozen_model_selection.json"
)

if not manifest_path.exists():
    raise FileNotFoundError(manifest_path)

with open(manifest_path, "r", encoding="utf-8") as f:
    frozen = json.load(f)

assert frozen["selection_status"] == "frozen_on_validation"
assert frozen["test_evaluated"] is False
assert frozen["n_candidates"] == 72

selected = frozen["selected_models"]

assert len(selected) == 3
assert {m["scenario"] for m in selected} == {
    "B0", "P1", "P2"
}

_project_print("Manifesto original recuperado.")
_project_print("Modelos selecionados:", len(selected))


# ============================================================
# 2. CONFERIR INTEGRIDADE DOS MODELOS
# ============================================================

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


integrity_records = []

for model in selected:

    path = Path(model["model_path"])

    if not path.exists():
        raise FileNotFoundError(path)

    actual_hash = sha256_file(path)
    expected_hash = model["model_sha256"]

    if actual_hash != expected_hash:
        raise RuntimeError(
            f"Modelo alterado após congelamento: {path}"
        )

    integrity_records.append({
        "scenario": model["scenario"],
        "algorithm": model["algorithm"],
        "validation_rmse": model["validation_rmse"],
        "sha256_verified": True,
        "model_path": str(path)
    })

integrity_df = pd.DataFrame(integrity_records)

_project_print("\nINTEGRIDADE DOS MODELOS")
_project_display(
    integrity_df[
        [
            "scenario",
            "algorithm",
            "validation_rmse",
            "sha256_verified"
        ]
    ].round(4)
)


# ============================================================
# 3. REGISTRAR EVIDÊNCIAS DA AUDITORIA DA FASE 2
# ============================================================
#
# Evidências obtidas por inspeção do código-fonte do
# notebook revisado 03_processos_operacionais.ipynb.
#
# A auditoria verifica a lógica implementada.
# Não constitui reexecução independente da Fase 2.
# ============================================================

audit_evidence = [
    {
        "component": "StandardScaler",
        "phase2_cell": "08",
        "finding": "fit somente com observacoes train",
        "status": "verified_in_source"
    },
    {
        "component": "Gaussian HMM",
        "phase2_cell": "09",
        "finding": "fit com sequencias e comprimentos train",
        "status": "verified_in_source"
    },
    {
        "component": "Inferencia HMM",
        "phase2_cell": "13",
        "finding": (
            "filtragem forward causal implementada; "
            "Viterbi mantido separadamente"
        ),
        "status": "verified_in_source"
    },
    {
        "component": "Teste de causalidade",
        "phase2_cell": "13",
        "finding": (
            "implementado teste de invariancia "
            "entre sequencia e prefixos"
        ),
        "status": "verified_in_source"
    },
    {
        "component": "Estados exportados",
        "phase2_cell": "30",
        "finding": (
            "hmm_state derivado de filtered_state; "
            "confianca e entropia da filtragem"
        ),
        "status": "verified_in_source"
    },
    {
        "component": "Defasagens",
        "phase2_cell": "30",
        "finding": (
            "load_lag e hmm_state_lag calculados "
            "por shift dentro dos ciclos"
        ),
        "status": "verified_in_source"
    },
    {
        "component": "Medias moveis",
        "phase2_cell": "30",
        "finding": (
            "rolling retrospectivo dentro dos ciclos; "
            "sem janela futura"
        ),
        "status": "verified_in_source"
    },
    {
        "component": "Duracoes operacionais",
        "phase2_cell": "30",
        "finding": (
            "contadores de tempo transcorrido "
            "nos estados e rampas"
        ),
        "status": "verified_in_source"
    }
]

audit_df = pd.DataFrame(audit_evidence)

_project_print("\nAUDITORIA DE PROVENIÊNCIA")
_project_display(audit_df)


# ============================================================
# 4. DEFINIR ESCOPO DAS CONCLUSÕES
# ============================================================

methodological_scope = {
    "task": "contemporaneous_hourly_co2_estimation",
    "target": "co2_metric_tons",
    "information_available": (
        "generation and operational measurements "
        "observed through hour t"
    ),
    "prospective_forecasting_before_hour_t": False,
    "hmm_parameters_fitted_on": "phase2_train_by_source_design",
    "phase2_vs_phase4_cutoffs": TEMPORAL_SOURCE_AUDIT,
    "foldwise_hmm_fit_verified": False,
    "hmm_inference": "forward_filtering",
    "test_partition_used_for_selection": False,
    "limitations": [
        (
            "Source-code audit, not independent "
            "reexecution of Phase 2."
        ),
        (
            "The temporal definition of complete "
            "operating cycles must be disclosed."
        ),
        (
            "The test was previously inspected in "
            "Phase 3; it is not a pristine holdout "
            "across the entire research project."
        ),
        (
            "Incremental gains from process mining "
            "must be separated from conventional "
            "load-history features by ablation."
        )
    ]
}


# ============================================================
# 5. CRIAR ADENDO SEM ALTERAR O ORIGINAL
# ============================================================

audit_report = {
    "phase": "phase_04_predictive_modeling",
    "cell": "15A",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "phase2_source_notebook": "notebooks/03_processos_operacionais.ipynb",
    "source_review_does_not_attest_runtime_artifact": True,
    "audit_type": "source_code_review",
    "audit_evidence": audit_evidence,
    "methodological_scope": methodological_scope,
    "selection_manifest": str(manifest_path),
    "selection_manifest_sha256": sha256_file(
        manifest_path
    ),
    "model_integrity": integrity_records,
    "test_evaluated_in_this_cell": False,
    "release_decision": "approved_with_qualifications",
    "release_scope": (
        "Evaluation of frozen B0, P1 and P2 "
        "models on the predefined test partition."
    ),
    "restrictions": [
        "Do not retune models using test results.",
        "Do not change selected hyperparameters.",
        "Do not modify the frozen feature sets.",
        "Report all three scenario champions.",
        "Interpret results as contemporaneous estimation."
    ]
}

report_path = (
    DIRS["reports"]
    / "cell15a_temporal_audit.json"
)

with open(report_path, "w", encoding="utf-8") as f:
    _project_dump(
        audit_report,
        f,
        indent=4,
        ensure_ascii=False
    )


# ============================================================
# 6. MANIFESTO DE LIBERAÇÃO
# ============================================================

release_manifest = {
    "phase": "phase_04_predictive_modeling",
    "cell": "15A",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "original_selection_manifest": str(
        manifest_path
    ),
    "original_selection_sha256": sha256_file(
        manifest_path
    ),
    "audit_report": str(report_path),
    "audit_report_sha256": sha256_file(
        report_path
    ),
    "selected_models": selected,
    "test_release_status": "approved_with_qualifications",
    "test_evaluated": False,
    "selection_frozen": True,
    "permitted_next_step": "cell16_final_test"
}

release_path = (
    DIRS["manifests"]
    / "cell15a_test_release.json"
)

with open(release_path, "w", encoding="utf-8") as f:
    _project_dump(
        release_manifest,
        f,
        indent=4,
        ensure_ascii=False
    )


# ============================================================
# 7. VERIFICAÇÕES FINAIS
# ============================================================

assert len(audit_df) == 8
assert integrity_df["sha256_verified"].all()
assert release_path.exists()
assert report_path.exists()

# Confirmar que o manifesto original permanece íntegro.
assert (
    sha256_file(manifest_path)
    == release_manifest["original_selection_sha256"]
)

_project_print("\n" + "=" * 70)
_project_print("FASE 4 — CÉLULA 15A")
_project_print("=" * 70)

_project_print("Auditoria do código da Fase 2: REGISTRADA")
_project_print("Integridade dos modelos: VERIFICADA")
_project_print("Seleção estatística: PRESERVADA")
_project_print("Teste avaliado nesta célula: NÃO")

_project_print("\nRelatório:", report_path)
_project_print("Manifesto de liberação:", release_path)

_project_print("\nSTATUS: TESTE LIBERADO COM RESSALVAS")
_project_print("=" * 70)

In [ ]:

# ============================================================
# FASE 4 — CÉLULA 16
# AVALIAÇÃO FINAL DOS MODELOS CONGELADOS
# ============================================================
#
# Objetivos:
# 1. Validar a autorização da Célula 15A.
# 2. Recuperar os três modelos congelados.
# 3. Avaliar B0, P1 e P2 nas mesmas horas de teste.
# 4. Calcular RMSE, MAE, R² e viés.
# 5. Comparar os resultados com a validação.
# 6. Salvar previsões, métricas, gráficos e relatório.
#
# NÃO ajusta modelos.
# NÃO altera hiperparâmetros.
# NÃO seleciona um novo modelo com base no teste.
#
# Unidade do alvo: toneladas métricas de CO2 por hora.
# ============================================================

import json
import hashlib
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)


# ============================================================
# 1. FUNÇÕES DE INTEGRIDADE E MÉTRICAS
# ============================================================

def cell16_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()


def cell16_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    if not np.isfinite(y_true).all():
        raise ValueError("Alvo contém valores não finitos.")

    if not np.isfinite(y_pred).all():
        raise ValueError("Previsões contêm valores não finitos.")

    return {
        "rmse": float(
            np.sqrt(mean_squared_error(y_true, y_pred))
        ),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
        # Viés positivo = modelo superestima emissões.
        "bias": float(np.mean(y_pred - y_true)),
    }


# ============================================================
# 2. VALIDAR MANIFESTOS E AUTORIZAÇÃO
# ============================================================

release_file16 = (
    DIRS["manifests"] / "cell15a_test_release.json"
)

if not release_file16.exists():
    raise FileNotFoundError(
        "Execute a Célula 15A antes da avaliação final."
    )

with open(release_file16, encoding="utf-8") as f:
    release16 = json.load(f)

assert (
    release16["test_release_status"]
    == "approved_with_qualifications"
)
assert release16["selection_frozen"] is True
assert release16["test_evaluated"] is False

frozen_file16 = Path(
    release16["original_selection_manifest"]
)

assert (
    cell16_sha256(frozen_file16)
    == release16["original_selection_sha256"]
), "O manifesto original foi modificado."

audit_file16 = Path(release16["audit_report"])

assert (
    cell16_sha256(audit_file16)
    == release16["audit_report_sha256"]
), "O relatório de auditoria foi modificado."

with open(frozen_file16, encoding="utf-8") as f:
    frozen16 = json.load(f)

selected16 = frozen16["selected_models"]

assert len(selected16) == 3
assert {m["scenario"] for m in selected16} == {
    "B0", "P1", "P2"
}

# Conferir se a autorização preserva a mesma seleção.
for model in selected16:
    match = [
        x for x in release16["selected_models"]
        if x["scenario"] == model["scenario"]
    ]
    assert len(match) == 1
    assert match[0] == model

_project_print("Autorização: VALIDADA")
_project_print("Seleção congelada: PRESERVADA")


# ============================================================
# 3. RECUPERAR O CONJUNTO DE TESTE
# ============================================================

if "split_data" not in globals():
    raise RuntimeError("Execute a Célula 07.")

if "processed_data" not in globals():
    raise RuntimeError("Execute a Célula 08.")

scenario_order16 = ["B0", "P1", "P2"]

reference16 = split_data["B0"]["test"]

y_test16 = np.asarray(
    reference16["y"],
    dtype=float
).reshape(-1)

timestamps16 = pd.to_datetime(
    reference16["timestamps"],
    utc=True
)

timestamps16 = pd.DatetimeIndex(timestamps16)

assert len(y_test16) == 1063
assert len(timestamps16) == 1063
assert timestamps16.is_unique
assert timestamps16.is_monotonic_increasing
assert np.isfinite(y_test16).all()

for scenario in scenario_order16:

    part = split_data[scenario]["test"]

    current_y = np.asarray(
        part["y"], dtype=float
    ).reshape(-1)

    current_timestamps = pd.DatetimeIndex(
        pd.to_datetime(
            part["timestamps"], utc=True
        )
    )

    np.testing.assert_array_equal(
        current_y, y_test16
    )

    assert current_timestamps.equals(timestamps16)

_project_print("Observações de teste:", len(y_test16))
_project_print("Primeira hora:", timestamps16.min())
_project_print("Última hora:", timestamps16.max())
_project_print("Alvos e timestamps entre cenários: IDÊNTICOS")


# ============================================================
# 4. CARREGAR MODELOS E GERAR PREVISÕES
# ============================================================

test_records16 = []
prediction_columns16 = {
    "timestamp": timestamps16,
    "partition": ["test"] * len(y_test16),
    "y_true": y_test16
}

loaded_models16 = {}

expected_features16 = {
    "B0": 1,
    "P1": 6,
    "P2": 35
}

for selected_model in selected16:

    scenario = selected_model["scenario"]
    algorithm = selected_model["algorithm"]

    model_path = Path(
        selected_model["model_path"]
    )

    if cell16_sha256(model_path) != (
        selected_model["model_sha256"]
    ):
        raise RuntimeError(
            f"Integridade comprometida: {scenario}"
        )

    model = joblib.load(model_path)

    # Usar exatamente o pré-processamento da Célula 08.
    # Não realizar fit ou fit_transform no teste.
    X_test = processed_data[
        scenario
    ][algorithm]["test"]

    assert X_test.shape == (
        1063,
        expected_features16[scenario]
    )

    if hasattr(X_test, "to_numpy"):
        numeric_X = X_test.to_numpy(dtype=float)
    else:
        numeric_X = np.asarray(X_test, dtype=float)

    assert np.isfinite(numeric_X).all()

    predictions = np.asarray(
        model.predict(X_test),
        dtype=float
    ).reshape(-1)

    assert len(predictions) == len(y_test16)

    metrics = cell16_metrics(
        y_test16, predictions
    )

    prediction_columns16[
        f"pred_{scenario}"
    ] = predictions

    prediction_columns16[
        f"error_{scenario}"
    ] = predictions - y_test16

    test_records16.append({
        "scenario": scenario,
        "algorithm": algorithm,
        "candidate_index": int(
            selected_model["candidate_index"]
        ),
        "n": len(y_test16),
        "partition": "test",
        **metrics,
        "validation_rmse": float(
            selected_model["validation_rmse"]
        ),
        "validation_mae": float(
            selected_model["validation_mae"]
        ),
        "validation_bias": float(
            selected_model["validation_bias"]
        ),
        "run_id": selected_model["run_id"],
        "model_sha256": selected_model["model_sha256"]
    })

    loaded_models16[scenario] = model

    _project_print(
        f"{scenario} — {algorithm}: "
        f"RMSE={metrics['rmse']:.4f} | "
        f"MAE={metrics['mae']:.4f} | "
        f"R²={metrics['r2']:.4f} | "
        f"Viés={metrics['bias']:+.4f}"
    )


# ============================================================
# 5. CONSOLIDAR RESULTADOS
# ============================================================

test_metrics16 = pd.DataFrame(test_records16)

test_metrics16 = (
    test_metrics16
    .set_index("scenario")
    .loc[scenario_order16]
    .reset_index()
)

predictions_df16 = pd.DataFrame(
    prediction_columns16
)

assert len(test_metrics16) == 3
assert len(predictions_df16) == 1063

test_metrics16["rmse_change_validation_to_test_pct"] = (
    100
    * (
        test_metrics16["rmse"]
        - test_metrics16["validation_rmse"]
    )
    / test_metrics16["validation_rmse"]
)

_project_print("\nDESEMPENHO FINAL — TESTE")
_project_display(
    test_metrics16[
        [
            "scenario", "algorithm", "n",
            "rmse", "mae", "r2", "bias",
            "validation_rmse",
            "rmse_change_validation_to_test_pct"
        ]
    ].round(4)
)


# ============================================================
# 6. COMPARAÇÃO PAREADA ENTRE CENÁRIOS
# ============================================================

rmse_by_scenario16 = (
    test_metrics16.set_index("scenario")["rmse"]
)

mae_by_scenario16 = (
    test_metrics16.set_index("scenario")["mae"]
)

comparisons16 = []

for reference, candidate in [
    ("B0", "P1"),
    ("B0", "P2"),
    ("P1", "P2")
]:

    rmse_ref = float(rmse_by_scenario16[reference])
    rmse_candidate = float(
        rmse_by_scenario16[candidate]
    )

    mae_ref = float(mae_by_scenario16[reference])
    mae_candidate = float(
        mae_by_scenario16[candidate]
    )

    comparisons16.append({
        "reference": reference,
        "candidate": candidate,
        "rmse_reference": rmse_ref,
        "rmse_candidate": rmse_candidate,
        "rmse_reduction_absolute": (
            rmse_ref - rmse_candidate
        ),
        "rmse_reduction_pct": (
            100 * (rmse_ref - rmse_candidate) / rmse_ref
        ),
        "mae_reduction_pct": (
            100 * (mae_ref - mae_candidate) / mae_ref
        )
    })

comparison_df16 = pd.DataFrame(comparisons16)

_project_print("\nCOMPARAÇÕES PAREADAS — TESTE")
_project_display(comparison_df16.round(4))


# ============================================================
# 7. GRÁFICO — VALIDAÇÃO VERSUS TESTE
# ============================================================

x16 = np.arange(3)
width16 = 0.35

fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(
    x16 - width16 / 2,
    test_metrics16["validation_rmse"],
    width16,
    label="Validação"
)

ax.bar(
    x16 + width16 / 2,
    test_metrics16["rmse"],
    width16,
    label="Teste"
)

ax.set_xticks(x16)
ax.set_xticklabels([
    f"{row.scenario}\n{row.algorithm}"
    for row in test_metrics16.itertuples()
])

ax.set_ylabel("RMSE (t CO₂/h)")
ax.set_title(
    "Iatan 2 — Desempenho na validação e no teste"
)
ax.legend()
ax.grid(axis="y", alpha=0.2)

fig.tight_layout()

fig_metrics16 = (
    DIRS["figures"]
    / "cell16_validation_vs_test_rmse.png"
)

fig.savefig(
    fig_metrics16, dpi=180, bbox_inches="tight"
)

plt.show()


# ============================================================
# 8. GRÁFICO — EMISSÕES OBSERVADAS E ESTIMADAS
# ============================================================

fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(
    predictions_df16["timestamp"],
    predictions_df16["y_true"],
    label="CO₂ observado",
    linewidth=1.7,
    color="black"
)

for scenario in scenario_order16:
    ax.plot(
        predictions_df16["timestamp"],
        predictions_df16[f"pred_{scenario}"],
        label=f"{scenario} estimado",
        linewidth=1.0,
        alpha=0.75
    )

ax.set_title(
    "Iatan 2 — Emissões observadas e estimadas no teste"
)
ax.set_xlabel("Data")
ax.set_ylabel("Emissões de CO₂ (t/h)")
ax.legend()
ax.grid(alpha=0.2)

fig.autofmt_xdate()
fig.tight_layout()

fig_series16 = (
    DIRS["figures"]
    / "cell16_test_observed_vs_predicted.png"
)

fig.savefig(
    fig_series16, dpi=180, bbox_inches="tight"
)

plt.show()


# ============================================================
# 9. GRÁFICO — DISTRIBUIÇÃO DOS ERROS
# ============================================================

fig, ax = plt.subplots(figsize=(10, 5))

error_data16 = [
    predictions_df16[f"error_{scenario}"].to_numpy()
    for scenario in scenario_order16
]

ax.boxplot(
    error_data16,
    tick_labels=scenario_order16,
    showfliers=False
)

ax.axhline(
    0,
    color="black",
    linestyle="--",
    linewidth=1
)

ax.set_ylabel("Erro: estimado − observado (t CO₂/h)")
ax.set_title("Iatan 2 — Distribuição dos erros no teste")
ax.grid(axis="y", alpha=0.2)

fig.tight_layout()

fig_errors16 = (
    DIRS["figures"]
    / "cell16_test_error_boxplot.png"
)

fig.savefig(
    fig_errors16, dpi=180, bbox_inches="tight"
)

plt.show()


# ============================================================
# 10. EXPORTAÇÃO DOS RESULTADOS
# ============================================================

metrics_file16 = (
    DIRS["tables"]
    / "cell16_final_test_metrics.csv"
)

comparison_file16 = (
    DIRS["tables"]
    / "cell16_final_test_comparisons.csv"
)

predictions_file16 = (
    DIRS["processed"]
    / "cell16_final_test_predictions.parquet"
)

report_file16 = (
    DIRS["reports"]
    / "cell16_final_test_report.json"
)

_project_csv(test_metrics16, 
    metrics_file16, index=False
)

_project_csv(comparison_df16, 
    comparison_file16, index=False
)

predictions_df16.to_parquet(
    predictions_file16, index=False
)

report16 = {
    "phase": "phase_04_predictive_modeling",
    "cell": "16",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "task": "contemporaneous_hourly_co2_estimation",
    "test_rows": int(len(y_test16)),
    "test_start": timestamps16.min().isoformat(),
    "test_end": timestamps16.max().isoformat(),
    "selection_frozen_before_test": True,
    "models_retrained": False,
    "hyperparameters_changed": False,
    "test_used_for_model_selection": False,
    "source_release_manifest": str(release_file16),
    "model_results": test_metrics16.to_dict(
        orient="records"
    ),
    "paired_comparisons": comparison_df16.to_dict(
        orient="records"
    ),
    "limitations": [
        (
            "The test partition had previously been "
            "inspected in Phase 3."
        ),
        (
            "Results represent contemporaneous "
            "estimation, not forecasting before hour t."
        ),
        (
            "The incremental contribution of process "
            "features versus load history remains "
            "to be isolated by ablation."
        ),
        (
            "Statistical uncertainty of paired "
            "performance differences remains "
            "to be evaluated."
        )
    ],
    "files": {
        "metrics": str(metrics_file16),
        "comparisons": str(comparison_file16),
        "predictions": str(predictions_file16),
        "figure_metrics": str(fig_metrics16),
        "figure_series": str(fig_series16),
        "figure_errors": str(fig_errors16)
    }
}

with open(report_file16, "w", encoding="utf-8") as f:
    _project_dump(
        report16,
        f,
        indent=4,
        ensure_ascii=False
    )


# ============================================================
# 11. VERIFICAÇÕES FINAIS
# ============================================================

assert len(test_metrics16) == 3
assert len(comparison_df16) == 3
assert len(predictions_df16) == 1063

assert predictions_df16["timestamp"].is_unique

assert np.isfinite(
    test_metrics16[
        ["rmse", "mae", "r2", "bias"]
    ].to_numpy(dtype=float)
).all()

for scenario in scenario_order16:
    assert np.isfinite(
        predictions_df16[f"pred_{scenario}"]
    ).all()

_project_print("\n" + "=" * 70)
_project_print("FASE 4 — CÉLULA 16")
_project_print("=" * 70)

_project_print("Modelos avaliados:", len(test_metrics16))
_project_print("Observações por modelo:", len(y_test16))
_project_print("Modelos reajustados: 0")
_project_print("Hiperparâmetros alterados: 0")
_project_print("Comparações pareadas:", len(comparison_df16))

_project_print("\nMétricas:", metrics_file16)
_project_print("Comparações:", comparison_file16)
_project_print("Previsões:", predictions_file16)
_project_print("Relatório:", report_file16)

_project_print("\nSTATUS: AVALIAÇÃO FINAL CONCLUÍDA")
_project_print("=" * 70)


In [ ]:

# ============================================================
# FASE 4 — CÉLULA 17
# COMPARAÇÃO PAREADA DOS ERROS NO TESTE
# ============================================================
#
# Objetivos:
# 1. Comparar erros nas mesmas 1.063 horas.
# 2. Quantificar quantas horas cada modelo vence.
# 3. Identificar períodos de maior deterioração.
# 4. Produzir gráficos de diferenças pareadas.
# 5. Exportar resultados para análises posteriores.
#
# NÃO treina, seleciona ou reajusta modelos.
# Inferência estatística com bootstrap: Célula 21.
# ============================================================

import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 1. CARREGAR PREVISÕES CONGELADAS
# ============================================================

source17 = (
    DIRS["processed"]
    / "cell16_final_test_predictions.parquet"
)

if not source17.exists():
    raise FileNotFoundError(source17)

df17 = pd.read_parquet(source17)

df17["timestamp"] = pd.to_datetime(
    df17["timestamp"], utc=True
)

required17 = [
    "timestamp", "partition", "y_true",
    "pred_B0", "pred_P1", "pred_P2"
]

assert set(required17).issubset(df17.columns)
assert len(df17) == 1063
assert df17["timestamp"].is_unique
assert df17["timestamp"].is_monotonic_increasing
assert (df17["partition"] == "test").all()

assert np.isfinite(
    df17[
        ["y_true", "pred_B0", "pred_P1", "pred_P2"]
    ].to_numpy(dtype=float)
).all()

scenarios17 = ["B0", "P1", "P2"]

for scenario in scenarios17:
    df17[f"residual_{scenario}"] = (
        df17["y_true"] - df17[f"pred_{scenario}"]
    )
    df17[f"ae_{scenario}"] = (
        df17[f"residual_{scenario}"].abs()
    )
    df17[f"se_{scenario}"] = (
        df17[f"residual_{scenario}"] ** 2
    )


# ============================================================
# 2. DIFERENÇAS PAREADAS
# ============================================================
#
# Diferença positiva:
# o candidato apresenta erro MENOR que a referência.
#
# dAE = AE_referencia - AE_candidato
# dSE = SE_referencia - SE_candidato
# ============================================================

pairs17 = [
    ("B0", "P1"),
    ("B0", "P2"),
    ("P1", "P2")
]

records17 = []

for reference, candidate in pairs17:

    d_ae = (
        df17[f"ae_{reference}"]
        - df17[f"ae_{candidate}"]
    )

    d_se = (
        df17[f"se_{reference}"]
        - df17[f"se_{candidate}"]
    )

    rmse_reference = np.sqrt(
        df17[f"se_{reference}"].mean()
    )

    rmse_candidate = np.sqrt(
        df17[f"se_{candidate}"].mean()
    )

    records17.append({
        "reference": reference,
        "candidate": candidate,
        "n": len(df17),
        "candidate_better_hours": int((d_ae > 0).sum()),
        "reference_better_hours": int((d_ae < 0).sum()),
        "tied_hours": int((d_ae == 0).sum()),
        "candidate_better_pct": float(
            100 * (d_ae > 0).mean()
        ),
        "mean_ae_improvement": float(d_ae.mean()),
        "median_ae_improvement": float(d_ae.median()),
        "mean_squared_error_improvement": float(
            d_se.mean()
        ),
        "rmse_reference": float(rmse_reference),
        "rmse_candidate": float(rmse_candidate),
        "rmse_reduction_pct": float(
            100 * (rmse_reference - rmse_candidate)
            / rmse_reference
        )
    })

    df17[f"dAE_{candidate}_vs_{reference}"] = d_ae
    df17[f"dSE_{candidate}_vs_{reference}"] = d_se

summary17 = pd.DataFrame(records17)

_project_print("COMPARAÇÃO PAREADA — 1.063 HORAS")
_project_display(summary17.round(4))


# ============================================================
# 3. DESEMPENHO POR DIA
# ============================================================
#
# Apenas dias com observações elegíveis.
# Dias com poucas horas não são diretamente
# comparáveis aos dias com cobertura completa.
# ============================================================

df17["date"] = df17["timestamp"].dt.date

daily17 = (
    df17.groupby("date", as_index=False)
    .agg(
        n=("y_true", "size"),
        rmse_B0=("se_B0", lambda x: np.sqrt(x.mean())),
        rmse_P1=("se_P1", lambda x: np.sqrt(x.mean())),
        rmse_P2=("se_P2", lambda x: np.sqrt(x.mean())),
        mae_B0=("ae_B0", "mean"),
        mae_P1=("ae_P1", "mean"),
        mae_P2=("ae_P2", "mean")
    )
)

daily17["rmse_gain_P2_vs_B0_pct"] = (
    100
    * (daily17["rmse_B0"] - daily17["rmse_P2"])
    / daily17["rmse_B0"].replace(0, np.nan)
)

daily17["rmse_gain_P1_vs_B0_pct"] = (
    100
    * (daily17["rmse_B0"] - daily17["rmse_P1"])
    / daily17["rmse_B0"].replace(0, np.nan)
)

_project_print("\nDIAS COM MAIOR DETERIORAÇÃO DE P2")
_project_display(
    daily17.sort_values(
        "rmse_gain_P2_vs_B0_pct"
    ).head(10).round(3)
)


# ============================================================
# 4. GRÁFICO — DIFERENÇAS ABSOLUTAS POR HORA
# ============================================================

fig, axes = plt.subplots(
    2, 1, figsize=(13, 8), sharex=True
)

for ax, candidate in zip(axes, ["P1", "P2"]):

    column = f"dAE_{candidate}_vs_B0"

    ax.plot(
        df17["timestamp"],
        df17[column],
        linewidth=0.8
    )

    ax.axhline(
        0, color="black", linestyle="--",
        linewidth=1
    )

    ax.set_ylabel("Ganho no erro absoluto (t CO₂/h)")
    ax.set_title(
        f"{candidate} versus B0 — positivo favorece {candidate}"
    )
    ax.grid(alpha=0.2)

axes[-1].set_xlabel("Data")
fig.autofmt_xdate()
fig.tight_layout()

fig_hourly17 = (
    DIRS["figures"]
    / "cell17_paired_hourly_absolute_errors.png"
)

fig.savefig(fig_hourly17, dpi=180, bbox_inches="tight")
plt.show()


# ============================================================
# 5. GRÁFICO — RMSE DIÁRIO
# ============================================================

fig, ax = plt.subplots(figsize=(13, 5))

for scenario in scenarios17:
    ax.plot(
        pd.to_datetime(daily17["date"]),
        daily17[f"rmse_{scenario}"],
        marker=".",
        linewidth=1.1,
        label=scenario
    )

ax.set_title("Iatan 2 — RMSE diário no teste")
ax.set_ylabel("RMSE (t CO₂/h)")
ax.set_xlabel("Data")
ax.legend()
ax.grid(alpha=0.2)

fig.autofmt_xdate()
fig.tight_layout()

fig_daily17 = (
    DIRS["figures"]
    / "cell17_daily_rmse.png"
)

fig.savefig(fig_daily17, dpi=180, bbox_inches="tight")
plt.show()


# ============================================================
# 6. GRÁFICO — DISTRIBUIÇÃO DAS DIFERENÇAS
# ============================================================

fig, ax = plt.subplots(figsize=(10, 5))

columns17 = [
    "dAE_P1_vs_B0",
    "dAE_P2_vs_B0",
    "dAE_P2_vs_P1"
]

ax.boxplot(
    [df17[c].to_numpy() for c in columns17],
    tick_labels=[
        "P1 − B0",
        "P2 − B0",
        "P2 − P1"
    ],
    showfliers=False
)

ax.axhline(
    0, color="black", linestyle="--"
)

ax.set_title(
    "Iatan 2 — Ganho pareado no erro absoluto"
)
ax.set_ylabel(
    "AE da referência − AE do candidato (t CO₂/h)"
)
ax.grid(axis="y", alpha=0.2)

fig.tight_layout()

fig_box17 = (
    DIRS["figures"]
    / "cell17_paired_error_differences.png"
)

fig.savefig(fig_box17, dpi=180, bbox_inches="tight")
plt.show()


# ============================================================
# 7. EXPORTAR RESULTADOS
# ============================================================

paired_file17 = (
    DIRS["processed"]
    / "cell17_paired_hourly_errors.parquet"
)

summary_file17 = (
    DIRS["tables"]
    / "cell17_paired_comparison.csv"
)

daily_file17 = (
    DIRS["tables"]
    / "cell17_daily_performance.csv"
)

report_file17 = (
    DIRS["reports"]
    / "cell17_paired_error_report.json"
)

df17.to_parquet(paired_file17, index=False)
_project_csv(summary17, summary_file17, index=False)
_project_csv(daily17, daily_file17, index=False)

report17 = {
    "phase": "phase_04_predictive_modeling",
    "cell": "17",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "test_observations": int(len(df17)),
    "days_with_test_observations": int(len(daily17)),
    "comparison_definition": (
        "Positive paired difference means "
        "the candidate has smaller error."
    ),
    "paired_results": summary17.to_dict(
        orient="records"
    ),
    "statistical_significance_tested": False,
    "models_retrained": False,
    "limitations": [
        "Hourly observations are temporally dependent.",
        "Win percentages are descriptive, not significance tests.",
        "Daily comparisons have unequal eligible-hour coverage.",
        "B0/P1 and P2 use different champion algorithms.",
        "Block-bootstrap inference is deferred to Cell 21."
    ],
    "files": {
        "paired_errors": str(paired_file17),
        "summary": str(summary_file17),
        "daily": str(daily_file17),
        "hourly_figure": str(fig_hourly17),
        "daily_figure": str(fig_daily17),
        "boxplot_figure": str(fig_box17)
    }
}

with open(report_file17, "w", encoding="utf-8") as f:
    _project_dump(
        report17, f, indent=4, ensure_ascii=False
    )

_project_print("\n" + "=" * 70)
_project_print("FASE 4 — CÉLULA 17")
_project_print("=" * 70)

_project_print("Horas comparadas:", len(df17))
_project_print("Dias analisados:", len(daily17))
_project_print("Comparações pareadas:", len(summary17))
_project_print("Modelos reajustados: 0")

_project_print("\nResumo:", summary_file17)
_project_print("Desempenho diário:", daily_file17)
_project_print("Erros horários:", paired_file17)
_project_print("Relatório:", report_file17)

_project_print("\nSTATUS: COMPARAÇÃO PAREADA CONCLUÍDA")
_project_print("=" * 70)


In [ ]:

# ============================================================
# FASE 4 — CÉLULA 18
# ERROS POR CARGA E REGIME OPERACIONAL
# ============================================================
#
# Investigar:
# 1. Desempenho por faixa de geração.
# 2. Desempenho por estado HMM.
# 3. Distribuição das condições no teste.
# 4. Regiões onde P2 melhora ou piora.
#
# Análise descritiva: sem testes de significância.
# ============================================================

import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# 1. RECUPERAR PREVISÕES E ATRIBUTOS OPERACIONAIS
# ------------------------------------------------------------

pred18 = pd.read_parquet(
    DIRS["processed"]
    / "cell16_final_test_predictions.parquet"
)

master18 = pd.read_parquet(
    DIRS["processed"]
    / "cell04_master_dataset.parquet"
)

pred18["timestamp"] = pd.to_datetime(
    pred18["timestamp"], utc=True
)

master18["timestamp"] = pd.to_datetime(
    master18["timestamp"], utc=True
)

operational_columns18 = [
    "timestamp",
    "generation_mw",
    "hmm_state",
    "state_changed",
    "delta_load_mw",
    "hours_in_current_state"
]

missing18 = (
    set(operational_columns18) - set(master18.columns)
)
if missing18:
    raise ValueError(
        f"Atributos operacionais ausentes: {missing18}"
    )

df18 = pred18.merge(
    master18[operational_columns18],
    on="timestamp",
    how="left",
    validate="one_to_one",
    indicator=True
)

assert len(df18) == 1063
assert (df18["_merge"] == "both").all()
df18 = df18.drop(columns="_merge")

assert df18["generation_mw"].notna().all()
assert df18["hmm_state"].notna().all()

df18["hmm_state"] = (
    df18["hmm_state"].astype(int)
)

assert df18["hmm_state"].between(0, 4).all()

scenarios18 = ["B0", "P1", "P2"]

for scenario in scenarios18:
    error = (
        df18[f"pred_{scenario}"] - df18["y_true"]
    )
    df18[f"error_{scenario}"] = error
    df18[f"ae_{scenario}"] = error.abs()
    df18[f"se_{scenario}"] = error ** 2

_project_print("Observações integradas:", len(df18))


# ------------------------------------------------------------
# 2. FUNÇÃO DE MÉTRICAS POR GRUPO
# ------------------------------------------------------------

def grouped_metrics18(data, group_column):
    records = []

    for group_value, group in data.groupby(
        group_column, observed=True, dropna=False
    ):
        record = {
            group_column: group_value,
            "n": len(group),
            "generation_mean_mw": float(
                group["generation_mw"].mean()
            ),
            "co2_mean_tph": float(
                group["y_true"].mean()
            )
        }

        for scenario in scenarios18:
            record[f"rmse_{scenario}"] = float(
                np.sqrt(group[f"se_{scenario}"].mean())
            )
            record[f"mae_{scenario}"] = float(
                group[f"ae_{scenario}"].mean()
            )
            record[f"bias_{scenario}"] = float(
                group[f"error_{scenario}"].mean()
            )

        record["rmse_gain_P2_vs_B0_pct"] = (
            100
            * (
                record["rmse_B0"]
                - record["rmse_P2"]
            )
            / record["rmse_B0"]
            if record["rmse_B0"] > 0
            else np.nan
        )

        records.append(record)

    return pd.DataFrame(records)


# ------------------------------------------------------------
# 3. FAIXAS FIXAS DE GERAÇÃO
# ------------------------------------------------------------
#
# Faixas definidas para análise descritiva.
# Não são utilizadas para otimizar modelos.
# ------------------------------------------------------------

load_edges18 = [
    -np.inf, 100, 300, 500, 700, 850, np.inf
]

load_labels18 = [
    "≤100",
    "100–300",
    "300–500",
    "500–700",
    "700–850",
    ">850"
]

df18["load_band"] = pd.cut(
    df18["generation_mw"],
    bins=load_edges18,
    labels=load_labels18,
    include_lowest=True
)

load_metrics18 = grouped_metrics18(
    df18, "load_band"
)

_project_print("\nDESEMPENHO POR FAIXA DE GERAÇÃO")
_project_display(load_metrics18.round(3))


# ------------------------------------------------------------
# 4. DESEMPENHO POR ESTADO HMM
# ------------------------------------------------------------

state_metrics18 = grouped_metrics18(
    df18, "hmm_state"
).sort_values("hmm_state")

_project_print("\nDESEMPENHO POR REGIME HMM")
_project_display(state_metrics18.round(3))


# ------------------------------------------------------------
# 5. DISTRIBUIÇÃO DOS REGIMES
# ------------------------------------------------------------

state_distribution18 = (
    df18["hmm_state"]
    .value_counts()
    .sort_index()
    .rename_axis("hmm_state")
    .reset_index(name="n")
)

state_distribution18["percentage"] = (
    100 * state_distribution18["n"] / len(df18)
)

_project_print("\nDISTRIBUIÇÃO DOS REGIMES NO TESTE")
_project_display(state_distribution18.round(2))


# ------------------------------------------------------------
# 6. GRÁFICO — RMSE POR FAIXA DE GERAÇÃO
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(11, 5))

x18 = np.arange(len(load_metrics18))
width18 = 0.25

for i, scenario in enumerate(scenarios18):
    ax.bar(
        x18 + (i - 1) * width18,
        load_metrics18[f"rmse_{scenario}"],
        width18,
        label=scenario
    )

ax.set_xticks(x18)
ax.set_xticklabels(
    load_metrics18["load_band"].astype(str)
)

ax.set_xlabel("Geração (MW)")
ax.set_ylabel("RMSE (t CO₂/h)")
ax.set_title("Iatan 2 — RMSE por faixa de geração")
ax.legend()
ax.grid(axis="y", alpha=0.2)

fig.tight_layout()

fig_load18 = (
    DIRS["figures"]
    / "cell18_rmse_by_load_band.png"
)

fig.savefig(fig_load18, dpi=180, bbox_inches="tight")
plt.show()


# ------------------------------------------------------------
# 7. GRÁFICO — RMSE POR REGIME HMM
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 5))

x18 = np.arange(len(state_metrics18))

for i, scenario in enumerate(scenarios18):
    ax.bar(
        x18 + (i - 1) * width18,
        state_metrics18[f"rmse_{scenario}"],
        width18,
        label=scenario
    )

ax.set_xticks(x18)
ax.set_xticklabels(
    state_metrics18["hmm_state"].astype(str)
)

ax.set_xlabel("Estado HMM")
ax.set_ylabel("RMSE (t CO₂/h)")
ax.set_title("Iatan 2 — RMSE por regime operacional")
ax.legend()
ax.grid(axis="y", alpha=0.2)

fig.tight_layout()

fig_state18 = (
    DIRS["figures"]
    / "cell18_rmse_by_hmm_state.png"
)

fig.savefig(fig_state18, dpi=180, bbox_inches="tight")
plt.show()


# ------------------------------------------------------------
# 8. GRÁFICO — GANHO P2 VERSUS B0 POR REGIME
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(
    state_metrics18["hmm_state"].astype(str),
    state_metrics18["rmse_gain_P2_vs_B0_pct"]
)

ax.axhline(
    0, color="black", linestyle="--", linewidth=1
)

ax.set_xlabel("Estado HMM")
ax.set_ylabel("Redução do RMSE de P2 versus B0 (%)")
ax.set_title(
    "Iatan 2 — Contribuição de P2 por regime"
)
ax.grid(axis="y", alpha=0.2)

fig.tight_layout()

fig_gain18 = (
    DIRS["figures"]
    / "cell18_p2_gain_by_hmm_state.png"
)

fig.savefig(fig_gain18, dpi=180, bbox_inches="tight")
plt.show()


# ------------------------------------------------------------
# 9. EXPORTAR TABELAS E RELATÓRIO
# ------------------------------------------------------------

load_file18 = (
    DIRS["tables"]
    / "cell18_performance_by_load_band.csv"
)

state_file18 = (
    DIRS["tables"]
    / "cell18_performance_by_hmm_state.csv"
)

distribution_file18 = (
    DIRS["tables"]
    / "cell18_hmm_state_distribution.csv"
)

detail_file18 = (
    DIRS["processed"]
    / "cell18_operational_test_errors.parquet"
)

report_file18 = (
    DIRS["reports"]
    / "cell18_operational_error_analysis.json"
)

_project_csv(load_metrics18, load_file18, index=False)
_project_csv(state_metrics18, state_file18, index=False)
_project_csv(state_distribution18, 
    distribution_file18, index=False
)
df18.to_parquet(detail_file18, index=False)

report18 = {
    "phase": "phase_04_predictive_modeling",
    "cell": "18",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "test_rows": int(len(df18)),
    "n_hmm_states": int(df18["hmm_state"].nunique()),
    "n_load_bands_present": int(
        df18["load_band"].nunique()
    ),
    "analysis_type": "descriptive_post_test",
    "models_retrained": False,
    "hyperparameters_changed": False,
    "statistical_significance_tested": False,
    "limitations": [
        "Operational subgroup analysis is exploratory.",
        "Subgroups may have different sample sizes.",
        "HMM state labels are categorical, not ordinal.",
        "Differences combine model and feature effects.",
        "Subgroup comparisons do not establish causality."
    ],
    "files": {
        "load_metrics": str(load_file18),
        "state_metrics": str(state_file18),
        "state_distribution": str(distribution_file18),
        "hourly_details": str(detail_file18),
        "figure_load": str(fig_load18),
        "figure_state": str(fig_state18),
        "figure_gain": str(fig_gain18)
    }
}

with open(report_file18, "w", encoding="utf-8") as f:
    _project_dump(
        report18,
        f,
        indent=4,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 10. RESUMO
# ------------------------------------------------------------

_project_print("\n" + "=" * 70)
_project_print("FASE 4 — CÉLULA 18")
_project_print("=" * 70)

_project_print("Horas analisadas:", len(df18))
_project_print("Regimes HMM:", df18["hmm_state"].nunique())
_project_print("Faixas de geração:", df18["load_band"].nunique())
_project_print("Modelos reajustados: 0")

_project_print("\nTabela por carga:", load_file18)
_project_print("Tabela por regime:", state_file18)
_project_print("Distribuição HMM:", distribution_file18)
_project_print("Relatório:", report_file18)

_project_print("\nSTATUS: ANÁLISE OPERACIONAL CONCLUÍDA")
_project_print("=" * 70)


In [ ]:

# ============================================================
# FASE 4 — CÉLULA 19
# TRANSIÇÕES, RAMPAS E PERMANÊNCIA OPERACIONAL
# ============================================================
#
# Objetivos:
# 1. Comparar horas com e sem transição de estado HMM.
# 2. Comparar subida, descida e estabilidade de carga.
# 3. Investigar a duração de permanência no estado.
# 4. Identificar condições em que P2 supera B0.
# 5. Exportar tabelas, figuras e relatório.
#
# Protocolo:
# - Utilizar somente o conjunto de teste já avaliado.
# - Preservar integralmente as previsões da Célula 16.
# - Não treinar ou selecionar novos modelos.
# - Não interpretar associações como relações causais.
# ============================================================

import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 1. RECUPERAR PREVISÕES E ATRIBUTOS DA FASE 2
# ============================================================

pred19 = pd.read_parquet(
    DIRS["processed"] /
    "cell16_final_test_predictions.parquet"
)

master19 = pd.read_parquet(
    DIRS["processed"] /
    "cell04_master_dataset.parquet"
)

for frame in [pred19, master19]:
    frame["timestamp"] = pd.to_datetime(
        frame["timestamp"], utc=True
    )

operational19 = [
    "timestamp",
    "generation_mw",
    "hmm_state",
    "state_changed",
    "delta_load_mw",
    "ramp_activity",
    "hours_in_current_state"
]

missing19 = set(operational19) - set(master19.columns)

if missing19:
    raise ValueError(
        f"Colunas ausentes no master: {sorted(missing19)}"
    )

df19 = pred19.merge(
    master19[operational19],
    on="timestamp",
    how="left",
    validate="one_to_one",
    indicator=True
)

assert len(df19) == 1063
assert df19["timestamp"].is_unique
assert (df19["_merge"] == "both").all()

df19 = df19.drop(columns="_merge")
df19 = df19.sort_values("timestamp").reset_index(drop=True)

scenarios19 = ["B0", "P1", "P2"]

for scenario in scenarios19:
    residual = (
        df19["y_true"] - df19[f"pred_{scenario}"]
    )
    df19[f"ae_{scenario}"] = residual.abs()
    df19[f"se_{scenario}"] = residual ** 2
    df19[f"residual_{scenario}"] = residual


# ============================================================
# 2. CLASSIFICAR TRANSIÇÕES
# ============================================================
#
# Usamos state_changed da Fase 2.
# Não recalculamos transições usando somente o teste,
# pois isso poderia perder a continuidade dos ciclos.
# ============================================================

def classify_transition19(value):
    if pd.isna(value):
        return "indefinida"
    if value in [True, 1]:
        return "transição"
    if value in [False, 0]:
        return "sem transição"
    return "indefinida"


df19["transition_group"] = (
    df19["state_changed"]
    .apply(classify_transition19)
)

_project_print("TRANSIÇÕES OPERACIONAIS")
_project_display(
    df19["transition_group"]
    .value_counts(dropna=False)
    .rename_axis("categoria")
    .reset_index(name="n")
)


# ============================================================
# 3. CLASSIFICAR RAMPAS
# ============================================================
#
# Prioridade: classificação original da Fase 2.
# Não criamos novos limiares de MW/h.
# ============================================================

df19["ramp_group"] = (
    df19["ramp_activity"]
    .astype("string")
    .fillna("indefinida")
)

_project_print("\nCATEGORIAS DE RAMPA DA FASE 2")
_project_display(
    df19["ramp_group"]
    .value_counts(dropna=False)
    .rename_axis("categoria")
    .reset_index(name="n")
)


# ============================================================
# 4. TEMPO DE PERMANÊNCIA NO ESTADO
# ============================================================
#
# Faixas descritivas em horas.
# Valores ausentes ou inválidos ficam separados.
# ============================================================

duration19 = pd.to_numeric(
    df19["hours_in_current_state"],
    errors="coerce"
)

duration19 = duration19.where(
    np.isfinite(duration19) & (duration19 >= 0)
)

df19["duration_group"] = pd.cut(
    duration19,
    bins=[-0.001, 1, 3, 6, 12, 24, np.inf],
    labels=[
        "0–1 h",
        ">1–3 h",
        ">3–6 h",
        ">6–12 h",
        ">12–24 h",
        ">24 h"
    ],
    include_lowest=True
)

df19["duration_group"] = (
    df19["duration_group"]
    .astype("string")
    .fillna("indefinida")
)


# ============================================================
# 5. FUNÇÃO DE COMPARAÇÃO POR GRUPO
# ============================================================

def compare_groups19(data, group_column):

    records = []

    for category, group in data.groupby(
        group_column, dropna=False, observed=True
    ):

        n = len(group)

        row = {
            "group": str(category),
            "n": int(n),
            "share_pct": float(
                100 * n / len(data)
            ),
            "mean_generation_mw": float(
                group["generation_mw"].mean()
            )
        }

        for scenario in scenarios19:

            row[f"rmse_{scenario}"] = float(
                np.sqrt(
                    group[f"se_{scenario}"].mean()
                )
            )

            row[f"mae_{scenario}"] = float(
                group[f"ae_{scenario}"].mean()
            )

            row[f"bias_{scenario}"] = float(
                group[f"residual_{scenario}"].mean()
            )

        row["p2_better_hours"] = int(
            (
                group["ae_P2"] <
                group["ae_B0"]
            ).sum()
        )

        row["p2_better_pct"] = float(
            100 * (
                group["ae_P2"] <
                group["ae_B0"]
            ).mean()
        )

        row["rmse_gain_P2_vs_B0_pct"] = (
            100
            * (
                row["rmse_B0"] -
                row["rmse_P2"]
            )
            / row["rmse_B0"]
            if row["rmse_B0"] > 0
            else np.nan
        )

        records.append(row)

    return pd.DataFrame(records)


transition_metrics19 = compare_groups19(
    df19, "transition_group"
)

ramp_metrics19 = compare_groups19(
    df19, "ramp_group"
)

duration_metrics19 = compare_groups19(
    df19, "duration_group"
)

_project_print("\nDESEMPENHO POR TRANSIÇÃO")
_project_display(transition_metrics19.round(3))

_project_print("\nDESEMPENHO POR TIPO DE RAMPA")
_project_display(ramp_metrics19.round(3))

_project_print("\nDESEMPENHO POR PERMANÊNCIA NO ESTADO")
_project_display(duration_metrics19.round(3))


# ============================================================
# 6. FUNÇÃO DE VISUALIZAÇÃO
# ============================================================

def plot_group_rmse19(metrics, title, output):

    fig, ax = plt.subplots(figsize=(11, 5))

    x = np.arange(len(metrics))
    width = 0.25

    for i, scenario in enumerate(scenarios19):

        ax.bar(
            x + (i - 1) * width,
            metrics[f"rmse_{scenario}"],
            width,
            label=scenario
        )

    ax.set_xticks(x)
    ax.set_xticklabels(
        metrics["group"],
        rotation=20,
        ha="right"
    )

    ax.set_ylabel("RMSE (t CO₂/h)")
    ax.set_title(title)
    ax.legend()
    ax.grid(axis="y", alpha=0.2)

    fig.tight_layout()
    fig.savefig(
        output, dpi=180, bbox_inches="tight"
    )
    plt.show()


# ============================================================
# 7. GRÁFICOS COMPARATIVOS
# ============================================================

figure_transition19 = (
    DIRS["figures"] /
    "cell19_rmse_by_transition.png"
)

figure_ramp19 = (
    DIRS["figures"] /
    "cell19_rmse_by_ramp.png"
)

figure_duration19 = (
    DIRS["figures"] /
    "cell19_rmse_by_state_duration.png"
)

plot_group_rmse19(
    transition_metrics19,
    "Iatan 2 — RMSE por transição HMM",
    figure_transition19
)

plot_group_rmse19(
    ramp_metrics19,
    "Iatan 2 — RMSE por atividade de rampa",
    figure_ramp19
)

plot_group_rmse19(
    duration_metrics19,
    "Iatan 2 — RMSE por permanência no estado",
    figure_duration19
)


# ============================================================
# 8. GRÁFICO — GANHO DE P2 POR TIPO DE RAMPA
# ============================================================

fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(
    ramp_metrics19["group"],
    ramp_metrics19["rmse_gain_P2_vs_B0_pct"]
)

ax.axhline(
    0,
    color="black",
    linestyle="--",
    linewidth=1
)

ax.set_ylabel("Redução do RMSE de P2 vs B0 (%)")
ax.set_xlabel("Atividade de rampa")
ax.set_title(
    "Iatan 2 — Ganho de P2 por atividade operacional"
)
ax.tick_params(axis="x", rotation=20)
ax.grid(axis="y", alpha=0.2)

fig.tight_layout()

figure_gain19 = (
    DIRS["figures"] /
    "cell19_p2_gain_by_ramp.png"
)

fig.savefig(
    figure_gain19, dpi=180, bbox_inches="tight"
)
plt.show()


# ============================================================
# 9. EXPORTAR RESULTADOS
# ============================================================

transition_file19 = (
    DIRS["tables"] /
    "cell19_performance_by_transition.csv"
)

ramp_file19 = (
    DIRS["tables"] /
    "cell19_performance_by_ramp.csv"
)

duration_file19 = (
    DIRS["tables"] /
    "cell19_performance_by_state_duration.csv"
)

detail_file19 = (
    DIRS["processed"] /
    "cell19_transition_ramp_errors.parquet"
)

report_file19 = (
    DIRS["reports"] /
    "cell19_transition_ramp_report.json"
)

_project_csv(transition_metrics19, 
    transition_file19, index=False
)

_project_csv(ramp_metrics19, 
    ramp_file19, index=False
)

_project_csv(duration_metrics19, 
    duration_file19, index=False
)

df19.to_parquet(
    detail_file19, index=False
)

report19 = {
    "phase": "phase_04_predictive_modeling",
    "cell": "19",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "test_observations": int(len(df19)),
    "transition_groups": int(
        len(transition_metrics19)
    ),
    "ramp_groups": int(
        len(ramp_metrics19)
    ),
    "duration_groups": int(
        len(duration_metrics19)
    ),
    "models_retrained": False,
    "hyperparameters_changed": False,
    "analysis_type": "exploratory_post_test",
    "limitations": [
        "Analysis is descriptive and exploratory.",
        "Operational groups can have unequal sizes.",
        "Group differences do not establish causality.",
        "Champion algorithms differ between scenarios.",
        "Small groups require cautious interpretation."
    ],
    "files": {
        "transitions": str(transition_file19),
        "ramps": str(ramp_file19),
        "durations": str(duration_file19),
        "hourly_details": str(detail_file19),
        "figure_transitions": str(figure_transition19),
        "figure_ramps": str(figure_ramp19),
        "figure_durations": str(figure_duration19),
        "figure_p2_gain": str(figure_gain19)
    }
}

with open(
    report_file19, "w", encoding="utf-8"
) as f:
    _project_dump(
        report19,
        f,
        indent=4,
        ensure_ascii=False
    )


# ============================================================
# 10. RESUMO FINAL
# ============================================================

_project_print("\n" + "=" * 70)
_project_print("FASE 4 — CÉLULA 19")
_project_print("=" * 70)

_project_print("Horas analisadas:", len(df19))
_project_print("Categorias de transição:", len(transition_metrics19))
_project_print("Categorias de rampa:", len(ramp_metrics19))
_project_print("Faixas de permanência:", len(duration_metrics19))

_project_print("Modelos reajustados: 0")
_project_print("Hiperparâmetros alterados: 0")

_project_print("\nTransições:", transition_file19)
_project_print("Rampas:", ramp_file19)
_project_print("Permanência:", duration_file19)
_project_print("Relatório:", report_file19)

_project_print("\nSTATUS: ANÁLISE DE TRANSIÇÕES CONCLUÍDA")
_project_print("=" * 70)


In [ ]:

# ============================================================
# FASE 4 — PREPARAÇÃO DA CÉLULA 20
# Localização dos modelos e das matrizes de atributos
# ============================================================

from pathlib import Path
import json

root = Path(DIRS["root"])

_project_print("ARQUIVOS DE MODELOS")
for p in sorted(root.rglob("*")):
    if p.is_file() and p.suffix.lower() in {
        ".joblib", ".pkl", ".pickle", ".json", ".ubj"
    }:
        if "model" in str(p).lower() or "manifest" in str(p).lower():
            _project_print(p.relative_to(root))

_project_print("\nMANIFESTOS DISPONÍVEIS")
for p in sorted(DIRS["manifests"].glob("*")):
    _project_print(p.name)

_project_print("\nOBJETOS DISPONÍVEIS NA SESSÃO")
for name in sorted(globals()):
    if any(term in name.lower() for term in [
        "model", "matrix", "feature", "selected", "manifest"
    ]):
        obj = globals()[name]
        _project_print(name, ":", type(obj).__name__)


In [ ]:

# ============================================================
# FASE 4 — CÉLULA 20
# IMPORTÂNCIA DOS ATRIBUTOS POR PERMUTAÇÃO
# ============================================================
#
# PROJETO:
# Modelagem das emissões horárias de CO2 da unidade 2 da
# usina Iatan (EPA Facility ID 6065), ano de 2025.
#
# OBJETIVOS:
# 1. Recuperar os modelos selecionados e congelados.
# 2. Recuperar os atributos originais da Célula 06.
# 3. Recuperar a validação do conjunto mestre da Célula 04.
# 4. Reproduzir os RMSE registrados na Célula 15.
# 5. Calcular a importância dos atributos por permutação.
# 6. Comparar B0, P1 e P2.
# 7. Exportar tabelas, gráficos e relatório.
#
# PROTOCOLO METODOLÓGICO:
# - Avaliação somente na validação.
# - Nenhum modelo é reajustado.
# - Nenhum hiperparâmetro é alterado.
# - Nenhuma observação de teste é utilizada.
# - A importância é calculada nos atributos originais,
#   antes do pré-processamento.
# - Importância preditiva não implica causalidade.
#
# ============================================================


# ============================================================
# 1. BIBLIOTECAS
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error

import json
import hashlib
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 2. CONFIGURAÇÕES E DIRETÓRIOS
# ============================================================

ROOT20 = Path(DIRS["root"])

MODEL_DIR20 = ROOT20 / "models"
MANIFEST_DIR20 = ROOT20 / "manifests"

TABLE_DIR20 = Path(DIRS["tables"])
FIGURE_DIR20 = Path(DIRS["figures"])
REPORT_DIR20 = Path(DIRS["reports"])

for directory in [
    TABLE_DIR20,
    FIGURE_DIR20,
    REPORT_DIR20
]:
    directory.mkdir(parents=True, exist_ok=True)

N_REPEATS20 = 10
RANDOM_STATE20 = 42

# Tolerância para diferenças de arredondamento.
RMSE_TOLERANCE20 = 0.01


# ============================================================
# 3. RECUPERAR MANIFESTOS
# ============================================================

selection_manifest_path20 = (
    MANIFEST_DIR20 /
    "cell15_frozen_model_selection.json"
)

feature_manifest_path20 = (
    MANIFEST_DIR20 /
    "cell06_feature_sets.json"
)

assert selection_manifest_path20.exists()
assert feature_manifest_path20.exists()

with open(
    selection_manifest_path20,
    "r",
    encoding="utf-8"
) as f:
    selection_manifest20 = json.load(f)

with open(
    feature_manifest_path20,
    "r",
    encoding="utf-8"
) as f:
    feature_manifest20 = json.load(f)

# A referência de integridade vem dos modelos congelados desta execução.
EXPECTED_RMSE20 = {
    row["scenario"]: float(row["validation_rmse"])
    for row in selection_manifest20["selected_models"]
}

_project_print("Manifesto de seleção recuperado.")
_project_print("Manifesto de atributos recuperado.")


# ============================================================
# 4. RECUPERAR MODELOS E PRÉ-PROCESSADORES
# ============================================================

MODEL_PATHS20 = {
    "B0": (
        MODEL_DIR20 /
        "cell12_b0_best_xgboost.joblib"
    ),
    "P1": (
        MODEL_DIR20 /
        "cell13_p1_best_xgboost.joblib"
    ),
    "P2": (
        MODEL_DIR20 /
        "cell14_p2_best_randomforest.joblib"
    )
}

PREPROCESSOR_PATHS20 = {
    "B0": (
        MODEL_DIR20 /
        "cell08_preprocessor_b0_xgboost.joblib"
    ),
    "P1": (
        MODEL_DIR20 /
        "cell08_preprocessor_p1_xgboost.joblib"
    ),
    "P2": (
        MODEL_DIR20 /
        "cell08_preprocessor_p2_randomforest.joblib"
    )
}

for scenario in ["B0", "P1", "P2"]:

    if not MODEL_PATHS20[scenario].exists():
        raise FileNotFoundError(
            MODEL_PATHS20[scenario]
        )

    if not PREPROCESSOR_PATHS20[scenario].exists():
        raise FileNotFoundError(
            PREPROCESSOR_PATHS20[scenario]
        )

models20 = {
    scenario: joblib.load(path)
    for scenario, path in MODEL_PATHS20.items()
}

preprocessors20 = {
    scenario: joblib.load(path)
    for scenario, path in PREPROCESSOR_PATHS20.items()
}

_project_print("\nModelos e pré-processadores carregados.")


# ============================================================
# 5. RECUPERAR CONJUNTO MESTRE — CORREÇÃO
# ============================================================
#
# A versão anterior tentou acessar partition em
# experimental_matrix_df, que não possuía essa coluna.
#
# Agora recuperamos diretamente o conjunto mestre da
# Célula 04, que contém a partição cronológica.
# ============================================================

MASTER_PATH20 = (
    Path(DIRS["processed"]) /
    "cell04_master_dataset.parquet"
)

if not MASTER_PATH20.exists():
    raise FileNotFoundError(
        f"Conjunto mestre não encontrado: {MASTER_PATH20}"
    )

matrix20 = pd.read_parquet(MASTER_PATH20)

required_base20 = {
    "timestamp",
    "partition",
    "eligible_for_modeling",
    "co2_metric_tons"
}

missing_base20 = (
    required_base20 - set(matrix20.columns)
)

if missing_base20:
    raise ValueError(
        "Colunas obrigatórias ausentes: "
        f"{sorted(missing_base20)}"
    )

matrix20["timestamp"] = pd.to_datetime(
    matrix20["timestamp"],
    utc=True
)

matrix20 = matrix20.loc[
    matrix20["eligible_for_modeling"].eq(True)
].copy()

matrix20 = matrix20.sort_values(
    "timestamp"
).reset_index(drop=True)

assert len(matrix20) == 6249, (
    f"Esperadas 6249 observações, obtidas {len(matrix20)}"
)

assert matrix20["timestamp"].is_unique, (
    "Existem timestamps duplicados."
)

_project_print("\nCONJUNTO MESTRE")
_project_print("Observações elegíveis:", len(matrix20))


# ============================================================
# 6. RECUPERAR ATRIBUTOS DOS TRÊS CENÁRIOS
# ============================================================
#
# Utilizamos os conjuntos definidos anteriormente na
# Célula 06, sem reconstruir as listas manualmente.
# ============================================================

if "FEATURE_SETS" not in globals():
    raise RuntimeError(
        "FEATURE_SETS não está disponível. "
        "Execute a Célula 06 antes desta célula."
    )

feature_sets20 = {
    scenario: list(FEATURE_SETS[scenario])
    for scenario in ["B0", "P1", "P2"]
}

expected_counts20 = {
    "B0": 1,
    "P1": 6,
    "P2": 35
}

for scenario, columns in feature_sets20.items():

    assert len(columns) == expected_counts20[scenario], (
        f"{scenario}: quantidade inesperada de atributos."
    )

    assert len(columns) == len(set(columns)), (
        f"{scenario}: atributos duplicados."
    )

    missing = [
        col for col in columns
        if col not in matrix20.columns
    ]

    if missing:
        raise ValueError(
            f"{scenario}: atributos ausentes: {missing}"
        )

_project_print("\nATRIBUTOS POR CENÁRIO")

for scenario, columns in feature_sets20.items():
    _project_print(f"{scenario}: {len(columns)} atributos")


# ============================================================
# 7. RECUPERAR EXCLUSIVAMENTE A VALIDAÇÃO
# ============================================================

validation20 = (
    matrix20.loc[
        matrix20["partition"].eq("validation")
    ]
    .sort_values("timestamp")
    .reset_index(drop=True)
)

assert len(validation20) == 687, (
    f"Validação inesperada: {len(validation20)}"
)

assert validation20["timestamp"].is_unique

TARGET20 = "co2_metric_tons"

y20 = validation20[
    TARGET20
].to_numpy(dtype=float)

assert np.isfinite(y20).all(), (
    "O alvo contém valores não finitos."
)

_project_print("\nPARTIÇÃO UTILIZADA")
_project_print("Validação:", len(validation20))
_project_print("Teste utilizado: NÃO")


# ============================================================
# 8. PREDITOR CONGELADO
# ============================================================
#
# A classe aplica o pré-processador previamente ajustado
# e depois utiliza o modelo congelado.
#
# O método fit é bloqueado para impedir treinamento.
# ============================================================

class FrozenPredictor20(RegressorMixin, BaseEstimator):

    def __init__(self, preprocessor, model):
        self.preprocessor = preprocessor
        self.model = model

    def fit(self, X, y=None):
        raise RuntimeError(
            "Treinamento proibido: modelo congelado."
        )

    def predict(self, X):
        X_processed = self.preprocessor.transform(X)
        return self.model.predict(X_processed)


# ============================================================
# 9. VERIFICAR REPRODUTIBILIDADE DOS MODELOS
# ============================================================
#
# Esta verificação é obrigatória.
#
# Caso o RMSE não reproduza o valor registrado na
# seleção congelada, a análise será interrompida.
# ============================================================

predictors20 = {}
baseline_rows20 = []

_project_print("\n" + "=" * 70)
_project_print("VERIFICAÇÃO DE REPRODUTIBILIDADE")
_project_print("=" * 70)

for scenario in ["B0", "P1", "P2"]:

    columns = feature_sets20[scenario]

    X_original = validation20[columns].copy()

    predictor = FrozenPredictor20(
        preprocessors20[scenario],
        models20[scenario]
    )

    predictions = np.asarray(
        predictor.predict(X_original),
        dtype=float
    ).reshape(-1)

    assert len(predictions) == len(y20)
    assert np.isfinite(predictions).all()

    rmse = float(
        np.sqrt(
            mean_squared_error(
                y20,
                predictions
            )
        )
    )

    expected = EXPECTED_RMSE20[scenario]

    difference = abs(rmse - expected)

    _project_print(
        f"{scenario}: "
        f"RMSE reproduzido={rmse:.4f} | "
        f"RMSE esperado={expected:.4f} | "
        f"diferença={difference:.6f}"
    )

    if difference > RMSE_TOLERANCE20:
        raise RuntimeError(
            f"{scenario}: o RMSE não reproduziu "
            "o resultado congelado. "
            "Interrompa e verifique a estrutura "
            "do pré-processador e do modelo."
        )

    predictors20[scenario] = predictor

    baseline_rows20.append({
        "scenario": scenario,
        "n": len(y20),
        "n_features": len(columns),
        "rmse_reproduced": rmse,
        "rmse_expected": expected,
        "absolute_difference": difference,
        "reproduction_verified": True
    })

baseline20 = pd.DataFrame(baseline_rows20)

_project_print("\nSTATUS: REPRODUTIBILIDADE VERIFICADA")


# ============================================================
# 10. IMPORTÂNCIA POR PERMUTAÇÃO
# ============================================================
#
# Métrica: aumento do RMSE quando um atributo é embaralhado.
#
# Interpretação:
# > 0: o embaralhamento prejudica a previsão.
# = 0: efeito praticamente nulo.
# < 0: o embaralhamento melhora a previsão.
#
# As permutações são realizadas sobre a validação.
# ============================================================

importance_rows20 = []

for scenario in ["B0", "P1", "P2"]:

    _project_print("\n" + "=" * 70)
    _project_print(f"IMPORTÂNCIA DOS ATRIBUTOS — {scenario}")
    _project_print("=" * 70)

    columns = feature_sets20[scenario]

    X_original = validation20[columns].copy()

    predictor = predictors20[scenario]

    result = permutation_importance(
        estimator=predictor,
        X=X_original,
        y=y20,
        scoring="neg_root_mean_squared_error",
        n_repeats=N_REPEATS20,
        random_state=RANDOM_STATE20,
        n_jobs=1
    )

    baseline_rmse = baseline20.loc[
        baseline20["scenario"].eq(scenario),
        "rmse_reproduced"
    ].iloc[0]

    for i, feature in enumerate(columns):

        importance_rows20.append({
            "scenario": scenario,
            "feature": feature,
            "importance_mean": float(
                result.importances_mean[i]
            ),
            "importance_std": float(
                result.importances_std[i]
            ),
            "importance_pct_rmse": float(
                100
                * result.importances_mean[i]
                / baseline_rmse
            ),
            "baseline_rmse": float(baseline_rmse),
            "n_repeats": N_REPEATS20
        })

    _project_print(
        f"Atributos avaliados: {len(columns)}"
    )

importance20 = pd.DataFrame(
    importance_rows20
)

importance20 = importance20.sort_values(
    ["scenario", "importance_mean"],
    ascending=[True, False]
).reset_index(drop=True)


# ============================================================
# 11. EXIBIR RESULTADOS
# ============================================================

_project_print("\nRMSE REPRODUZIDOS")
_project_display(baseline20.round(5))

_project_print("\nATRIBUTOS MAIS IMPORTANTES")

for scenario in ["B0", "P1", "P2"]:

    _project_print("\n" + "-" * 60)
    _project_print(f"CENÁRIO {scenario}")
    _project_print("-" * 60)

    subset = importance20.loc[
        importance20["scenario"].eq(scenario)
    ].head(12)

    _project_display(
        subset[
            [
                "feature",
                "importance_mean",
                "importance_std",
                "importance_pct_rmse"
            ]
        ].round(4)
    )


# ============================================================
# 12. GRÁFICOS DE IMPORTÂNCIA
# ============================================================

FIGURE_PATH20 = (
    FIGURE_DIR20 /
    "cell20_permutation_importance.png"
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(19, 7),
    constrained_layout=True
)

for ax, scenario in zip(
    axes,
    ["B0", "P1", "P2"]
):

    subset = importance20.loc[
        importance20["scenario"].eq(scenario)
    ].head(12)

    subset = subset.sort_values(
        "importance_mean"
    )

    ax.barh(
        subset["feature"],
        subset["importance_mean"],
        xerr=subset["importance_std"],
        capsize=3
    )

    ax.axvline(
        0,
        color="black",
        linestyle="--",
        linewidth=0.8
    )

    ax.set_title(
        f"{scenario} — importância por permutação"
    )

    ax.set_xlabel(
        "Aumento do RMSE (t CO₂/h)"
    )

    ax.grid(
        axis="x",
        alpha=0.2
    )

fig.suptitle(
    "Iatan 2 — Importância dos atributos na validação",
    fontsize=15
)

fig.savefig(
    FIGURE_PATH20,
    dpi=200,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 13. EXPORTAR TABELAS
# ============================================================

IMPORTANCE_FILE20 = (
    TABLE_DIR20 /
    "cell20_feature_importance.csv"
)

BASELINE_FILE20 = (
    TABLE_DIR20 /
    "cell20_validation_baselines.csv"
)

_project_csv(importance20, 
    IMPORTANCE_FILE20,
    index=False
)

_project_csv(baseline20, 
    BASELINE_FILE20,
    index=False
)


# ============================================================
# 14. RELATÓRIO METODOLÓGICO
# ============================================================

REPORT_FILE20 = (
    REPORT_DIR20 /
    "cell20_feature_importance_report.json"
)

report20 = {
    "phase": "phase_04_predictive_modeling",
    "cell": 20,
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "method": "permutation_importance",
    "scoring": "negative_root_mean_squared_error",
    "evaluation_partition": "validation",
    "n_observations": int(len(validation20)),
    "n_repeats": N_REPEATS20,
    "random_state": RANDOM_STATE20,
    "reproduction_tolerance": RMSE_TOLERANCE20,
    "reproduction_verified": True,
    "models_refitted": False,
    "hyperparameters_changed": False,
    "test_evaluated": False,
    "scenarios": baseline_rows20,
    "limitations": [
        "Permutation importance is not a causal effect.",
        "Correlated predictors can share predictive information.",
        "Individual importance can be reduced by collinearity.",
        "Importance depends on the fitted model.",
        "Permutation can create unrealistic feature combinations.",
        "Validation was previously used for model selection.",
        "Results are descriptive and post-selection."
    ],
    "files": {
        "importance_table": str(IMPORTANCE_FILE20),
        "baseline_table": str(BASELINE_FILE20),
        "figure": str(FIGURE_PATH20)
    }
}

with open(
    REPORT_FILE20,
    "w",
    encoding="utf-8"
) as f:

    _project_dump(
        report20,
        f,
        indent=4,
        ensure_ascii=False
    )


# ============================================================
# 15. RESUMO FINAL
# ============================================================

_project_print("\n" + "=" * 70)
_project_print("FASE 4 — CÉLULA 20")
_project_print("=" * 70)

_project_print(
    "Observações analisadas:",
    len(validation20)
)

_project_print(
    "Cenários analisados:",
    len(predictors20)
)

_project_print(
    "Atributos avaliados:",
    len(importance20)
)

_project_print(
    "Permutações por atributo:",
    N_REPEATS20
)

_project_print("Modelos reajustados: 0")
_project_print("Hiperparâmetros alterados: 0")
_project_print("Teste avaliado nesta célula: NÃO")

_project_print("\nTabela de importâncias:")
_project_print(IMPORTANCE_FILE20)

_project_print("\nGráfico:")
_project_print(FIGURE_PATH20)

_project_print("\nRelatório:")
_project_print(REPORT_FILE20)

_project_print(
    "\nSTATUS: IMPORTÂNCIA DOS ATRIBUTOS CONCLUÍDA"
)

_project_print("=" * 70)


### Incerteza em erros pareados

[Künsch (1989)](https://doi.org/10.1214/aos/1176347265) fundamenta a reamostragem por blocos para séries dependentes. Comprimento de bloco, suporte e número de réplicas são escolhas do estudo. Os intervalos são exploratórios e condicionados aos modelos já ajustados.


In [ ]:

# ============================================================
# FASE 4 — CÉLULA 21
# BOOTSTRAP EM BLOCOS TEMPORAIS
# ============================================================
#
# PROJETO:
# Iatan — Unidade 2 — EPA Facility ID 6065 — 2025
#
# OBJETIVO:
# Quantificar a incerteza das diferenças de desempenho
# entre os modelos B0, P1 e P2 no conjunto de teste.
#
# MÉTODO:
# Moving Block Bootstrap (MBB), com blocos de 24 horas
# consecutivas e 2.000 reamostragens.
#
# PROTOCOLO:
# 1. Utilizar somente previsões congeladas da Célula 16.
# 2. Preservar o pareamento entre os três cenários.
# 3. Não criar blocos através de lacunas temporais.
# 4. Não reajustar modelos ou hiperparâmetros.
# 5. Não utilizar resultados para nova seleção.
#
# INTERPRETAÇÃO:
# Ganho percentual positivo = candidato melhor.
# Ganho percentual negativo = candidato pior.
#
# ATENÇÃO:
# Os intervalos são condicionais aos modelos selecionados.
# O teste já foi examinado nas etapas anteriores, portanto
# não constitui um holdout inteiramente intocado.
# ============================================================


# ============================================================
# 1. BIBLIOTECAS E CONFIGURAÇÕES
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

N_BOOT21 = 2000
BLOCK_LENGTH21 = 24
CONFIDENCE21 = 0.95
RANDOM_STATE21 = 42

rng21 = np.random.default_rng(RANDOM_STATE21)

ROOT21 = Path(DIRS["root"])
DATA_DIR21 = Path(DIRS["processed"])
TABLE_DIR21 = Path(DIRS["tables"])
FIGURE_DIR21 = Path(DIRS["figures"])
REPORT_DIR21 = Path(DIRS["reports"])

for directory in [
    DATA_DIR21,
    TABLE_DIR21,
    FIGURE_DIR21,
    REPORT_DIR21
]:
    directory.mkdir(parents=True, exist_ok=True)

INPUT_FILE21 = (
    DATA_DIR21 /
    "cell16_final_test_predictions.parquet"
)

assert INPUT_FILE21.exists(), (
    f"Arquivo não encontrado: {INPUT_FILE21}"
)


# ============================================================
# 2. CARREGAR E INSPECIONAR PREVISÕES
# ============================================================

raw21 = pd.read_parquet(INPUT_FILE21)

_project_print("Arquivo recuperado:")
_project_print(INPUT_FILE21)

_project_print("\nDimensões:", raw21.shape)
_project_print("\nColunas disponíveis:")
_project_print(raw21.columns.tolist())

# A Célula 16 pode ter exportado previsões em formato
# longo (scenario + prediction) ou largo (prediction_B0,
# prediction_P1, prediction_P2).
#
# A função abaixo reconhece os dois formatos, mas
# interrompe a execução se houver ambiguidade.

def find_column21(columns, candidates):

    for candidate in candidates:
        if candidate in columns:
            return candidate

    return None


TIMESTAMP_CANDIDATES21 = [
    "timestamp",
    "datetime",
    "date_time"
]

TARGET_CANDIDATES21 = [
    "co2_metric_tons",
    "y_true",
    "observed",
    "actual",
    "target",
    "co2_observed"
]

SCENARIO_CANDIDATES21 = [
    "scenario",
    "cenario"
]

PREDICTION_CANDIDATES21 = [
    "y_pred",
    "prediction",
    "predicted",
    "y_prediction",
    "co2_predicted",
    "co2_estimated"
]

timestamp_col21 = find_column21(
    raw21.columns,
    TIMESTAMP_CANDIDATES21
)

target_col21 = find_column21(
    raw21.columns,
    TARGET_CANDIDATES21
)

scenario_col21 = find_column21(
    raw21.columns,
    SCENARIO_CANDIDATES21
)

prediction_col21 = find_column21(
    raw21.columns,
    PREDICTION_CANDIDATES21
)

if timestamp_col21 is None:
    raise ValueError(
        "Coluna de timestamp não identificada."
    )

if target_col21 is None:
    raise ValueError(
        "Coluna de CO2 observado não identificada."
    )


# ============================================================
# 3. PADRONIZAR O FORMATO DAS PREVISÕES
# ============================================================

SCENARIOS21 = ["B0", "P1", "P2"]

if (
    scenario_col21 is not None
    and prediction_col21 is not None
):

    # FORMATO LONGO
    long21 = raw21[
        [
            timestamp_col21,
            target_col21,
            scenario_col21,
            prediction_col21
        ]
    ].copy()

    long21.columns = [
        "timestamp",
        "observed",
        "scenario",
        "prediction"
    ]

    long21["scenario"] = (
        long21["scenario"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    assert set(long21["scenario"].unique()) == set(
        SCENARIOS21
    ), "Cenários inesperados no arquivo."

    assert not long21.duplicated(
        ["timestamp", "scenario"]
    ).any(), "Previsões duplicadas."

    # Conferir que o alvo é idêntico nos três cenários.
    target_consistency21 = long21.groupby(
        "timestamp"
    )["observed"].nunique(dropna=False)

    assert target_consistency21.eq(1).all(), (
        "Alvos divergentes entre cenários."
    )

    wide21 = long21.pivot(
        index="timestamp",
        columns="scenario",
        values="prediction"
    )

    observed21 = long21.groupby(
        "timestamp"
    )["observed"].first()

    wide21 = wide21.join(
        observed21.rename("observed")
    ).reset_index()

    wide21 = wide21.rename(
        columns={
            "B0": "pred_B0",
            "P1": "pred_P1",
            "P2": "pred_P2"
        }
    )

else:

    # FORMATO LARGO
    wide21 = raw21.copy()

    wide21 = wide21.rename(
        columns={
            timestamp_col21: "timestamp",
            target_col21: "observed"
        }
    )

    prediction_options21 = {
        "B0": [
            "pred_B0",
            "prediction_B0",
            "y_pred_B0",
            "B0_pred"
        ],
        "P1": [
            "pred_P1",
            "prediction_P1",
            "y_pred_P1",
            "P1_pred"
        ],
        "P2": [
            "pred_P2",
            "prediction_P2",
            "y_pred_P2",
            "P2_pred"
        ]
    }

    rename21 = {}

    for scenario, candidates in prediction_options21.items():

        found = find_column21(
            wide21.columns,
            candidates
        )

        if found is None:
            raise ValueError(
                f"Previsão de {scenario} não identificada. "
                f"Colunas: {wide21.columns.tolist()}"
            )

        rename21[found] = f"pred_{scenario}"

    wide21 = wide21.rename(
        columns=rename21
    )

    wide21 = wide21[
        [
            "timestamp",
            "observed",
            "pred_B0",
            "pred_P1",
            "pred_P2"
        ]
    ].copy()


# ============================================================
# 4. VERIFICAÇÕES DE INTEGRIDADE
# ============================================================

wide21["timestamp"] = pd.to_datetime(
    wide21["timestamp"],
    utc=True
)

wide21 = (
    wide21
    .sort_values("timestamp")
    .reset_index(drop=True)
)

assert len(wide21) == 1063, (
    f"Esperadas 1063 horas; encontradas {len(wide21)}."
)

assert wide21["timestamp"].is_unique, (
    "Timestamps duplicados."
)

assert wide21[
    [
        "observed",
        "pred_B0",
        "pred_P1",
        "pred_P2"
    ]
].notna().all().all(), (
    "Valores ausentes nas previsões."
)

for col in [
    "observed",
    "pred_B0",
    "pred_P1",
    "pred_P2"
]:
    assert np.isfinite(
        wide21[col].to_numpy(dtype=float)
    ).all(), f"Valores não finitos: {col}"


# ============================================================
# 5. REPRODUZIR MÉTRICAS DA CÉLULA 16
# ============================================================

# Confere as previsões persistidas contra as métricas da mesma execução.
test_metrics21 = pd.read_csv(Path(DIRS["tables"]) / "cell16_final_test_metrics.csv")
assert test_metrics21["scenario"].is_unique
EXPECTED_RMSE21 = test_metrics21.set_index("scenario")["rmse"].to_dict()

def rmse21(y, yhat):
    return float(
        np.sqrt(
            np.mean((y - yhat) ** 2)
        )
    )

def mae21(y, yhat):
    return float(
        np.mean(np.abs(y - yhat))
    )

y21 = wide21["observed"].to_numpy(dtype=float)

point_metrics21 = {}

_project_print("\n" + "=" * 70)
_project_print("REPRODUÇÃO DAS MÉTRICAS DO TESTE")
_project_print("=" * 70)

for scenario in SCENARIOS21:

    pred = wide21[
        f"pred_{scenario}"
    ].to_numpy(dtype=float)

    current_rmse = rmse21(y21, pred)
    current_mae = mae21(y21, pred)

    assert abs(
        current_rmse - EXPECTED_RMSE21[scenario]
    ) < 0.01, (
        f"RMSE de {scenario} não reproduzido."
    )

    point_metrics21[scenario] = {
        "rmse": current_rmse,
        "mae": current_mae
    }

    _project_print(
        f"{scenario}: "
        f"RMSE={current_rmse:.4f} | "
        f"MAE={current_mae:.4f}"
    )

_project_print("\nIntegridade: VERIFICADA")


# ============================================================
# 6. IDENTIFICAR SEGMENTOS TEMPORAIS CONTÍNUOS
# ============================================================
#
# Uma diferença de exatamente 1 hora indica continuidade.
# Diferenças maiores iniciam um novo segmento.
#
# Isso evita construir blocos artificiais atravessando
# períodos sem observações elegíveis.
# ============================================================

time_diff21 = wide21[
    "timestamp"
].diff()

new_segment21 = (
    time_diff21.ne(pd.Timedelta(hours=1))
)

wide21["segment_id"] = (
    new_segment21.cumsum()
)

segments21 = []

for _, group in wide21.groupby(
    "segment_id",
    sort=True
):

    indices = group.index.to_numpy(dtype=int)

    segments21.append(indices)

segment_sizes21 = [
    len(segment)
    for segment in segments21
]

_project_print("\nSEGMENTAÇÃO TEMPORAL")
_project_print("Segmentos:", len(segments21))
_project_print("Maior segmento:", max(segment_sizes21))
_project_print("Menor segmento:", min(segment_sizes21))
_project_print(
    "Segmentos com 24h ou mais:",
    sum(
        size >= BLOCK_LENGTH21
        for size in segment_sizes21
    )
)


# ============================================================
# 7. CONSTRUIR BLOCOS VÁLIDOS
# ============================================================
#
# Os blocos completos têm 24 observações horárias.
#
# Segmentos com menos de 24 horas também são preservados:
# cada segmento curto entra como um bloco indivisível.
#
# Esse procedimento evita descartar observações elegíveis,
# mas a heterogeneidade dos comprimentos deve ser relatada.
# ============================================================

blocks21 = []

for segment in segments21:

    n_segment = len(segment)

    if n_segment >= BLOCK_LENGTH21:

        for start in range(
            n_segment - BLOCK_LENGTH21 + 1
        ):

            block = segment[
                start:start + BLOCK_LENGTH21
            ]

            blocks21.append(block)

    else:

        blocks21.append(segment.copy())

assert len(blocks21) > 0

block_lengths21 = np.array(
    [len(block) for block in blocks21]
)

_project_print("\nBLOCOS CANDIDATOS")
_project_print("Total:", len(blocks21))
_project_print(
    "Blocos completos:",
    int(np.sum(
        block_lengths21 == BLOCK_LENGTH21
    ))
)
_project_print(
    "Blocos curtos:",
    int(np.sum(
        block_lengths21 < BLOCK_LENGTH21
    ))
)


# ============================================================
# 8. FUNÇÃO DE REAMOSTRAGEM
# ============================================================
#
# Os mesmos índices são utilizados para B0, P1 e P2.
#
# A amostra é formada pela concatenação de blocos
# selecionados com reposição.
#
# O último bloco pode ser truncado para obter N=1063.
# ============================================================

N21 = len(wide21)

def sample_indices21():

    selected_indices = []
    current_size = 0

    while current_size < N21:

        block = blocks21[
            rng21.integers(
                0,
                len(blocks21)
            )
        ]

        selected_indices.append(block)

        current_size += len(block)

    sample = np.concatenate(
        selected_indices
    )[:N21]

    return sample


# ============================================================
# 9. EXECUTAR BOOTSTRAP
# ============================================================

predictions21 = {
    scenario: wide21[
        f"pred_{scenario}"
    ].to_numpy(dtype=float)
    for scenario in SCENARIOS21
}

COMPARISONS21 = [
    ("B0", "P1"),
    ("B0", "P2"),
    ("P1", "P2")
]

bootstrap_rows21 = []

_project_print("\n" + "=" * 70)
_project_print("BOOTSTRAP EM BLOCOS TEMPORAIS")
_project_print("=" * 70)

for repetition in range(N_BOOT21):

    idx = sample_indices21()

    y_sample = y21[idx]

    metrics = {}

    for scenario in SCENARIOS21:

        pred_sample = predictions21[
            scenario
        ][idx]

        metrics[scenario] = {
            "rmse": rmse21(
                y_sample,
                pred_sample
            ),
            "mae": mae21(
                y_sample,
                pred_sample
            )
        }

    row = {
        "replication": repetition + 1
    }

    for scenario in SCENARIOS21:

        row[
            f"rmse_{scenario}"
        ] = metrics[scenario]["rmse"]

        row[
            f"mae_{scenario}"
        ] = metrics[scenario]["mae"]

    for reference, candidate in COMPARISONS21:

        comparison_name = (
            f"{candidate}_vs_{reference}"
        )

        ref_rmse = metrics[
            reference
        ]["rmse"]

        cand_rmse = metrics[
            candidate
        ]["rmse"]

        ref_mae = metrics[
            reference
        ]["mae"]

        cand_mae = metrics[
            candidate
        ]["mae"]

        row[
            f"rmse_gain_pct_{comparison_name}"
        ] = (
            100
            * (ref_rmse - cand_rmse)
            / ref_rmse
        )

        row[
            f"mae_gain_pct_{comparison_name}"
        ] = (
            100
            * (ref_mae - cand_mae)
            / ref_mae
        )

        row[
            f"rmse_gain_absolute_{comparison_name}"
        ] = (
            ref_rmse - cand_rmse
        )

    bootstrap_rows21.append(row)

    if (repetition + 1) % 500 == 0:

        _project_print(
            f"Reamostragens concluídas: "
            f"{repetition + 1}/{N_BOOT21}"
        )

bootstrap21 = pd.DataFrame(
    bootstrap_rows21
)


# ============================================================
# 10. INTERVALOS DE CONFIANÇA PERCENTÍLICOS
# ============================================================

alpha21 = 1 - CONFIDENCE21

lower_q21 = alpha21 / 2
upper_q21 = 1 - alpha21 / 2

def interval21(values):

    values = np.asarray(
        values,
        dtype=float
    )

    return (
        float(np.quantile(values, lower_q21)),
        float(np.quantile(values, upper_q21))
    )


# ============================================================
# 11. INTERVALOS PARA RMSE E MAE
# ============================================================

model_ci_rows21 = []

for scenario in SCENARIOS21:

    for metric in ["rmse", "mae"]:

        values = bootstrap21[
            f"{metric}_{scenario}"
        ]

        lower, upper = interval21(values)

        model_ci_rows21.append({
            "scenario": scenario,
            "metric": metric,
            "point_estimate": point_metrics21[
                scenario
            ][metric],
            "ci_lower_95": lower,
            "ci_upper_95": upper,
            "bootstrap_mean": float(
                values.mean()
            ),
            "bootstrap_std": float(
                values.std(ddof=1)
            ),
            "n_bootstrap": N_BOOT21
        })

model_ci21 = pd.DataFrame(
    model_ci_rows21
)


# ============================================================
# 12. INTERVALOS PARA GANHOS PAREADOS
# ============================================================

comparison_ci_rows21 = []

for reference, candidate in COMPARISONS21:

    comparison_name = (
        f"{candidate}_vs_{reference}"
    )

    for metric in ["rmse", "mae"]:

        ref_point = point_metrics21[
            reference
        ][metric]

        cand_point = point_metrics21[
            candidate
        ][metric]

        point_gain = (
            100
            * (ref_point - cand_point)
            / ref_point
        )

        values = bootstrap21[
            f"{metric}_gain_pct_{comparison_name}"
        ]

        lower, upper = interval21(values)

        # Frequência descritiva das reamostragens
        # com ganho positivo. Não é p-valor.
        positive_frequency = float(
            np.mean(values > 0)
        )

        if lower > 0:
            interpretation = (
                "intervalo inteiramente positivo"
            )

        elif upper < 0:
            interpretation = (
                "intervalo inteiramente negativo"
            )

        else:
            interpretation = (
                "intervalo inclui zero"
            )

        comparison_ci_rows21.append({
            "reference": reference,
            "candidate": candidate,
            "metric": metric,
            "point_gain_pct": point_gain,
            "ci_lower_95": lower,
            "ci_upper_95": upper,
            "positive_bootstrap_fraction": (
                positive_frequency
            ),
            "interpretation": interpretation,
            "n_bootstrap": N_BOOT21
        })

comparison_ci21 = pd.DataFrame(
    comparison_ci_rows21
)


# ============================================================
# 13. EXIBIR RESULTADOS
# ============================================================

_project_print("\n" + "=" * 70)
_project_print("INTERVALOS DE CONFIANÇA DOS MODELOS")
_project_print("=" * 70)

_project_display(
    model_ci21.round(4)
)

_project_print("\n" + "=" * 70)
_project_print("INTERVALOS DE CONFIANÇA DOS GANHOS PAREADOS")
_project_print("=" * 70)

_project_display(
    comparison_ci21.round(4)
)


# ============================================================
# 14. GRÁFICO DOS INTERVALOS DE RMSE
# ============================================================

rmse_ci21 = model_ci21.loc[
    model_ci21["metric"].eq("rmse")
].copy()

rmse_ci21 = rmse_ci21.set_index(
    "scenario"
).loc[SCENARIOS21].reset_index()

fig, ax = plt.subplots(
    figsize=(9, 5)
)

x21 = np.arange(len(rmse_ci21))

points21 = rmse_ci21[
    "point_estimate"
].to_numpy()

lower_errors21 = (
    points21
    - rmse_ci21["ci_lower_95"].to_numpy()
)

upper_errors21 = (
    rmse_ci21["ci_upper_95"].to_numpy()
    - points21
)

# Intervalos percentílicos podem não conter a estimativa
# original; nesse caso exibimos a faixa com seus limites
# e a estimativa pontual separadamente.

for i, row in rmse_ci21.iterrows():

    ax.plot(
        [i, i],
        [
            row["ci_lower_95"],
            row["ci_upper_95"]
        ],
        linewidth=3
    )

    ax.scatter(
        i,
        row["point_estimate"],
        s=75,
        zorder=3
    )

ax.set_xticks(
    x21,
    SCENARIOS21
)

ax.set_ylabel(
    "RMSE (t CO₂/h)"
)

ax.set_title(
    "Iatan 2 — RMSE no teste e IC bootstrap de 95%"
)

ax.grid(
    axis="y",
    alpha=0.25
)

FIGURE_RMSE21 = (
    FIGURE_DIR21 /
    "cell21_bootstrap_rmse_ci.png"
)

fig.savefig(
    FIGURE_RMSE21,
    dpi=200,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 15. GRÁFICO DOS GANHOS PAREADOS
# ============================================================

plot_comparison21 = comparison_ci21.loc[
    comparison_ci21["metric"].eq("rmse")
].copy()

fig, ax = plt.subplots(
    figsize=(10, 5)
)

for i, row in plot_comparison21.reset_index(
    drop=True
).iterrows():

    ax.plot(
        [
            row["ci_lower_95"],
            row["ci_upper_95"]
        ],
        [i, i],
        linewidth=3
    )

    ax.scatter(
        row["point_gain_pct"],
        i,
        s=75,
        zorder=3
    )

ax.axvline(
    0,
    color="black",
    linestyle="--",
    linewidth=1
)

labels21 = [
    f"{row['candidate']} vs {row['reference']}"
    for _, row in plot_comparison21.iterrows()
]

ax.set_yticks(
    range(len(labels21)),
    labels21
)

ax.set_xlabel(
    "Redução percentual do RMSE (%)"
)

ax.set_title(
    "Iatan 2 — Ganhos pareados e IC bootstrap de 95%"
)

ax.grid(
    axis="x",
    alpha=0.25
)

FIGURE_GAIN21 = (
    FIGURE_DIR21 /
    "cell21_bootstrap_paired_gain_ci.png"
)

fig.savefig(
    FIGURE_GAIN21,
    dpi=200,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 16. EXPORTAR RESULTADOS
# ============================================================

BOOTSTRAP_FILE21 = (
    DATA_DIR21 /
    "cell21_bootstrap_replicates.parquet"
)

MODEL_CI_FILE21 = (
    TABLE_DIR21 /
    "cell21_model_confidence_intervals.csv"
)

COMPARISON_CI_FILE21 = (
    TABLE_DIR21 /
    "cell21_paired_confidence_intervals.csv"
)

bootstrap21.to_parquet(
    BOOTSTRAP_FILE21,
    index=False
)

_project_csv(model_ci21, 
    MODEL_CI_FILE21,
    index=False
)

_project_csv(comparison_ci21, 
    COMPARISON_CI_FILE21,
    index=False
)


# ============================================================
# 17. RELATÓRIO METODOLÓGICO
# ============================================================

REPORT_FILE21 = (
    REPORT_DIR21 /
    "cell21_block_bootstrap_report.json"
)

report21 = {
    "phase": "phase_04_predictive_modeling",
    "cell": 21,
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "method": "moving_block_bootstrap",
    "block_length_hours": BLOCK_LENGTH21,
    "bootstrap_repetitions": N_BOOT21,
    "confidence_level": CONFIDENCE21,
    "random_state": RANDOM_STATE21,
    "n_test_observations": N21,
    "n_continuous_segments": len(segments21),
    "segment_sizes": segment_sizes21,
    "n_candidate_blocks": len(blocks21),
    "n_short_blocks": int(
        np.sum(
            block_lengths21 < BLOCK_LENGTH21
        )
    ),
    "paired_resampling": True,
    "models_refitted": False,
    "hyperparameters_changed": False,
    "model_selection_changed": False,
    "limitations": [
        "Intervals are conditional on frozen models.",
        "Test data were previously examined.",
        "Short continuous segments are sampled as blocks.",
        "Blocks are sampled with equal probability.",
        "Bootstrap intervals do not prove causality.",
        "Results depend on block length and stationarity assumptions."
    ],
    "files": {
        "replicates": str(BOOTSTRAP_FILE21),
        "model_intervals": str(MODEL_CI_FILE21),
        "paired_intervals": str(COMPARISON_CI_FILE21),
        "rmse_figure": str(FIGURE_RMSE21),
        "paired_figure": str(FIGURE_GAIN21)
    }
}

with open(
    REPORT_FILE21,
    "w",
    encoding="utf-8"
) as f:

    _project_dump(
        report21,
        f,
        indent=4,
        ensure_ascii=False
    )


# ============================================================
# 18. RESUMO FINAL
# ============================================================

_project_print("\n" + "=" * 70)
_project_print("FASE 4 — CÉLULA 21")
_project_print("=" * 70)

_project_print(
    "Observações de teste:",
    N21
)

_project_print(
    "Segmentos temporais:",
    len(segments21)
)

_project_print(
    "Blocos candidatos:",
    len(blocks21)
)

_project_print(
    "Reamostragens:",
    N_BOOT21
)

_project_print(
    "Comparações pareadas:",
    len(COMPARISONS21)
)

_project_print("Modelos reajustados: 0")
_project_print("Hiperparâmetros alterados: 0")
_project_print("Seleção modificada: NÃO")

_project_print("\nIntervalos dos modelos:")
_project_print(MODEL_CI_FILE21)

_project_print("\nIntervalos pareados:")
_project_print(COMPARISON_CI_FILE21)

_project_print("\nRelatório:")
_project_print(REPORT_FILE21)

_project_print(
    "\nSTATUS: BOOTSTRAP TEMPORAL CONCLUÍDO"
)

_project_print("=" * 70)


In [ ]:

# ============================================================
# FASE 4 — CÉLULA 22
# ESTUDO DE ABLAÇÃO DOS ATRIBUTOS DE PROCESSOS
# ============================================================
#
# OBJETIVO
# ------------------------------------------------------------
# Investigar, exclusivamente na validação, a contribuição
# incremental de grupos de atributos do cenário P2.
#
# ESTRATÉGIA
# ------------------------------------------------------------
# 1. Recuperar a matriz experimental e os manifestos.
# 2. Recuperar o Random Forest originalmente selecionado.
# 3. Construir variantes retirando grupos de atributos.
# 4. Ajustar cada variante somente com o treinamento.
# 5. Avaliar exclusivamente na validação.
# 6. Comparar os resultados com P2 e B0 originais.
# 7. Exportar métricas, configurações e figuras.
#
# RESTRIÇÕES METODOLÓGICAS
# ------------------------------------------------------------
# - Não utilizar observações do teste.
# - Não alterar a seleção congelada da Célula 15.
# - Não otimizar hiperparâmetros.
# - Não substituir os modelos originais.
# - Interpretar ablações como análises exploratórias.
# ============================================================

import json
import hashlib
import copy
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from joblib import load
from sklearn.base import clone
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# ============================================================
# 1. DIRETÓRIOS E ARQUIVOS
# ============================================================

ROOT22 = Path(PHASE_ROOT)

PROCESSED22 = ROOT22 / "data" / "processed"
MODELS22 = ROOT22 / "models"
TABLES22 = ROOT22 / "tables"
FIGURES22 = ROOT22 / "figures"
REPORTS22 = ROOT22 / "reports"
MANIFESTS22 = ROOT22 / "manifests"

for directory in [
    TABLES22, FIGURES22, REPORTS22, MANIFESTS22
]:
    directory.mkdir(parents=True, exist_ok=True)

FEATURE_MANIFEST22 = (
    MANIFESTS22 / "cell06_feature_sets.json"
)

FROZEN_MANIFEST22 = (
    MANIFESTS22 / "cell15_frozen_model_selection.json"
)

MASTER_FILE22 = (
    PROCESSED22 / "cell04_master_dataset.parquet"
)

P2_MODEL_FILE22 = (
    MODELS22 / "cell14_p2_best_randomforest.joblib"
)

P2_PREPROCESSOR_FILE22 = (
    MODELS22 / "cell08_preprocessor_p2_randomforest.joblib"
)

BASELINE_METRICS22 = (
    TABLES22 / "cell15_best_12_models.csv"
)

for required_file in [
    FEATURE_MANIFEST22,
    FROZEN_MANIFEST22,
    MASTER_FILE22,
    P2_MODEL_FILE22,
    P2_PREPROCESSOR_FILE22,
    BASELINE_METRICS22,
]:
    assert required_file.exists(), (
        f"Arquivo obrigatório não encontrado: {required_file}"
    )

_project_print("Arquivos necessários: LOCALIZADOS")

# ============================================================
# 2. RECUPERAÇÃO DOS ATRIBUTOS E DADOS
# ============================================================

with open(FEATURE_MANIFEST22, "r", encoding="utf-8") as f:
    feature_manifest22 = json.load(f)

with open(FROZEN_MANIFEST22, "r", encoding="utf-8") as f:
    frozen_manifest22 = json.load(f)

# O objeto FEATURE_SETS foi produzido na Célula 06.
# Usamos a versão em memória quando disponível.
# Caso contrário, recuperamos a lista pelo manifesto.

if (
    "FEATURE_SETS" in globals()
    and isinstance(FEATURE_SETS, dict)
    and "P2" in FEATURE_SETS
):
    p2_features22 = list(FEATURE_SETS["P2"])

elif (
    "feature_sets" in globals()
    and isinstance(feature_sets, dict)
    and "P2" in feature_sets
):
    p2_features22 = list(feature_sets["P2"])

else:
    # Estruturas alternativas do manifesto.
    candidates22 = [
        feature_manifest22.get("feature_sets", {}),
        feature_manifest22.get("scenarios", {}),
        feature_manifest22,
    ]

    p2_features22 = None

    for candidate in candidates22:
        if not isinstance(candidate, dict):
            continue

        value = candidate.get("P2")

        if isinstance(value, list):
            p2_features22 = value
            break

        if isinstance(value, dict):
            for key in ["features", "feature_columns", "columns"]:
                if isinstance(value.get(key), list):
                    p2_features22 = value[key]
                    break

        if p2_features22 is not None:
            break

    assert p2_features22 is not None, (
        "Não foi possível recuperar os atributos P2."
    )

assert len(p2_features22) == 35, (
    f"Esperados 35 atributos P2; encontrados {len(p2_features22)}."
)

assert len(set(p2_features22)) == len(p2_features22), (
    "Existem atributos duplicados em P2."
)

master22 = pd.read_parquet(MASTER_FILE22)

required_columns22 = (
    ["partition", "co2_metric_tons"] + p2_features22
)

missing_columns22 = [
    c for c in required_columns22
    if c not in master22.columns
]

assert not missing_columns22, (
    f"Colunas ausentes no master: {missing_columns22}"
)

train22 = master22.loc[
    master22["partition"].eq("train")
].copy()

validation22 = master22.loc[
    master22["partition"].eq("validation")
].copy()

assert len(train22) == 4499, (
    f"Treinamento inesperado: {len(train22)}"
)

assert len(validation22) == 687, (
    f"Validação inesperada: {len(validation22)}"
)

assert train22["co2_metric_tons"].notna().all()
assert validation22["co2_metric_tons"].notna().all()

# Nenhuma observação de teste é incorporada à análise.
assert not train22["partition"].eq("test").any()
assert not validation22["partition"].eq("test").any()

X_train22 = train22[p2_features22]
X_validation22 = validation22[p2_features22]

y_train22 = train22["co2_metric_tons"].to_numpy()
y_validation22 = (
    validation22["co2_metric_tons"].to_numpy()
)

_project_print(
    f"Treinamento: {len(train22)} | "
    f"Validação: {len(validation22)}"
)
_project_print(f"Atributos P2: {len(p2_features22)}")

# ============================================================
# 3. RECUPERAÇÃO DO RANDOM FOREST ORIGINAL
# ============================================================

original_model22 = load(P2_MODEL_FILE22)
original_preprocessor22 = load(
    P2_PREPROCESSOR_FILE22
)

# O arquivo de modelo pode conter o estimador diretamente
# ou um pipeline. Identificamos o estimador de forma explícita.

if isinstance(original_model22, Pipeline):
    original_estimator22 = (
        original_model22.steps[-1][1]
    )
else:
    original_estimator22 = original_model22

assert hasattr(original_estimator22, "get_params")
assert hasattr(original_estimator22, "predict")

# Recupera os parâmetros do estimador selecionado.
frozen_params22 = copy.deepcopy(
    original_estimator22.get_params(deep=False)
)

_project_print(
    "Estimador original:",
    type(original_estimator22).__name__
)

# ============================================================
# 4. DEFINIÇÃO DOS GRUPOS DE ABLAÇÃO
# ============================================================
#
# Os grupos são definidos a partir dos nomes efetivamente
# utilizados em P2, evitando incluir colunas inexistentes.
#
# A análise "leave-one-group-out" retira um grupo por vez.
#
# ============================================================

GROUP_DEFINITIONS22 = {

    "defasagens_geracao": [
        c for c in p2_features22
        if c.startswith("load_lag_")
    ],

    "estatisticas_moveis": [
        c for c in p2_features22
        if (
            c.startswith("load_mean_last_")
            or c.startswith("load_std_last_")
        )
    ],

    "historico_transicoes": [
        c for c in p2_features22
        if (
            c.startswith("state_changes_last_")
            or c in [
                "state_changed",
                "cumulative_state_changes",
                "previous_hmm_state",
                "has_previous_state",
            ]
            or c.startswith("hmm_state_lag_")
        )
    ],

    "duracoes_operacionais": [
        c for c in p2_features22
        if c in [
            "hours_since_cycle_start",
            "hours_in_current_state",
            "hours_in_current_ramp",
        ]
    ],

    "atividades_rampa": [
        c for c in p2_features22
        if (
            c.startswith("is_ramp_")
            or c in [
                "delta_load_mw",
                "ramp_defined",
            ]
        )
    ],

    "estados_hmm": [
        c for c in p2_features22
        if (
            c.startswith("is_regime_s")
            or c in [
                "hmm_state",
                "hmm_confidence",
                "hmm_entropy",
            ]
        )
    ],
}

# Mantém somente grupos efetivamente presentes.
GROUP_DEFINITIONS22 = {
    name: columns
    for name, columns in GROUP_DEFINITIONS22.items()
    if len(columns) > 0
}

# Verificação de sobreposição entre grupos.
all_grouped22 = [
    feature
    for group in GROUP_DEFINITIONS22.values()
    for feature in group
]

duplicates22 = sorted({
    c for c in all_grouped22
    if all_grouped22.count(c) > 1
})

assert not duplicates22, (
    f"Grupos de ablação sobrepostos: {duplicates22}"
)

group_rows22 = []

for group_name, columns in GROUP_DEFINITIONS22.items():
    group_rows22.append({
        "group": group_name,
        "n_features": len(columns),
        "features": ", ".join(columns),
    })

groups_df22 = pd.DataFrame(group_rows22)

_project_print("\nGRUPOS DE ABLAÇÃO")
_project_display(groups_df22)

# ============================================================
# 5. CONSTRUÇÃO DOS PIPELINES
# ============================================================
#
# O pré-processador é reconstruído com a mesma classe e
# parâmetros do original, mas reajustado somente no treino.
#
# O Random Forest mantém os hiperparâmetros congelados.
#
# ============================================================

def build_ablation_pipeline22(selected_features):

    # Clonagem preserva a configuração do pré-processador
    # sem transportar estatísticas já ajustadas.
    preprocessor = clone(original_preprocessor22)

    estimator = clone(original_estimator22)

    # Se o pré-processador for um ColumnTransformer,
    # seus seletores de colunas podem referenciar atributos
    # removidos. Nesse caso, utilizamos a mesma estratégia
    # de imputação do P2 original: mediana do treinamento.
    #
    # Para Random Forest, escalonamento não é necessário.

    from sklearn.compose import ColumnTransformer

    if isinstance(preprocessor, ColumnTransformer):

        pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "model",
                estimator
            ),
        ])

    elif isinstance(preprocessor, Pipeline):

        # Pipelines genéricos podem conter transformadores
        # que exigem um número fixo de atributos.
        # A imputação por mediana mantém a regra original
        # usada para as variáveis numéricas de P2.
        pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "model",
                estimator
            ),
        ])

    else:

        pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "model",
                estimator
            ),
        ])

    return pipeline

# ============================================================
# 6. MÉTRICAS
# ============================================================

def calculate_metrics22(y_true, y_pred):

    residual = (
        np.asarray(y_true)
        - np.asarray(y_pred)
    )

    return {
        "rmse": float(
            np.sqrt(
                mean_squared_error(y_true, y_pred)
            )
        ),
        "mae": float(
            mean_absolute_error(y_true, y_pred)
        ),
        "r2": float(
            r2_score(y_true, y_pred)
        ),
        "bias": float(np.mean(residual)),
    }

# ============================================================
# 7. REFERÊNCIAS CONGELADAS
# ============================================================

best12_22 = pd.read_csv(BASELINE_METRICS22)

def get_frozen_rmse22(scenario, algorithm):

    subset = best12_22.loc[
        best12_22["scenario"].eq(scenario)
        & best12_22["algorithm"].eq(algorithm)
    ]

    assert len(subset) == 1, (
        f"Referência não encontrada: {scenario}/{algorithm}"
    )

    return float(subset.iloc[0]["rmse"])

rmse_b0_frozen22 = get_frozen_rmse22(
    "B0", "XGBoost"
)

rmse_p2_frozen22 = get_frozen_rmse22(
    "P2", "RandomForest"
)

frozen_rmse22 = {
    row["scenario"]: float(row["validation_rmse"])
    for row in frozen_manifest22["selected_models"]
}
assert np.isclose(rmse_b0_frozen22, frozen_rmse22["B0"], atol=0.01)
assert np.isclose(rmse_p2_frozen22, frozen_rmse22["P2"], atol=0.01)

# ============================================================
# 8. EXECUÇÃO DAS ABLAÇÕES
# ============================================================

ablation_results22 = []
ablation_predictions22 = {}

# Inclui uma reconstrução do P2 completo para verificar
# a reprodutibilidade antes das ablações.
variants22 = {
    "P2_completo_reajustado": []
}

for group_name, group_features in GROUP_DEFINITIONS22.items():
    variants22[
        f"sem_{group_name}"
    ] = group_features

_project_print("\n" + "=" * 70)
_project_print("ESTUDO DE ABLAÇÃO — RANDOM FOREST P2")
_project_print("=" * 70)

for variant_name, removed_features in variants22.items():

    selected_features = [
        feature
        for feature in p2_features22
        if feature not in removed_features
    ]

    assert "generation_mw" in selected_features, (
        "A geração contemporânea deve ser preservada."
    )

    assert len(selected_features) > 0

    model22 = build_ablation_pipeline22(
        selected_features
    )

    model22.fit(
        train22[selected_features],
        y_train22
    )

    predictions22 = model22.predict(
        validation22[selected_features]
    )

    metrics22 = calculate_metrics22(
        y_validation22,
        predictions22
    )

    rmse_reduction22 = (
        100.0
        * (
            rmse_p2_frozen22
            - metrics22["rmse"]
        )
        / rmse_p2_frozen22
    )

    ablation_results22.append({
        "variant": variant_name,
        "n_features": len(selected_features),
        "n_removed": len(removed_features),
        "removed_features": ", ".join(removed_features),
        "rmse": metrics22["rmse"],
        "mae": metrics22["mae"],
        "r2": metrics22["r2"],
        "bias": metrics22["bias"],
        "rmse_gain_vs_frozen_p2_pct": rmse_reduction22,
        "evaluation_partition": "validation",
    })

    ablation_predictions22[variant_name] = (
        predictions22
    )

    _project_print(
        f"{variant_name:35s} | "
        f"atributos={len(selected_features):2d} | "
        f"RMSE={metrics22['rmse']:.4f} | "
        f"ganho={rmse_reduction22:+.2f}%"
    )

results22 = pd.DataFrame(
    ablation_results22
).sort_values("rmse").reset_index(drop=True)

# ============================================================
# 9. VERIFICAÇÃO DE REPRODUTIBILIDADE
# ============================================================

full_rmse22 = float(
    results22.loc[
        results22["variant"].eq(
            "P2_completo_reajustado"
        ),
        "rmse"
    ].iloc[0]
)

reproduction_difference22 = abs(
    full_rmse22 - rmse_p2_frozen22
)

_project_print("\nREPRODUÇÃO DO P2 ORIGINAL")
_project_print(
    f"RMSE congelado: {rmse_p2_frozen22:.4f}"
)
_project_print(
    f"RMSE reconstruído: {full_rmse22:.4f}"
)
_project_print(
    f"Diferença absoluta: "
    f"{reproduction_difference22:.6f}"
)

# Uma divergência relevante exige investigação,
# pois a comparação deve manter o mesmo protocolo.
if reproduction_difference22 > 0.05:
    warnings.warn(
        "O P2 completo reconstruído divergiu do modelo "
        "congelado. As ablações devem ser interpretadas "
        "com cautela até confirmar a equivalência "
        "do pré-processamento."
    )

# ============================================================
# 10. TABELA DE RESULTADOS
# ============================================================

_project_print("\nRESULTADOS DAS ABLAÇÕES")
_project_display(
    results22[
        [
            "variant",
            "n_features",
            "n_removed",
            "rmse",
            "mae",
            "r2",
            "bias",
            "rmse_gain_vs_frozen_p2_pct",
        ]
    ].round(4)
)

# ============================================================
# 11. FIGURA — RMSE DAS VARIANTES
# ============================================================

plot22 = results22.sort_values(
    "rmse",
    ascending=True
)

fig, ax = plt.subplots(
    figsize=(11, 6)
)

ax.barh(
    plot22["variant"],
    plot22["rmse"]
)

ax.axvline(
    rmse_p2_frozen22,
    linestyle="--",
    linewidth=1.5,
    label=(
        f"P2 congelado ({rmse_p2_frozen22:.2f})"
    )
)

ax.axvline(
    rmse_b0_frozen22,
    linestyle=":",
    linewidth=1.5,
    label=(
        f"B0 congelado ({rmse_b0_frozen22:.2f})"
    )
)

ax.set_xlabel(
    "RMSE de validação (t CO₂/h)"
)

ax.set_ylabel(
    "Variante experimental"
)

ax.set_title(
    "Iatan 2 — Ablação dos atributos de processos"
)

ax.legend()
ax.grid(axis="x", alpha=0.2)

plt.tight_layout()

FIGURE_FILE22 = (
    FIGURES22 / "cell22_ablation_rmse.png"
)

fig.savefig(
    FIGURE_FILE22,
    dpi=200,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 12. EXPORTAÇÃO
# ============================================================

RESULTS_FILE22 = (
    TABLES22 / "cell22_ablation_results.csv"
)

GROUPS_FILE22 = (
    TABLES22 / "cell22_ablation_groups.csv"
)

PREDICTIONS_FILE22 = (
    PROCESSED22 / "cell22_validation_predictions.parquet"
)

REPORT_FILE22 = (
    REPORTS22 / "cell22_ablation_report.json"
)

_project_csv(results22, 
    RESULTS_FILE22,
    index=False
)

_project_csv(groups_df22, 
    GROUPS_FILE22,
    index=False
)

prediction_table22 = pd.DataFrame({
    "timestamp": validation22["timestamp"].to_numpy(),
    "y_true": y_validation22,
})

for variant_name, values in ablation_predictions22.items():
    prediction_table22[
        f"pred_{variant_name}"
    ] = values

prediction_table22.to_parquet(
    PREDICTIONS_FILE22,
    index=False
)

report22 = {
    "cell": 22,
    "study": "exploratory_leave_one_group_out_ablation",
    "unit": "Iatan 2",
    "year": 2025,
    "algorithm": "RandomForest",
    "train_observations": int(len(train22)),
    "validation_observations": int(len(validation22)),
    "test_observations_evaluated": 0,
    "frozen_p2_validation_rmse": rmse_p2_frozen22,
    "frozen_b0_validation_rmse": rmse_b0_frozen22,
    "reconstructed_p2_validation_rmse": full_rmse22,
    "reproduction_difference": reproduction_difference22,
    "frozen_selection_modified": False,
    "hyperparameters_reoptimized": False,
    "groups": GROUP_DEFINITIONS22,
    "results": results22.to_dict(
        orient="records"
    ),
    "interpretation": (
        "Exploratory validation-only ablation. "
        "Results must not be used to replace "
        "the previously frozen final models."
    ),
}

with open(
    REPORT_FILE22,
    "w",
    encoding="utf-8"
) as f:
    _project_dump(
        report22,
        f,
        indent=2,
        ensure_ascii=False,
        default=str
    )

# ============================================================
# 13. RESUMO FINAL
# ============================================================

_project_print("\n" + "=" * 70)
_project_print("FASE 4 — CÉLULA 22")
_project_print("=" * 70)

_project_print(
    f"Variantes avaliadas: {len(results22)}"
)
_project_print(
    f"Grupos investigados: {len(GROUP_DEFINITIONS22)}"
)
_project_print(
    f"Observações de treinamento: {len(train22)}"
)
_project_print(
    f"Observações de validação: {len(validation22)}"
)
_project_print("Observações de teste avaliadas: 0")
_project_print("Seleção congelada modificada: NÃO")

_project_print("\nArquivos:")
_project_print(RESULTS_FILE22)
_project_print(GROUPS_FILE22)
_project_print(PREDICTIONS_FILE22)
_project_print(FIGURE_FILE22)
_project_print(REPORT_FILE22)

_project_print(
    "\nSTATUS: ESTUDO DE ABLAÇÃO CONCLUÍDO"
)
_project_print("=" * 70)


In [ ]:

# ============================================================
# FASE 4 — CÉLULA 22A
# AUDITORIA DE EQUIVALÊNCIA DO PRÉ-PROCESSAMENTO P2
# ============================================================
#
# OBJETIVO
# 1. Recuperar o modelo P2 RandomForest congelado.
# 2. Recuperar seu pré-processador original.
# 3. Reproduzir as previsões na validação.
# 4. Comparar o RMSE com o resultado da Célula 15.
# 5. Investigar a divergência observada na Célula 22.
#
# IMPORTANTE
# - Não utiliza dados de teste.
# - Não reajusta modelos.
# - Não modifica a seleção congelada.
# - Não sobrescreve arquivos anteriores.
# ============================================================

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error, mean_absolute_error

_project_print("=" * 70)
_project_print("FASE 4 — CÉLULA 22A")
_project_print("AUDITORIA DE EQUIVALÊNCIA DO P2")
_project_print("=" * 70)

# ------------------------------------------------------------
# 1. LOCALIZAÇÃO DOS ARQUIVOS
# ------------------------------------------------------------

ROOT22A = Path(PHASE_ROOT)

model_file22a = (
    ROOT22A / "models/cell14_p2_best_randomforest.joblib"
)

preprocessor_file22a = (
    ROOT22A /
    "models/cell08_preprocessor_p2_randomforest.joblib"
)

manifest_file22a = (
    ROOT22A / "manifests/cell06_feature_sets.json"
)

master_file22a = (
    ROOT22A / "data/processed/cell04_master_dataset.parquet"
)

if not master_file22a.exists():
    master_file22a = (
        ROOT22A / "data/processed/cell07_modeling_dataset.parquet"
    )

for path in [
    model_file22a,
    preprocessor_file22a,
    manifest_file22a
]:
    assert path.exists(), f"Arquivo não encontrado: {path}"

_project_print("\nArquivos de modelo e pré-processamento: LOCALIZADOS")

# ------------------------------------------------------------
# 2. RECUPERAÇÃO DOS OBJETOS ORIGINAIS
# ------------------------------------------------------------

model22a = joblib.load(model_file22a)
preprocessor22a = joblib.load(preprocessor_file22a)

with open(manifest_file22a, "r", encoding="utf-8") as f:
    feature_manifest22a = json.load(f)

_project_print("\nModelo:", type(model22a).__name__)
_project_print("Pré-processador:", type(preprocessor22a).__name__)

# ------------------------------------------------------------
# 3. IDENTIFICAÇÃO DOS ATRIBUTOS P2
# ------------------------------------------------------------

# Preferimos os atributos utilizados na Célula 22,
# preservando a ordem original da matriz.

if "p2_features" in globals():
    features22a = list(p2_features)
elif "FEATURE_SETS" in globals():
    features22a = list(FEATURE_SETS["P2"])
elif "feature_sets" in globals():
    features22a = list(feature_sets["P2"])
else:
    raise RuntimeError(
        "Não foi possível recuperar os atributos P2 "
        "da sessão. Execute anteriormente a Célula 06."
    )

assert len(features22a) == 35, (
    f"Esperados 35 atributos P2; encontrados {len(features22a)}."
)

_project_print("Atributos P2:", len(features22a))

# ------------------------------------------------------------
# 4. RECUPERAÇÃO DA MATRIZ DE MODELAGEM
# ------------------------------------------------------------

if "modeling_data" in globals():
    df22a = modeling_data.copy()
elif master_file22a.exists():
    df22a = pd.read_parquet(master_file22a)
else:
    raise RuntimeError(
        "Matriz de modelagem não disponível."
    )

# Se a partição não estiver na matriz, recuperar do master.

if "partition" not in df22a.columns:
    master22a = pd.read_parquet(
        ROOT22A / "data/processed/cell04_master_dataset.parquet"
    )

    df22a["timestamp"] = pd.to_datetime(
        df22a["timestamp"], utc=True
    )

    master22a["timestamp"] = pd.to_datetime(
        master22a["timestamp"], utc=True
    )

    df22a = df22a.merge(
        master22a[["timestamp", "partition"]],
        on="timestamp",
        how="left",
        validate="one_to_one"
    )

assert "partition" in df22a.columns

validation22a = df22a.loc[
    df22a["partition"].eq("validation")
].copy()

assert len(validation22a) == 687, (
    f"Validação inesperada: {len(validation22a)} linhas."
)

missing22a = [
    c for c in features22a
    if c not in validation22a.columns
]

assert not missing22a, (
    f"Atributos ausentes: {missing22a}"
)

X22a = validation22a[features22a].copy()

if "co2_metric_tons" in validation22a.columns:
    y22a = validation22a["co2_metric_tons"].to_numpy()
elif "y_true" in validation22a.columns:
    y22a = validation22a["y_true"].to_numpy()
else:
    raise RuntimeError(
        "Variável-alvo não localizada na matriz."
    )

assert np.isfinite(y22a).all()

_project_print("\nObservações de validação:", len(X22a))
_project_print("Dimensões de X:", X22a.shape)

# ------------------------------------------------------------
# 5. INSPEÇÃO DO PRÉ-PROCESSADOR
# ------------------------------------------------------------

_project_print("\n" + "=" * 70)
_project_print("INSPEÇÃO DO PRÉ-PROCESSADOR")
_project_print("=" * 70)

if hasattr(preprocessor22a, "feature_names_in_"):
    original_features22a = list(
        preprocessor22a.feature_names_in_
    )

    _project_print(
        "Atributos esperados pelo pré-processador:",
        len(original_features22a)
    )

    missing_original22a = [
        c for c in original_features22a
        if c not in X22a.columns
    ]

    extra_original22a = [
        c for c in X22a.columns
        if c not in original_features22a
    ]

    _project_print("Ausentes:", missing_original22a)
    _project_print("Extras:", extra_original22a)

    assert not missing_original22a

    X22a = X22a[original_features22a]

if hasattr(preprocessor22a, "named_steps"):
    _project_print(
        "Etapas:",
        list(preprocessor22a.named_steps.keys())
    )

# ------------------------------------------------------------
# 6. REPRODUÇÃO COM PRÉ-PROCESSAMENTO ORIGINAL
# ------------------------------------------------------------

X_transformed22a = preprocessor22a.transform(X22a)

pred22a = np.asarray(
    model22a.predict(X_transformed22a)
).reshape(-1)

assert len(pred22a) == len(y22a)
assert np.isfinite(pred22a).all()

rmse22a = float(
    np.sqrt(mean_squared_error(y22a, pred22a))
)

mae22a = float(
    mean_absolute_error(y22a, pred22a)
)

with (Path(DIRS["root"]) / "manifests/cell15_frozen_model_selection.json").open(
    encoding="utf-8"
) as manifest_file22a:
    selection22a = json.load(manifest_file22a)
frozen_p2_22a = [row for row in selection22a["selected_models"] if row["scenario"] == "P2"]
assert len(frozen_p2_22a) == 1
assert frozen_p2_22a[0]["algorithm"] == "RandomForest"
original_rmse22a = float(frozen_p2_22a[0]["validation_rmse"])

difference22a = abs(
    rmse22a - original_rmse22a
)

_project_print("\n" + "=" * 70)
_project_print("REPRODUÇÃO DO P2 CONGELADO")
_project_print("=" * 70)

_project_print(f"RMSE original registrado: {original_rmse22a:.4f}")
_project_print(f"RMSE reproduzido:         {rmse22a:.4f}")
_project_print(f"MAE reproduzido:          {mae22a:.4f}")
_project_print(f"Diferença absoluta:       {difference22a:.6f}")

# A checagem reporta equivalência a quatro casas decimais.
equivalent22a = (
    round(rmse22a, 4) == round(original_rmse22a, 4)
)

_project_print(
    "\nEquivalência na precisão registrada:",
    "VERIFICADA" if equivalent22a else "NÃO VERIFICADA"
)

# ------------------------------------------------------------
# 7. REGISTRO DA AUDITORIA
# ------------------------------------------------------------

report22a = {
    "cell": "22A",
    "purpose": "P2 preprocessing equivalence audit",
    "scenario": "P2",
    "algorithm": "RandomForest",
    "validation_n": int(len(y22a)),
    "n_features": int(X22a.shape[1]),
    "original_rmse": original_rmse22a,
    "reproduced_rmse": rmse22a,
    "reproduced_mae": mae22a,
    "absolute_difference": difference22a,
    "equivalent_at_4_decimals": bool(equivalent22a),
    "model_refitted": False,
    "test_evaluated": False
}

report_path22a = (
    ROOT22A /
    "reports/cell22a_preprocessing_equivalence.json"
)

report_path22a.parent.mkdir(
    parents=True, exist_ok=True
)

with open(report_path22a, "w", encoding="utf-8") as f:
    _project_dump(
        report22a,
        f,
        indent=2,
        ensure_ascii=False
    )

_project_print("\nRelatório:", report_path22a)

_project_print("\n" + "=" * 70)
_project_print("STATUS:", (
    "EQUIVALÊNCIA VERIFICADA"
    if equivalent22a
    else "DIVERGÊNCIA A INVESTIGAR"
))
_project_print("=" * 70)


In [ ]:

# ============================================================
# FASE 4 — CÉLULA 23
# CONSOLIDAÇÃO, EXPORTAÇÃO E RASTREABILIDADE
# ============================================================
#
# PROJETO:
# Estimativa horária de emissões de CO2 — Iatan, Unidade 2
#
# OBJETIVOS:
# 1. Recuperar os resultados das células anteriores.
# 2. Verificar arquivos e integridade dos experimentos.
# 3. Consolidar desempenho em validação e teste.
# 4. Registrar os intervalos de confiança.
# 5. Consolidar os resultados de ablação como exploratórios.
# 6. Produzir tabelas e manifesto para o relatório final.
#
# PROTOCOLO:
# - Nenhum modelo será treinado.
# - Nenhuma previsão será recalculada.
# - Nenhum hiperparâmetro será alterado.
# - A seleção congelada será preservada.
# - Os resultados existentes serão apenas recuperados.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import json
import hashlib

# ------------------------------------------------------------
# 1. CONFIGURAÇÃO
# ------------------------------------------------------------

ROOT23 = Path(PHASE_ROOT)

TABLES23 = ROOT23 / "tables"
REPORTS23 = ROOT23 / "reports"
MANIFESTS23 = ROOT23 / "manifests"
PROCESSED23 = ROOT23 / "data/processed"
MODELS23 = ROOT23 / "models"

for directory in [
    TABLES23, REPORTS23, MANIFESTS23
]:
    directory.mkdir(parents=True, exist_ok=True)

_project_print("=" * 70)
_project_print("FASE 4 — CÉLULA 23")
_project_print("CONSOLIDAÇÃO DOS RESULTADOS")
_project_print("=" * 70)

# ------------------------------------------------------------
# 2. FUNÇÕES AUXILIARES
# ------------------------------------------------------------

def read_csv23(filename, required=True):
    path = TABLES23 / filename

    if not path.exists():
        if required:
            raise FileNotFoundError(path)
        return None

    return pd.read_csv(path)


def read_json23(directory, filename, required=True):
    path = directory / filename

    if not path.exists():
        if required:
            raise FileNotFoundError(path)
        return None

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def sha256_23(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            digest.update(block)

    return digest.hexdigest()


def save_json23(data, path):
    with open(path, "w", encoding="utf-8") as f:
        _project_dump(
            data,
            f,
            indent=2,
            ensure_ascii=False,
            default=str
        )


# ------------------------------------------------------------
# 3. RECUPERAÇÃO DOS RESULTADOS PRINCIPAIS
# ------------------------------------------------------------

validation23 = read_csv23(
    "cell15_best_12_models.csv"
)

test23 = read_csv23(
    "cell16_final_test_metrics.csv"
)

comparisons23 = read_csv23(
    "cell16_final_test_comparisons.csv"
)

paired23 = read_csv23(
    "cell17_paired_comparison.csv"
)

ci_models23 = read_csv23(
    "cell21_model_confidence_intervals.csv"
)

ci_paired23 = read_csv23(
    "cell21_paired_confidence_intervals.csv"
)

audit22a23 = read_json23(
    REPORTS23,
    "cell22a_preprocessing_equivalence.json"
)

selection23 = read_json23(
    MANIFESTS23,
    "cell15_frozen_model_selection.json"
)

release23 = read_json23(
    MANIFESTS23,
    "cell15a_test_release.json"
)

_project_print("\nArquivos principais: RECUPERADOS")

# ------------------------------------------------------------
# 4. VERIFICAÇÕES ESTRUTURAIS
# ------------------------------------------------------------

required_validation23 = {
    "scenario", "algorithm", "rmse", "mae"
}

required_test23 = {
    "scenario", "algorithm", "rmse",
    "mae", "r2", "bias"
}

assert required_validation23.issubset(
    validation23.columns
), "Estrutura inesperada na tabela de validação."

assert required_test23.issubset(
    test23.columns
), "Estrutura inesperada na tabela de teste."

assert set(test23["scenario"]) == {
    "B0", "P1", "P2"
}, "Os três cenários não estão presentes no teste."

assert len(test23) == 3, (
    "Esperadas exatamente três avaliações finais."
)

assert len(validation23) == 12, (
    "Esperadas 12 combinações principais."
)

assert bool(
    audit22a23["equivalent_at_4_decimals"]
), "A equivalência da Célula 22A não foi confirmada."

_project_print("Cenários finais: B0, P1, P2")
_project_print("Combinações principais:", len(validation23))
_project_print("Modelos avaliados no teste:", len(test23))
_project_print("Equivalência P2 — Célula 22A: VERIFICADA")

# ------------------------------------------------------------
# 5. CONSOLIDAÇÃO VALIDAÇÃO × TESTE
# ------------------------------------------------------------

validation_best23 = (
    validation23
    .sort_values("rmse")
    .groupby("scenario", as_index=False)
    .first()
)

validation_best23 = validation_best23[
    ["scenario", "algorithm", "rmse", "mae"]
].rename(
    columns={
        "algorithm": "algorithm_validation",
        "rmse": "rmse_validation",
        "mae": "mae_validation"
    }
)

test_summary23 = test23[
    ["scenario", "algorithm", "rmse",
     "mae", "r2", "bias"]
].rename(
    columns={
        "algorithm": "algorithm_test",
        "rmse": "rmse_test",
        "mae": "mae_test",
        "r2": "r2_test",
        "bias": "bias_test"
    }
)

summary23 = validation_best23.merge(
    test_summary23,
    on="scenario",
    how="inner",
    validate="one_to_one"
)

assert (
    summary23["algorithm_validation"]
    == summary23["algorithm_test"]
).all(), (
    "Algoritmo final diferente do selecionado "
    "na validação."
)

summary23["rmse_change_pct"] = (
    (
        summary23["rmse_test"]
        - summary23["rmse_validation"]
    )
    / summary23["rmse_validation"]
    * 100
)

summary23 = (
    summary23
    .sort_values(
        "scenario",
        key=lambda s: s.map(
            {"B0": 0, "P1": 1, "P2": 2}
        )
    )
    .reset_index(drop=True)
)

summary_path23 = (
    TABLES23 / "cell23_validation_test_summary.csv"
)

_project_csv(summary23, 
    summary_path23,
    index=False
)

_project_print("\n" + "=" * 70)
_project_print("DESEMPENHO CONSOLIDADO")
_project_print("=" * 70)

_project_display(
    summary23.round(4)
)

# ------------------------------------------------------------
# 6. CONSOLIDAÇÃO DOS INTERVALOS DE CONFIANÇA
# ------------------------------------------------------------

ci_models_export23 = ci_models23.copy()
ci_paired_export23 = ci_paired23.copy()

_project_csv(ci_models_export23, 
    TABLES23 / "cell23_model_confidence_intervals.csv",
    index=False
)

_project_csv(ci_paired_export23, 
    TABLES23 / "cell23_paired_confidence_intervals.csv",
    index=False
)

_project_print("\nIntervalos de confiança: CONSOLIDADOS")

# ------------------------------------------------------------
# 7. RESULTADOS DE ABLAÇÃO
# ------------------------------------------------------------

# Recuperamos o arquivo existente, sem recalcular
# variantes ou modificar os modelos originais.

ablation_files23 = sorted(
    TABLES23.glob("cell22*.csv")
)

ablation_inventory23 = []

for path in ablation_files23:
    df_tmp = pd.read_csv(path)

    ablation_inventory23.append({
        "filename": path.name,
        "rows": int(len(df_tmp)),
        "columns": int(len(df_tmp.columns)),
        "sha256": sha256_23(path)
    })

ablation_status23 = (
    "exploratory_preprocessing_not_fully_equivalent"
)

_project_print("\nArquivos de ablação encontrados:",
      len(ablation_inventory23))

_project_print(
    "Interpretação:",
    "EXPLORATÓRIA — equivalência das variantes "
    "ainda não demonstrada"
)

# ------------------------------------------------------------
# 8. INVENTÁRIO DE ARQUIVOS DA FASE 4
# ------------------------------------------------------------

inventory23 = []

categories23 = {
    "models": MODELS23,
    "tables": TABLES23,
    "reports": REPORTS23,
    "manifests": MANIFESTS23,
    "processed": PROCESSED23
}

for category, directory in categories23.items():

    if not directory.exists():
        continue

    for path in sorted(directory.rglob("*")):

        if not path.is_file():
            continue

        inventory23.append({
            "category": category,
            "relative_path": str(
                path.relative_to(ROOT23)
            ),
            "size_bytes": int(path.stat().st_size),
            "sha256": sha256_23(path)
        })

inventory_df23 = pd.DataFrame(inventory23)

inventory_path23 = (
    TABLES23 / "cell23_artifact_inventory.csv"
)

_project_csv(inventory_df23, 
    inventory_path23,
    index=False
)

_project_print("\nArquivos inventariados:", len(inventory_df23))

_project_print("\nDistribuição por categoria:")
_project_display(
    inventory_df23.groupby(
        "category", as_index=False
    ).size()
)

# ------------------------------------------------------------
# 9. REGISTRO DAS CONCLUSÕES
# ------------------------------------------------------------

rmse23 = dict(
    zip(
        summary23["scenario"],
        summary23["rmse_test"]
    )
)

gain_p1_23 = (
    (rmse23["B0"] - rmse23["P1"])
    / rmse23["B0"] * 100
)

gain_p2_23 = (
    (rmse23["B0"] - rmse23["P2"])
    / rmse23["B0"] * 100
)

conclusions23 = {
    "primary_metric": "RMSE",
    "unit": "metric_tons_CO2_per_hour",
    "test_observations": 1063,
    "b0_rmse_test": float(rmse23["B0"]),
    "p1_rmse_test": float(rmse23["P1"]),
    "p2_rmse_test": float(rmse23["P2"]),
    "p1_rmse_gain_vs_b0_pct": float(gain_p1_23),
    "p2_rmse_gain_vs_b0_pct": float(gain_p2_23),
    "interpretation": (
        "P1 apresentou ganho pontual muito pequeno "
        "em RMSE, sem evidência robusta de superioridade. "
        "P2 apresentou deterioração no teste, apesar "
        "do melhor desempenho na validação."
    ),
    "limitations": [
        "Estudo de caso de uma única unidade termelétrica.",
        "Amostra temporal restrita ao ano de 2025.",
        "O teste já havia sido examinado na Fase 3.",
        "Ablações exploratórias utilizaram reconstrução "
        "de pré-processamento não integralmente equivalente."
    ]
}

# ------------------------------------------------------------
# 10. MANIFESTO FINAL
# ------------------------------------------------------------

manifest23 = {
    "phase": "04",
    "cell": "23",
    "project": "co2-cems-thermal-modeling",
    "facility": "Iatan",
    "unit": "2",
    "year": 2025,
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "scenarios": ["B0", "P1", "P2"],
    "algorithms_evaluated": [
        "ElasticNet",
        "RandomForest",
        "XGBoost",
        "HistGradientBoosting"
    ],
    "validation_combinations": 12,
    "hyperparameter_candidates": 72,
    "final_test_models": 3,
    "model_selection_frozen": True,
    "test_recomputed_in_cell23": False,
    "models_refitted_in_cell23": False,
    "preprocessing_equivalence": {
        "scenario": "P2",
        "status": "verified",
        "rmse_original": audit22a23["original_rmse"],
        "rmse_reproduced": audit22a23["reproduced_rmse"]
    },
    "ablation": {
        "status": ablation_status23,
        "files": ablation_inventory23
    },
    "conclusions": conclusions23,
    "source_manifests": [
        "cell15_frozen_model_selection.json",
        "cell15a_test_release.json",
        "cell06_feature_sets.json"
    ],
    "outputs": {
        "summary": str(summary_path23),
        "inventory": str(inventory_path23),
        "model_confidence_intervals": str(
            TABLES23 /
            "cell23_model_confidence_intervals.csv"
        ),
        "paired_confidence_intervals": str(
            TABLES23 /
            "cell23_paired_confidence_intervals.csv"
        )
    }
}

manifest_path23 = (
    MANIFESTS23 /
    "cell23_final_results_manifest.json"
)

save_json23(
    manifest23,
    manifest_path23
)

report_path23 = (
    REPORTS23 /
    "cell23_consolidation_report.json"
)

save_json23(
    {
        "status": "completed",
        "summary": summary23.to_dict(
            orient="records"
        ),
        "conclusions": conclusions23,
        "artifact_count": len(inventory_df23),
        "ablation_status": ablation_status23
    },
    report_path23
)

# ------------------------------------------------------------
# 11. RESUMO FINAL
# ------------------------------------------------------------

_project_print("\n" + "=" * 70)
_project_print("RESULTADOS CENTRAIS")
_project_print("=" * 70)

_project_print(
    f"B0 — RMSE teste: {rmse23['B0']:.4f}"
)

_project_print(
    f"P1 — RMSE teste: {rmse23['P1']:.4f}"
)

_project_print(
    f"P2 — RMSE teste: {rmse23['P2']:.4f}"
)

_project_print(
    f"\nGanho P1 versus B0: {gain_p1_23:+.4f}%"
)

_project_print(
    f"Ganho P2 versus B0: {gain_p2_23:+.4f}%"
)

_project_print("\n" + "=" * 70)
_project_print("FASE 4 — CÉLULA 23")
_project_print("=" * 70)

_project_print("Modelos reajustados: 0")
_project_print("Previsões recalculadas: 0")
_project_print("Seleção congelada: PRESERVADA")
_project_print("Equivalência P2: VERIFICADA")
_project_print("Ablação: EXPLORATÓRIA")

_project_print("\nResumo:", summary_path23)
_project_print("Inventário:", inventory_path23)
_project_print("Manifesto:", manifest_path23)
_project_print("Relatório:", report_path23)

_project_print("\nSTATUS: CONSOLIDAÇÃO CONCLUÍDA")
_project_print("=" * 70)


In [ ]:

# ============================================================
# FASE 4 — CÉLULA 24
# RELATÓRIO CIENTÍFICO E ENCERRAMENTO DA FASE
# ============================================================
#
# PROJETO:
# Modelagem de emissões horárias de CO2 — Iatan, Unidade 2
#
# OBJETIVOS:
# 1. Consolidar a metodologia experimental.
# 2. Documentar resultados de validação e teste.
# 3. Incorporar os intervalos de confiança.
# 4. Registrar conclusões e limitações.
# 5. Gerar relatório científico em Markdown.
# 6. Registrar o encerramento da Fase 4.
#
# NÃO REALIZA:
# - Treinamento ou reajuste de modelos.
# - Seleção de hiperparâmetros.
# - Novas previsões.
# - Novas análises sobre o conjunto de teste.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. DIRETÓRIOS
# ------------------------------------------------------------

ROOT24 = Path(PHASE_ROOT)

TABLES24 = ROOT24 / "tables"
REPORTS24 = ROOT24 / "reports"
MANIFESTS24 = ROOT24 / "manifests"

REPORTS24.mkdir(parents=True, exist_ok=True)
MANIFESTS24.mkdir(parents=True, exist_ok=True)

_project_print("=" * 70)
_project_print("FASE 4 — CÉLULA 24")
_project_print("RELATÓRIO CIENTÍFICO E ENCERRAMENTO")
_project_print("=" * 70)

# ------------------------------------------------------------
# 2. RECUPERAÇÃO DOS RESULTADOS
# ------------------------------------------------------------

summary24 = pd.read_csv(
    TABLES24 / "cell23_validation_test_summary.csv"
)

ci_models24 = pd.read_csv(
    TABLES24 / "cell23_model_confidence_intervals.csv"
)

ci_paired24 = pd.read_csv(
    TABLES24 / "cell23_paired_confidence_intervals.csv"
)

with open(
    MANIFESTS24 / "cell23_final_results_manifest.json",
    "r",
    encoding="utf-8"
) as f:
    manifest23_24 = json.load(f)

with open(
    REPORTS24 / "cell22a_preprocessing_equivalence.json",
    "r",
    encoding="utf-8"
) as f:
    audit22a24 = json.load(f)

assert set(summary24["scenario"]) == {"B0", "P1", "P2"}
assert len(summary24) == 3
assert audit22a24["equivalent_at_4_decimals"] is True

summary24 = summary24.set_index("scenario")

_project_print("\nResultados consolidados: RECUPERADOS")
_project_print("Equivalência P2: VERIFICADA")

# ------------------------------------------------------------
# 3. ORGANIZAÇÃO DAS MÉTRICAS
# ------------------------------------------------------------

def get_metrics24(scenario):
    row = summary24.loc[scenario]

    return {
        "algorithm": str(row["algorithm_test"]),
        "rmse_validation": float(row["rmse_validation"]),
        "mae_validation": float(row["mae_validation"]),
        "rmse_test": float(row["rmse_test"]),
        "mae_test": float(row["mae_test"]),
        "r2_test": float(row["r2_test"]),
        "bias_test": float(row["bias_test"])
    }


results24 = {
    scenario: get_metrics24(scenario)
    for scenario in ["B0", "P1", "P2"]
}

rmse_b0 = results24["B0"]["rmse_test"]
rmse_p1 = results24["P1"]["rmse_test"]
rmse_p2 = results24["P2"]["rmse_test"]

gain_p1 = 100 * (rmse_b0 - rmse_p1) / rmse_b0
gain_p2 = 100 * (rmse_b0 - rmse_p2) / rmse_b0

# ------------------------------------------------------------
# 4. EXTRAÇÃO DOS INTERVALOS DE CONFIANÇA
# ------------------------------------------------------------

required_ci24 = {
    "reference",
    "candidate",
    "metric",
    "ci_lower_95",
    "ci_upper_95"
}

assert required_ci24.issubset(ci_paired24.columns)

def get_paired_ci24(reference, candidate, metric="rmse"):

    subset = ci_paired24.loc[
        (ci_paired24["reference"] == reference)
        & (ci_paired24["candidate"] == candidate)
        & (ci_paired24["metric"] == metric)
    ]

    assert len(subset) == 1, (
        f"Intervalo não localizado: "
        f"{reference} versus {candidate}, {metric}"
    )

    row = subset.iloc[0]

    return {
        "lower": float(row["ci_lower_95"]),
        "upper": float(row["ci_upper_95"])
    }


ci_p1 = get_paired_ci24("B0", "P1")
ci_p2 = get_paired_ci24("B0", "P2")

# ------------------------------------------------------------
# 5. SÍNTESE CIENTÍFICA
# ------------------------------------------------------------

scientific_summary24 = {
    "study": {
        "facility": "Iatan",
        "unit": "2",
        "year": 2025,
        "data_source": "US EPA CEMS",
        "target": "hourly_CO2_metric_tons",
        "experimental_protocol": (
            "Contemporaneous hourly emission estimation "
            "using information available through hour t"
        )
    },
    "experimental_design": {
        "scenarios": {
            "B0": "Current generation",
            "P1": "Generation plus HMM operational states",
            "P2": "Generation, HMM states and process features"
        },
        "algorithms": [
            "ElasticNet",
            "RandomForest",
            "XGBoost",
            "HistGradientBoosting"
        ],
        "candidate_configurations": 72,
        "validation_combinations": 12,
        "test_models": 3,
        "train_observations": 4499,
        "validation_observations": 687,
        "test_observations": 1063,
        "primary_metric": "RMSE"
    },
    "results": results24,
    "paired_rmse_comparison": {
        "P1_vs_B0": {
            "gain_pct": float(gain_p1),
            "ci_lower_95": ci_p1["lower"],
            "ci_upper_95": ci_p1["upper"]
        },
        "P2_vs_B0": {
            "gain_pct": float(gain_p2),
            "ci_lower_95": ci_p2["lower"],
            "ci_upper_95": ci_p2["upper"]
        }
    },
    "methodological_integrity": {
        "selection_frozen_before_phase4_test": True,
        "hmm_provenance_audited": True,
        "p2_preprocessing_reproduced": True,
        "ablation_results_exploratory": True,
        "test_previously_examined_in_phase3": True
    },
    "conclusion": (
        "The addition of HMM states produced no robust "
        "improvement over the generation-only baseline. "
        "The richer process-feature model improved validation "
        "performance but deteriorated on the temporal test set."
    )
}

# ------------------------------------------------------------
# 6. RELATÓRIO CIENTÍFICO EM MARKDOWN
# ------------------------------------------------------------

report24 = f"""# Fase 4 — Modelagem preditiva e avaliação comparativa

## 1. Objetivo

Avaliar o valor incremental de atributos derivados de mineração
de processos na estimativa horária das emissões de CO₂ de uma
unidade termelétrica, utilizando registros CEMS da US EPA.

**Unidade estudada:** Iatan, Unidade 2.

**Período:** 2025.

**Variável-alvo:** emissões horárias de CO₂ em toneladas métricas.

O protocolo corresponde à estimativa contemporânea das emissões,
considerando informações operacionais disponíveis até a hora t.
Não corresponde à previsão antecipada de emissões futuras.

## 2. Desenho experimental

Foram comparados três cenários:

- **B0:** geração elétrica contemporânea.
- **P1:** geração elétrica e cinco indicadores de estados HMM.
- **P2:** geração, estados HMM e atributos derivados de processos.

Foram avaliados quatro algoritmos:

- ElasticNet;
- Random Forest;
- XGBoost;
- HistGradientBoosting.

O experimento compreendeu 72 configurações candidatas e
12 combinações principais de cenário e algoritmo.

### Partições temporais

| Partição | Observações |
|---|---:|
| Treinamento | 4.499 |
| Validação | 687 |
| Teste | 1.063 |

A seleção dos modelos foi realizada utilizando a validação.
Os três modelos selecionados foram congelados antes da
avaliação de teste realizada na Fase 4.

## 3. Resultados de validação

| Cenário | Algoritmo | RMSE (t CO₂/h) |
|---|---|---:|
| B0 | {results24['B0']['algorithm']} | {results24['B0']['rmse_validation']:.4f} |
| P1 | {results24['P1']['algorithm']} | {results24['P1']['rmse_validation']:.4f} |
| P2 | {results24['P2']['algorithm']} | {results24['P2']['rmse_validation']:.4f} |

Na validação, o cenário P2 apresentou o menor RMSE.

## 4. Resultados finais no teste

| Cenário | RMSE | MAE | R² | Viés |
|---|---:|---:|---:|---:|
| B0 | {results24['B0']['rmse_test']:.4f} | {results24['B0']['mae_test']:.4f} | {results24['B0']['r2_test']:.4f} | {results24['B0']['bias_test']:+.4f} |
| P1 | {results24['P1']['rmse_test']:.4f} | {results24['P1']['mae_test']:.4f} | {results24['P1']['r2_test']:.4f} | {results24['P1']['bias_test']:+.4f} |
| P2 | {results24['P2']['rmse_test']:.4f} | {results24['P2']['mae_test']:.4f} | {results24['P2']['r2_test']:.4f} | {results24['P2']['bias_test']:+.4f} |

Os resultados mostram que o cenário P1 apresentou desempenho
muito próximo ao baseline convencional.

O cenário P2, embora superior na validação, apresentou maior
erro no conjunto de teste.

## 5. Comparações pareadas e incerteza

A comparação entre os cenários utilizou bootstrap temporal
em blocos, com 2.000 reamostragens.

### P1 versus B0

- Redução pontual do RMSE: {gain_p1:+.4f}%.
- IC bootstrap de 95%: [{ci_p1['lower']:.4f}%, {ci_p1['upper']:.4f}%].

O intervalo inclui zero, não fornecendo evidência robusta
de superioridade de P1 sobre B0.

### P2 versus B0

- Redução pontual do RMSE: {gain_p2:+.4f}%.
- IC bootstrap de 95%: [{ci_p2['lower']:.4f}%, {ci_p2['upper']:.4f}%].

O intervalo é inteiramente negativo, indicando deterioração
do desempenho de P2 em relação a B0 neste teste.

Esses intervalos dependem das hipóteses do procedimento
de reamostragem temporal adotado.

## 6. Interpretação dos resultados

A análise demonstra que a inclusão de atributos derivados
de mineração de processos não assegura melhoria na capacidade
de generalização temporal.

Os estados HMM isoladamente acrescentaram pouco ao desempenho
do baseline baseado em geração.

A combinação ampliada de atributos de processos apresentou
melhor desempenho na validação, mas não sustentou essa vantagem
no período de teste.

As análises condicionais indicaram heterogeneidade do erro
entre faixas de geração, estados HMM e atividades operacionais.

## 7. Importância dos atributos

A análise por permutação identificou a geração elétrica
contemporânea como o atributo dominante nos três cenários.

No cenário P2, atributos históricos de geração apresentaram
contribuições adicionais, embora substancialmente menores.

A importância por permutação mede sensibilidade preditiva
do modelo e não estabelece causalidade.

## 8. Análise de ablação

A ablação exploratória indicou que retirar defasagens de
geração ou estatísticas móveis deteriorou o desempenho
de validação.

Entretanto, as variantes de ablação utilizaram uma
reconstrução de pré-processamento cuja equivalência integral
ao procedimento original ainda não foi demonstrada.

Por isso, os resultados de ablação são classificados como
exploratórios e não sustentam conclusões confirmatórias.

A Célula 22A confirmou a reprodução do P2 congelado:

- RMSE original: {audit22a24['original_rmse']:.4f}.
- RMSE reproduzido: {audit22a24['reproduced_rmse']:.4f}.

Essa verificação não demonstra, por si só, a equivalência
das variantes reajustadas.

## 9. Integridade e rastreabilidade

Foram adotados:

- Separação cronológica de treinamento, validação e teste;
- Ajuste do pré-processamento utilizando treinamento;
- Seleção dos modelos exclusivamente pela validação na Fase 4;
- Congelamento dos modelos antes do teste da Fase 4;
- Registro dos experimentos em MLflow;
- Auditoria de proveniência temporal do HMM;
- Verificação de integridade dos modelos;
- Reprodução do P2 congelado.

## 10. Limitações

1. Estudo restrito a uma única unidade termelétrica.
2. Dados correspondentes a um único ano.
3. Diferenças de distribuição entre períodos temporais.
4. Atributos de processos possivelmente redundantes com geração.
5. Resultados de ablação ainda exploratórios.
6. O conjunto de teste já havia sido examinado na Fase 3,
   de modo que não constitui um holdout completamente inédito
   em relação ao projeto como um todo.
7. Os resultados descrevem estimativa contemporânea de
   emissões, não previsão antecipada nem inferência causal.

## 11. Conclusão

Os resultados não sustentam a hipótese de que a incorporação
de atributos de mineração de processos melhora necessariamente
a estimativa horária de emissões de CO₂.

O cenário P1 apresentou desempenho praticamente equivalente
ao baseline B0.

O cenário P2 apresentou ganho na validação, mas deterioração
estatisticamente consistente no teste temporal, segundo o
bootstrap em blocos empregado.

Assim, a principal contribuição empírica é evidenciar os
limites de generalização de representações operacionais
derivadas de mineração de processos para esta unidade e
este período.

Essa conclusão deve ser interpretada no contexto do desenho
experimental e não generalizada automaticamente para outras
usinas ou tecnologias.
"""

report_md_path24 = (
    REPORTS24 / "cell24_scientific_report.md"
)

_project_write_text(report_md_path24, 
    report24,
    encoding="utf-8"
)

# ------------------------------------------------------------
# 7. EXPORTAÇÃO DA SÍNTESE E MANIFESTO
# ------------------------------------------------------------

summary_json_path24 = (
    REPORTS24 / "cell24_scientific_summary.json"
)

with open(
    summary_json_path24,
    "w",
    encoding="utf-8"
) as f:
    _project_dump(
        scientific_summary24,
        f,
        indent=2,
        ensure_ascii=False
    )

completion_manifest24 = {
    "phase": "04",
    "status": "completed",
    "completed_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "cells_completed": [
        *list(range(1, 25))
    ],
    "additional_audits": ["15A", "22A"],
    "models_refitted_in_cell24": 0,
    "test_predictions_recomputed_in_cell24": 0,
    "model_selection_modified": False,
    "primary_result": (
        "No robust temporal generalization improvement "
        "from process-mining features"
    ),
    "limitations_recorded": True,
    "ablation_status": "exploratory",
    "outputs": {
        "scientific_report": str(report_md_path24),
        "scientific_summary": str(summary_json_path24)
    }
}

completion_path24 = (
    MANIFESTS24 / "cell24_phase_completion.json"
)

with open(
    completion_path24,
    "w",
    encoding="utf-8"
) as f:
    _project_dump(
        completion_manifest24,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 8. VERIFICAÇÃO DOS ARQUIVOS PRODUZIDOS
# ------------------------------------------------------------

outputs24 = [
    report_md_path24,
    summary_json_path24,
    completion_path24
]

for path in outputs24:
    assert path.exists()
    assert path.stat().st_size > 0

_project_print("\n" + "=" * 70)
_project_print("RESULTADOS FINAIS")
_project_print("=" * 70)

for scenario in ["B0", "P1", "P2"]:
    m = results24[scenario]
    _project_print(
        f"{scenario}: "
        f"RMSE={m['rmse_test']:.4f} | "
        f"MAE={m['mae_test']:.4f} | "
        f"R²={m['r2_test']:.4f}"
    )

_project_print(
    f"\nP1 versus B0: {gain_p1:+.4f}% "
    f"IC95% [{ci_p1['lower']:.4f}, "
    f"{ci_p1['upper']:.4f}]"
)

_project_print(
    f"P2 versus B0: {gain_p2:+.4f}% "
    f"IC95% [{ci_p2['lower']:.4f}, "
    f"{ci_p2['upper']:.4f}]"
)

_project_print("\n" + "=" * 70)
_project_print("ARQUIVOS GERADOS")
_project_print("=" * 70)

for path in outputs24:
    _project_print(path)

_project_print("\n" + "=" * 70)
_project_print("ENCERRAMENTO DA FASE 4")
_project_print("=" * 70)

_project_print("Modelos reajustados: 0")
_project_print("Previsões recalculadas: 0")
_project_print("Seleção congelada: PRESERVADA")
_project_print("Rastreabilidade: DOCUMENTADA")
_project_print("Limitações: REGISTRADAS")
_project_print("Ablação: EXPLORATÓRIA")
_project_print("Relatório científico: GERADO")

_project_print("\nSTATUS: FASE 4 CONCLUÍDA")
_project_print("=" * 70)


In [ ]:
# FASE 4 V2 — AUDITORIA CIENTÍFICA, NÃO DESTRUTIVA
# Executar após a Célula 24. Sem treinamento ou escolha baseada no teste.
from pathlib import Path
import json, hashlib
import pandas as pd
import numpy as np
from IPython.display import display

review_dir = Path(PHASE_ROOT) / 'reports' / 'v2_scientific_audit'
review_dir.mkdir(parents=True, exist_ok=True)
summary_path = Path(DIRS['tables']) / 'cell23_validation_test_summary.csv'
if not summary_path.exists():
    raise FileNotFoundError(f'Execute a consolidação da Célula 23: {summary_path}')
summary = pd.read_csv(summary_path)
_project_print('Colunas disponíveis:', list(summary.columns))
_project_display(summary)
_project_csv(summary, review_dir / 'original_validation_test_summary_snapshot.csv',index=False)

# Inspeção descritiva de mudanças temporais; sem recalibração e sem escolha de modelos.
master_candidates = [Path(PHASE_ROOT) / 'data/processed/cell04_master_dataset.parquet',
                     Path(PHASE_ROOT) / 'data/processed/cell07_modeling_dataset.parquet']
master_path = next((p for p in master_candidates if p.exists()),None)
if master_path is not None:
    master = pd.read_parquet(master_path)
    columns = list(master.columns)
    _project_print('Dataset mestre:', master_path.name, 'linhas:', len(master))
    _project_print('Colunas temporais/carga/CO2 candidatas:',
          [c for c in columns if any(s in c.lower() for s in ('date','time','load','co2','gross','generation'))][:35])
else:
    _project_print('Base mestre não encontrada; auditoria temporal quantitativa pendente.')

manifest = {
    'phase':'04', 'revision':'v2.0', 'year':2025,
    'frozen_models_unchanged':True,
    'original_test_already_examined':True,
    'test_not_eligible_for_new_model_selection':True,
    'limitations':['Validação set-out atípica', 'Teste nov-dez previamente consultado',
                   'Comparação entre algoritmos distintos não isola efeito das features',
                   'Novo teste independente requer dados não examinados'],
    'next_experiments':['Validação temporal aninhada somente no desenvolvimento',
                        'Comparação pareada B0/P1/P2 sob mesmo algoritmo e mesmas linhas',
                        'Diagnóstico de erro por mês, carga e início do ciclo',
                        'Auditoria da intensidade de carbono por faixa de carga'],
    'summary_sha256':hashlib.sha256(summary_path.read_bytes()).hexdigest(),
}
with open(review_dir / 'phase04_v2_scientific_manifest.json','w',encoding='utf-8') as f:
    _project_dump(manifest,f,indent=2,ensure_ascii=False)
_project_print('Auditoria v2 registrada em:',review_dir)


## Auditorias complementares — Fase 4 (não destrutivas)

As células seguintes **não alteram os modelos congelados**. Avaliam a sensibilidade a três horas extremas do treinamento, a mudança de distribuição entre partições e a rastreabilidade das ablações. Os resultados são **exploratórios**: o teste já foi consultado e não deve orientar nova seleção de modelos. Execute após todas as células anteriores, no mesmo ambiente Colab.

### Sensibilidade e composição das partições

[Cerqueira et al. (2020)](https://arxiv.org/abs/1905.11744) discutem a não estacionariedade. Os testes KS são descritivos aqui, com dependência entre horas. As três horas extremas são uma análise de sensibilidade do treino.


In [ ]:
# AUDITORIA 25 — Sensibilidade às três horas extremas (somente treinamento)
# Mantém o teste intacto. Não substitui modelos congelados.
import numpy as np, pandas as pd, json, joblib
from pathlib import Path
from sklearn.base import clone
from sklearn.metrics import mean_squared_error, mean_absolute_error
from datetime import datetime, timezone

AUD4 = Path(PHASE_ROOT) / "reports" / "auditorias_complementares"
AUD4.mkdir(parents=True, exist_ok=True)
EXTREME_DATE = "2025-06-18"
EXTREME_HOURS = {11,12,13}
records25 = []
for chosen in selected_models15:
    scenario, algorithm = chosen["scenario"], chosen["algorithm"]
    train = split_data[scenario]["train"]
    test = split_data[scenario]["test"]
    ts = pd.to_datetime(train["timestamps"])
    # Não presume timezone; utiliza o horário tal como registrado na base.
    mask = (ts.dt.strftime("%Y-%m-%d") == EXTREME_DATE) & ts.dt.hour.isin(EXTREME_HOURS)
    mask = np.asarray(mask, dtype=bool)
    Xtrain = train["X"]
    ytrain = np.asarray(train["y"])
    Xtest = test["X"]
    ytest = np.asarray(test["y"])
    if mask.sum() != 3:
        _project_print(f"ATENÇÃO {scenario}: {mask.sum()} registros localizados; conferir timezone/horários. Experimento não executado.")
        records25.append({"scenario":scenario,"algorithm":algorithm,"status":"pendente_identificacao","extreme_rows":int(mask.sum())})
        continue
    model_original = joblib.load(chosen["model_path"])
    original_preprocessor = preprocessors[scenario][algorithm]
    for variant, keep in [("com_extremos", np.ones(len(mask), dtype=bool)),
                          ("sem_3_extremos", ~mask)]:
        prep = clone(original_preprocessor)
        model = clone(model_original)
        Xfit = Xtrain.iloc[keep] if hasattr(Xtrain,"iloc") else Xtrain[keep]
        yfit = ytrain[keep]
        Xt = prep.fit_transform(Xfit)
        model.fit(Xt,yfit)
        pred = model.predict(prep.transform(Xtest))
        records25.append({"scenario":scenario,"algorithm":algorithm,"variant":variant,
             "status":"executado_exploratorio","extreme_rows":int(mask.sum()),
             "train_n":int(len(yfit)),"test_n":int(len(ytest)),
             "rmse":float(np.sqrt(mean_squared_error(ytest,pred))),
             "mae":float(mean_absolute_error(ytest,pred)),
             "bias":float(np.mean(pred-ytest))})
result25 = pd.DataFrame(records25)
_project_csv(result25, AUD4 / "sensibilidade_extremos.csv",index=False)
_project_print(result25.to_string(index=False))
_project_print("ATENÇÃO: modelos reestimados para análise exploratória; NÃO substituem os congelados.")


In [ ]:
# AUDITORIA 26 — Distribuição por partição e estabilidade operacional
import pandas as pd, numpy as np
from scipy.stats import ks_2samp
ref26 = split_data["B0"]
rows26=[]
for part in ("train","validation","test"):
    item = ref26[part]
    yv = np.asarray(item["y"],dtype=float)
    xv = item["X"]
    rows26.append({"partition":part,"n":len(yv),"co2_mean":float(np.mean(yv)),
       "co2_median":float(np.median(yv)),"co2_std":float(np.std(yv)),
       "co2_p05":float(np.percentile(yv,5)),"co2_p95":float(np.percentile(yv,95)),
       "start":str(pd.to_datetime(item["timestamps"]).min()),
       "end":str(pd.to_datetime(item["timestamps"]).max())})
summary26=pd.DataFrame(rows26)
_project_csv(summary26, AUD4 / "distribuicao_particoes.csv",index=False)
_project_print(summary26.to_string(index=False))
for a,b in (("train","validation"),("validation","test"),("train","test")):
    ya=np.asarray(ref26[a]["y"],dtype=float); yb=np.asarray(ref26[b]["y"],dtype=float)
    ks=ks_2samp(ya,yb)
    _project_print(f"KS CO2 {a} vs {b}: D={ks.statistic:.4f}, p={ks.pvalue:.3g} (descritivo; autocorrelação viola independência)")
# Diagnóstico de geração: usa a variável explicativa do B0, sem assumir nome fixo.
cols26=list(ref26["train"]["X"].columns)
_project_print("Variáveis B0:",cols26)
for col in cols26:
    _project_print("Variável:",col)
    for part in ("train","validation","test"):
        vals=pd.to_numeric(ref26[part]["X"][col],errors="coerce")
        _project_print(part,"n=",vals.notna().sum(),"mediana=",round(float(vals.median()),3),"p05=",round(float(vals.quantile(.05)),3),"p95=",round(float(vals.quantile(.95)),3))


In [ ]:
# AUDITORIA 27 — Verificação da ablação e manifesto final (sem alterar modelos)
import json, hashlib
ablation27 = {
    "status":"exploratoria_nao_confirmatoria",
    "frozen_p2_validation_rmse":float(rmse_p2_frozen22),
    "reconstructed_p2_validation_rmse":float(full_rmse22),
    "difference":float(abs(full_rmse22-rmse_p2_frozen22)),
    "original_pipeline_reproduced_at_4_decimals":bool(equivalent22a),
    "ablation_variants_equivalent_to_original_pipeline":"nao_demonstrado",
    "note":"A reprodução do pipeline completo não demonstra equivalência das variantes reduzidas. Não usar ablação como prova causal.",
}
with open(AUD4 / "auditoria_ablacao.json","w",encoding="utf-8") as f:
    _project_dump(ablation27,f,indent=2,ensure_ascii=False)
manifest27={
 "phase":"04","kind":"auditorias_complementares","generated_utc":datetime.now(timezone.utc).isoformat(),
 "frozen_models_modified":False,
 "test_previously_examined":True,
 "sensitivity":"exploratory_only_not_for_model_selection",
 "limitations":["Teste não é independente para seleção adicional", "Ablações ainda não têm equivalência demonstrada", "KS descritivo ignora autocorrelação"],
 "artifacts":{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in AUD4.iterdir() if p.is_file()}
}
with open(AUD4 / "manifesto_auditorias.json","w",encoding="utf-8") as f:_project_dump(manifest27,f,indent=2,ensure_ascii=False)
_project_print("Auditorias gravadas em:",AUD4)
_project_print("Equivalência do pipeline completo:",equivalent22a)
_project_print("Ablações: exploratórias; não foram alteradas.")


## Experimento 28 — Contribuição incremental controlada dos atributos

**Objetivo:** testar se a adição de estados HMM e atributos de processos melhora a previsão em comparação com geração apenas, **mantendo fixos o algoritmo XGBoost, seus hiperparâmetros, as partições e o protocolo de pré-processamento**. O teste de 2025 não será consultado nesta seção, pois já foi utilizado nas avaliações anteriores. A comparação aqui é **complementar e exploratória na validação**, não uma nova confirmação independente. Os modelos congelados permanecem intactos.

O grupo `PROCESS_FEATURES` contém tanto variáveis de processos quanto histórico temporal de geração; portanto, o efeito desse grupo não é atribuível exclusivamente ao Process Mining. Não há nova otimização de hiperparâmetros.


### Ablação controlada

Aula 08, pp. 8–9 e 23–32. Mesmos dados e hiperparâmetros permitem comparar representações. O contraste M4 × M2 mede ganho adicional dos atributos de processo, com proveniência temporal ainda exploratória. A unidade de reamostragem no Exp29 é a observação ON, que pode ter lacunas no relógio. [Künsch (1989)](https://doi.org/10.1214/aos/1176347265)


In [ ]:
# ============================================================
# FASE 4 — EXPERIMENTO 28
# CONTRIBUIÇÃO INCREMENTAL CONTROLADA DOS ATRIBUTOS
# Comparação na VALIDAÇÃO; sem acesso ao teste nesta célula.
# ============================================================

import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from IPython.display import display

for required in ("split_data", "FEATURE_SETS", "HMM_FEATURES", "PROCESS_FEATURES", "selected_models15", "DIRS"):
    if required not in globals():
        raise RuntimeError(f"Execute as células anteriores: variável ausente {required}")

b0_reference28 = next(
    (r for r in selected_models15 if r["scenario"] == "B0"), None
)
if b0_reference28 is None or b0_reference28["algorithm"].lower() != "xgboost":
    raise RuntimeError("Referência B0/XGBoost congelada não identificada.")

# Recupera parâmetros do modelo selecionado, sem usar dados de teste.
import joblib
frozen_b0_28 = joblib.load(b0_reference28["model_path"])
if not isinstance(frozen_b0_28, XGBRegressor):
    raise TypeError("Modelo B0 congelado não é um XGBRegressor isolado.")
fixed_params28 = frozen_b0_28.get_params(deep=False)

# Mesmo algoritmo e mesma parametrização para TODOS os cenários.
# Imputação mediana ajustada SOMENTE no treinamento.
# Reajuste controlado de todos os modelos, inclusive B0.
feature_groups28 = {
    "G0_geracao": list(FEATURE_SETS["B0"]),
    "G1_geracao_HMM": list(FEATURE_SETS["B0"]) + list(HMM_FEATURES),
    "G2_geracao_processos_temporais": list(FEATURE_SETS["B0"]) + list(PROCESS_FEATURES),
    "G3_geracao_HMM_processos_temporais": list(FEATURE_SETS["P2"]),
}

base_train28 = split_data["B0"]["train"]
base_val28 = split_data["B0"]["validation"]
y_train28 = np.asarray(base_train28["y"], dtype=float)
y_val28 = np.asarray(base_val28["y"], dtype=float)
assert len(y_train28) == 4499 and len(y_val28) == 687

results28 = []
predictions28 = {}
models28 = {}
for label, columns in feature_groups28.items():
    assert len(columns) == len(set(columns)), f"Colunas duplicadas: {label}"
    scenario = "B0" if label == "G0_geracao" else ("P1" if label == "G1_geracao_HMM" else "P2")
    # Para G2 usamos as colunas P2, removendo apenas os estados HMM.
    raw_train = split_data[scenario]["train"]["X"].loc[:, columns].copy()
    raw_val = split_data[scenario]["validation"]["X"].loc[:, columns].copy()
    assert raw_train.columns.tolist() == raw_val.columns.tolist()
    assert raw_train.shape[0] == len(y_train28) and raw_val.shape[0] == len(y_val28)
    assert np.array_equal(np.asarray(split_data[scenario]["train"]["y"], dtype=float), y_train28)
    assert np.array_equal(np.asarray(split_data[scenario]["validation"]["y"], dtype=float), y_val28)
    for frame in (raw_train, raw_val):
        if not all(pd.api.types.is_numeric_dtype(frame[c]) for c in frame.columns):
            raise TypeError(f"Atributos não numéricos em {label}")
    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("model", XGBRegressor(**fixed_params28)),
    ])
    pipeline.fit(raw_train, y_train28)
    prediction = np.asarray(pipeline.predict(raw_val), dtype=float)
    predictions28[label] = prediction
    models28[label] = pipeline
    results28.append({
        "scenario": label, "n_features": len(columns),
        "rmse_validation": float(np.sqrt(mean_squared_error(y_val28, prediction))),
        "mae_validation": float(mean_absolute_error(y_val28, prediction)),
        "r2_validation": float(r2_score(y_val28, prediction)),
        "bias_validation": float(np.mean(prediction - y_val28)),
    })

results_df28 = pd.DataFrame(results28)
reference_rmse28 = float(results_df28.loc[
    results_df28["scenario"] == "G0_geracao", "rmse_validation"
].iloc[0])
results_df28["gain_vs_generation_pct"] = (
    100 * (reference_rmse28 - results_df28["rmse_validation"]) / reference_rmse28
)
_project_print("EXPERIMENTO CONTROLADO — MESMO XGBOOST, HIPERPARÂMETROS E PARTIÇÕES")
_project_display(results_df28)

# Bootstrap pareado em blocos contíguos da validação (exploratório).
rng28 = np.random.default_rng(20261108)
block_size28 = 24
n_boot28 = 1000
n28 = len(y_val28)
base_error28 = (y_val28 - predictions28["G0_geracao"]) ** 2
bootstrap_rows28 = []
for label, pred in predictions28.items():
    if label == "G0_geracao":
        continue
    model_error = (y_val28 - pred) ** 2
    gains = []
    for _ in range(n_boot28):
        indices = []
        while len(indices) < n28:
            start = int(rng28.integers(0, n28))
            indices.extend((start + np.arange(block_size28)) % n28)
        idx = np.asarray(indices[:n28], dtype=int)
        baseline_rmse = float(np.sqrt(base_error28[idx].mean()))
        alternative_rmse = float(np.sqrt(model_error[idx].mean()))
        gains.append(100 * (baseline_rmse - alternative_rmse) / baseline_rmse)
    bootstrap_rows28.append({
        "scenario": label,
        "gain_pct": float(100 * (np.sqrt(base_error28.mean()) - np.sqrt(model_error.mean())) / np.sqrt(base_error28.mean())),
        "ci95_low": float(np.quantile(gains, 0.025)),
        "ci95_high": float(np.quantile(gains, 0.975)),
    })
bootstrap_df28 = pd.DataFrame(bootstrap_rows28)
_project_print("BOOTSTRAP PAREADO EM BLOCOS — VALIDAÇÃO (EXPLORATÓRIO)")
_project_display(bootstrap_df28)

# Persistência: apenas tabelas e manifesto; não sobrescreve modelos congelados.
out_dir28 = Path(DIRS["tables"])
out_dir28.mkdir(parents=True, exist_ok=True)
_project_csv(results_df28, out_dir28 / "cell28_controlled_incremental_validation.csv", index=False)
_project_csv(bootstrap_df28, out_dir28 / "cell28_controlled_bootstrap_validation.csv", index=False)
manifest28 = {
    "phase": "phase_04_predictive_modeling", "experiment": "28",
    "evaluation_partition": "validation_only",
    "test_reused": False,
    "reference_algorithm": "XGBoost",
    "hyperparameters_source": str(b0_reference28["model_path"]),
    "fixed_hyperparameters": fixed_params28,
    "feature_groups": feature_groups28,
    "preprocessing": "SimpleImputer(median, train-only) for all groups",
    "interpretation": "exploratory; process group includes generation history",
}
manifest_path28 = Path(DIRS["manifests"]) / "cell28_controlled_incremental.json"
manifest_path28.parent.mkdir(parents=True, exist_ok=True)
with open(manifest_path28, "w", encoding="utf-8") as f:
    _project_dump(manifest28, f, ensure_ascii=False, indent=2, default=str)
_project_print("Arquivos:", out_dir28 / "cell28_controlled_incremental_validation.csv", manifest_path28)
_project_print("STATUS: experimento 28 concluído na validação; modelos congelados preservados.")


## Experimento 29 — Ablação controlada: HMM, processos e históricos de geração

Seis modelos XGBoost com hiperparâmetros fixos e mesma amostra de validação. Compara contribuição incremental de processos além dos históricos. **Análise exploratória**, sem reutilizar o teste. Os atributos de processos incluem eventos baseados em estados HMM, portanto não são independentes do HMM.

In [ ]:
# ============================================================
# FASE 4 — EXPERIMENTO 29: ABLAÇÃO CONTROLADA POR GRUPO
# Estimativa contemporânea de CO2; avaliação na validação.
# NÃO altera os modelos congelados nem utiliza o conjunto teste.
# ============================================================
import json
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
from IPython.display import display

for name29 in ("split_data", "FEATURE_SETS", "PROCESS_FEATURES", "HMM_FEATURES", "DIRS", "fixed_params28"):
    if name29 not in globals():
        raise RuntimeError(f"Execute as células anteriores: variável ausente {name29}")

# Grupos disjuntos, conforme nomes definidos na Célula 06.
# 'Processos' inclui eventos/rampas e transições derivadas de HMM;
# por isso não representa Process Mining independente do HMM.
TEMPORAL29 = [c for c in PROCESS_FEATURES if c.startswith(("load_mean_last_", "load_std_last_", "load_lag_"))]
PROCESSOS29 = [c for c in PROCESS_FEATURES if c not in TEMPORAL29]
HMM29 = list(HMM_FEATURES)
GERACAO29 = list(FEATURE_SETS["B0"])
assert TEMPORAL29 and PROCESSOS29 and HMM29 and GERACAO29
assert not (set(TEMPORAL29) & set(PROCESSOS29))
assert not (set(HMM29) & (set(TEMPORAL29) | set(PROCESSOS29)))
assert set(TEMPORAL29 + PROCESSOS29) == set(PROCESS_FEATURES)
assert set(GERACAO29 + HMM29 + TEMPORAL29 + PROCESSOS29) == set(FEATURE_SETS["P2"])

GRUPOS29 = {
    "M0_geracao": GERACAO29,
    "M1_geracao_HMM": GERACAO29 + HMM29,
    "M2_geracao_temporal": GERACAO29 + TEMPORAL29,
    "M3_geracao_processos": GERACAO29 + PROCESSOS29,
    "M4_geracao_temporal_processos": GERACAO29 + TEMPORAL29 + PROCESSOS29,
    "M5_completo": GERACAO29 + TEMPORAL29 + PROCESSOS29 + HMM29,
}

# Usar exatamente as mesmas linhas e o mesmo alvo para TODOS.
# P2 contém a união de todos os atributos, sem precisar unir tabelas.
train29 = split_data["P2"]["train"]
val29 = split_data["P2"]["validation"]
Xtrain29, Xval29 = train29["X"], val29["X"]
ytrain29 = np.asarray(train29["y"], dtype=float)
yval29 = np.asarray(val29["y"], dtype=float)
assert len(ytrain29) == 4499 and len(yval29) == 687
assert np.array_equal(ytrain29, np.asarray(split_data["B0"]["train"]["y"], dtype=float))
assert np.array_equal(yval29, np.asarray(split_data["B0"]["validation"]["y"], dtype=float))
assert Xtrain29.columns.is_unique and Xval29.columns.is_unique
assert np.isfinite(ytrain29).all() and np.isfinite(yval29).all()

pred29 = {}
models29 = {}
rows29 = []
for nome29, colunas29 in GRUPOS29.items():
    assert len(colunas29) == len(set(colunas29)), nome29
    ausentes29 = sorted(set(colunas29) - set(Xtrain29.columns) | set(colunas29) - set(Xval29.columns))
    if ausentes29:
        raise RuntimeError(f"Atributos ausentes em {nome29}: {ausentes29}")
    xtr29 = Xtrain29.loc[:, colunas29]
    xva29 = Xval29.loc[:, colunas29]
    if not all(pd.api.types.is_numeric_dtype(xtr29[c]) for c in colunas29):
        raise TypeError(f"Atributos não numéricos: {nome29}")
    if xtr29.isna().all().any():
        raise ValueError(f"Coluna inteiramente ausente no treino: {nome29}")
    pipe29 = Pipeline([
        ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("model", XGBRegressor(**fixed_params28)),
    ])
    pipe29.fit(xtr29, ytrain29)
    p29 = np.asarray(pipe29.predict(xva29), dtype=float)
    assert np.isfinite(p29).all()
    pred29[nome29] = p29
    models29[nome29] = pipe29
    rows29.append({
        "modelo": nome29,
        "n_atributos": len(colunas29),
        "rmse_validacao": float(np.sqrt(mean_squared_error(yval29, p29))),
        "mae_validacao": float(mean_absolute_error(yval29, p29)),
        "r2_validacao": float(r2_score(yval29, p29)),
        "vies_validacao": float(np.mean(p29 - yval29)),
    })
resultados29 = pd.DataFrame(rows29)
rmse_base29 = resultados29.loc[resultados29.modelo == "M0_geracao", "rmse_validacao"].iloc[0]
resultados29["ganho_vs_M0_pct"] = 100 * (rmse_base29 - resultados29.rmse_validacao) / rmse_base29
_project_print("EXPERIMENTO 29 — ABLAÇÃO CONTROLADA (VALIDAÇÃO)")
_project_display(resultados29)

# Bootstrap pareado circular em blocos de 24 observações operacionais.
# ATENÇÃO: blocos de observações ON, não necessariamente 24 horas consecutivas.
comparacoes29 = [
    ("M1_geracao_HMM", "M0_geracao", "HMM além da geração"),
    ("M2_geracao_temporal", "M0_geracao", "histórico de geração"),
    ("M3_geracao_processos", "M0_geracao", "processos além da geração"),
    ("M4_geracao_temporal_processos", "M2_geracao_temporal", "processos além dos históricos"),
    ("M5_completo", "M4_geracao_temporal_processos", "HMM além de processos e históricos"),
]
rng29 = np.random.default_rng(20261108)
B29, L29, N29 = 2000, 24, len(yval29)
indices29 = np.empty((B29, N29), dtype=np.int32)
for b29 in range(B29):
    starts29 = rng29.integers(0, N29, size=(N29 + L29 - 1) // L29)
    indices29[b29] = ((starts29[:, None] + np.arange(L29)) % N29).ravel()[:N29]
bootstrap_rows29 = []
for alt29, ref29, pergunta29 in comparacoes29:
    erro_ref29 = (yval29 - pred29[ref29]) ** 2
    erro_alt29 = (yval29 - pred29[alt29]) ** 2
    ganhos29 = 100 * (
        1 - np.sqrt(erro_alt29[indices29].mean(axis=1)) /
        np.sqrt(erro_ref29[indices29].mean(axis=1))
    )
    bootstrap_rows29.append({
        "comparacao": f"{alt29} vs {ref29}",
        "pergunta": pergunta29,
        "ganho_rmse_pct": float(100 * (1 - np.sqrt(erro_alt29.mean() / erro_ref29.mean()))),
        "ic95_inferior": float(np.quantile(ganhos29, 0.025)),
        "ic95_superior": float(np.quantile(ganhos29, 0.975)),
        "inclui_zero": bool(np.quantile(ganhos29, 0.025) <= 0 <= np.quantile(ganhos29, 0.975)),
    })
comparacoes_df29 = pd.DataFrame(bootstrap_rows29)
_project_print("GANHOS INCREMENTAIS — BOOTSTRAP EXPLORATÓRIO")
_project_display(comparacoes_df29)

# Registrar grupos e resultados sem sobrescrever saídas anteriores.
tabelas29 = Path(DIRS["tables"])
manifestos29 = Path(DIRS["manifests"])
tabelas29.mkdir(parents=True, exist_ok=True)
manifestos29.mkdir(parents=True, exist_ok=True)
_project_csv(resultados29, tabelas29 / "cell29_ablation_validation.csv", index=False)
_project_csv(comparacoes_df29, tabelas29 / "cell29_incremental_bootstrap.csv", index=False)
manifesto29 = {
    "fase": "04", "experimento": 29,
    "executado_em_utc": datetime.now(timezone.utc).isoformat(),
    "particao": "validation_only", "test_reused": False,
    "algoritmo": "XGBoost", "hiperparametros": fixed_params28,
    "protocolo": "mesmos hiperparametros e particoes; imputacao mediana ajustada no treino",
    "grupos": {"geracao": GERACAO29, "hmm": HMM29, "temporais": TEMPORAL29, "processos": PROCESSOS29},
    "modelos": GRUPOS29,
    "limites": [
        "Analise exploratoria: validacao usada previamente para escolhas de modelagem.",
        "Atributos de processos incluem transicoes de estados inferidos por HMM; nao isolam Process Mining independente de HMM.",
        "Bootstrap em blocos de observacoes ON, nao de horas consecutivas.",
        "Hiperparametros fixos provenientes do B0; nao demonstram otimalidade para cada grupo.",
        "Estimativa contemporanea: pressupoe disponibilidade dos atributos ate a hora t.",
    ],
}
with open(manifestos29 / "cell29_controlled_ablation.json", "w", encoding="utf-8") as f29:
    _project_dump(manifesto29, f29, ensure_ascii=False, indent=2, default=str)
_project_print("STATUS: EXPERIMENTO 29 CONCLUÍDO NA VALIDAÇÃO; TESTE E MODELOS CONGELADOS PRESERVADOS.")


## Experimento 30 — DoE multifatorial, robustez temporal e MLflow

**Objetivo:** avaliar a interação entre quatro famílias de algoritmos e seis representações de atributos, usando **somente a partição de treinamento** para busca e comparação temporal interna. O teste e a validação externa não são utilizados nesta seção.

**Hipóteses:** H1: históricos temporais contribuem além da geração; H2: processos acrescentam informação além dos históricos; H3: a contribuição de processos varia conforme o algoritmo.

**Fatores:** algoritmo (4 níveis), representação (6 níveis), hiperparâmetros (até 8 candidatos por algoritmo) e bloco temporal (3 janelas expansivas). **Resposta principal:** RMSE; secundárias: MAE, viés e R².

**Limitação essencial:** os atributos HMM/processos foram preparados nas Fases 2/3. Uma avaliação temporal verdadeiramente prospectiva exige verificar que o ajuste do HMM, as transformações e quaisquer parâmetros aprendidos não utilizaram observações posteriores ao início de cada janela. Este experimento é **exploratório**, não confirmação externa, até que a proveniência temporal seja comprovada. O bootstrap da Fase 4 não substitui replicação por janelas.

**Execução:** primeiro rode 30A (auditoria/plano), depois 30B (treinamento; pode demorar), 30C (síntese). Todos os artefatos novos usam prefixo `cell30_`; nenhuma saída anterior é sobrescrita.

### DoE 30 e trilha de decisão

Aula 08, pp. 31–32 e 53–56. A grade, a matriz, os folds, os IDs e as métricas documentam seleção e reajuste. Os folds são expansivos na amostra do regressor e os atributos de processo já chegam materializados. [Cerqueira et al. (2020)](https://arxiv.org/abs/1905.11744) · [Zaharia et al. (2018)](https://people.eecs.berkeley.edu/~matei/papers/2018/ieee_mlflow.pdf)


In [ ]:
# FASE 4 — EXPERIMENTO 30A: PLANO, FATORES E AUDITORIA (SEM TREINAMENTO)
import json, hashlib, itertools, platform
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from sklearn.model_selection import ParameterGrid
from IPython.display import display

required30 = ['split_data','GRUPOS29','DIRS','PARAM_GRIDS']
missing30 = [k for k in required30 if k not in globals()]
if missing30:
    raise RuntimeError('Execute as células anteriores: ' + ', '.join(missing30))

ALGORITHMS30 = ['ElasticNet','RandomForest','XGBoost','HistGradientBoosting']
FEATURES30 = {k: list(v) for k,v in GRUPOS29.items()}
TRAIN30 = split_data['P2']['train']
X30 = TRAIN30['X'].reset_index(drop=True)
y30 = np.asarray(TRAIN30['y'], dtype=float)
ts30 = pd.to_datetime(TRAIN30['timestamps'], utc=True).reset_index(drop=True)
assert len(X30) == len(y30) == len(ts30)
assert ts30.is_monotonic_increasing and not ts30.duplicated().any()
assert np.isfinite(y30).all() and X30.columns.is_unique
for label, cols in FEATURES30.items():
    assert len(cols) == len(set(cols)), label
    assert set(cols).issubset(X30.columns), label
    assert all(pd.api.types.is_numeric_dtype(X30[c]) for c in cols), label
    assert not X30[cols].isna().all().any(), f'Coluna vazia: {label}'

# Origem móvel expansiva: todas as janelas ficam DENTRO do treino original.
# Observações ON não são horas consecutivas; o timestamp é preservado.
N30 = len(y30)
initial30 = int(np.floor(0.60 * N30))
fold_size30 = (N30 - initial30) // 3
FOLDS30 = []
for k in range(3):
    a = initial30 + k * fold_size30
    b = N30 if k == 2 else initial30 + (k + 1) * fold_size30
    FOLDS30.append({'fold': k+1, 'train_end': a, 'val_start': a, 'val_end': b,
                    'train_last_utc': str(ts30.iloc[a-1]),
                    'val_first_utc': str(ts30.iloc[a]),
                    'val_last_utc': str(ts30.iloc[b-1])})
assert all(f['val_start'] > 0 and f['val_end'] > f['val_start'] for f in FOLDS30)

# Grade original, com subamostragem DETERMINÍSTICA quando >8 candidatos.
# As mesmas configurações de um algoritmo são usadas em todos os grupos.
MAX_TRIALS30 = 8
SEED30 = 42
CANDIDATES30 = {}
for algo in ALGORITHMS30:
    grid = list(ParameterGrid(PARAM_GRIDS[algo]))
    if len(grid) > MAX_TRIALS30:
        indices = np.linspace(0, len(grid)-1, MAX_TRIALS30, dtype=int)
        grid = [grid[int(i)] for i in indices]
    CANDIDATES30[algo] = [dict(p) for p in grid]

PLAN30 = pd.DataFrame([
    {'algorithm': algo, 'feature_group': name, 'n_features': len(cols),
     'n_trials': len(CANDIDATES30[algo]), 'n_folds': len(FOLDS30),
     'planned_fits': len(CANDIDATES30[algo])*len(FOLDS30)}
    for algo in ALGORITHMS30 for name,cols in FEATURES30.items()
])
_project_print('MATRIZ DoE —',len(PLAN30),'combinações; ajustes planejados:',int(PLAN30.planned_fits.sum()))
_project_display(PLAN30)
_project_print('JANELAS TEMPORAIS (somente TREINO):')
_project_display(pd.DataFrame(FOLDS30))

# Aviso explícito: auditamos nomes, não certificamos reestimação HMM por fold.
PROVENANCE_STATUS30 = 'NOT_CERTIFIED_FOLDWISE_HMM_OR_PROCESS_FEATURE_FITTING'
_project_print('ATENÇÃO:', PROVENANCE_STATUS30)
_project_print('Resultados exploratórios até comprovar causalidade temporal do ajuste HMM/atributos.')

out30 = Path(DIRS['tables']); out30.mkdir(parents=True,exist_ok=True)
man30 = Path(DIRS['manifests']); man30.mkdir(parents=True,exist_ok=True)
planpath30 = out30/'cell30_experimental_plan.csv'
_project_csv(PLAN30, planpath30,index=False)
manifest30 = {
 'experiment':'30','design':'4 algorithms x 6 feature representations x 3 expanding train-only folds',
 'seed':SEED30,'max_trials_per_algorithm':MAX_TRIALS30,
 'training_rows':N30,'train_first_timestamp':str(ts30.iloc[0]),'train_last_timestamp':str(ts30.iloc[-1]),
 'folds':FOLDS30,'feature_groups':FEATURES30,'candidate_grids':CANDIDATES30,
 'source_partition':'train_only_for_regressor',
 'external_validation_used_for_regressor':False,'test_used_for_regressor':False,
 'phase2_vs_phase4_cutoffs': TEMPORAL_SOURCE_AUDIT,
 'provenance_status':PROVENANCE_STATUS30,
 'limitations':['HMM and process features were materialized before the internal folds.',
                'Pre-existing validation/test inspection makes this an exploratory robustness analysis.',
                'Rows correspond to ON observations, not continuous hours.'],
 'created_utc':datetime.now(timezone.utc).isoformat()
}
with open(man30/'cell30_experimental_protocol.json','w',encoding='utf-8') as f:
    _project_dump(manifest30,f,ensure_ascii=False,indent=2,default=str)
_project_print('Plano salvo:',planpath30)

In [ ]:
# EXPERIMENTO 30B — DoE com ledger e IDs reais do MLflow.
# O regressor usa treino original. O ajuste do HMM por janela segue pendente.
# Cawley e Talbot (2010): https://jmlr.org/papers/v11/cawley10a.html
# Aula 08, pp. 23–32 e 53–56: fatores, respostas e proveniência.
import time, json, hashlib, warnings
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import mlflow
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score
from xgboost import XGBRegressor
from IPython.display import display

resultpath30 = Path(DIRS['tables']) / 'cell30_all_trials_train_only.csv'
execution_path30 = Path(DIRS['manifests']) / 'cell30_execution.json'
if resultpath30.exists() or execution_path30.exists():
    raise FileExistsError('Há uma tentativa do Exp30 nesta pasta. Inicie outra execução da Fase 4 para recalcular.')
if mlflow.active_run() is not None:
    raise RuntimeError('Finalize a execução MLflow ativa antes de executar o DoE 30.')
globals().pop('all_trials30', None)
activate_phase4_experiment('Iatan2_Phase4_Experiment30_DoE_Exploratory', 'doe30')

def model30(algorithm, params):
    if algorithm == 'ElasticNet':
        estimator = ElasticNet(**params,random_state=SEED30,max_iter=30000,tol=1e-5)
        return Pipeline([('imputer',SimpleImputer(strategy='median',keep_empty_features=True)),
                         ('scaler',StandardScaler()),('model',estimator)])
    if algorithm == 'RandomForest':
        estimator = RandomForestRegressor(**params,random_state=SEED30,n_jobs=2)
    elif algorithm == 'XGBoost':
        estimator = XGBRegressor(**params,random_state=SEED30,objective='reg:squarederror',
                                 tree_method='hist',n_jobs=2,verbosity=0)
    elif algorithm == 'HistGradientBoosting':
        estimator = HistGradientBoostingRegressor(**params,random_state=SEED30,early_stopping=False)
    else:
        raise ValueError(algorithm)
    return Pipeline([('imputer',SimpleImputer(strategy='median',keep_empty_features=True)),('model',estimator)])


def atomic_json(path, payload):
    temporary = path.with_suffix(path.suffix + '.tmp')
    _project_write_text(temporary, _project_dumps(payload, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
    temporary.replace(path)

def fingerprint(payload):
    return hashlib.sha256(_project_dumps(payload, sort_keys=True, default=str).encode('utf-8')).hexdigest()

def training_hash(features, target, timestamps):
    digest = hashlib.sha256()
    for values in (features, pd.Series(target), timestamps):
        digest.update(pd.util.hash_pandas_object(values, index=True).to_numpy().tobytes())
    return digest.hexdigest()

spec30 = {
    'phase4_run_id': PHASE4_RUN_ID, 'notebook_file_sha256': SOURCE_NOTEBOOK_SHA256,
    'phase2_run_id': PHASE2_REFERENCE_RUN, 'phase3_run_id': PHASE3_REFERENCE_RUN,
    'input_file_hashes': {key: item['sha256'] for key, item in file_metadata.items()},
    'train_matrix_sha256': training_hash(X30, y30, ts30), 'features': FEATURES30,
    'folds': FOLDS30, 'seed': SEED30, 'candidate_grids': CANDIDATES30, 'versions': versions,
    'provenance_status': PROVENANCE_STATUS30,
}
signature30 = fingerprint(spec30)
run_manifest30 = {'status': 'running', 'signature': signature30, 'specification': spec30}
atomic_json(execution_path30, run_manifest30)
rows30 = []
try:
    for algo30 in ALGORITHMS30:
        for group30, cols30 in FEATURES30.items():
            with mlflow.start_run(run_name=f'DoE30_{algo30}_{group30}') as parent30:
                mlflow.set_tags({'phase':'04','experiment':'30','algorithm':algo30,
                                 'feature_group':group30,'evaluation':'train_only_expanding_cv',
                                 'interpretation':'exploratory','hmm_fold_provenance':'unverified',
                                 'protocol_signature':signature30})
                mlflow.log_params({'seed':SEED30,'n_features':len(cols30),
                                   'n_folds':len(FOLDS30),'n_trials':len(CANDIDATES30[algo30])})
                mlflow.log_dict(spec30,'doe30_protocol.json')
                xgroup30 = X30.loc[:,cols30]
                for trial30, params30 in enumerate(CANDIDATES30[algo30]):
                    with mlflow.start_run(run_name=f'candidate_{trial30:02d}', nested=True) as candidate30:
                        mlflow.set_tags({'algorithm':algo30,'feature_group':group30,'trial':str(trial30),
                                         'protocol_signature':signature30})
                        mlflow.log_params({f'model_{k}':v for k,v in params30.items()})
                        mlflow.log_param('candidate_params_json', _project_dumps(params30, sort_keys=True, default=str))
                        trial_rows30 = []
                        for fold30 in FOLDS30:
                            a30,b30 = fold30['val_start'],fold30['val_end']
                            xtr30,xva30 = xgroup30.iloc[:a30],xgroup30.iloc[a30:b30]
                            ytr30,yva30 = y30[:a30],y30[a30:b30]
                            if xtr30.isna().all().any():
                                raise RuntimeError(f'Coluna vazia no fold: {algo30}/{group30}/{fold30["fold"]}')
                            t0_30 = time.perf_counter()
                            with warnings.catch_warnings(record=True) as caught30:
                                warnings.simplefilter('always')
                                pipeline30 = model30(algo30,params30)
                                pipeline30.fit(xtr30,ytr30)
                            pred30 = np.asarray(pipeline30.predict(xva30),dtype=float)
                            if not np.isfinite(pred30).all():
                                raise RuntimeError('Predição não finita')
                            metrics30 = {
                                'rmse':float(np.sqrt(mean_squared_error(yva30,pred30))),
                                'mae':float(mean_absolute_error(yva30,pred30)),
                                'bias':float(np.mean(pred30-yva30)),
                                'r2':float(r2_score(yva30,pred30)),
                                'elapsed_seconds':float(time.perf_counter()-t0_30),
                                'warning_count':len(caught30),
                            }
                            mlflow.log_metrics({f'fold{fold30["fold"]}_{key}':value
                                                for key,value in metrics30.items()})
                            trial_rows30.append({
                                'algorithm':algo30,'feature_group':group30,
                                'trial':trial30,'fold':fold30['fold'],
                                'n_train':a30,'n_validation':b30-a30,
                                'params_json':_project_dumps(params30,sort_keys=True,default=str),
                                'parent_run_id':parent30.info.run_id,
                                'candidate_run_id':candidate30.info.run_id,
                                'protocol_signature':signature30,
                                'warning_types':','.join(sorted({item.category.__name__ for item in caught30})),
                                **metrics30,
                            })
                        mlflow.log_metric('mean_rmse', float(np.mean([row['rmse'] for row in trial_rows30])))
                        mlflow.log_dict({'rows': trial_rows30}, 'candidate_fold_ledger.json')
                        rows30.extend(trial_rows30)
            backup_mlflow_database()
            _project_print('Concluído:',algo30,group30)
    ledger30 = pd.DataFrame(rows30)
    expected30 = int(PLAN30.planned_fits.sum())
    keys30 = ['algorithm','feature_group','trial','fold']
    if len(ledger30) != expected30 or ledger30.duplicated(keys30).any():
        raise RuntimeError(f'Ledger incompleto ou duplicado: {len(ledger30)}/{expected30}.')
    temporary30 = resultpath30.with_suffix('.csv.tmp')
    _project_csv(ledger30, temporary30,index=False)
    temporary30.replace(resultpath30)
    run_manifest30.update(status='completed_exploratory', ledger_sha256=calculate_sha256(resultpath30),
                          n_fits=len(ledger30), n_candidate_runs=ledger30.candidate_run_id.nunique())
    atomic_json(execution_path30, run_manifest30)
    all_trials30 = ledger30
except BaseException as exc:
    run_manifest30.update(status='interrupted', completed_fold_rows=len(rows30),
                          error_type=type(exc).__name__)
    atomic_json(execution_path30, run_manifest30)
    raise
finally:
    backup_mlflow_database()
_project_print('Linhas registradas:',len(all_trials30),'em',resultpath30)

In [ ]:
# FASE 4 — EXPERIMENTO 30C: SELEÇÃO INTERNA, INTERAÇÕES E EXPORTAÇÃO
# Seleciona hiperparâmetros pela MÉDIA dos RMSEs nos folds internos.
# A análise é descritiva/exploratória: folds expansivos compartilham treino.
import json
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
if 'all_trials30' not in globals():
    raise RuntimeError('Execute 30B nesta pasta de saída antes de selecionar os modelos.')
if set(all_trials30['protocol_signature']) != {signature30}:
    raise RuntimeError('A tabela 30B não corresponde à assinatura desta execução.')
trial_summary30=(all_trials30.groupby(['algorithm','feature_group','trial','params_json',
                                      'candidate_run_id','parent_run_id','protocol_signature'],as_index=False)
                .agg(mean_rmse=('rmse','mean'),mean_mae=('mae','mean'),
                     mean_bias=('bias','mean'),std_rmse=('rmse','std'),n_folds=('fold','nunique')))
assert (trial_summary30.n_folds==len(FOLDS30)).all()
best30=(trial_summary30.sort_values(['mean_rmse','trial'])
        .groupby(['algorithm','feature_group'],as_index=False).first())
assert len(best30)==24
_project_print('MELHOR CONFIGURAÇÃO POR ALGORITMO × GRUPO (seleção interna):')
_project_display(best30.sort_values(['algorithm','mean_rmse']))

pivot30=best30.pivot(index='algorithm',columns='feature_group',values='mean_rmse')
comparisons30=[
 ('M2_geracao_temporal','M0_geracao','temporal_alem_geracao'),
 ('M4_geracao_temporal_processos','M2_geracao_temporal','processos_alem_temporal'),
 ('M1_geracao_HMM','M0_geracao','HMM_alem_geracao'),
 ('M5_completo','M4_geracao_temporal_processos','HMM_alem_temporal_processos')]
effects30=[]
for algo30 in pivot30.index:
    for alt30,ref30,contrast30 in comparisons30:
        ref_rmse30=float(pivot30.loc[algo30,ref30]); alt_rmse30=float(pivot30.loc[algo30,alt30])
        effects30.append({'algorithm':algo30,'contrast':contrast30,
                          'rmse_reference':ref_rmse30,'rmse_alternative':alt_rmse30,
                          'gain_pct':100*(ref_rmse30-alt_rmse30)/ref_rmse30})
effects_df30=pd.DataFrame(effects30)
_project_print('EFEITOS DESCRITIVOS — GANHO POSITIVO INDICA MELHORA:')
_project_display(effects_df30.pivot(index='algorithm',columns='contrast',values='gain_pct').round(2))

# Comparação em cada fold dos candidatos escolhidos pela CV completa: DESCRITIVA,
# não uma nova estimativa independente de generalização.
selected_folds30=all_trials30.merge(best30[['algorithm','feature_group','trial']],
                                     on=['algorithm','feature_group','trial'],how='inner')
fold_pivot30=selected_folds30.pivot_table(index=['algorithm','fold'],columns='feature_group',values='rmse')
fold_effects30=[]
for (algo30,fold30), row30 in fold_pivot30.iterrows():
    for alt30,ref30,contrast30 in comparisons30:
        fold_effects30.append({'algorithm':algo30,'fold':fold30,'contrast':contrast30,
                              'gain_pct':100*(row30[ref30]-row30[alt30])/row30[ref30]})
fold_effects_df30=pd.DataFrame(fold_effects30)
_project_print('ESTABILIDADE DOS CONTRASTES POR JANELA (DESCRITIVA):')
_project_display(fold_effects_df30.sort_values(['contrast','algorithm','fold']))

out30=Path(DIRS['tables']);out30.mkdir(parents=True,exist_ok=True)
_project_csv(best30, out30/'cell30_best_configurations.csv',index=False)
_project_csv(effects_df30, out30/'cell30_algorithm_feature_effects.csv',index=False)
_project_csv(fold_effects_df30, out30/'cell30_foldwise_effects_exploratory.csv',index=False)
summary30={
 'experiment':'30','status':'completed_exploratory_train_only',
 'n_algorithms':4,'n_feature_groups':6,'n_combinations':24,
 'n_folds':len(FOLDS30),'n_fits':len(all_trials30),
 'selection':'minimum mean RMSE across expanding folds within original train',
 'external_validation_used':False,'test_used':False,
 'scientific_limitations':['No independent holdout for this DoE; foldwise performance after selection is optimistic.',
 'HMM/process feature extraction may not be foldwise refitted; provenance audit required.',
 'Temporal ON observations not necessarily consecutive hours.',
 'No formal factorial ANOVA inference; effects are descriptive and correlated.']}
with open(Path(DIRS['manifests'])/'cell30_final_summary.json','w',encoding='utf-8') as f:
    _project_dump(summary30,f,ensure_ascii=False,indent=2)
_project_print('EXPERIMENTO 30 concluído (exploratório). Resultados em',out30)

## Experimento 31 — Auditoria de proveniência e síntese sem novos treinamentos

**Correção metodológica:** os 360 ajustes do Experimento 30 são preservados. Esta seção não transforma atributos HMM/processos pré-computados em atributos certificados por janela; isso exigiria comprovar o ajuste exclusivamente com dados anteriores ao corte de cada fold, ou reconstruí-los por fold a partir dos artefatos das Fases 2/3. A auditoria registra explicitamente o que é verificável e o que permanece pendente.

**Correção do viés de seleção:** além das médias originalmente selecionadas e avaliadas nos mesmos folds (Experimento 30C), calculamos uma avaliação *leave-one-fold-out* de escolha de hiperparâmetros: para cada fold, selecionamos o candidato pelos **outros dois** folds e avaliamos no fold excluído. Isso reduz o viés de selecionar e avaliar no mesmo fold, mas **não** constitui validação independente: os modelos foram treinados com janelas expansivas sobrepostas, e a proveniência de HMM/processos permanece não certificada.

**Síntese:** comparar resultados dos Experimentos 29 e 30 sem misturar suas populações de avaliação, métricas e regimes de hiperparâmetros. Nenhum teste é executado.

In [ ]:
# FASE 4 — EXPERIMENTO 31A: AUDITORIA EXPLÍCITA DA PROVENIÊNCIA TEMPORAL
# Não treina modelos nem sobrescreve resultados anteriores.
import json, re, hashlib
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from IPython.display import display

for required31 in ('FOLDS30','FEATURES30','X30','ts30','DIRS','all_trials30'):
    if required31 not in globals():
        raise RuntimeError('Execute as células anteriores para disponibilizar '+required31)

# Regras de disponibilidade contemporânea: verificar nomes NÃO comprova causalidade.
# Atestado somente com proveniência de ajuste por fold, das Fases 2 e 3.
source_groups31 = {
    'geracao': list(globals().get('GERACAO29', [])),
    'hmm': list(globals().get('HMM29', [])),
    'temporais': list(globals().get('TEMPORAL29', [])),
    'processos': list(globals().get('PROCESSOS29', [])),
}
if not all(source_groups31.values()):
    raise RuntimeError('Grupos 29 incompletos: executar Experimento 29 antes desta auditoria.')

# Classificação de risco, não diagnóstico definitivo de vazamento.
all_features31 = sorted(set(sum((list(v) for v in FEATURES30.values()), [])))
rows31=[]
for feature31 in all_features31:
    memberships31=[g for g, cols in source_groups31.items() if feature31 in cols]
    suspicious31=bool(re.search(r'(^|_)(co2|emission|target|label|y_true|future|lead)(_|$)',feature31,re.I))
    derived31=any(g in ('hmm','processos') for g in memberships31)
    rows31.append({
        'feature':feature31,'groups':','.join(memberships31),
        'name_target_or_future_flag':suspicious31,
        'requires_foldwise_fit_provenance':derived31,
        'causal_availability_verified':False if derived31 else None,
        'observation':'Nomes e matriz disponíveis; origem temporal não certificada' if derived31 else
                      'Verificar se transformação é causal e disponível na hora t',
    })
feature_audit31=pd.DataFrame(rows31)

fold_rows31=[]
for f31 in FOLDS30:
    a31,b31=int(f31['val_start']),int(f31['val_end'])
    fold_rows31.append({
        'fold':int(f31['fold']), 'train_end_utc':str(ts30.iloc[a31-1]),
        'validation_start_utc':str(ts30.iloc[a31]),
        'validation_end_utc':str(ts30.iloc[b31-1]),
        'train_precedes_validation':bool(ts30.iloc[a31-1]<ts30.iloc[a31]),
        'hmm_fitted_using_only_train_prefix':'NOT_VERIFIED',
        'process_features_recomputed_without_future':'NOT_VERIFIED',
        'temporal_lags_past_only':'NOT_VERIFIED_BY_THIS_CELL',
        'fold_provenance_certified':False,
    })
fold_audit31=pd.DataFrame(fold_rows31)
if not fold_audit31['train_precedes_validation'].all():
    raise AssertionError('Ordem temporal dos folds violada.')

# Não inferir certificação de uma coluna já materializada no dataframe.
provenance31={
 'status':'NOT_CERTIFIED_FOLDWISE_HMM_OR_PROCESS_FEATURE_FITTING',
 'what_was_verified':['Ordem cronológica dos cortes de validação',
                      'Composição nominal dos grupos de atributos',
                      'Existência dos atributos no conjunto de treinamento'],
 'what_is_not_verified':['Data de corte e amostra utilizada para ajustar HMM em cada fold',
                         'Se decodificação de estados HMM usa smoothing bidirecional ou futuro',
                         'Se atributos de processo, transições e agregações usam observações futuras',
                         'Se parâmetros de transformação das Fases 2/3 foram aprendidos apenas no prefixo',
                         'Disponibilidade operacional dos atributos na hora t'],
 'certification_requirement':'Reestimar/reproduzir HMM e atributos dependentes por fold, ou fornecer evidência auditável de geração causal prefix-only.',
 'interpretation':'Resultados com HMM/processos continuam exploratórios; não declarar generalização prospectiva.',
 'created_utc':datetime.now(timezone.utc).isoformat(),
}
out31=Path(DIRS['tables']); out31.mkdir(parents=True,exist_ok=True)
man31=Path(DIRS['manifests']); man31.mkdir(parents=True,exist_ok=True)
_project_csv(feature_audit31, out31/'cell31_feature_provenance_audit.csv',index=False)
_project_csv(fold_audit31, out31/'cell31_fold_provenance_audit.csv',index=False)
with open(man31/'cell31_provenance_status.json','w',encoding='utf-8') as f:
    _project_dump(provenance31,f,ensure_ascii=False,indent=2)
_project_print('AUDITORIA 31A — NÃO CERTIFICADA: requer evidências/reconstrução das Fases 2/3.')
_project_print('Folds cronológicos auditados:',len(fold_audit31),'| atributos:',len(feature_audit31))
_project_print('Atributos com nomes potencialmente sensíveis:')
_project_display(feature_audit31.loc[feature_audit31.name_target_or_future_flag,['feature','groups']])
_project_display(fold_audit31)


In [ ]:
# FASE 4 — EXPERIMENTO 31B: SELEÇÃO LEAVE-ONE-FOLD-OUT (SEM NOVOS TREINOS)
# Para cada fold, escolher hiperparâmetros APENAS pelos outros folds.
# Esta sensibilidade usa os demais folds inclusive posteriores ao fold avaliado.
# Treinos expansivos se sobrepõem; os escores não equivalem a um teste prospectivo.
import json
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

if 'all_trials30' not in globals() or 'FOLDS30' not in globals():
    raise RuntimeError('Execute Experimento 30 antes da análise 31B.')
required_cols31={'algorithm','feature_group','trial','fold','rmse','params_json'}
if not required_cols31.issubset(all_trials30.columns):
    raise RuntimeError('Arquivo 30B incompleto: '+str(required_cols31-set(all_trials30.columns)))
fold_ids31=sorted(int(f['fold']) for f in FOLDS30)
if len(fold_ids31)<3:
    raise RuntimeError('A seleção leave-one-fold-out exige ao menos 3 folds.')
if all_trials30.duplicated(['algorithm','feature_group','trial','fold']).any():
    raise RuntimeError('Há execuções duplicadas por algoritmo/grupo/trial/fold.')

selected31=[]
for (algo31,group31), sub31 in all_trials30.groupby(['algorithm','feature_group'],sort=True):
    for held31 in fold_ids31:
        selection31=sub31.loc[sub31.fold!=held31].groupby('trial',as_index=False).agg(
            selection_rmse=('rmse','mean'),n_selection_folds=('fold','nunique'))
        selection31=selection31.loc[selection31.n_selection_folds==len(fold_ids31)-1]
        if selection31.empty:
            raise RuntimeError(f'Faltam candidatos completos: {algo31}/{group31}/{held31}')
        winner31=selection31.sort_values(['selection_rmse','trial']).iloc[0]
        heldrow31=sub31.loc[(sub31.fold==held31)&(sub31.trial==winner31.trial)]
        if len(heldrow31)!=1:
            raise RuntimeError(f'Fold de avaliação ausente: {algo31}/{group31}/{held31}')
        r31=heldrow31.iloc[0]
        selected31.append({'algorithm':algo31,'feature_group':group31,'fold':held31,
                           'trial_selected_without_held_fold':int(winner31.trial),
                           'selection_rmse_other_folds':float(winner31.selection_rmse),
                           'held_fold_rmse':float(r31.rmse),
                           'held_fold_mae':float(r31.mae),
                           'held_fold_bias':float(r31.bias),
                           'params_json':r31.params_json})
loo_trials31=pd.DataFrame(selected31)
assert len(loo_trials31)==len(fold_ids31)*24
summary_loo31=loo_trials31.groupby(['algorithm','feature_group'],as_index=False).agg(
    mean_held_rmse=('held_fold_rmse','mean'),
    std_held_rmse=('held_fold_rmse','std'),
    mean_held_mae=('held_fold_mae','mean'),
    n_folds=('fold','nunique'))

comparisons31=[
 ('M2_geracao_temporal','M0_geracao','temporais_alem_geracao'),
 ('M4_geracao_temporal_processos','M2_geracao_temporal','processos_alem_temporais'),
 ('M1_geracao_HMM','M0_geracao','HMM_alem_geracao'),
 ('M5_completo','M4_geracao_temporal_processos','HMM_alem_temporais_processos'),
]
wide31=loo_trials31.pivot(index=['algorithm','fold'],columns='feature_group',values='held_fold_rmse')
contrasts31=[]
for (algo31,fold31), row31 in wide31.iterrows():
    for alt31,ref31,name31 in comparisons31:
        refval31=float(row31[ref31]);altval31=float(row31[alt31])
        contrasts31.append({'algorithm':algo31,'fold':int(fold31),'contrast':name31,
                            'rmse_ref':refval31,'rmse_alt':altval31,
                            'gain_pct':100*(refval31-altval31)/refval31})
fold_contrasts31=pd.DataFrame(contrasts31)
summary_contrasts31=fold_contrasts31.groupby(['algorithm','contrast'],as_index=False).agg(
    mean_gain_pct=('gain_pct','mean'),
    min_gain_pct=('gain_pct','min'),max_gain_pct=('gain_pct','max'),
    positive_folds=('gain_pct',lambda s:int((s>0).sum())),n_folds=('fold','nunique'))

out31=Path(DIRS['tables']);out31.mkdir(parents=True,exist_ok=True)
_project_csv(loo_trials31, out31/'cell31_held_fold_selection.csv',index=False)
_project_csv(summary_loo31, out31/'cell31_held_fold_summary.csv',index=False)
_project_csv(fold_contrasts31, out31/'cell31_held_fold_contrasts.csv',index=False)
_project_csv(summary_contrasts31, out31/'cell31_contrasts_summary.csv',index=False)
_project_print('EXPERIMENTO 31B — seleção sem usar o fold avaliado; SEM NOVOS TREINOS.')
_project_print('GANHOS POR ALGORITMO (%; positivos = redução de RMSE):')
_project_display(summary_contrasts31.pivot(index='algorithm',columns='contrast',values='mean_gain_pct').round(2))
_project_print('ESTABILIDADE DOS CONTRASTES:')
_project_display(summary_contrasts31.sort_values(['contrast','algorithm']))
_project_print('ATENÇÃO: não é teste independente nem corrige proveniência HMM/processos.')

In [ ]:
# FASE 4 — EXPERIMENTO 31C: SÍNTESE 29 × 30 × 31 E LIMITAÇÕES
# Não comparar RMSE absolutos entre amostras de validação diferentes.
import json
from pathlib import Path
from datetime import datetime,timezone
import pandas as pd
from IPython.display import display

for k31 in ('comparacoes_df29','effects_df30','summary_contrasts31','provenance31'):
    if k31 not in globals():
        raise RuntimeError('Execute as células 29, 30C, 31A e 31B: falta '+k31)

# Mesma pergunta científica, protocolos distintos: NÃO combinar percentuais por média.
rows_synthesis31=[]
for _,r31 in comparacoes_df29.iterrows():
    if 'M4_geracao_temporal_processos vs M2_geracao_temporal' in str(r31['comparacao']):
        rows_synthesis31.append({'experiment':'29','algorithm':'XGBoost',
            'evaluation':'validação externa previamente explorada',
            'selection':'hiperparâmetros fixos de B0',
            'process_gain_pct':float(r31['ganho_rmse_pct']),
            'provenance_certified':False})
for _,r31 in effects_df30.loc[effects_df30.contrast=='processos_alem_temporal'].iterrows():
    rows_synthesis31.append({'experiment':'30','algorithm':r31['algorithm'],
        'evaluation':'folds internos; mesmos folds usados na seleção',
        'selection':'mínimo RMSE médio nos 3 folds',
        'process_gain_pct':float(r31['gain_pct']),
        'provenance_certified':False})
for _,r31 in summary_contrasts31.loc[summary_contrasts31.contrast=='processos_alem_temporais'].iterrows():
    rows_synthesis31.append({'experiment':'31','algorithm':r31['algorithm'],
        'evaluation':'fold excluído da seleção de parâmetros; treino expansivo sobreposto',
        'selection':'seleção pelos outros 2 folds',
        'process_gain_pct':float(r31['mean_gain_pct']),
        'provenance_certified':False})
synthesis_df31=pd.DataFrame(rows_synthesis31)
_project_print('SÍNTESE DESCRITIVA — CONTRASTE PROCESSOS ALÉM DOS TEMPORAIS:')
_project_display(synthesis_df31)

status31={
 'status':'EXPLORATORY_COMPLETED_PROVENANCE_NOT_CERTIFIED',
 'training_repeated':False,'previous_models_preserved':True,
 'experiment_29':'external validation previously explored; fixed XGBoost parameters',
 'experiment_30':'train-only expanding folds; selected and evaluated on same folds',
 'experiment_31':'leave-one-fold-out parameter selection using existing results; overlapping training sets',
 'hmm_process_provenance':provenance31['status'],
 'interpretation':'Diferenças de ganho entre 29/30/31 não provam causa; períodos, amostras e seleção diferem.',
 'next_step_for_prospective_claim':'Obter artefatos das Fases 2/3 e verificar/reconstruir HMM e atributos por corte temporal; depois avaliar em dados realmente novos.',
 'created_utc':datetime.now(timezone.utc).isoformat(),
}
out31=Path(DIRS['tables']);out31.mkdir(parents=True,exist_ok=True)
man31=Path(DIRS['manifests']);man31.mkdir(parents=True,exist_ok=True)
_project_csv(synthesis_df31, out31/'cell31_experiments_29_30_31_synthesis.csv',index=False)
with open(man31/'cell31_scientific_status.json','w',encoding='utf-8') as f:
    _project_dump(status31,f,ensure_ascii=False,indent=2)
_project_print('STATUS: análises exploratórias consolidadas; proveniência foldwise NÃO CERTIFICADA.')


## Experimento 32 — Consolidação e encerramento da Fase 4 (2025)

**Objetivo:** congelar a decisão experimental para o estudo multianual da **mesma usina Iatan 2** (Fase 5). Não repete os 360 ajustes do DoE, não consulta o teste reservado e não cria novas auditorias. Mantém M0 como referência, M2 para atributos temporais e M4 para processos. A seleção usa exclusivamente os resultados internos exploratórios do Experimento 30, com o Experimento 31 como análise de sensibilidade; não constitui validação independente. Os modelos finais são reajustados **uma vez** no treinamento original, apenas para serialização e posterior reutilização. Os artefatos não são modelos validados em anos externos.

In [ ]:
# FASE 4 — EXPERIMENTO 32A: DECISÃO EXPERIMENTAL E MANIFESTO
import json
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
from IPython.display import display
required32 = ['best30','summary_loo31','summary_contrasts31','FEATURES30','DIRS','TRAIN30','FOLDS30']
missing32 = [k for k in required32 if k not in globals()]
if missing32:
    raise RuntimeError('Execute as células anteriores antes do Experimento 32: '+', '.join(missing32))
GROUPS32 = ['M0_geracao','M2_geracao_temporal','M4_geracao_temporal_processos']
# Seleção pré-especificada para preservar uma referência e duas ablações.
# XGBoost e RandomForest são retidos para comparação na Fase 5;
# ElasticNet/HGB permanecem documentados no DoE, sem serem apagados.
ALGOS32 = ['XGBoost','RandomForest']
selected32 = best30.loc[best30.algorithm.isin(ALGOS32) & best30.feature_group.isin(GROUPS32)].copy()
assert len(selected32)==len(GROUPS32)*len(ALGOS32)
selected32=selected32.merge(summary_loo31[['algorithm','feature_group','mean_held_rmse','std_held_rmse']],
                            on=['algorithm','feature_group'],how='left',validate='one_to_one')
assert selected32.mean_held_rmse.notna().all()
selected32['n_features']=selected32.feature_group.map(lambda g:len(FEATURES30[g]))
selected32['selection_status']='exploratory_train_only'
selected32=selected32.sort_values(['algorithm','feature_group']).reset_index(drop=True)
OUT32=Path(DIRS['tables']); OUT32.mkdir(parents=True,exist_ok=True)
MAN32=Path(DIRS['manifests']); MAN32.mkdir(parents=True,exist_ok=True)
_project_csv(selected32, OUT32/'cell32_selected_finalists.csv',index=False)
if 'synthesis_df31' in globals():
    _project_csv(synthesis_df31, OUT32/'cell32_prior_experiments_synthesis.csv',index=False)
manifest32={
 'phase':'04','experiment':'32','plant':'Iatan 2','reference_year':2025,
 'purpose':'Close phase 4 and hand off frozen configurations for multiyear phase 5',
 'phase_5':'Multiyear evaluation of Iatan 2, not cross-plant transfer',
 'algorithms':ALGOS32,'feature_groups':GROUPS32,
 'selection_basis':'experiment 30 mean expanding-fold RMSE on original train; experiment 31 descriptive sensitivity',
 'original_training_rows':int(len(TRAIN30['y'])),
 'selection_signature': signature30,
 'selected_candidate_run_ids': selected32.candidate_run_id.tolist(),
 'external_validation_used_for_selection':False,'test_used_for_selection':False,
 'scientific_status':'exploratory; foldwise HMM/process feature fitting not certified; test previously inspected in project',
 'not_claimed':'No independent test validation, no prospective generalization, no Brazilian applicability demonstrated',
 'retraining':'only 6 final refits on original train in cell 32B; no 360-fit DoE rerun',
 'created_utc':datetime.now(timezone.utc).isoformat()
}
with open(MAN32/'cell32_phase4_closure_protocol.json','w',encoding='utf-8') as f:
    _project_dump(manifest32,f,indent=2,ensure_ascii=False)
_project_print('FINALISTAS DA FASE 4 — parâmetros selecionados sem usar teste:')
_project_display(selected32[['algorithm','feature_group','n_features','mean_rmse','mean_held_rmse','std_held_rmse','params_json']])
_project_print('Atenção: métricas internas exploratórias, NÃO desempenho multianual ou teste independente.')

In [ ]:
# EXPERIMENTO 32B — seis reajustes rastreados e ligação ao candidato.
# A Fase 5 deve reavaliar a estratégia em blocos cronológicos.
import json
from pathlib import Path
import joblib
from IPython.display import display

MODELS32 = Path(DIRS['models']) / 'experiment_32'
if MODELS32.exists():
    raise FileExistsError('Há uma tentativa do Exp32 nesta pasta. Inicie outra execução da Fase 4 para recalcular.')
if mlflow.active_run() is not None:
    raise RuntimeError('Finalize a execução MLflow ativa antes do Exp32.')
activate_phase4_experiment('Iatan2_Phase4_Experiment32_Refit_Exploratory', 'refit32')
Xfit32 = TRAIN30['X'].reset_index(drop=True)
yfit32 = np.asarray(TRAIN30['y'], dtype=float)
assert len(Xfit32) == len(yfit32) and np.isfinite(yfit32).all()
assert set(selected32['protocol_signature']) == {signature30}
tsfit32 = pd.to_datetime(TRAIN30['timestamps'], utc=True).reset_index(drop=True)
if fingerprint(spec30) != signature30 or training_hash(Xfit32, yfit32, tsfit32) != spec30['train_matrix_sha256']:
    raise RuntimeError('Especificação ou amostra de treino diferente da usada na seleção do DoE.')
MODELS32.mkdir(parents=True, exist_ok=False)
records32 = []
try:
    for r32 in selected32.itertuples(index=False):
        group32 = r32.feature_group
        algo32 = r32.algorithm
        cols32 = list(FEATURES30[group32])
        params32 = json.loads(r32.params_json)
        selected_run32 = client.get_run(r32.candidate_run_id)
        if (selected_run32.info.status != 'FINISHED'
                or selected_run32.data.tags.get('protocol_signature') != signature30
                or selected_run32.data.tags.get('algorithm') != algo32
                or selected_run32.data.tags.get('feature_group') != group32
                or selected_run32.data.params.get('candidate_params_json') != _project_dumps(params32, sort_keys=True, default=str)):
            raise RuntimeError(f'Candidato MLflow divergente para {algo32}/{group32}.')
        path32 = MODELS32 / f'{algo32}__{group32}.joblib'
        pipeline32 = model30(algo32, params32)
        with mlflow.start_run(run_name=f'Refit32_{algo32}_{group32}') as run32:
            mlflow.set_tags({'phase':'04','experiment':'32','algorithm':algo32,'feature_group':group32,
                             'interpretation':'exploratory','selected_candidate_run_id':r32.candidate_run_id,
                             'protocol_signature':signature30})
            mlflow.log_params({'n_train':len(yfit32),'n_features':len(cols32),
                               **{f'model_{key}':value for key,value in params32.items()}})
            pipeline32.fit(Xfit32.loc[:,cols32],yfit32)
            temporary32 = path32.with_suffix('.joblib.tmp')
            joblib.dump({'pipeline':pipeline32,'features':cols32,'algorithm':algo32,
                         'feature_group':group32,'params':params32,'fit_partition':'original_train_2025',
                         'interpretation':'exploratory_baseline_not_prospectively_validated',
                         'protocol_signature':signature30,'selected_candidate_run_id':r32.candidate_run_id,
                         'refit_run_id':run32.info.run_id}, temporary32)
            temporary32.replace(path32)
            record32 = {
                'algorithm':algo32,'feature_group':group32,'model_path':str(path32),
                'sha256':calculate_sha256(path32),'n_features':len(cols32),'status':'fitted_and_saved',
                'selected_candidate_run_id':r32.candidate_run_id,'refit_run_id':run32.info.run_id,
                'protocol_signature':signature30,
            }
            mlflow.log_artifact(str(path32), artifact_path='model')
            mlflow.log_dict({
                **record32,'features':cols32,'effective_model_parameters':pipeline32['model'].get_params(),
                'train_matrix_sha256':spec30['train_matrix_sha256'],
            }, 'refit_manifest.json')
            records32.append(record32)
        backup_mlflow_database()
finally:
    backup_mlflow_database()

models_manifest32 = pd.DataFrame(records32)
_project_csv(models_manifest32, OUT32/'cell32_model_artifacts.csv',index=False)
atomic_json(MAN32/'cell32_phase5_handoff.json', {
    'phase_5':'multiyear Iatan 2',
    'input_requirement':'same features, ordering, hourly time semantics and process transformations',
    'scientific_status':'exploratory; no independent prospective validation',
    'models':records32,'created_utc':datetime.now(timezone.utc).isoformat(),
})
_project_print('MODELOS CONGELADOS — referência exploratória para estudo multianual:')
_project_display(models_manifest32)

## Exp33 · Ablação com ajuste de HMM e DFG dentro de cada janela

A geração da própria hora está disponível. Os quatro cenários usam o mesmo XGBoost, os parâmetros do Exp29 e as mesmas horas ON em cada janela. A usa apenas a carga atual. B inclui seis defasagens e oito estatísticas de carga dentro do ciclo. C inclui as probabilidades filtradas dos cinco estados e seus diagnósticos. D acrescenta três atributos do grafo de sucessão direta (DFG) descoberto no log de treino. O contraste D − C mede o ganho incremental desses três atributos. Um delta negativo indica menor RMSE.

As três janelas internas mantêm os cortes do Exp30. Validação e teste usam os recortes históricos e o mesmo treino de 4.499 horas ON. Para cada janela, o scaler, o HMM, o DFG e o imputador são ajustados somente com dados de treino. O filtro forward mantém o contexto dos ciclos e usa observações até a hora estimada. O HMM usa carga e diferença dentro de cada ciclo. As convenções de emissão e primeira hora vêm do piloto `iatan/processes.py`. Os parâmetros do HMM foram fixados a partir da análise anterior. A avaliação continua retrospectiva.

**Para executar só a ablação.** Prepare o insumo com o notebook 00. Abra o 05 e execute as três células deste bloco em ordem. Em um Colab novo, instale as dependências do arquivo `requirements-iatan2025.txt` no mesmo runtime do 05, como orientado no 00. A primeira célula usa a raiz padrão do Drive ou `IATAN_PROJECT_ROOT`. O Parquet bruto preservado é o único insumo. O bloco grava previsões pareadas, log e DFG de treino, ajustes e backup SQLite do MLflow em uma pasta nova.

[Diba et al., extração e abstração de eventos](https://doi.org/10.1002/widm.1346) · [Rabiner, filtragem HMM](https://doi.org/10.1109/5.18626) · [Cerqueira et al., avaliação temporal](https://arxiv.org/abs/1905.11744) · [Chen e Guestrin, XGBoost](https://doi.org/10.1145/2939672.2939785) · [MLflow Tracking](https://mlflow.org/docs/latest/ml/tracking/tracking-api/)

A Aula 08 de e-Science e DoE, pp. 31–32 e 53–56, orienta o registro dos fatores, das respostas e da proveniência.

**Monitoramento da execução.** O MLflow registra um trace com etapas de leitura, janelas, ajustes,
previsões e auditoria. O run principal registra CPU e memória do ambiente a cada segundo.
Os runs incluem a linhagem dos conjuntos de treino e avaliação, vinculada ao Parquet bruto.
Para executar este bloco, instale também `psutil` no ambiente indicado no notebook 00.
O SQLite, os artefatos e os inventários de spans, dados e métricas do sistema ficam na pasta da rodada.

[Traces no MLflow](https://mlflow.org/docs/latest/genai/tracing/) ·
[Métricas do sistema](https://mlflow.org/docs/latest/ml/tracking/system-metrics/) ·
[Rastreamento de dados](https://mlflow.org/docs/latest/ml/tracking/)


In [ ]:
# Registros do experimento usam caminhos relativos à raiz configurada.
# Versões de bibliotecas e resultados são preservados sem identidade da sessão.
import builtins as _builtins
import json as _json
import os as _os
import re as _re
import warnings as _warnings
from pathlib import Path as _Path

def _project_root():
    for name in ("PROJECT_ROOT", "ROOT33", "ROOT", "PROJECT", "SOURCE", "root"):
        value = globals().get(name)
        if isinstance(value, _Path) and value.is_dir():
            return value.resolve()
    return _Path.cwd()

def portable_path(path, root=None):
    path = _Path(path).expanduser()
    root = _Path(root) if root is not None else _project_root()
    try:
        return path.resolve().relative_to(root.resolve()).as_posix()
    except (ValueError, OSError):
        return "external/" + path.name

def public_record(value):
    if isinstance(value, _Path):
        return portable_path(value)
    if isinstance(value, dict):
        return {k: public_record(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [public_record(v) for v in value]
    if isinstance(value, str):
        root = _project_root()
        for prefix in {str(root), root.as_posix()}:
            if value.startswith(prefix + "/") or value.startswith(prefix + "\\"):
                return value[len(prefix)+1:].replace("\\", "/")
            value = value.replace(prefix + "/", "").replace(prefix + "\\", "")
            value = value.replace(prefix, ".")
        if _re.match(r"^(?:[A-Za-z]:[\\/]|/(?:content|tmp|home|Users|mnt)/)", value):
            return "external/" + value.replace("\\", "/").rstrip("/").rsplit("/", 1)[-1]
        return value
    # Cópias de tabelas exportadas normalizam somente colunas textuais.
    if type(value).__module__.startswith("pandas.") and hasattr(value, "columns"):
        result = value.copy()
        for column in result.select_dtypes(include=["object", "string"]).columns:
            result[column] = result[column].map(public_record)
        return result
    return value

def _project_print(*values, **kwargs):
    root = _project_root()
    if root.is_dir():
        _os.chdir(root)
    return _builtins.print(*(public_record(v) for v in values), **kwargs)

def _project_dump(value, *args, **kwargs):
    return _json.dump(public_record(value), *args, **kwargs)

def _project_dumps(value, *args, **kwargs):
    return _json.dumps(public_record(value), *args, **kwargs)

def _project_csv(frame, *args, **kwargs):
    return public_record(frame).to_csv(*args, **kwargs)

def _project_write_text(path, text, *args, **kwargs):
    return path.write_text(public_record(text), *args, **kwargs)

def _project_display(*values, **kwargs):
    from IPython.display import display as _display
    return _display(*(public_record(v) for v in values), **kwargs)

# Avisos mantêm o conteúdo sem diretórios temporários ou IDs do kernel.
_warnings.formatwarning = lambda message, category, *args, **kwargs: (
    f"{category.__name__}: {public_record(str(message))}\n"
)
_os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
_context = _json.loads(_os.environ.get("MLFLOW_RUN_CONTEXT", "{}"))
_context.update({
    "mlflow.user": "equipe", "mlflow.source.name": "iatan",
    "mlflow.source.type": "LOCAL", "mlflow.source.git.commit": "",
    "mlflow.source.git.branch": "", "mlflow.source.git.repoURL": "",
    "mlflow.source.git.diff": "", "mlflow.source.git.dirty": "",
})
_os.environ["MLFLOW_RUN_CONTEXT"] = _json.dumps(_context)

# Exp33 · preparação independente no Colab.
from pathlib import Path
import os
import sys

if "google.colab" in sys.modules and not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
default33 = Path("/content/drive/MyDrive/Ciência de Dados - Iatan - Process Mining e Experimentos"
                 "/Iatan 2025 - entrega revisada/execucao")
ROOT33 = Path(os.environ.get("IATAN_PROJECT_ROOT", str(default33))).expanduser().resolve()
if not (ROOT33 / "pipeline/selected_unit_hourly_raw.parquet").is_file():
    raise FileNotFoundError(f"Insumo horário ausente em {ROOT33}. Prepare a pasta com o notebook 00.")

PROCESS_SOURCE33 = '28ba5f8e6834e1036adee7cfa8f16c94fd60db06:iatan/processes.py'
PROCESS_SHA33 = 'e454a719abc2227f0c5c2f12bfb54ee49b0add31e2d2e0b09002947f4cf89b1c'
EXP_CODE_SHA33 = 'f5a312b647e93256f06ab5e7d7916c9e4425ab21727da14cff1479ea2057f6f1'

# Rotina incorporada de iatan/processes.py para permitir uso direto no Colab.
from dataclasses import dataclass
import numpy as np
import pandas as pd
from scipy.special import logsumexp
from sklearn.preprocessing import StandardScaler
from hmmlearn.hmm import GaussianHMM

LOAD = "Gross Load (MW)"
OP = "Operating Time"

def observations(raw):
    d = raw.sort_values('timestamp').reset_index(drop=True).copy()
    contiguous = d.timestamp.diff().eq(pd.Timedelta(hours=1))
    on = d[OP].gt(0)
    d['cycle_id'] = (~(on & on.shift(fill_value=False) & contiguous)).cumsum()
    load = d[LOAD].where(on, 0.)
    delta = load.diff().where(contiguous & on & on.shift(fill_value=False))
    d['hmm_load'] = load
    d['hmm_delta'] = delta
    d['hmm_valid'] = on & np.isfinite(load) & np.isfinite(delta)
    return d


def lengths_for_valid(d):
    part = d.loc[d.hmm_valid]
    cut = part.cycle_id.ne(part.cycle_id.shift()) | part.timestamp.diff().ne(pd.Timedelta(hours=1))
    return part.groupby(cut.cumsum(), sort=False).size().tolist()


@dataclass
class ProcessModel:
    k: int = 5
    seed: int = 42
    n_iter: int = 100
    ramp_threshold: float = 10.

    def fit(self, raw):
        d = observations(raw)
        x = d.loc[d.hmm_valid, ['hmm_load', 'hmm_delta']].to_numpy(float)
        if len(x) < 20 * self.k:
            raise ValueError('Amostra insuficiente para HMM')
        self.scaler = StandardScaler().fit(x)
        z = self.scaler.transform(x)
        self.hmm = GaussianHMM(n_components=self.k, covariance_type='diag', random_state=self.seed,
                               n_iter=self.n_iter, tol=1e-3, min_covar=1e-3)
        self.hmm.fit(z, lengths=lengths_for_valid(d))
        # Rank-based names avoid claiming physical identity of heuristic labels.
        self.order = np.argsort(self.hmm.means_[:, 0], kind='stable')
        self.train_end = d.timestamp.max()
        self.edges = {}; self.totals = {}; self.activities = []
        labels = self.transform(raw)
        train_events = events(labels)
        for _, g in train_events.groupby('cycle_id', sort=False):
            acts = g.activity.tolist()
            for a, b in zip(acts[:-1], acts[1:]):
                self.edges[(a, b)] = self.edges.get((a, b), 0) + 1
                self.totals[a] = self.totals.get(a, 0) + 1
        self.activities = sorted(train_events.activity.unique())
        self.training_events = train_events
        self.audit = {'train_until': str(self.train_end), 'rows_fit': len(x), 'k': self.k,
                      'seed': self.seed, 'iterations': self.hmm.monitor_.iter,
                      'converged': bool(self.hmm.monitor_.converged),
                      'decoding': 'forward_per_contiguous_on_sequence', 'co2_used': False,
                      'learned_edges': len(self.edges), 'ramp_threshold_mw': self.ramp_threshold}
        return self

    def transform(self, raw):
        d = observations(raw)
        columns = [f'hmm_p_{i}' for i in range(self.k)]
        d[columns] = 0.
        d['hmm_available'] = d.hmm_valid.astype('int8')
        d['hmm_confidence'] = 0.; d['hmm_entropy'] = 0.
        d['activity'] = np.where(d[OP].gt(0), 'ON_UNRESOLVED', 'OFF')
        valid = d.loc[d.hmm_valid]
        if not valid.empty:
            x = self.scaler.transform(valid[['hmm_load', 'hmm_delta']].to_numpy(float))
            emission = self.hmm._compute_log_likelihood(x)
            cut = valid.cycle_id.ne(valid.cycle_id.shift()) | valid.timestamp.diff().ne(pd.Timedelta(hours=1))
            probs = np.empty((len(x), self.k)); alpha = None
            log_trans = np.log(np.clip(self.hmm.transmat_, 1e-300, None))
            log_start = np.log(np.clip(self.hmm.startprob_, 1e-300, None))
            for i, reset in enumerate(cut):
                alpha = emission[i] + (log_start if reset else logsumexp(alpha[:, None] + log_trans, axis=0))
                alpha -= logsumexp(alpha)
                probs[i] = np.exp(alpha)[self.order]
            d.loc[valid.index, columns] = probs
            d.loc[valid.index, 'hmm_confidence'] = probs.max(axis=1)
            d.loc[valid.index, 'hmm_entropy'] = -(probs * np.log(np.clip(probs, 1e-300, None))).sum(axis=1)
            states = probs.argmax(axis=1)
            directions = np.where(valid.hmm_delta > self.ramp_threshold, 'UP',
                                  np.where(valid.hmm_delta < -self.ramp_threshold, 'DOWN', 'FLAT'))
            d.loc[valid.index, 'activity'] = [f'E{s}_{direction}' for s, direction in zip(states, directions)]
        derived = np.zeros((len(d), 6), dtype=float)
        previous = None; previous_time = None; previous_cycle = None
        elapsed = age = changes = 0; probability = 0.; known = 0
        for i, row in d.iterrows():
            if row[OP] <= 0:
                previous = previous_time = previous_cycle = None
                continue
            reset = previous_time is None or row.timestamp - previous_time != pd.Timedelta(hours=1) or row.cycle_id != previous_cycle
            activity = row.activity
            if reset:
                elapsed = age = 1; changes = 0; probability = 0.; known = 0
            else:
                age += 1
                if activity == previous: elapsed += 1
                else:
                    elapsed = 1; changes += 1
                    count = self.edges.get((previous, activity), 0)
                    probability = (count + 1.) / (self.totals.get(previous, 0) + max(1, len(self.activities) + 1))
                    known = int(count > 0)
            derived[i] = [
                elapsed, age, changes, probability, -np.log(probability) if probability > 0 else 0., known]
            previous, previous_time, previous_cycle = activity, row.timestamp, row.cycle_id
        d[['proc_elapsed_activity_h', 'proc_elapsed_cycle_h', 'proc_changes',
           'proc_transition_probability', 'proc_transition_surprise', 'proc_edge_known']] = derived
        return d


def events(labels):
    d = labels.loc[labels[OP].gt(0)].copy()
    if d.empty: return pd.DataFrame(columns=['cycle_id', 'activity', 'start', 'end', 'duration_h'])
    cut = d.cycle_id.ne(d.cycle_id.shift()) | d.activity.ne(d.activity.shift()) | d.timestamp.diff().ne(pd.Timedelta(hours=1))
    return d.groupby(cut.cumsum(), sort=False).agg(cycle_id=('cycle_id', 'first'), activity=('activity', 'first'),
            start=('timestamp', 'min'), end=('timestamp', 'max'), duration_h=('timestamp', 'size')).reset_index(drop=True)


In [ ]:
# Exp33 · delineamento, corte temporal e registro por janela.
import json
import sqlite3
from contextlib import closing, contextmanager
from datetime import datetime, timezone
from hashlib import file_digest
from importlib.metadata import version
from pathlib import Path
from tempfile import TemporaryDirectory
from uuid import uuid4

import joblib
import mlflow
import psutil
from mlflow.entities import SpanStatusCode
from mlflow.entities.trace_location import MlflowExperimentLocation
from mlflow.environment_variables import (
    MLFLOW_SYSTEM_METRICS_SAMPLING_INTERVAL,
    MLFLOW_SYSTEM_METRICS_SAMPLES_BEFORE_LOGGING,
)
from mlflow.tracking import MlflowClient
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from xgboost import XGBRegressor

RAW_INPUT33 = "pipeline/selected_unit_hourly_raw.parquet"
RAW_SHA33 = "0613b96dfbf0209ed830d36bcf185f5ba89b2cfb8edc1daca276a12e0b4d8911"
TARGET33 = "co2_metric_tons"
SYSTEM33 = (
    "system/cpu_utilization_percentage",
    "system/system_memory_usage_megabytes",
    "system/system_memory_usage_percentage",
)
MONITOR33 = "exp33_traces_system_dataset_v1"
HISTORY33 = [f"load_{stat}_last_{n}h" for n in (3, 6, 12, 24)
             for stat in ("mean", "std")] + [f"load_lag_{n}h" for n in (1, 2, 3, 6, 12, 24)]
HMM33 = [f"hmm_p_{n}" for n in range(5)] + ["hmm_available", "hmm_confidence", "hmm_entropy"]
DFG33 = ["proc_transition_probability", "proc_transition_surprise", "proc_edge_known"]
COLUMNS33 = {
    "A": ["generation_mw"],
    "B": ["generation_mw", *HISTORY33],
    "C": ["generation_mw", *HISTORY33, *HMM33],
    "D": ["generation_mw", *HISTORY33, *HMM33, *DFG33],
}
XGB33 = dict(n_estimators=200, max_depth=3, learning_rate=.05, subsample=.9,
             colsample_bytree=.9, random_state=42, objective="reg:squarederror",
             tree_method="hist", n_jobs=2, verbosity=0, enable_categorical=True)
PROCESS33 = dict(k=5, seed=11, n_iter=300, ramp_threshold=20.)
# Índices na série de horas ON, com fim exclusivo.
WINDOWS33 = (
    ("interna_1", 2699, 2699, 3299, "06-03 18:00", "06-03 19:00", "06-30 18:00"),
    ("interna_2", 3299, 3299, 3899, "06-30 18:00", "06-30 19:00", "07-28 16:00"),
    ("interna_3", 3899, 3899, 4499, "07-28 16:00", "07-28 17:00", "09-04 15:00"),
    ("validacao", 4499, 4499, 5186, "09-04 15:00", "09-04 16:00", "11-05 23:00"),
    ("teste", 4499, 5186, 6249, "09-04 15:00", "11-06 00:00", "12-31 23:00"),
)

def digest33(path):
    with path.open("rb") as stream:
        return file_digest(stream, "sha256").hexdigest()

def write33(path, data):
    _project_write_text(path, _project_dumps(data, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    return path

def read_calendar33(root):
    path = root / RAW_INPUT33
    if digest33(path) != RAW_SHA33:
        raise ValueError("O Parquet horário difere do insumo preservado de Iatan 2. Confira o pacote do notebook 00.")
    target = "CO2 Mass (short tons)"
    d = pd.read_parquet(path, columns=["timestamp", "Facility ID", "Unit ID", LOAD, OP, target])
    d = d.sort_values("timestamp").reset_index(drop=True)
    expected = pd.date_range("2025-01-01", periods=8760, freq="h")
    if not pd.DatetimeIndex(d.timestamp).equals(expected):
        raise ValueError("Esperado calendário horário contínuo de 2025 no eixo original da fonte.")
    if not d["Facility ID"].eq(6065).all() or not d["Unit ID"].astype(str).eq("2").all():
        raise ValueError("O insumo deve corresponder à Iatan 2 (6065/2).")
    on = d[OP].gt(0)
    if on.sum() != 6249 or not np.isfinite(d.loc[on, [LOAD, target]]).all().all():
        raise ValueError("As 6.249 horas ON devem ter carga e CO₂ finitos.")
    return d[["timestamp", LOAD, OP]], d[target] * 0.90718474, np.flatnonzero(on)

def features33(operational, process):
    frame = process.transform(operational)
    frame["generation_mw"] = frame[LOAD]
    # As estatísticas incluem t e as defasagens reiniciam em cada ciclo observado, como no C30.
    load = frame["generation_mw"].where(frame[OP].gt(0))
    within_cycle = load.groupby(frame["cycle_id"], sort=False)
    for n in (3, 6, 12, 24):
        frame[f"load_mean_last_{n}h"] = within_cycle.transform(lambda series: series.rolling(n, min_periods=1).mean())
        frame[f"load_std_last_{n}h"] = within_cycle.transform(lambda series: series.rolling(n, min_periods=2).std(ddof=0)).fillna(0.)
    for n in (1, 2, 3, 6, 12, 24):
        frame[f"load_lag_{n}h"] = within_cycle.shift(n)
    return frame[["timestamp", "cycle_id", *COLUMNS33["D"]]]

def backup33(source, target):
    partial = target.with_suffix(".tmp")
    with closing(sqlite3.connect(source)) as original, closing(sqlite3.connect(partial)) as copy:
        original.backup(copy)
        if copy.execute("PRAGMA integrity_check").fetchone() != ("ok",):
            raise RuntimeError("Falha na integridade do backup MLflow.")
    partial.replace(target)

@contextmanager
def tracked33(experiment_id, name, parent=None):
    tags = {"study": "iatan2_ablation_foldwise", "protocol": "exp33_v1", "monitoring": MONITOR33}
    if parent:
        tags["mlflow.parentRunId"] = parent
    with mlflow.start_run(experiment_id=experiment_id, run_name=name, tags=tags,
                          nested=parent is not None, log_system_metrics=parent is None) as run:
        yield run.info.run_id


@contextmanager
def monitored_tracking33(uri):
    if mlflow.active_run() is not None:
        raise RuntimeError("Encerre a execução ativa do MLflow antes de iniciar a ablação.")
    original_uri = mlflow.get_tracking_uri()
    interval = MLFLOW_SYSTEM_METRICS_SAMPLING_INTERVAL.get()
    samples = MLFLOW_SYSTEM_METRICS_SAMPLES_BEFORE_LOGGING.get()
    mlflow.set_tracking_uri(uri)
    mlflow.set_system_metrics_sampling_interval(1)
    mlflow.set_system_metrics_samples_before_logging(1)
    try:
        yield
    finally:
        mlflow.set_tracking_uri(original_uri)
        mlflow.set_system_metrics_sampling_interval(interval)
        mlflow.set_system_metrics_samples_before_logging(samples)


@contextmanager
def span33(name, attributes=None, **kwargs):
    with mlflow.start_span(name, attributes=attributes, **kwargs) as span:
        yield span
        span.set_status(SpanStatusCode.OK)


def log_dataset33(frame, source, name, role):
    dataset = mlflow.data.from_pandas(frame, source=source, name=name, targets=TARGET33)
    mlflow.log_input(dataset, context=role, tags={"raw_sha256": RAW_SHA33,
        "first_timestamp": frame.timestamp.iloc[0].isoformat(),
        "last_timestamp": frame.timestamp.iloc[-1].isoformat(),
        "rows": str(len(frame))})


def audit_monitor33(client, experiment_id, trace_id, summary_run, audits, rows, folder):
    trace = client.get_trace(trace_id, display=False)
    spans = trace.data.spans
    expected = {
        "Exp33": 1, "Ler e validar calendário": 1, "Registrar linhagem do calendário": 1,
        "Janela cronológica": 5, "Ajustar scaler, HMM e DFG": 5,
        "Construir atributos forward": 5, "Auditar prefixo, cortes e posterior": 5,
        "Registrar linhagem por janela": 5, "Cenário XGBoost": 20,
        "Registrar linhagem do cenário": 20, "Ajustar XGBoost e imputador": 20,
        "Prever CO2": 20, "Auditar previsões e métricas": 20,
        "Consolidar métricas e previsões": 1,
    }
    observed = pd.Series([span.name for span in spans]).value_counts().to_dict()
    if observed != expected or any(span.status.status_code != SpanStatusCode.OK for span in spans):
        raise RuntimeError(f"Rastro incompleto ou com falhas: {observed}.")
    ids = [summary_run] + [a["mlflow_janela"] for a in audits] + [r["mlflow_run_id"] for r in rows]
    for rid in ids:
        client.link_traces_to_run([trace_id], rid)
        client.set_tag(rid, "exp33_trace_id", trace_id)
    rows_spans = [{
        "trace_id": span.trace_id, "span_id": span.span_id, "parent_id": span.parent_id,
        "nome": span.name, "estado": span.status.status_code.value,
        "inicio_ns": span.start_time_ns, "fim_ns": span.end_time_ns,
        "duracao_ms": (span.end_time_ns - span.start_time_ns) / 1e6,
        "janela": span.attributes.get("janela"), "cenario": span.attributes.get("cenario"),
        "mlflow_run_id": span.attributes.get("run_id"),
    } for span in spans]
    _project_csv(pd.DataFrame(rows_spans), folder / "inventario_spans.csv", index=False)
    resources = {name: client.get_metric_history(summary_run, name) for name in SYSTEM33}
    if any(not history for history in resources.values()):
        raise RuntimeError("O monitor nativo do MLflow não registrou CPU e RAM.")
    _project_csv(pd.DataFrame([{"metrica": name, "passo": point.step, "timestamp_ms": point.timestamp,
                   "valor": point.value}
                  for name, history in resources.items() for point in history]
                 ), folder / "metricas_sistema.csv", index=False)
    lineage = []
    for rid in ids:
        for item in client.get_run(rid).inputs.dataset_inputs:
            lineage.append({"mlflow_run_id": rid, "dataset": item.dataset.name,
                            "digest": item.dataset.digest,
                            **{tag.key: tag.value for tag in item.tags}})
    if len(lineage) != 51:
        raise RuntimeError(f"Esperadas 51 associações de entrada, obtidas {len(lineage)}.")
    _project_csv(pd.DataFrame(lineage), folder / "linhagem_mlflow.csv", index=False)
    return dict(trace_id=trace_id, experiment_id=experiment_id, runs_associados=len(ids),
                spans=len(spans), spans_por_etapa=observed, vinculos_dados=len(lineage),
                amostras_por_metrica={name: len(history) for name, history in resources.items()},
                intervalo_amostragem_s=1, psutil=psutil.__version__,
                tipo="execução offline, métricas do ambiente durante o ajuste e a avaliação")


def log_params33(client, rid, params):
    for key, value in params.items():
        client.log_param(rid, key, value)

def fit_window33(window, operational, y, positions, folder, client, experiment_id, parent, dataset_source):
    name, train_stop, first, end, expected_train, expected_first, expected_last = window
    train_idx, eval_idx = positions[:train_stop], positions[first:end]
    train_raw = operational.iloc[:train_idx[-1] + 1]
    through_eval = operational.iloc[:eval_idx[-1] + 1]
    fit_end = train_raw.timestamp.iloc[-1]
    eval_start, eval_end = operational.timestamp.iloc[eval_idx[[0, -1]]]
    cuts = tuple(t.strftime("%m-%d %H:%M") for t in (fit_end, eval_start, eval_end))
    if cuts != (expected_train, expected_first, expected_last) or fit_end >= eval_start:
        raise ValueError(f"Os cortes da janela {name} não correspondem ao protocolo.")
    out = folder / name
    out.mkdir()
    with tracked33(experiment_id, name, parent) as fold_run, span33("Janela cronológica",
            {"janela": name, "run_id": fold_run, "n_treino": len(train_idx),
             "n_avaliacao": len(eval_idx), "ajuste_ate": fit_end.isoformat(),
             "avaliacao_inicio": eval_start.isoformat(), "avaliacao_fim": eval_end.isoformat()}) as fold_span:
        with span33("Ajustar scaler, HMM e DFG", {"janela": name, "run_id": fold_run, **PROCESS33}) as fit_span:
            process = ProcessModel(**PROCESS33).fit(train_raw)
            if process.train_end != fit_end or process.training_events["end"].max() > fit_end:
                raise RuntimeError("O ajuste da representação operacional avançou além do treino.")
            fit_span.set_attributes({"iteracoes": process.hmm.monitor_.iter, "arestas": len(process.edges),
                                     "eventos": len(process.training_events), "linhas": process.audit["rows_fit"]})
        graph = pd.DataFrame(
            [(a, b, n) for (a, b), n in sorted(process.edges.items())],
            columns=["origem", "destino", "frequencia"],
        )
        _project_csv(graph, out / "dfg_treino.csv", index=False)
        _project_csv(process.training_events, out / "log_eventos_treino.csv", index=False)
        history = list(process.hmm.monitor_.history)
        increment = float(history[-1] - history[-2])
        converged = 0 <= increment < process.hmm.tol and process.hmm.monitor_.iter < process.hmm.n_iter
        if not converged:
            raise RuntimeError(f"HMM sem convergência confirmada em {name}. Incremento {increment:.6g}.")
        model_state = write33(out / "hmm_scaler_dfg.json", {
            "scaler_mean": process.scaler.mean_.tolist(), "scaler_scale": process.scaler.scale_.tolist(),
            "startprob": process.hmm.startprob_.tolist(), "transmat": process.hmm.transmat_.tolist(),
            "means": process.hmm.means_.tolist(), "covars": process.hmm.covars_.tolist(),
            "state_order_by_load": process.order.tolist(), "dfg": graph.to_dict(orient="records"),
        })
        with span33("Construir atributos forward", {"janela": name, "run_id": fold_run}) as feature_span:
            frame = features33(through_eval, process)
            feature_span.set_attributes({"linhas": len(frame), "atributos": len(COLUMNS33["D"])})
        with span33("Auditar prefixo, cortes e posterior", {"janela": name, "run_id": fold_run}) as audit_span:
            # Truncar o calendário no treino e no meio da avaliação deve preservar todos os atributos do prefixo.
            probe_stop = int(eval_idx[len(eval_idx) // 2]) + 1
            for stop in (len(train_raw), probe_stop):
                pd.testing.assert_frame_equal(features33(through_eval.iloc[:stop], process), frame.iloc[:stop],
                                              rtol=1e-12, atol=1e-12)
            if not np.isfinite(frame.loc[eval_idx, HMM33 + DFG33]).all().all():
                raise RuntimeError(f"Atributos de HMM/DFG inválidos em {name}.")
            posterior = frame.loc[eval_idx, HMM33[:5]].sum(axis=1)
            np.testing.assert_allclose(posterior, frame.loc[eval_idx, "hmm_available"], atol=1e-12)
            audit_span.set_attributes({"prefixo_verificado": True, "posterior_normalizado": True,
                                       "ajuste_estritamente_anterior": fit_end < eval_start})
        cut_record = dict(
            janela=name, n_treino=len(train_idx), n_avaliacao=len(eval_idx),
            treino_inicio=operational.timestamp.iloc[train_idx[0]].isoformat(),
            ajuste_ate=fit_end.isoformat(), avaliacao_inicio=eval_start.isoformat(),
            avaliacao_fim=eval_end.isoformat(), n_ciclos_avaliacao=int(frame.loc[eval_idx, "cycle_id"].nunique()),
            n_hmm_disponivel=int(frame.loc[eval_idx, "hmm_available"].sum()),
            n_eventos_treino=len(process.training_events), n_arestas_treino=len(graph),
            hmm_iteracoes=process.hmm.monitor_.iter, hmm_incremento_final=increment, hmm_convergiu=converged,
            prefixo_verificado=True, hmm_dfg_sha256=digest33(model_state),
            log_eventos_sha256=digest33(out / "log_eventos_treino.csv"),
            dfg_sha256=digest33(out / "dfg_treino.csv"), mlflow_janela=fold_run,
        )
        log_params33(client, fold_run, {**cut_record, **PROCESS33})
        write33(out / "auditoria_da_janela.json", cut_record)
        client.log_artifacts(fold_run, str(out), artifact_path="treino")
        predictions = pd.DataFrame({
            "janela": name, "timestamp": operational.timestamp.iloc[eval_idx].to_numpy(),
            "ciclo": frame.loc[eval_idx, "cycle_id"].to_numpy(), TARGET33: y.iloc[eval_idx].to_numpy(),
        })
        with span33("Registrar linhagem por janela", {"janela": name, "run_id": fold_run}):
            for role, idx in (("training", train_idx), ("evaluation", eval_idx)):
                data = frame.loc[idx, ["timestamp", *COLUMNS33["D"]]].assign(**{TARGET33: y.iloc[idx]})
                log_dataset33(data, dataset_source, f"iatan2025_{name}_{role}", role)
        rows = []
        for scenario, cols in COLUMNS33.items():
            with tracked33(experiment_id, scenario, fold_run) as scenario_run, span33("Cenário XGBoost",
                    {"janela": name, "cenario": scenario, "run_id": scenario_run,
                     "n_atributos": len(cols), "n_treino": len(train_idx),
                     "n_avaliacao": len(eval_idx)}) as scenario_span:
                operation = {"janela": name, "cenario": scenario, "run_id": scenario_run}
                with span33("Registrar linhagem do cenário", operation):
                    for role, idx in (("training", train_idx), ("evaluation", eval_idx)):
                        data = frame.loc[idx, ["timestamp", *cols]].assign(**{TARGET33: y.iloc[idx]})
                        log_dataset33(data, dataset_source, f"iatan2025_{name}_{scenario}_{role}", role)
                with span33("Ajustar XGBoost e imputador", operation, span_type="MODEL"):
                    model = make_pipeline(SimpleImputer(strategy="median", keep_empty_features=True),
                                          XGBRegressor(**XGB33))
                    model.fit(frame.loc[train_idx, cols], y.iloc[train_idx])
                with span33("Prever CO2", operation, span_type="MODEL"):
                    pred = model.predict(frame.loc[eval_idx, cols]).astype(float)
                with span33("Auditar previsões e métricas", operation) as score_span:
                    if len(pred) != len(eval_idx) or not np.isfinite(pred).all():
                        raise RuntimeError(f"Previsões inválidas para {name}/{scenario}.")
                    predictions[scenario] = pred
                    error = pred - predictions[TARGET33].to_numpy()
                    row = dict(janela=name, cenario=scenario, n_treino=len(train_idx), n_avaliacao=len(eval_idx),
                               rmse=float(np.sqrt(np.mean(error ** 2))), mae=float(np.mean(np.abs(error))),
                               vies=float(np.mean(error)), mlflow_run_id=scenario_run)
                    score_span.set_attributes({key: row[key] for key in ("rmse", "mae", "vies")})
                path = out / f"xgboost_{scenario}.joblib"
                joblib.dump(model, path, compress=3)
                features_path = write33(out / f"atributos_{scenario}.json", {
                    "cenario": scenario, "atributos_ordenados": cols, "modelo_sha256": digest33(path),
                    "xgboost_version": version("xgboost"), "hmm_dfg_sha256": cut_record["hmm_dfg_sha256"],
                })
                log_params33(client, scenario_run, {**XGB33, "janela": name, "cenario": scenario,
                                                   "n_atributos": len(cols), "n_avaliacao": len(eval_idx),
                                                   "ajuste_ate": fit_end.isoformat(),
                                                   "avaliacao_inicio": eval_start.isoformat(),
                                                   "avaliacao_fim": eval_end.isoformat(),
                                                   "xgboost_version": version("xgboost")})
                for key in ("rmse", "mae", "vies"):
                    client.log_metric(scenario_run, key, row[key])
                client.log_artifact(scenario_run, str(path), artifact_path="modelo")
                client.log_artifact(scenario_run, str(features_path), artifact_path="modelo")
                rows.append(row)
                scenario_span.set_attributes({key: row[key] for key in ("rmse", "mae", "vies")})
        _project_csv(predictions, out / "previsoes_pareadas.csv", index=False)
        client.log_artifact(fold_run, str(out / "previsoes_pareadas.csv"))
        fold_span.set_attributes({"n_eventos_treino": cut_record["n_eventos_treino"],
                                  "n_arestas_treino": cut_record["n_arestas_treino"],
                                  "hmm_incremento_final": increment, "prefixo_verificado": True})
    return cut_record, rows, predictions

def run_ablation33(root):
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
    folder = root / "outputs" / "phase_04_ablation_foldwise" / run_id
    folder.mkdir(parents=True)
    protocol = {
        "experimento": "33", "monitoramento": MONITOR33, "run_directory_id": run_id, "unidade": "EPA 6065/2", "ano": 2025,
        "raw_sha256": RAW_SHA33, "fonte_codigo_processo": PROCESS_SOURCE33, "sha256_codigo_processo": PROCESS_SHA33,
        "sha256_codigo_experimento": EXP_CODE_SHA33,
        "tempo": "eixo horário preservado da fonte, sem conversão de fuso", "alvo": "toneladas métricas de CO₂/h",
        "xgboost": XGB33, "hmm": PROCESS33, "atributos": COLUMNS33,
        "janelas": [list(window) for window in WINDOWS33],
        "versions": {name: version(name) for name in ("numpy", "pandas", "hmmlearn", "scikit-learn",
                                                      "xgboost", "mlflow", "pyarrow", "joblib", "psutil")},
        "controle": "mesmos parâmetros, mesmos alvos e timestamps. Imputação mediana ajustada no treino.",
        "historia": "6 defasagens e 8 estatísticas incluindo t, dentro dos ciclos e sem unir horas OFF.",
        "dfg": "frequências de sucessão de atividades comprimidas no log de treino. Em D entram probabilidade, surpresa e indicador de aresta conhecida.",
        "limites": "Iatan 2, 2025 já consultado. Parâmetros fixados a partir do histórico do projeto. Validação e teste externos ao treino são retrospectivos.",
    }
    protocol_path = write33(folder / "protocolo.json", protocol)
    predictions = []
    audits = []
    rows = []
    artifacts = folder / "mlflow_artifacts"
    artifacts.mkdir()
    with TemporaryDirectory(prefix="iatan_ablacao_") as temporary, monitored_tracking33(
            f"sqlite:///{(Path(temporary) / 'mlflow.db').as_posix()}"):
        database = Path(temporary) / "mlflow.db"
        client = MlflowClient()
        exp = client.create_experiment("Iatan 2 · Exp33 monitorado", artifact_location=artifacts.as_uri())
        try:
            with tracked33(exp, "Exp33 · A/B/C/D monitorado") as summary_run, span33(
                "Exp33", {"run_id": summary_run, "unidade": "EPA 6065/2", "ano": 2025,
                          "raw_sha256": RAW_SHA33, "protocolo": MONITOR33},
                run_id=summary_run, trace_destination=MlflowExperimentLocation(experiment_id=exp)
            ) as root_span:
                trace_id = root_span.trace_id
                client.set_tag(summary_run, "exp33_trace_id", trace_id)
                with span33("Ler e validar calendário", {"raw_sha256": RAW_SHA33}) as source_span:
                    operational, y, positions = read_calendar33(root)
                    source_span.set_attributes({"horas_calendario": len(operational),
                                                "horas_operacionais": len(positions)})
                dataset_source = (root / RAW_INPUT33).as_uri()
                with span33("Registrar linhagem do calendário", {"run_id": summary_run}):
                    calendar = operational.assign(**{TARGET33: y})
                    log_dataset33(calendar, dataset_source, "iatan2025_calendario", "source")
                client.log_artifact(summary_run, str(protocol_path))
                for window in WINDOWS33:
                    audit, metrics, pairs = fit_window33(window, operational, y, positions, folder, client, exp, summary_run, dataset_source)
                    audits.append(audit)
                    rows.extend(metrics)
                    predictions.append(pairs)
                    backup33(database, folder / "mlflow.db")
                    _project_print(window[0], "concluída", {row["cenario"]: round(row["rmse"], 4) for row in metrics}, flush=True)

                with span33("Consolidar métricas e previsões", {"run_id": summary_run}) as summary_span:
                    metric_table = pd.DataFrame(rows)
                    wide = metric_table.pivot(index="janela", columns="cenario", values="rmse").loc[[w[0] for w in WINDOWS33]]
                    wide["B_menos_A"] = wide["B"] - wide["A"]
                    wide["C_menos_B"] = wide["C"] - wide["B"]
                    wide["D_menos_C"] = wide["D"] - wide["C"]
                    paired = pd.concat(predictions, ignore_index=True)
                    pooled = paired.loc[paired["janela"].str.startswith("interna_")]
                    pooled_rmse = {s: float(np.sqrt(np.mean((pooled[s] - pooled[TARGET33]) ** 2))) for s in COLUMNS33}
                    _project_csv(metric_table, folder / "metricas.csv", index=False)
                    _project_csv(wide, folder / "rmse_por_janela.csv")
                    _project_csv(paired, folder / "previsoes_pareadas.csv", index=False)
                    write33(folder / "auditoria_das_janelas.json", audits)
                    summary = {
                        "run_directory_id": run_id, "mlflow_experiment_id": exp,
                        "mlflow_run_id": summary_run, "n_janelas": len(audits), "n_ajustes_xgboost": len(rows),
                        "n_pares_avaliacao": len(paired), "rmse_internas_agrupado": pooled_rmse,
                        "dfg_melhor_em_janelas_internas": int(wide.loc[pooled["janela"].unique(), "D_menos_C"].lt(0).sum()),
                    }
                    for scenario, score in pooled_rmse.items():
                        client.log_metric(summary_run, f"rmse_internas_agrupado_{scenario}", score)
                    for name in ("metricas.csv", "rmse_por_janela.csv", "previsoes_pareadas.csv",
                                 "auditoria_das_janelas.json"):
                        client.log_artifact(summary_run, str(folder / name))
                    summary_span.set_attributes({"n_janelas": len(audits), "n_pares": len(paired),
                                                 "n_ajustes": len(rows)})
                root_span.set_attributes({"n_janelas": len(audits), "n_pares": len(paired),
                                          "n_ajustes": len(rows), "rmse_internas": pooled_rmse})
            if client.get_run(summary_run).info.status != "FINISHED":
                raise RuntimeError("O registro de síntese do MLflow não foi finalizado.")
            mlflow.flush_trace_async_logging()
            summary["monitoramento"] = audit_monitor33(client, exp, trace_id, summary_run, audits, rows, folder)
            for name in ("inventario_spans.csv", "metricas_sistema.csv", "linhagem_mlflow.csv"):
                client.log_artifact(summary_run, str(folder / name), artifact_path="monitoramento")
            summary["status"] = "FINISHED"
            client.log_artifact(summary_run, str(write33(folder / "resumo.json", summary)))
        finally:
            mlflow.flush_trace_async_logging()
            backup33(database, folder / "mlflow.db")
            client._tracking_client.store.engine.dispose()
    _project_print("Artefatos e backup do MLflow em", folder)
    return folder, wide


In [ ]:
ABLATION_ROOT33, ABLATION_RMSE33 = run_ablation33(ROOT33)
_project_display(ABLATION_RMSE33.round(4))
